# InsightStay · 7개 지역 장기상품 전략 — 분석 코드 통합 노트북

**대상 보고서:** 7개지역_장기상품_통합전략.html (2026-10-03의 현재 파일).

이 노트북은 공통 분석표 준비 → 2×3 섹터·시설 정의 → 연관규칙·구성 보정 → 시설 군집 → 7지역 후보 → 조정회귀 → 손익분기 → HTML 보고서 생성 과정을 한 파일에 모았다.

- **본문:** 원자료가 아닌 기존 공통 분석표에서 후보 수·연관지표·군집 연결·손익을 다시 계산하고 저장값과 대조한다. 회귀와 군집 품질은 저장 결과를 읽는다.
- **부록:** 실제 사용한 원본 Python/JavaScript 코드를 전문으로 보존한다. `REPLAY_STAGES`에 단계를 지정하면 원본 입력 스냅샷으로 다시 계산한다. 군집 재학습·회귀 재적합은 기본 실행에서 생략한다.
- **입력:** CSV.gz 등 대용량 데이터는 노트북에 내장하지 않았다. 현재 컴퓨터의 경로를 기본값으로 사용한다. 다른 컴퓨터에서는 아래 경로와 필요한 입력 파일을 함께 옮겨야 한다.
- 원자료와 기존 보고서는 읽기만 한다. 실행 산출물은 별도 `notebook_runs` 폴더로 저장한다. 입력의 전체 전처리 이력을 새로 설계하지 않고, 기존 정제 CSV·고정 시설표·호스트 분할을 전제로 한다.
- 3+6은 시설 참조의 주 분석, 3번 단독은 보조 분석이다. 시설 후보·E4는 실제 장기예약 성공이나 법적 적격을 의미하지 않는다. 규제는 보고서 작성 당시의 조사 기록이며 노트북 실행으로 재검증되지 않는다.


In [1]:
from pathlib import Path
from datetime import datetime
import json, hashlib, ast, re, shutil, sys, io, contextlib, traceback
import numpy as np
import pandas as pd

PROJECT = Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
REPORT_PROJECT = Path('/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities')
REPORT_DIR = REPORT_PROJECT / 'outputs/7개지역_장기상품_통합전략_20261003'
COMMON_DIR = PROJECT / 'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터'
CLUSTER_DIR = COMMON_DIR.parent / '01_상품군집'
ASSOC_DIR = PROJECT / 'outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
REGRESSION_DIR = REPORT_PROJECT / 'outputs/7개지역_생활업무시설_조정회귀_20261003'
RUN_DIR = REPORT_DIR / 'notebook_runs' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
# 예: {'association', 'adjustment', 'sector3', 'regression'}
# cluster는 기존 선택을 재사용하지 않고 재학습하므로 시간이 걸린다.
REPLAY_STAGES = set()
SOURCE_CODE, SOURCE_META = {}, {}

def show(x):
    print(x.to_string(index=False) if isinstance(x, pd.DataFrame) else x)
def load_json(p): return json.loads(Path(p).read_text(encoding='utf-8'))
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20), b''): h.update(b)
    return h.hexdigest()
print('결과 저장:', RUN_DIR)


결과 저장: /Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_장기상품_통합전략_20261003/notebook_runs/20261003_195008_519881


## 1. 입력 파일과 분석 대상 확인
`source == 0 & availability_365 > 0`인 공통 표본이다. ID는 정밀도 손실 방지를 위해 문자열로 읽는다. 결측 시설을 미등록 0으로 바꾸지 않는다.

| 최근 12개월 리뷰 | minimum_nights 1~3 | 4~29 | 30 이상 |
|---|---:|---:|---:|
| 양수 | 1 | 2 | 3 |
| 0 | 4 | 5 | 6 |


In [2]:
FUNCTIONS = ['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']
FCOLS = ['function__'+f for f in FUNCTIONS]
REGIONS = ['london','pays-basque','oslo','munich','malaga','sevilla','valencia']
EXPECTED = dict(zip(REGIONS,[13317,1563,1700,606,2266,2107,1663]))
INPUTS = [COMMON_DIR/'공통분석표.csv.gz', CLUSTER_DIR/'숙소별_군집.csv.gz',
          REPORT_DIR/'7개지역_시설후보_ID.csv', REPORT_DIR/'연관분석_원지표.csv',
          REPORT_DIR/'연관분석_구성보정.csv', REGRESSION_DIR/'결과요약.json']
for p in INPUTS:
    if not p.is_file(): raise FileNotFoundError(f'입력 파일이 필요합니다: {p}')
input_hashes = {str(p):sha(p) for p in INPUTS}
cols=['id','host_id','source','availability_365','minimum_nights','number_of_reviews_ltm',
      'sector','split','region','room_type','property_type','accommodates']+FCOLS
items=['kitchen','stove','oven','microwave','refrigerator','cooking_basics','dishes_silverware','washer','wifi','workspace']
d=pd.read_csv(COMMON_DIR/'공통분석표.csv.gz',usecols=cols+['item__'+v for v in items],dtype={'id':'string','host_id':'string'})
assert len(d)==677678 and d.id.is_unique and d.source.eq(0).all() and d.availability_365.gt(0).all()
assert d.minimum_nights.ge(1).all() and d.number_of_reviews_ltm.ge(0).all()
band=np.select([d.minimum_nights.le(3),d.minimum_nights.le(29)],[1,2],default=3)
assert np.array_equal(d.sector,band+np.where(d.number_of_reviews_ltm.gt(0),0,3))
assert d.groupby('host_id').split.nunique().max()==1
show(d.groupby('sector').agg(숙소수=('id','size'),호스트수=('host_id','nunique')).reset_index())


 sector    숙소수   호스트수
      1 418643 212534
      2  43259  31754
      3   9555   5002
      4 144542  76812
      5  33330  23276
      6  28349  16659


## 2. 시설 조건과 후보 선별
7기능은 연구자가 정한 기능 분류이다. `heating_food`는 난방이 아니라 stove/oven/microwave 중 하나 이상, `tools`는 cooking_basics와 dishes_silverware의 평균이다. `tools == 1`은 두 항목 모두 등록됐다는 뜻이다.

생활형 L=앞 5기능, 업무형 W=Wi-Fi·업무공간, 최종 조건 LW=L∩W. 7지역의 집 전체·1/2/4/5번 중 LW 조건을 모두 충족한 후보를 선별한다. maximum_nights·호스트 평점은 이 선별 기준에 넣지 않았다.


In [3]:
heat=d[['item__stove','item__oven','item__microwave']]
expected_heat=np.where(heat.eq(1).any(axis=1),1,np.where(heat.eq(0).all(axis=1),0,np.nan))
expected_tools=d[['item__cooking_basics','item__dishes_silverware']].sum(axis=1,min_count=2)/2
assert np.allclose(d.function__heating_food,expected_heat,equal_nan=True)
assert np.allclose(d.function__tools,expected_tools,equal_nan=True)
for f in ['kitchen','refrigerator','washer','wifi','workspace']:
    assert np.allclose(d['function__'+f],d['item__'+f],equal_nan=True)
d['observed7']=d[FCOLS].notna().all(axis=1)
d['L']=d[FCOLS[:5]].eq(1).all(axis=1)
d['W']=d[FCOLS[5:]].eq(1).all(axis=1)
d['LW']=d.L & d.W
candidate=d[d.sector.isin([1,2,4,5]) & d.region.isin(REGIONS) & d.room_type.eq('Entire home/apt') & d.LW].copy()
saved=pd.read_csv(REPORT_DIR/'7개지역_시설후보_ID.csv',dtype={'id':'string','host_id':'string'})
assert set(candidate.id)==set(saved.id) and len(candidate)==23222
assert candidate.host_id.nunique()==13255
assert candidate.groupby('region').size().to_dict()==EXPECTED
counts=candidate.groupby('region').agg(숙소수=('id','size'),호스트수=('host_id','nunique')).reindex(REGIONS).reset_index()
show(counts)
candidate.to_csv(RUN_DIR/'재계산_후보.csv.gz',index=False,compression='gzip')


     region   숙소수  호스트수
     london 13317  6757
pays-basque  1563  1393
       oslo  1700  1585
     munich   606   493
     malaga  2266  1190
    sevilla  2107   980
   valencia  1663   875


## 3. 세 연관규칙을 원자료 행에서 재계산
R1: 취사 4기능→세탁기, R2: Wi-Fi→업무공간, R3: 생활형→업무형.
지지도=P(A∩B), 신뢰도=P(B|A), 향상도=P(A∩B)/(P(A)P(B)). 신뢰도는 통계적 확신이나 예약 성공률을 뜻하지 않는다. 자유로운 Apriori 탐색으로 최적 조건을 발견했다고 설명하지 않는다.


In [4]:
RULES={'R1':(FCOLS[:4],[FCOLS[4]]),'R2':([FCOLS[5]],[FCOLS[6]]),'R3':(FCOLS[:5],FCOLS[5:])}
v=d[d.observed7].copy()
v['pool']=np.where(v.sector.isin([3,6]),'reference','target')
for name,(aa,bb) in RULES.items():
    v['A_'+name]=v[aa].eq(1).all(axis=1).astype(int)
    v['B_'+name]=v[bb].eq(1).all(axis=1).astype(int)
    v['AB_'+name]=v['A_'+name]*v['B_'+name]
raw=[]
for pool,mask in {'pooled':v.sector.isin([3,6]),'sector3':v.sector.eq(3),'target':v.sector.isin([1,2,4,5])}.items():
    g=v[mask]
    for rule in RULES:
        pa,pb,pab=g[['A_'+rule,'B_'+rule,'AB_'+rule]].mean()
        raw.append(dict(pool=pool,rule=rule,n=len(g),support=pab,confidence=pab/pa,lift=pab/(pa*pb)))
raw=pd.DataFrame(raw)
reference=pd.read_csv(REPORT_DIR/'연관분석_원지표.csv')
for row in raw.itertuples():
    x=reference[(reference.pool==row.pool)&(reference.rule==row.rule)&(reference.weighting=='listing')&(reference.split=='all')].iloc[0]
    for key in ['support','confidence','lift']: assert abs(getattr(row,key)-x[key])<1e-10
show(raw)
raw.to_csv(RUN_DIR/'재계산_연관규칙.csv',index=False)


   pool rule      n  support  confidence     lift
 pooled   R1  37701 0.325270    0.887273 1.134704
 pooled   R2  37701 0.411952    0.491114 1.092553
 pooled   R3  37701 0.192011    0.590312 1.432964
sector3   R1   9516 0.516393    0.919192 1.034907
sector3   R2   9516 0.558743    0.579762 1.015464
sector3   R3   9516 0.334594    0.647945 1.159647
 target   R1 636938 0.412547    0.827399 1.144425
 target   R2 636938 0.444521    0.466762 1.027631
 target   R3 636938 0.213380    0.517226 1.163560


## 4. 지역·상품 구성을 맞춘 연관지표
region × room_type × property_type × 수용인원 구간(1–2/3–4/5–6/7+)에서 양쪽 집단 각각 30곳 이상인 층을 사용한다. 3+6의 구성비를 두 집단에 공통 적용한다. 아래는 주 분석의 숙소 가중 계산을 새로 실행한다. 호스트 가중·다른 비교층·3번 단독 보조 분석 코드는 부록에 전문으로 포함했다.


In [5]:
v['accommodates_band']=np.select([v.accommodates.between(1,2),v.accommodates.between(3,4),v.accommodates.between(5,6),v.accommodates.ge(7)],['1-2','3-4','5-6','7+'],default='unknown')
keys=['region','room_type','property_type','accommodates_band']
n=v.groupby(keys+['pool'],dropna=False).size().unstack('pool',fill_value=0)
support=n[(n.reference>=30)&(n.target>=30)]
assert len(support)==163
weights=support.reference/support.reference.sum()
adjusted=[]
for pool in ['reference','target']:
    g=v[v.pool==pool]
    means=g.groupby(keys,dropna=False)[[prefix+rule for rule in RULES for prefix in ['A_','B_','AB_']]].mean().reindex(support.index)
    for rule in RULES:
        a,b,ab=[means[prefix+rule] for prefix in ['A_','B_','AB_']]
        pa,pb,pab=[float((weights*x).sum()) for x in [a,b,ab]]
        adjusted.append(dict(pool=pool,rule=rule,pA_standardized=pa,pB_standardized=pb,support_standardized=pab,
            confidence_standardized=pab/pa,lift_standardized=pab/(pa*pb),lift_within_strata=pab/float((weights*a*b).sum())))
adjusted=pd.DataFrame(adjusted)
old=pd.read_csv(REPORT_DIR/'연관분석_구성보정.csv')
for row in adjusted.to_dict('records'):
    x=old[old.primary & old.weighting.eq('listing') & old.pool.eq(row['pool']) & old.rule.eq(row['rule'])].iloc[0]
    for name,value in row.items():
        if name not in ['pool','rule']:assert abs(value-x[name])<1e-10
show(adjusted)
adjusted.to_csv(RUN_DIR/'재계산_구성보정.csv',index=False)


     pool rule  pA_standardized  pB_standardized  support_standardized  confidence_standardized  lift_standardized  lift_within_strata
reference   R1         0.377608         0.801492              0.337053                 0.892600           1.113673            1.038479
reference   R2         0.835662         0.451132              0.413832                 0.495215           1.097717            1.037119
reference   R3         0.337053         0.413832              0.200025                 0.593454           1.434045            1.258986
   target   R1         0.479213         0.709844              0.387416                 0.808442           1.138901            1.053661
   target   R2         0.936230         0.441489              0.427930                 0.457077           1.035309            1.025743
   target   R3         0.387416         0.427930              0.194205                 0.501282           1.171412            1.162904


## 5. 군집 품질과 E4 연결
기존 K-means는 1/2/4/5번을 학습 대상으로 삼았다. 리뷰·규제를 군집 입력으로 사용하지 않았다. 7개 기능을 0–1 척도로 두고 취사·세탁·업무 블록의 제곱거리 기여를 각 1/3로 배분했다. 일반 표준화와 다르다.

현재 셀은 저장된 군집 배정을 연결한다. 재학습 코드는 부록의 `cluster` 단계에 포함돼 있다. E4 전체와 고정 시설 후보는 동일 집단이 아니며 후보가 E4의 부분집합이다. 같은 시설 입력의 연결이므로 독립적인 장기 성공 검증으로 해석하지 않는다.


In [6]:
k=pd.read_csv(CLUSTER_DIR/'숙소별_군집.csv.gz',usecols=['id','sector','region','room_type','cluster_label','ood_any'],dtype={'id':'string'})
joined=candidate[['id','region']].merge(k[['id','cluster_label','ood_any']],on='id',how='left',validate='one_to_one')
assert joined.cluster_label.eq('E_K4_4').all() and not joined.ood_any.any()
e4=k[k.sector.isin([1,2,4,5])&k.region.isin(REGIONS)&k.room_type.eq('Entire home/apt')&k.cluster_label.eq('E_K4_4')]
assert len(e4)==31224
connection=pd.DataFrame({'E4전체':e4.groupby('region').size(),'시설후보':joined.groupby('region').size()}).reindex(REGIONS)
show(connection.reset_index())
ks=load_json(CLUSTER_DIR/'결과요약.json')
quality=ks['test']['results']['Entire home/apt']
show(quality)
assert abs(quality['silhouette_test']-0.5616693439230752)<1e-10
connection.to_csv(RUN_DIR/'재계산_E4연결.csv')


     region  E4전체  시설후보
     london 17438 13317
pays-basque  2192  1563
       oslo  2570  1700
     munich   855   606
     malaga  3055  2266
    sevilla  2821  2107
   valencia  2293  1663
{'adopted': True, 'reason': 'test_pass', 'test_evaluated': True, 'test_n': 75850, 'test_cluster_n': [26132, 16612, 9010, 24096], 'silhouette_test': 0.5616693439230752, 'silhouette_sample_n': 1500, 'test_min_share': 0.11878707976268951}


## 6. 추가 조정회귀: 시설 완전 등록이 장기 설정에 더 흔한가
종속변수=bundle_registered, 설명변수=long_setting 및 지역·상품 조건. 최근 리뷰 양수 3번 대 1/2번이 주 분석이다. 호스트 단위 표준오차, 동일 침실·욕실 관측 표본 비교, 호스트 가중 및 3+6 보조 분석을 포함한다.

주 분석 조정차이 약 +4.87%p의 95% 구간은 0을 포함한다. 시설의 일반적인 장기 특성이나 전환 효과가 회귀로 확정됐다고 해석하지 않는다. 아래는 기존 결과 표시이고, 원본 회귀 전체 실행은 부록의 regression 단계다.


In [7]:
reg=load_json(REGRESSION_DIR/'결과요약.json')
reg_table=pd.DataFrame(reg['models'])
show(reg_table[['model','n','long_n','difference_pp','ci_low_pp','ci_high_pp','converged']])
m=reg_table.set_index('model').loc['M1_primary']
assert abs(m.difference_pp-4.872684552002971)<1e-8 and m.ci_low_pp<0<m.ci_high_pp


                   model     n  long_n  difference_pp  ci_low_pp  ci_high_pp  converged
           M0_unadjusted 42757     216       3.087218  -4.475700   10.650135       True
              M1_primary 42757     216       4.872685  -2.967485   12.712854       True
       M1_CC_same_sample 36609     184       4.649762  -3.856395   13.155919       True
          M2_CC_bed_bath 36609     184       3.893270  -4.630238   12.416778       True
       S_HOST_equal_host 42757     216      11.859547   3.657781   20.061313       True
S_ALL_review_interaction 70085     787      -3.055176  -7.663161    1.552808       True
           REGION_london 23190      88       2.851824  -7.975896   13.679543       True
         REGION_valencia  3891      64      -4.180767 -20.124606   11.763073       True


## 7. 손익분기 시나리오
모든 월세·비용·포기 관광 판매일은 가상의 입력이다. 장기 거래로 대체되는 관광 순기여와 플랫폼 수수료를 차감한다. 15.5%는 과제 가정이고 실제 주거상품 요율을 입증한 값이 아니다. 23,222곳에 가상 이익을 곱해 실현 기대효과로 제시하지 않는다.


In [8]:
def calc(x):
    gl=x['months']*x['monthlyRent'];gs=x['displacedNights']*x['shortNightlyPrice']
    opp=max(x['displacedNights']*(x['shortNightlyPrice']*(1-x['hostShortFee'])-x['shortNightlyCost']),x['otherHostAlternative'])
    hb=gl*(1-x['hostLongFee'])-x['months']*x['longMonthlyCost']-x['hostSetupCost']-opp
    pb=x['platformLongFee']*gl-x['platformShortFee']*gs-x['platformCost']-x['otherPlatformLoss']
    hf=max(0,(x['months']*x['longMonthlyCost']+x['hostSetupCost']+opp-x['support'])/(x['months']*(1-x['hostLongFee'])))
    pf=max(0,(x['platformShortFee']*gs+x['platformCost']+x['support']+x['otherPlatformLoss'])/(x['months']*x['platformLongFee']))
    return dict(longGMV=gl,displacedGMV=gs,hostOpportunity=opp,hostBeforeSupport=hb,platformBeforeSupport=pb,hostChange=hb+x['support'],platformChange=pb-x['support'],hostRentFloor=hf,platformRentFloor=pf,jointRentFloor=max(hf,pf),supportMin=max(0,-hb),supportMax=pb,supportFeasible=pb>=max(0,-hb),bothNonnegative=hb+x['support']>=0 and pb-x['support']>=0)

economics=load_json(REPORT_DIR/'손익분기_가정과계산.json')
defaults=economics['defaults']
scenario=[]
for nights in [0,45,90]:
    inputs=dict(defaults,displacedNights=nights)
    scenario.append(dict(포기관광박수=nights,**calc(inputs)))
scenario=pd.DataFrame(scenario)
show(scenario[['포기관광박수','hostChange','platformChange','jointRentFloor','supportFeasible']])
assert abs(calc(defaults)['hostChange']-3022)<1e-8
assert abs(calc(defaults)['platformChange']-308)<1e-8
scenario.to_csv(RUN_DIR/'재계산_손익분기.csv',index=False)


 포기관광박수  hostChange  platformChange  jointRentFloor  supportFeasible
      0      6910.0          1145.0      208.476518             True
     45      3022.0           308.0      779.211470             True
     90      -866.0          -529.0     1379.211470            False


## 8. 지역별 제안과 조사 출처
지역 선택은 군집의 자동 출력이 아니다. 저장된 보고서의 제안과 공식 출처를 연결한다. 개별 주소의 허가·주거 사용·연속 가용·호스트 의사·장기 수요는 후보 수만으로 확인되지 않는다. 다음 표는 기존 보고서의 기록이며 새로운 법률 조사 결과가 아니다.


In [9]:
meta=load_json(REPORT_DIR/'선별기준_출처_검산.json')
products=['91박 이상 실제 거주 상품','학생 최소 9개월 + 여름 관광 3개월','가구 포함 주거임대',
          '연수·파견 근무자 임시주거','근무·프로젝트 주거','학업·연구 체류 주거','학업·근무 임시주거']
show(pd.DataFrame({'region':REGIONS,'시설후보':[EXPECTED[r] for r in REGIONS],'보고서_제안':products}))
show(pd.DataFrame(meta['sources'])[['title','url']])


     region  시설후보                보고서_제안
     london 13317       91박 이상 실제 거주 상품
pays-basque  1563 학생 최소 9개월 + 여름 관광 3개월
       oslo  1700            가구 포함 주거임대
     munich   606        연수·파견 근무자 임시주거
     malaga  2266            근무·프로젝트 주거
    sevilla  2107           학업·연구 체류 주거
   valencia  1663            학업·근무 임시주거
                        title                                                                                                                                                                                      url
   City of London · 단기임대와 90박                                                                                                https://www.cityoflondon.gov.uk/services/planning/planning-enforcement/short-term-letting
           Camden · 단기임대 계획허가                                                                                                                        https://www.camden.gov.uk/short-term-lettings-planning-permission
  영국 정부 · Renters’ Rights Act      

## 9. 실제 원본 코드 전문 · 재계산 선택
아래 코드 셀은 각 원본 파일을 문자열로 등록한다. 코드를 축약하거나 외부 `.py`를 읽어서 대신 실행하지 않는다. 별도 `.py`가 없어도 원본 전문을 이 노트북에서 추출할 수 있다.

맨 위 `REPLAY_STAGES`에서 `common`, `association`, `adjustment`, `sector3`, `cluster`, `regression`, `report`, `report_update`를 선택하면 마지막 실행 셀이 새 폴더에서 실행한다. **각 단계는 기존의 고정 입력 스냅샷을 사용해 재계산**한다. 앞 단계에서 새로 만든 결과를 자동으로 모든 후속 입력에 대체하는 전체 데이터 변경 파이프라인은 아니다.

`common`은 기존 정제 CSV·호스트 분할·시설표를 입력으로 공통표를 재작성한다. 전처리 이전 원본 데이터 정제 전체 코드를 의미하지 않는다. 기본 실행은 등록만 하며 대규모 재학습을 하지 않는다.


### common — 공통표 병합·기능 파생·ID와 결측 검산
원본: `/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/prepare_common.py`
SHA256: `e4ba0ce8f35992979a9c4e223854f3acff90463cdc917b344cf619ab82394a2e`


In [10]:
SOURCE_CODE['common'] = """from pathlib import Path
import json, hashlib, math, os, datetime
import numpy as np
import pandas as pd
O=Path(__file__).resolve().parent;CONTRACT=O.parent/'00_공통분석계약.json';contract=json.loads(CONTRACT.read_text());assert str(O)==contract['facility_plan']['folder']
RAW=Path('/Users/ohgyuwon/Desktop/dev/26-2 1차인콘/data/InsightStay_data.csv');CLEAN=Path(contract['inputs']['source_csv']);SHARED=Path(contract['inputs']['shared']);FAC=Path(contract['inputs']['existing_facilities']);DICT=Path(contract['inputs']['dictionary']);SOURCE_SPEC=Path(contract['inputs']['source_spec'])
if (O/'READY.json').exists():raise RuntimeError('READY already exists; do not overwrite a published common table without a reviewed change')
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(8388608),b''):h.update(b)
 return h.hexdigest()
def js(v,name):
 def safe(x):
  if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
  if isinstance(x,(list,tuple)):return [safe(v) for v in x]
  if isinstance(x,np.generic):return safe(x.item())
  if isinstance(x,float) and not math.isfinite(x):return None
  return x
 (O/name).write_text(json.dumps(safe(v),ensure_ascii=False,indent=2,allow_nan=False,default=str)+'\\n')
def status(stage):
 js({'done':False,'stage':stage,'date_kst':'2026-10-03','remaining':['共通' if False else '공통표검산및원자적인계']},'진행상태.json');print(stage,flush=True)
checks=[]
def ck(n,v):
 checks.append({'check':n,'passed':bool(v)});assert v,n
def num(v):return pd.to_numeric(v,errors='coerce')
status('원본스키마·고정시설·호스트분할준비')
inputs=[CONTRACT,SOURCE_SPEC,RAW,CLEAN,SHARED,FAC,DICT];hashes={str(p):sha(p) for p in inputs};ck('현재CSV지정해시',hashes[str(CLEAN)]==contract['inputs']['source_expected_sha256']);js(hashes,'입력해시.json')
rawcols=pd.read_csv(RAW,nrows=0).columns.tolist();cleancols=pd.read_csv(CLEAN,nrows=0).columns.tolist();fcols=pd.read_csv(FAC,nrows=0).columns.tolist();definition=json.loads(DICT.read_text())
shared=pd.read_csv(SHARED,dtype={'id':'string','host_id':'string'},low_memory=False);ck('공통원분모677678',len(shared)==677678);ck('shared문자열ID유일',shared.id.is_unique and shared.id.notna().all() and shared.host_id.notna().all());ck('shared필터',shared.source.eq(0).all() and shared.availability_365.gt(0).all())
idset=set(shared.id);sector_map={'1-3__positive':1,'4-29__positive':2,'30+__positive':3,'1-3__zero':4,'4-29__zero':5,'30+__zero':6};shared['sector']=shared.matrix_cell.map(sector_map).astype('int8');expected={int(k):v for k,v in contract['population']['sector_n'].items()};ck('6섹터분모',shared.groupby('sector').size().to_dict()==expected);ck('군집639774',shared.sector.isin([1,2,4,5]).sum()==639774);ck('참조37904',shared.sector.isin([3,6]).sum()==37904)
required_clean=['id','host_id','source','country','region','room_type','property_type','accommodates','minimum_nights','maximum_nights','availability_365','number_of_reviews_ltm','number_of_reviews','host_listings_total','host_listings_is_unknown','host_meta_is_unknown','flag_max_nights_default']
required_raw=['id','host_id','source','bedrooms','beds','bathrooms','hosts_time_as_host_years','hosts_time_as_host_months','calculated_host_listings_count_entire_homes','calculated_host_listings_count_private_rooms','calculated_host_listings_count_shared_rooms']
for c in required_clean:ck('현재필수컬럼 '+c,c in cleancols)
for c in required_raw:ck('원본필수컬럼 '+c,c in rawcols)
optional_raw=[c for c in ['number_of_reviews_ly','last_review','scrape_date'] if c in rawcols];optional_clean=[c for c in ['number_of_reviews_ly','last_review','scrape_date'] if c in cleancols];unavailable=[]
if 'scrape_date' not in rawcols and 'scrape_date' not in cleancols:unavailable.append({'requested':'scrape_date_if_exists','output_column':None,'reason':'원본·현재CSV에scrape_date컬럼없음;수집일추정생성안함','available':False})
cc=pd.read_csv(CLEAN,usecols=required_clean+optional_clean,dtype={'id':'string','host_id':'string'},low_memory=False);ck('현재원ID유일',cc.id.is_unique);cc=cc[cc.id.isin(idset)].copy();ck('현재공통ID전수커버',len(cc)==len(shared));cc=cc.set_index('id').reindex(shared.id).reset_index();ck('현재host와공통일치',np.array_equal(cc.host_id,shared.host_id));ck('기준최소박·가용일·리뷰원값동일',all(np.allclose(cc[c],shared[c],equal_nan=True) for c in ['minimum_nights','availability_365','number_of_reviews_ltm','number_of_reviews']))
parts=[];raw_rows_read=0
for chunk in pd.read_csv(RAW,usecols=required_raw+optional_raw,dtype={'id':'string','host_id':'string'},chunksize=200000,low_memory=False):
 raw_rows_read+=len(chunk);parts.append(chunk[chunk.id.isin(idset)])
rr=pd.concat(parts,ignore_index=True);del parts;ck('원본공통ID커버·유일',len(rr)==len(shared) and rr.id.is_unique);rr=rr.set_index('id').reindex(shared.id).reset_index();ck('원본host·source동일',np.array_equal(rr.host_id,shared.host_id) and np.array_equal(rr.source,shared.source))
itemcols=[c for c in fcols if c.startswith('item__')];token_cols=[c for c in fcols if c.startswith('raw_tokens__')];outside_cols=[c for c in fcols if c.endswith('__outside_category_registered_count')];fc=pd.read_csv(FAC,usecols=['id','host_id','source','sector','matrix_cell','split','observation_state','region','room_type','property_type','accommodates','raw_registered_count','unclassified_token_count']+itemcols+token_cols+outside_cols,dtype={'id':'string','host_id':'string'},low_memory=False)
ck('기존전체시설677678ID유일',len(fc)==677678 and fc.id.is_unique);ck('시설ID전수일치',set(fc.id)==idset);fc=fc.set_index('id').reindex(shared.id).reset_index();ck('시설hostsplitsector동일',np.array_equal(fc.host_id,shared.host_id) and np.array_equal(fc.split,shared.split) and np.array_equal(fc.sector,shared.sector))
fv=fc.observation_state.eq('valid');ck('시설유효행상품키원본동일',all(np.array_equal(fc.loc[fv,c],cc.loc[fv,c]) for c in ['region','room_type','property_type','accommodates']));ck('시설미관측행키빈값범위확인',(~fv).sum()==3039 and fc.loc[~fv,['region','room_type','property_type','accommodates']].isna().all().all())
js({'rows_with_old_facility_context_blank':3039,'nonmissing_valid_facility_context_matches_current':True,'resolution':'기존시설표는3039미관측시설행의상품키도비워두었음.이번공통표는ID로현재CSV의실제지역·상품조건을연결하고시설항목NA는그대로유지.서로다른상품조건을바꾼것이아님','control_authority':str(CLEAN),'facility_authority':str(FAC),'ultra_trigger':False},'상품조건_입력대조.json')
status('공통변수병합·원관측및7기능정의검산')
out=shared.copy();schema={}
def spec(c,origin,role,meaning,source_field=None):schema[c]={'source_path':str(origin) if origin else None,'source_field':source_field or c,'role':role,'meaning':meaning}
for c in shared:spec(c,SHARED,'identifier' if c in ['id','host_id'] else ('split' if c=='split' else 'population_or_context'),'기존공통표값유지;섹터는고정matrix_cell에대응')
for c in ['country','region','room_type','property_type','accommodates','maximum_nights','flag_max_nights_default','host_meta_is_unknown','host_listings_is_unknown']:
 out[c]=cc[c].to_numpy();spec(c,CLEAN,'control' if c not in ['flag_max_nights_default','host_meta_is_unknown','host_listings_is_unknown'] else 'source_flag','현재파일의실제컬럼;새대체·인코딩없음')
for f in ['bedrooms','beds','bathrooms']:
 v=num(rr[f]);ok=v.ge(0)&np.isfinite(v);c=f+'_observed';out[c]=v.where(ok).to_numpy();out[c+'_is_missing']=~ok.to_numpy();spec(c,RAW,'control','원본의유한한음수아닌관측값만;현재impute컬럼사용안함',f);spec(c+'_is_missing',RAW,'observation_flag','원본결측/무효표시',f)
for f in ['hosts_time_as_host_years','hosts_time_as_host_months']:
 v=num(rr[f]);ok=v.ge(0)&v.mod(1).eq(0)&np.isfinite(v)
 if f.endswith('months'):ok &= v.le(11)
 out[f]=v.where(ok).to_numpy();spec(f,RAW,'control','원본정수관측값유지;월은0~11.호스트경력≠숙소등록연령')
out['host_tenure_months_observed']=out.hosts_time_as_host_years*12+out.hosts_time_as_host_months;spec('host_tenure_months_observed',RAW,'control','관측년*12+관측월;둘중하나미상이면결측','hosts_time_as_host_years + hosts_time_as_host_months')
hf=[c for c in required_raw if c.startswith('calculated_')];hv=rr[hf].apply(pd.to_numeric,errors='coerce');hv=hv.where(hv.ge(0)&np.isfinite(hv));hostsum=hv.sum(axis=1,min_count=3).where(~cc.host_listings_is_unknown.fillna(True).astype(bool));ck('관측host총량현재파일과일치',np.allclose(hostsum.dropna(),cc.loc[hostsum.notna(),'host_listings_total']))
out['host_listings_total']=hostsum.to_numpy();spec('host_listings_total',RAW,'control','원자료세calculated_host_listings_count합;현재unknownflag인행은결측.관측규모이며플랫폼전체보유로단정안함','+'.join(hf))
for c in ['number_of_reviews_ly','last_review','scrape_date']:
 if c in rr:
  out[c]=rr[c].to_numpy();spec(c,RAW,'context_only_not_model_input','전년도1/1~12/31리뷰수;rolling12개월아님' if c=='number_of_reviews_ly' else '원본관측텍스트유지;주예측입력금지')
 elif c in cc:
  out[c]=cc[c].to_numpy();spec(c,CLEAN,'context_only_not_model_input','현재실제컬럼존재값유지;주예측입력금지')
for c in ['number_of_reviews','number_of_reviews_ltm','recent_review_group']:schema[c]['role']='target_or_sensitivity_only_not_model_input'
for c in ['observation_state','raw_registered_count','unclassified_token_count']+itemcols+token_cols+outside_cols:
 out[c]=fc[c].to_numpy();spec(c,FAC,'facility_item' if c.startswith('item__') else 'facility_observation_or_count','기존고정사전·안전파싱시설표재사용.0은미등록,NA는관측판단불가.원토큰기여와항목이진값은다름')
valid=out.observation_state.eq('valid');ck('시설유효674639',valid.sum()==674639);ck('시설42항목과원토큰42',len(itemcols)==42 and len(token_cols)==42);ck('시설원항목미관측NA보존',out.loc[~valid,itemcols].isna().all().all());ck('유효시설이진값정의',out.loc[valid,itemcols].isin([0.,1.]).all().all());ck('유효원토큰0일때이진0',all(np.array_equal(out.loc[valid,c].to_numpy()>0,out.loc[valid,c.replace('raw_tokens__','item__')].to_numpy()==1) for c in token_cols))
ck('등록수안전파싱shared와일치',np.allclose(out.loc[valid,'raw_registered_count'],out.loc[valid,'amenity_count_recounted']));ck('원토큰총기여초과없음',(out.loc[valid,token_cols].sum(axis=1)<=out.loc[valid,'raw_registered_count']).all());ck('미분류기여일치',np.allclose(out.loc[valid,token_cols].sum(axis=1)+out.loc[valid,'unclassified_token_count'],out.loc[valid,'raw_registered_count']))
for category,cfg in definition['categories'].items():
 c=category+'__outside_category_registered_count';summed=out[['raw_tokens__'+f for f in cfg['items']]].sum(axis=1,min_count=len(cfg['items']));ck(category+'전체범주밖기여정의',np.allclose(out.loc[valid,c],(out.raw_registered_count-summed).loc[valid]));ck(category+'밖등록량음수없음',out.loc[valid,c].ge(0).all())
function_defs={}
for f in ['kitchen','refrigerator','washer','wifi','workspace']:
 c='function__'+f;out[c]=out['item__'+f];function_defs[c]={'inputs':['item__'+f],'formula':'identity; NA preserved','scale':'0/1/NA','role':'primary_fixed_cluster_function'}
h=out[['item__stove','item__oven','item__microwave']].astype('Float64').eq(1);out['function__heating_food']=(h.item__stove|h.item__oven|h.item__microwave).astype('Float64');function_defs['function__heating_food']={'inputs':h.columns.tolist(),'formula':'nullable OR: any observed1 ->1; all observed0 ->0; otherwiseNA. No missing=0 filling','scale':'0/1/NA','role':'primary_fixed_cluster_function'}
out['function__tools']=out[['item__cooking_basics','item__dishes_silverware']].sum(axis=1,min_count=2)/2;function_defs['function__tools']={'inputs':['item__cooking_basics','item__dishes_silverware'],'formula':'(cooking_basics+dishes_silverware)/2; any inputNA ->NA','scale':'0/.5/1/NA','role':'primary_fixed_cluster_function'}
ordered_functions=['function__'+f for f in ['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']]
for c,fd in function_defs.items():spec(c,FAC,'primary_fixed_cluster_function',fd['formula'],'+'.join(fd['inputs']))
ck('미観測' if False else '7기능미관측NA유지',out.loc[~valid,ordered_functions].isna().all().all());ck('7기능유효행완전',out.loc[valid,ordered_functions].notna().all().all())
out['host_rows_common']=out.groupby('host_id').id.transform('size');out['host_weight_common']=1./out.host_rows_common;out['host_rows_sector']=out.groupby(['sector','host_id']).id.transform('size');out['host_weight_sector']=1./out.host_rows_sector
for c in ['host_rows_common','host_weight_common','host_rows_sector','host_weight_sector']:spec(c,SHARED,'weight_or_denominator','common=전체6섹터677678내host행수및역수;sector=해당섹터내host행수및역수.이질적인범위가중치를서로대체하지않음','id/host_id/sector')
ck('동일host분할배타',out.groupby('host_id').split.nunique().max()==1);ck('기존분할ID별전수일치',np.array_equal(out.split,shared.split));ck('분할명3종',set(out.split)=={'train','validation','test'});ck('global호스트가중합1',np.allclose(out.groupby('host_id').host_weight_common.sum(),1));ck('섹터호스트가중합1',np.allclose(out.groupby(['sector','host_id']).host_weight_sector.sum(),1))
ck('부호와구간에따른섹터직접검산',all((out.loc[out.sector.eq(t),'number_of_reviews_ltm'].eq(0).all() and out.loc[out.sector.eq(r),'number_of_reviews_ltm'].gt(0).all() and out.loc[out.sector.isin([t,r]),'minimum_nights_band'].eq(b).all()) for b,(t,r) in {'1-3':(4,1),'4-29':(5,2),'30+':(6,3)}.items()))
ck('minimum_nights변경없음',np.array_equal(out.minimum_nights,shared.minimum_nights));ck('평점·리뷰파생지위입력없음',not any(c.startswith('review_scores_') for c in out.columns) and 'host_is_superhost' not in out.columns)
flow=[]
for sector in range(1,7):
 z=out[out.sector.eq(sector)]
 for split in ['all','train','validation','test']:
  s=z if split=='all' else z[z.split.eq(split)]
  flow.append({'sector':sector,'split':split,'population_n':len(s),'unique_hosts':s.host_id.nunique(),'regions':s.region.nunique(),'facility_valid_n':int(s.observation_state.eq('valid').sum()),'facility_unknown_n':int((~s.observation_state.eq('valid')).sum()),'seven_functions_complete_n':int(s[ordered_functions].notna().all(axis=1).sum()),'entire_home_n':int(s.room_type.eq('Entire home/apt').sum()),'private_room_n':int(s.room_type.eq('Private room').sum()),'other_room_type_n':int((~s.room_type.isin(['Entire home/apt','Private room'])).sum()),'bedrooms_observed_n':int(s.bedrooms_observed.notna().sum()),'beds_observed_n':int(s.beds_observed.notna().sum()),'bathrooms_observed_n':int(s.bathrooms_observed.notna().sum()),'host_scale_observed_n':int(s.host_listings_total.notna().sum()),'host_tenure_observed_n':int(s.host_tenure_months_observed.notna().sum())})
pd.DataFrame(flow).to_csv(O/'표본흐름.csv',index=False,encoding='utf-8-sig')
for c in out.columns:schema[c].update(dtype=str(out[c].dtype),missing_n=int(out[c].isna().sum()),rows=len(out))
js({'contract_path':str(CONTRACT),'contract_sha256':hashes[str(CONTRACT)],'actual_original_columns':rawcols,'actual_cleaned_columns':cleancols,'columns':schema,'unavailable_requested_columns':unavailable,'existing_flags_used_not_fabricated':['flag_max_nights_default','host_meta_is_unknown','host_listings_is_unknown'],'imputation_or_encoding_fitted':False,'review_exclusion_policy':'number_of_reviews*和last_review仅目标/语境/敏感度' if False else 'number_of_reviews*とlast_review仅target/context/sensitivity' if False else 'number_of_reviews*와last_review는목표·맥락·민감도로만사용.주요예측입력제외','max_nights_limit':'현재maximum_nights와실제flag_max_nights_default유지;법적가능기간·연속가용일로간주안함'},'컬럼명세.json')
js({'version':'common-long-product-20261003-v1','frozen_dictionary_version':definition['version'],'dictionary_source':str(DICT),'dictionary_sha256':hashes[str(DICT)],'frozen_dictionary':definition,'fixed_cluster_functions':function_defs,'fixed_cluster_function_order':ordered_functions,'existing_item_columns':itemcols,'existing_raw_token_columns':token_cols,'audit_only_not_additional_model_search':'42개원항목및원토큰은고정기여검산·추적용.주군집은계약7기능만.기존12조합을입력으로가져오지않음','registered_zero_meaning':'미등록≠실제부재','missing_policy':'unknown/empty/unseen alias observations retained asNA, not0','ly_definition':contract['user_decisions']['ly'],'prior_test_exposure':'기존host분할내부검증재사용;외부독립확증아님'},'피처사전.json')
status('공통표저장및내보낸파일재검산')
temp=O/'공통분석표.csv.gz.tmp';out.to_csv(temp,index=False,encoding='utf-8-sig',compression={'method':'gzip','compresslevel':5,'mtime':0});os.replace(temp,O/'공통분석표.csv.gz')
export=pd.read_csv(O/'공통분석표.csv.gz',dtype={'id':'string','host_id':'string'},low_memory=False)
ck('저장후행수ID문자열순서',len(export)==677678 and export.id.is_unique and np.array_equal(export.id,out.id) and np.array_equal(export.host_id,out.host_id));ck('저장후섹터분할값보존',np.array_equal(export.sector,out.sector) and np.array_equal(export.split,out.split));ck('저장후7기능NA및숫자보존',all(np.allclose(pd.to_numeric(export[c]),pd.to_numeric(out[c]),equal_nan=True) for c in ordered_functions));ck('저장후시설관측상태보존',np.array_equal(export.observation_state,out.observation_state));ck('저장후物理' if False else '저장후물리원관측NA보존',all(np.allclose(export[c],out[c],equal_nan=True) for c in ['bedrooms_observed','beds_observed','bathrooms_observed','host_listings_total','host_tenure_months_observed']));ck('저장후미관측원항목NA보존',export.loc[~valid,itemcols].isna().all().all())
for p,h0 in hashes.items():ck('입력보존 '+Path(p).name,sha(p)==h0)
split_rows=out.groupby('split').size().to_dict();split_hosts=out.groupby('split').host_id.nunique().to_dict();sector_split=out.groupby(['sector','split']).size().rename('rows').reset_index().to_dict('records');datahash=sha(O/'공통분석표.csv.gz');functions_complete=int(out[ordered_functions].notna().all(axis=1).sum());observation_counts=out.observation_state.value_counts(dropna=False).to_dict()
audit={'all_passed':True,'check_count':len(checks),'checks':checks,'rows':len(out),'sector_counts':expected,'cluster_population_n':639774,'reference_n':37904,'id_dtype_read':'string','host_id_dtype_read':'string','id_unique':True,'global_host_split_exclusive':True,'global_host_weight_sum_one':True,'sector_host_weight_sum_one':True,'split_rows':split_rows,'split_hosts':split_hosts,'sector_split_rows':sector_split,'facility_observation_counts':observation_counts,'seven_functions_complete_n':functions_complete,'physical_imputation_used':False,'new_models_or_rules_fit':False,'source_preservation':True,'original_rows_scanned_selective_columns':raw_rows_read,'input_sha256':hashes,'common_data_sha256':datahash,'missing_requested_columns':unavailable,'ultra_required':False}
js(audit,'검산.json')
summary={'done':True,'stage':'common_data_complete','date_kst':'2026-10-03','why':'최소박설정을유지한지역특화장기상품추가후보검증의두분석이같은시설·조건·호스트분할표를사용하도록인계','how':['기존shared-ID/host분할과전체시설표재사용','원본beds/bedrooms/bathrooms·host경력과관측host규모를ID로연결','고정7기능결측보존및원항목/범주밖토큰검산','전체6섹터·관측결측·host가중치·내보낸파일·원본해시검산'],'counts':{'rows':len(out),'columns':len(out.columns),'sector_n':expected,'cluster_population_n':639774,'reference_n':37904,'facility_valid_n':int(valid.sum()),'facility_unknown_n':int((~valid).sum()),'seven_functions_complete_n':functions_complete,'split_rows':split_rows},'result':'공통표준비완료.군집・장기리뷰비교의입력만제공;공급후보채택/효과판정은후속담당범위','limitations':['scrape_date가원본/현재파일모두없어생성하지않음','number_of_reviews_ly는전년도calendar-year리뷰수이며rolling12개월아님','시설0은미등록,원관측NA는유지;actualfacilities/법적적격/장기수요/수익효과미확인','관측host규모와host행수·가중치는서로다른변수;host경력≠숙소등록연령','1/2/4/5에상품을추가해도섹터3자동이동아님','기존분할/시설정의와test자료재사용;외부독립검증아님'],'missing_requested_columns':unavailable,'verification':{'all_passed':True,'check_count':len(checks),'source_preserved':True,'export_reloaded':True},'data_sha256':datahash,'artifacts':{f:str(O/f) for f in ['공통분석표.csv.gz','피처사전.json','컬럼명세.json','표본흐름.csv','검산.json','READY.json']},'next':['상품군집담당은READY해시확인후1/2/4/5 train의7기능설계진행','장기리뷰비교담당은같은공통표3/6의고정블록·지원조건분석진행'],'remaining_in_assigned_scope':[],'ultra_required':False,'model_requested':'GPT-6.1 Sol / High','runtime_model_verified':False}
js(summary,'결과요약.json');js({'done':True,'stage':'common_data_complete','date_kst':'2026-10-03','remaining':[],'pending_downstream':['군집학습및장기리뷰분석은각담당범위'],'data_sha256':datahash},'진행상태.json')
output_hashes={f:sha(O/f) for f in ['공통분석표.csv.gz','피처사전.json','컬럼명세.json','표본흐름.csv','검산.json','결과요약.json']}
ready={'ready':True,'status':'READY','done':True,'contract_version':contract['version'],'contract_sha256':hashes[str(CONTRACT)],'date_kst':'2026-10-03','data_path':str(O/'공통분석표.csv.gz'),'data_sha256':datahash,'data_hash':datahash,'rows':len(out),'row_count':len(out),'sector_counts':expected,'cluster_population_n':639774,'reference_n':37904,'split_counts':split_rows,'split_verification':{'same_existing_host_split':True,'host_id_exclusive_across_splits':True,'global_host_weight_sum_one':True,'sector_host_weight_sum_one':True},'id_columns_as_string':['id','host_id'],'feature_columns':ordered_functions,'observation_counts':observation_counts,'seven_functions_complete_n':functions_complete,'missing_requested_columns':unavailable,'output_paths':{f:str(O/f) for f in output_hashes},'output_sha256':output_hashes,'all_checks_passed':True,'verification_path':str(O/'검산.json'),'no_new_fit':True,'ultra_required':False,'atomic_publish':True}
tmp=O/'READY.json.tmp'
with tmp.open('w') as f:json.dump(ready,f,ensure_ascii=False,indent=2);f.write('\\n');f.flush();os.fsync(f.fileno())
os.replace(tmp,O/'READY.json')
print({'READY':str(O/'READY.json'),'rows':len(out),'columns':len(out.columns),'checks':len(checks),'data_sha256':datahash,'split_rows':split_rows,'facility_states':observation_counts},flush=True)
"""
SOURCE_META['common'] = {'path': '/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/prepare_common.py', 'sha256': 'e4ba0ce8f35992979a9c4e223854f3acff90463cdc917b344cf619ab82394a2e', 'language': 'python'}


### association — 고정 시설묶음 탐색·확인·원지표·후보 매핑
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/analyze_facilities.py`
SHA256: `b433c682a8e747d88e6ca92c3592e7285ca8a2669813ca40e6af2532b8573595`


In [11]:
SOURCE_CODE['association'] = """import argparse, hashlib, itertools, json, math
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

ROOT = Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
COMMON = ROOT / 'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터'
OUT = ROOT / 'outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
TASK = 'POOLED_LONG_PRODUCT_20261003'
PLAN = json.loads((OUT / '분석기준_사전고정.json').read_text())
FUNCTIONS = PLAN['functions']
BUNDLES = {b['id']: b for b in PLAN['candidate_bundles']}
INPUTS = ['kitchen', 'refrigerator', 'washer', 'wifi', 'workspace', 'stove', 'oven', 'microwave', 'cooking_basics', 'dishes_silverware']

def now(): return datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()
def write_json(name, obj):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
def write_csv(name, rows):
    d = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows)
    d.to_csv(OUT / name, index=False, encoding='utf-8-sig', compression='gzip' if name.endswith('.gz') else None)
def ratio(a, b): return float(a/b) if b else None
def progress(stage, completed, remaining):
    write_json('진행상태.json', dict(task_id=TASK, status='running', updated_at=now(), stage=stage, completed=completed, remaining=remaining))

def load():
    cols = ['id', 'host_id', 'source', 'availability_365', 'minimum_nights', 'number_of_reviews_ltm', 'sector', 'split', 'region', 'room_type', 'property_type', 'accommodates', 'observation_state']
    cols += ['function__'+f for f in FUNCTIONS] + ['item__'+f for f in INPUTS]
    d = pd.read_csv(COMMON / '공통분석표.csv.gz', usecols=cols, dtype={'id':'string', 'host_id':'string'}, encoding='utf-8-sig')
    audit = {'task_id': TASK, 'created_at':now(), 'rows':len(d), 'id_missing_n':int(d.id.isna().sum()), 'id_duplicate_n':int(d.id.duplicated().sum()), 'host_missing_n':int(d.host_id.isna().sum()), 'population_filter_violations':int(((d.source != 0) | (d.availability_365 <= 0)).sum()), 'sector_counts':{str(k):int(v) for k,v in d.sector.value_counts().sort_index().items()}, 'host_split_crossing_n':int((d.groupby('host_id').split.nunique()>1).sum()), 'split_counts':{str(k):int(v) for k,v in d.split.value_counts().items()}, 'definition_mismatch':{}, 'function_invalid_values':{}}
    expected = np.select([(d.minimum_nights<=3)&(d.number_of_reviews_ltm>0), (d.minimum_nights.between(4,29))&(d.number_of_reviews_ltm>0), (d.minimum_nights>=30)&(d.number_of_reviews_ltm>0), (d.minimum_nights<=3)&(d.number_of_reviews_ltm==0), (d.minimum_nights.between(4,29))&(d.number_of_reviews_ltm==0), (d.minimum_nights>=30)&(d.number_of_reviews_ltm==0)], [1,2,3,4,5,6], default=-1)
    audit['sector_definition_mismatch_n'] = int((d.sector != expected).sum())
    for f in ['kitchen','refrigerator','washer','wifi','workspace']:
        a,b=d['function__'+f],d['item__'+f]
        audit['definition_mismatch'][f]=int((~(a.eq(b)|(a.isna()&b.isna()))).sum())
    a=d[['item__stove','item__oven','item__microwave']]
    heat=np.where(a.eq(1).any(axis=1),1,np.where(a.eq(0).all(axis=1),0,np.nan))
    a=d[['item__cooking_basics','item__dishes_silverware']]
    tools=a.sum(axis=1,min_count=2)/2
    for f,b in [('heating_food',pd.Series(heat,index=d.index)),('tools',tools)]:
        a=d['function__'+f]
        audit['definition_mismatch'][f]=int((~(a.eq(b)|(a.isna()&b.isna()))).sum())
    for f in FUNCTIONS:
        allowed=[0,0.5,1] if f=='tools' else [0,1]
        audit['function_invalid_values'][f]=int((d['function__'+f].notna()&~d['function__'+f].isin(allowed)).sum())
    assert audit['rows']==677678 and audit['id_missing_n']==audit['id_duplicate_n']==audit['host_missing_n']==audit['population_filter_violations']==audit['host_split_crossing_n']==audit['sector_definition_mismatch_n']==0, audit
    assert sum(audit['definition_mismatch'].values())==sum(audit['function_invalid_values'].values())==0, audit
    assert set(d.split.unique())=={'train','validation','test'}, audit['split_counts']
    d['pool']=np.where(d.sector.isin([3,6]),'reference','target')
    d['observed_all7']=d[['function__'+f for f in FUNCTIONS]].notna().all(axis=1)
    d['accommodates_band']=np.select([d.accommodates.between(1,2),d.accommodates.between(3,4),d.accommodates.between(5,6),d.accommodates.ge(7)],['1-2','3-4','5-6','7+'],default='unknown')
    for bid,b in BUNDLES.items():
        a=d[['function__'+f for f in b['required_full']]]
        d['observed_'+bid]=a.notna().all(axis=1)
        fails=(a.notna()&a.ne(1)).any(axis=1)
        d['bundle_'+bid]=pd.Series(np.where(fails,0,np.where(a.eq(1).all(axis=1),1,np.nan)),index=d.index,dtype='Float64')
    d['host_weight']=1/d.groupby(['pool','host_id']).id.transform('size')
    audit['reference_n']=int(d.pool.eq('reference').sum()); audit['target_n']=int(d.pool.eq('target').sum())
    assert audit['reference_n']==37904 and audit['target_n']==639774
    audit['host_weight_pool_sums']={p:float(g.host_weight.sum()) for p,g in d.groupby('pool')}
    audit['unique_hosts_pool']={p:int(g.host_id.nunique()) for p,g in d.groupby('pool')}
    assert all(abs(audit['host_weight_pool_sums'][p]-audit['unique_hosts_pool'][p])<1e-7 for p in audit['unique_hosts_pool'])
    write_json('입력검증.json',audit)
    return d,audit

def bundle_stats(d,bid):
    obs=d['observed_'+bid]; match=d['bundle_'+bid].eq(1).fillna(False); weight=d.host_weight
    match_obs=match&obs
    return dict(bundle=bid,label=BUNDLES[bid]['label'] if bid in BUNDLES else '선정묶음 합집합',pool_n=len(d),pool_host_n=int(d.host_id.nunique()),observed_n=int(obs.sum()),unobserved_n=int((~obs).sum()),match_n=int(match.sum()),match_observed_n=int(match_obs.sum()),match_host_n=int(d.loc[match,'host_id'].nunique()),unknown_membership_n=int(d['bundle_'+bid].isna().sum()),support_observed=ratio(match_obs.sum(),obs.sum()),support_all=ratio(match.sum(),len(d)),host_weighted_support_observed=ratio(weight[match_obs].sum(),weight[obs].sum()),host_weighted_support_all=ratio(weight[match].sum(),weight.sum()),host_weight_observed_sum=float(weight[obs].sum()),host_weight_match_sum=float(weight[match].sum()))

def discover_confirm(d):
    ref=d[d.pool.eq('reference')]
    train=ref[ref.split.eq('train')]
    rows=[]; accepted_train=[]
    t=PLAN['selection']['train']
    for bid in BUNDLES:
        r=bundle_stats(train,bid)
        passed=(r['support_observed']>=t['support_observed_min'] and r['match_n']>=t['matched_listing_min'] and r['match_host_n']>=t['matched_host_min'] and r['host_weighted_support_observed']>=t['host_weighted_support_observed_min'])
        r['train_pass']=bool(passed); rows.append(r)
        if passed: accepted_train.append(bid)
    write_json('탐색_train_고정.json',dict(task_id=TASK,created_at=now(),criteria_sha256=sha(OUT/'분석기준_사전고정.json'),candidate_bundles=list(BUNDLES),accepted_train=accepted_train,train_only_results=rows,validation_used_in_discovery=False,target_used_in_discovery=False,test_used_in_discovery=False))
    print('TRAIN', json.dumps(rows,ensure_ascii=False),flush=True)
    val=ref[ref.split.eq('validation')]; v=PLAN['selection']['validation']; s=PLAN['selection']['stability']
    confirmed=[]; validations=[]
    for tr in rows:
        bid=tr['bundle']; r=bundle_stats(val,bid)
        r['listing_gap']=r['support_observed']-tr['support_observed']; r['listing_ratio']=r['support_observed']/tr['support_observed']
        r['host_gap']=r['host_weighted_support_observed']-tr['host_weighted_support_observed']; r['host_ratio']=r['host_weighted_support_observed']/tr['host_weighted_support_observed']
        checks=dict(train_pass=tr['train_pass'],validation_support=r['support_observed']>=v['support_observed_min'],validation_count=r['match_n']>=v['matched_listing_min'],validation_hosts=r['match_host_n']>=v['matched_host_min'],validation_host_support=r['host_weighted_support_observed']>=v['host_weighted_support_observed_min'],listing_gap=abs(r['listing_gap'])<=s['listing_absolute_support_gap_max'],listing_ratio=s['listing_validation_train_ratio_min']<=r['listing_ratio']<=s['listing_validation_train_ratio_max'],host_gap=abs(r['host_gap'])<=s['host_absolute_support_gap_max'],host_ratio=s['host_validation_train_ratio_min']<=r['host_ratio']<=s['host_validation_train_ratio_max'])
        r['confirmation_checks']=checks; r['confirmed']=all(checks.values()); validations.append(r)
        if r['confirmed']:confirmed.append(bid)
    write_json('확인_validation.json',dict(task_id=TASK,created_at=now(),train_lock_sha256=sha(OUT/'탐색_train_고정.json'),results=validations,confirmed=confirmed,external_independent_confirmation=False))
    write_json('선정묶음_최종고정.json',dict(task_id=TASK,created_at=now(),selected=confirmed,bundle_definitions=[BUNDLES[b] for b in confirmed],selection_source='reference pooled 3+6 train discovery then validation confirmation',target_used_in_selection=False,test_used_in_selection=False,prior_exposure_limit=PLAN['prior_exposure_limit']))
    write_csv('묶음_탐색확인.csv',[dict(phase='train',**{k:v for k,v in r.items() if not isinstance(v,dict)}) for r in rows]+[dict(phase='validation',**{k:v for k,v in r.items() if not isinstance(v,dict)}) for r in validations])
    print('VALIDATION',json.dumps(validations,ensure_ascii=False),flush=True)
    return confirmed

def association(d,rule):
    fs=list(dict.fromkeys(rule['antecedent']+rule['consequent']))
    eligible=d[['function__'+f for f in fs]].notna().all(axis=1)
    e=d[eligible]
    a=e[['function__'+f for f in rule['antecedent']]].eq(1).all(axis=1)
    b=e[['function__'+f for f in rule['consequent']]].eq(1).all(axis=1)
    ab=a&b
    conf=ratio(ab.sum(),a.sum()); base=ratio(b.sum(),len(e))
    w=e.host_weight
    wc=ratio(w[ab].sum(),w[a].sum()); wb=ratio(w[b].sum(),w.sum())
    return dict(rule=rule['id'],antecedent='+'.join(rule['antecedent']),consequent='+'.join(rule['consequent']),observed_n=len(e),A_n=int(a.sum()),B_n=int(b.sum()),AB_n=int(ab.sum()),support=ratio(ab.sum(),len(e)),confidence=conf,consequent_rate=base,lift=conf/base if conf is not None and base else None,host_support=ratio(w[ab].sum(),w.sum()),host_confidence=wc,host_consequent_rate=wb,host_lift=wc/wb if wc is not None and wb else None)

def aggregations(d,selected):
    # ANY observation is stricter than membership: all selected requirements observed.
    if selected:
        flags=d[['bundle_'+b for b in selected]]
        d['bundle_ANY']=pd.Series(np.where(flags.eq(1).fillna(False).any(axis=1),1,np.where(flags.eq(0).fillna(False).all(axis=1),0,np.nan)),index=d.index,dtype='Float64')
        d['observed_ANY']=d[['observed_'+b for b in selected]].all(axis=1)
    else:
        d['bundle_ANY']=0; d['observed_ANY']=True
    stats=[]
    for p,g in d.groupby('pool'):
        for bid in list(BUNDLES)+['ANY']:
            stats.append(dict(pool=p,split='all',**bundle_stats(g,bid)))
        if p=='reference':
            for split,ss in g.groupby('split'):
                for bid in BUNDLES:stats.append(dict(pool=p,split=split,**bundle_stats(ss,bid)))
    write_csv('묶음_전체집계.csv',stats)
    items=[]; observation=[]; distributions=[]; patterns=[]; associations=[]
    for p,g in d.groupby('pool'):
        for state,gg in g.groupby('observation_state',dropna=False):
            observation.append(dict(pool=p,observation_state=str(state),n=len(gg),host_n=int(gg.host_id.nunique()),all7_observed_n=int(gg.observed_all7.sum())))
        for f in FUNCTIONS:
            a=g['function__'+f];obs=a.notna();w=g.host_weight
            for value,label in [(0,'미등록'),(.5,'부분등록'),(1,'완전등록'),(None,'결측')]:
                if f!='tools' and value==.5:continue
                match=a.isna() if value is None else a.eq(value)
                items.append(dict(pool=p,function=f,state=label,n=int(match.sum()),observed_n=int(obs.sum()),pool_n=len(g),rate_all=ratio(match.sum(),len(g)),rate_observed=ratio(match.sum(),obs.sum()) if value is not None else None,host_weighted_rate_observed=ratio(w[match].sum(),w[obs].sum()) if value is not None else None))
        for group,fs in [('식생활4기능',['kitchen','heating_food','refrigerator','tools']),('전체7기능',FUNCTIONS)]:
            a=g[['function__'+f for f in fs]];obs=a.notna().all(axis=1)
            counts=a.loc[obs].eq(1).sum(axis=1).value_counts().sort_index()
            for count,n in counts.items():distributions.append(dict(pool=p,group=group,full_registered_functions=int(count),n=int(n),observed_n=int(obs.sum()),rate_observed=float(n/obs.sum())))
            distributions.append(dict(pool=p,group=group,full_registered_functions='unobserved',n=int((~obs).sum()),observed_n=int(obs.sum()),rate_observed=None))
        for subset,spl in [(g,'all')]+([(g[g.split.eq('train')],'train')] if p=='reference' else []):
            a=subset.loc[subset.observed_all7,['function__'+f for f in FUNCTIONS]].copy()
            c=a.value_counts().reset_index(name='n').head(10)
            for rank,(_,r) in enumerate(c.iterrows(),1):
                patterns.append(dict(pool=p,split=spl,rank=rank,n=int(r['n']),support_observed=float(r['n']/len(a)),**{f:float(r['function__'+f]) for f in FUNCTIONS}))
        for subset,spl in [(g,'all')]+([(g[g.split.eq(s)],s) for s in ['train','validation','test']] if p=='reference' else []):
            for rule in PLAN['fixed_association_rules']:associations.append(dict(pool=p,split=spl,**association(subset,rule)))
    write_csv('기능_등록률.csv',items);write_csv('시설_관측상태.csv',observation);write_csv('생활기능_분포.csv',distributions);write_csv('정확등록패턴_top10.csv',patterns);write_csv('조건부동시등록_3규칙.csv',associations)
    groups=[]
    for p,g in d.groupby('pool'):
        for dims in [['region'],['room_type'],['property_type'],['accommodates_band']]+([['sector'],['region','sector']] if p=='target' else []):
            for key,sub in g.groupby(dims,dropna=False,sort=True):
                key=key if isinstance(key,tuple) else (key,)
                for bid in selected+['ANY']:
                    r=bundle_stats(sub,bid)
                    groups.append(dict(pool=p,dimension='+'.join(dims),group=' | '.join(str(k) for k in key),**dict(zip(dims,key)),**r))
    gd=pd.DataFrame(groups);write_csv('묶음_지역섹터유형별.csv',gd)
    write_csv('대상_지역섹터별.csv',gd[(gd.pool=='target')&(gd.dimension=='region+sector')])
    # Pairwise listing/host intersections and observed, mutually exclusive patterns.
    overlap=[];partition=[]
    for p,g in d.groupby('pool'):
        for a,b in itertools.combinations(selected,2):
            ma=g['bundle_'+a].eq(1).fillna(False);mb=g['bundle_'+b].eq(1).fillna(False);mask=ma&mb
            ha=set(g.loc[ma,'host_id']);hb=set(g.loc[mb,'host_id'])
            overlap.append(dict(pool=p,bundle_a=a,bundle_b=b,intersection_n=int(mask.sum()),intersection_host_n=int(g.loc[mask,'host_id'].nunique()),host_overlap_any_listing_n=len(ha&hb),host_union_n=len(ha|hb)))
        pattern=g[['bundle_'+b for b in selected]].astype('string').fillna('NA').agg('|'.join,axis=1) if selected else pd.Series('NONE',index=g.index)
        for value,sub in g.groupby(pattern):partition.append(dict(pool=p,pattern=value,n=len(sub),host_n=int(sub.host_id.nunique()),bundle_order='|'.join(selected)))
    write_csv('묶음_중복.csv',overlap);write_csv('묶음_배타패턴.csv',partition)
    region_stability=[]
    ref=d[d.pool=='reference']
    for region,g in ref.groupby('region'):
        for bid in selected:
            tr=bundle_stats(g[g.split=='train'],bid); va=bundle_stats(g[g.split=='validation'],bid)
            eligible=tr['observed_n']>=100 and va['observed_n']>=100
            region_stability.append(dict(region=region,bundle=bid,train_observed_n=tr['observed_n'],validation_observed_n=va['observed_n'],train_support=tr['support_observed'],validation_support=va['support_observed'],gap=(va['support_observed']-tr['support_observed']) if tr['support_observed'] is not None and va['support_observed'] is not None else None,reporting_eligible=eligible))
    write_csv('지역_train_validation_안정성.csv',region_stability)
    return stats,items,associations,groups,overlap,partition

def standardizations(d,selected):
    rows=[];detail=[]
    for dims in [['region'],['room_type'],['property_type'],['accommodates_band'],['region','room_type','accommodates_band']]:
        for bid in selected:
            summaries=[]
            for pool in ['reference','target']:
                sub=d[(d.pool==pool)&d['observed_'+bid]].copy()
                sub['match']=sub['bundle_'+bid].eq(1).astype(int)
                grouped=sub.groupby(dims,dropna=False).agg(observed_n=('id','size'),match_n=('match','sum'))
                grouped.columns=[pool+'_'+c for c in grouped.columns]
                summaries.append(grouped)
            joined=summaries[0].join(summaries[1],how='outer').fillna(0)
            joined['eligible']=(joined.reference_observed_n>=30)&(joined.target_observed_n>=30)
            j=joined[joined.eligible]
            ref_den=float(j.reference_observed_n.sum());tar_den=float(j.target_observed_n.sum())
            ref_total=float(joined.reference_observed_n.sum());tar_total=float(joined.target_observed_n.sum())
            ref_rate=float(j.reference_match_n.sum()/ref_den) if ref_den else None
            tar_std=float(((j.target_match_n/j.target_observed_n)*j.reference_observed_n).sum()/ref_den) if ref_den else None
            r=dict(dimension='+'.join(dims),bundle=bid,included_strata_n=len(j),excluded_strata_n=int((~joined.eligible).sum()),reference_coverage=ratio(ref_den,ref_total),target_coverage=ratio(tar_den,tar_total),reference_rate_common=ref_rate,target_rate_standardized_to_reference=tar_std,standardized_gap=ref_rate-tar_std if ref_rate is not None else None,reference_raw_rate=ratio(joined.reference_match_n.sum(),ref_total),target_raw_rate=ratio(joined.target_match_n.sum(),tar_total))
            rows.append(r)
            for key,v in joined.iterrows():
                key=key if isinstance(key,tuple) else (key,)
                detail.append(dict(dimension='+'.join(dims),bundle=bid,group=' | '.join(str(k) for k in key),**{k:int(v[k]) for k in ['reference_observed_n','reference_match_n','target_observed_n','target_match_n']},eligible=bool(v.eligible)))
    write_csv('구성표준화.csv',rows);write_csv('구성표준화_층별근거.csv',detail)
    return rows

def mapping_and_checks(d,selected,audit,stats):
    target=d[d.pool=='target']
    mapping=pd.DataFrame({'id':target.id})
    for b in selected:mapping['bundle_'+b]=target['bundle_'+b].astype('Int8')
    mapping['facility_observed_all7']=target.observed_all7.astype('int8')
    for b in selected:mapping['facility_observed_'+b]=target['observed_'+b].astype('int8')
    write_csv('후보매핑.csv.gz',mapping)
    reread=pd.read_csv(OUT/'후보매핑.csv.gz',dtype={'id':'string'},encoding='utf-8-sig')
    checks=dict(task_id=TASK,created_at=now(),input=audit,mapping_rows=len(reread),mapping_id_missing_n=int(reread.id.isna().sum()),mapping_id_duplicate_n=int(reread.id.duplicated().sum()),missing_target_ids_n=len(set(target.id)-set(reread.id)),unexpected_ids_n=len(set(reread.id)-set(target.id)),mapping_membership_count_matches={},unknowns_preserved={},weight_origin='recalculated 1/host rows within reference pooled 3+6 and separately within target 1+2+4+5',region_n_all=int(d.region.nunique()),region_n_reference=int(d[d.pool=='reference'].region.nunique()),region_n_target=int(target.region.nunique()))
    for b in selected:
        a=int(target['bundle_'+b].eq(1).sum());z=int(reread['bundle_'+b].eq(1).sum())
        checks['mapping_membership_count_matches'][b]=dict(source=a,saved=z,pass_=a==z)
        checks['unknowns_preserved'][b]=dict(source=int(target['bundle_'+b].isna().sum()),saved=int(reread['bundle_'+b].isna().sum()))
    checks['listing_union_inclusion_exclusion']={}
    if 'L' in selected and 'W' in selected:
        l=target.bundle_L.eq(1).fillna(False);w=target.bundle_W.eq(1).fillna(False)
        n=int(l.sum()+w.sum()-(l&w).sum());un=int(target.bundle_ANY.eq(1).sum())
        checks['listing_union_inclusion_exclusion']=dict(L_n=int(l.sum()),W_n=int(w.sum()),intersection_n=int((l&w).sum()),calculated_union=n,observed_union=un,pass_=n==un)
        lh=set(target.loc[l,'host_id']);wh=set(target.loc[w,'host_id']);anyh=set(target.loc[target.bundle_ANY.eq(1).fillna(False),'host_id'])
        checks['host_union_inclusion_exclusion']=dict(L_hosts=len(lh),W_hosts=len(wh),host_overlap_any_listing_n=len(lh&wh),calculated_union=len(lh)+len(wh)-len(lh&wh),observed_union=len(anyh),same_listing_both_host_n=int(target.loc[l&w,'host_id'].nunique()),pass_=(lh|wh)==anyh)
        if 'LW' in selected:checks['LW_equals_L_intersect_W']=bool(target.bundle_LW.eq(1).fillna(False).equals(l&w))
    checks['id_sets_exact']=checks['mapping_id_missing_n']==checks['mapping_id_duplicate_n']==checks['missing_target_ids_n']==checks['unexpected_ids_n']==0 and len(reread)==639774
    checks['mapping_columns_minimal']=all(c=='id' or c.startswith('bundle_') or c.startswith('facility_observed_') for c in reread.columns)
    checks['all7_unknown_union_membership_preserved']=bool(d.loc[~d.observed_all7,'bundle_ANY'].isna().all())
    checks['all_pass']=checks['id_sets_exact'] and checks['mapping_columns_minimal'] and all(x['pass_'] for x in checks['mapping_membership_count_matches'].values()) and all(x['source']==x['saved'] for x in checks['unknowns_preserved'].values()) and checks.get('LW_equals_L_intersect_W',True) and checks.get('listing_union_inclusion_exclusion',{}).get('pass_',True) and checks.get('host_union_inclusion_exclusion',{}).get('pass_',True)
    assert checks['all_pass'],checks
    write_json('검산.json',checks)
    return checks

if __name__=='__main__':
    arg=argparse.ArgumentParser();arg.add_argument('--phase',choices=['selection','full'],required=True);args=arg.parse_args()
    progress('loading_and_definition_audit',['task specification','dictionary/schema','criteria frozen'],['train discovery','validation','pooled summaries','target mapping','report'])
    d,audit=load()
    if args.phase=='selection':
        selected=discover_confirm(d)
        progress('selection_locked',['definition audit','train discovery','validation confirmation','final selection lock'],['pooled summaries','composition','target mapping','report verification'])
        print('SELECTED',selected,flush=True)
    else:
        selected=json.loads((OUT/'선정묶음_최종고정.json').read_text())['selected']
        progress('aggregate_and_target_mapping',['selection locked'],['pooled summaries','composition','target mapping','report verification'])
        stats,items,associations,groups,overlap,partition=aggregations(d,selected)
        std=standardizations(d,selected)
        checks=mapping_and_checks(d,selected,audit,stats)
        summary=dict(task_id=TASK,created_at=now(),selected_bundles=selected,bundle_definitions=[BUNDLES[b] for b in selected],populations={p:{'n':len(g),'host_n':int(g.host_id.nunique()),'all7_observed_n':int(g.observed_all7.sum()),'region_n':int(g.region.nunique())} for p,g in d.groupby('pool')},pooled_bundles=[r for r in stats if r['split']=='all'],fixed_association_rules=[r for r in associations if r['pool']=='reference' and r['split']=='all'],composition_standardization=std,overlap=overlap,exclusive_patterns=partition,definition=PLAN['interpretation'],prior_exposure_limit=PLAN['prior_exposure_limit'],interpretation_limits=['minimum_nights>=30 설정 공급; 실제 장기예약 확인 아님','시설0=미등록; 실제 부재 아님. 시설등록은 기능 품질·전용성·속도·접근권한 보증 아님','취사용 가열은 stove/oven/microwave 중 하나로 구성; 난방과 구분','세탁기에는 공용·유료 등록도 포함; 주방은 kitchenette 포함','업무형은 Wi-Fi·전용 작업공간 등록만 요구하며 생활기능을 보증하지 않음','시설 반복·조건부 동시등록은 같은 묶음 선호·신규 장기수요·예약효과·성공확률·수익증가 입증 아님','법정 필수시설·법적 적격 검증 아님','target pool의 기존 minimum_nights 유지와 상품 추가 검토용; 실제 판매 가능성은 별도 확인','지역·유형 구성표준화는 기술적 비교이며 인과추정 아님','호스트 수는 묶음·지역·섹터 사이에 중복되어 단순 합산 불가'],source_paths={'common':str(COMMON/'공통분석표.csv.gz'),'dictionary':str(COMMON/'피처사전.json'),'schema':str(COMMON/'컬럼명세.json'),'criteria':str(OUT/'분석기준_사전고정.json')},check_pass=checks['all_pass'])
        write_json('결과요약.json',summary)
        progress('report_pending',['pooled aggregation','target counts','composition standardization','mapping and checks'],['report','independent recount','HTML visual verification','READY'])
        print('POOLED',json.dumps(summary['pooled_bundles'],ensure_ascii=False),flush=True)
        print('STANDARDIZATION',json.dumps(std,ensure_ascii=False),flush=True)
"""
SOURCE_META['association'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/analyze_facilities.py', 'sha256': 'b433c682a8e747d88e6ca92c3592e7285ca8a2669813ca40e6af2532b8573595', 'language': 'python'}


### adjustment — 시설 연관규칙 구성 보정·호스트 가중·민감도
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/adjust_associations.py`
SHA256: `a955ca50a8e208f34158ce834cfc193d900d49744b4be0193cdddfdb48d3c281`


In [12]:
SOURCE_CODE['adjustment'] = """import hashlib,json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

ROOT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
COMMON=ROOT/'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/공통분석표.csv.gz'
OUT=ROOT/'outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
PLAN=json.loads((OUT/'연관보정_사전기준.json').read_text());OLD=json.loads((OUT/'분석기준_사전고정.json').read_text())
RULES=OLD['fixed_association_rules'];FUNCS=OLD['functions']
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def stamp():return datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
def dump(name,obj):(OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
def metric(a,b,ab):
    return {'pA':float(a),'pB':float(b),'support':float(ab),'confidence':float(ab/a) if a else None,'lift':float(ab/(a*b)) if a*b else None}

mapping_hash_before=sha(OUT/'후보매핑.csv.gz')
cols=['id','host_id','source','availability_365','sector','region','room_type','property_type','accommodates']+['function__'+f for f in FUNCS]
d=pd.read_csv(COMMON,usecols=cols,dtype={'id':'string','host_id':'string'},encoding='utf-8-sig')
assert len(d)==677678 and d.id.is_unique and not d.host_id.isna().any()
assert d.source.eq(0).all() and d.availability_365.gt(0).all()
d['pool']=np.where(d.sector.isin([3,6]),'reference','target')
d['host_weight']=1/d.groupby(['pool','host_id']).id.transform('size')
d['accommodates_band']=np.select([d.accommodates.between(1,2),d.accommodates.between(3,4),d.accommodates.between(5,6),d.accommodates.ge(7)],['1-2','3-4','5-6','7+'],default='unknown')
observed=d[['function__'+f for f in FUNCS]].notna()
assert observed.eq(observed.iloc[:,0],axis=0).all().all(), 'Observation masks differ; must use rule-specific masks.'
valid=d[observed.all(axis=1)].copy()
assert len(valid[valid.pool=='reference'])==37701 and len(valid[valid.pool=='target'])==636938
for rule in RULES:
    r=rule['id'];valid['A_'+r]=valid[['function__'+f for f in rule['antecedent']]].eq(1).all(axis=1).astype(int)
    valid['B_'+r]=valid[['function__'+f for f in rule['consequent']]].eq(1).all(axis=1).astype(int)
    valid['AB_'+r]=valid['A_'+r]*valid['B_'+r]
    for prefix in ['A','B','AB']:valid['w'+prefix+'_'+r]=valid['host_weight']*valid[prefix+'_'+r]
oldmetrics=pd.read_csv(OUT/'조건부동시등록_3규칙.csv')
rawchecks=[];globalmetrics={}
for pool,g in valid.groupby('pool'):
    for rule in RULES:
        r=rule['id'];n=len(g);w=g.host_weight.sum()
        met=metric(g['A_'+r].sum()/n,g['B_'+r].sum()/n,g['AB_'+r].sum()/n)
        hm=metric(g['wA_'+r].sum()/w,g['wB_'+r].sum()/w,g['wAB_'+r].sum()/w)
        old=oldmetrics[(oldmetrics.pool==pool)&(oldmetrics.split=='all')&(oldmetrics.rule==r)].iloc[0]
        assert all(abs(met[k]-old[k])<1e-10 for k in ['support','confidence','lift'])
        assert all(abs(hm[k]-old['host_'+k])<1e-10 for k in ['support','confidence','lift'])
        rawchecks.append({'pool':pool,'rule':r,'raw_metrics_match_existing':True,'host_metrics_match_existing':True})
        globalmetrics[(pool,r,'listing')]=met;globalmetrics[(pool,r,'host')]=hm
cases=[PLAN['primary_stratum']]+PLAN['sensitivity_strata']
results=[];cellrows=[];casesummary=[];old_support_checks=[]
oldstd=pd.read_csv(OUT/'구성표준화.csv')
for case_no,dims in enumerate(cases):
    name='+'.join(dims);primary=case_no==0;aggs=[]
    agg={'n':('id','size'),'hosts_n':('host_id','nunique'),'w':('host_weight','sum')}
    for rule in RULES:
        for prefix in ['A','B','AB','wA','wB','wAB']:
            key=prefix+'_'+rule['id'];agg[key]=(key,'sum')
    for pool in ['reference','target']:
        a=valid[valid.pool==pool].groupby(dims,dropna=False).agg(**agg)
        a.columns=[pool+'_'+c for c in a.columns];aggs.append(a)
    allcells=aggs[0].join(aggs[1],how='outer').fillna(0)
    ok=(allcells.reference_n>=30)&(allcells.target_n>=30)
    cells=allcells[ok].copy();assert len(cells)>0
    qn=cells.reference_n/cells.reference_n.sum();qw=cells.reference_w/cells.reference_w.sum()
    assert abs(qn.sum()-1)<1e-12 and abs(qw.sum()-1)<1e-12
    info={'dimension':name,'primary':primary,'included_strata_n':len(cells),'excluded_strata_n':int((~ok).sum()),'coverage':{}}
    for pool in ['reference','target']:
        v=valid[valid.pool==pool];ix=v.set_index(dims).index;included=ix.isin(cells.index);sub=v[included]
        cov={'all_observed_n':len(v),'included_observed_n':len(sub),'observed_listing_coverage':float(len(sub)/len(v)),'all_observed_hosts_n':int(v.host_id.nunique()),'included_observed_hosts_n':int(sub.host_id.nunique()),'observed_host_coverage':float(sub.host_id.nunique()/v.host_id.nunique()),'all_observed_weight':float(v.host_weight.sum()),'included_observed_weight':float(sub.host_weight.sum()),'observed_weight_coverage':float(sub.host_weight.sum()/v.host_weight.sum())}
        info['coverage'][pool]=cov
        assert len(sub)==int(cells[pool+'_n'].sum())
        for rule in RULES:
            r=rule['id']
            assert (allcells[pool+'_AB_'+r]<=allcells[pool+'_A_'+r]+1e-9).all()
            assert (allcells[pool+'_AB_'+r]<=allcells[pool+'_B_'+r]+1e-9).all()
            assert (allcells[pool+'_A_'+r]+allcells[pool+'_B_'+r]-allcells[pool+'_AB_'+r]<=allcells[pool+'_n']+1e-9).all()
            for weighting,q,den,prefix in [('listing',qn,cells[pool+'_n'],''),('host',qw,cells[pool+'_w'],'w')]:
                pa=cells[pool+'_'+prefix+'A_'+r]/den;pb=cells[pool+'_'+prefix+'B_'+r]/den;pab=cells[pool+'_'+prefix+'AB_'+r]/den
                m=metric((q*pa).sum(),(q*pb).sum(),(q*pab).sum())
                expected=float((q*pa*pb).sum());conditional_lift=m['support']/expected if expected else None
                baseline_conditional_b=expected/m['pA'] if m['pA'] else None
                restricted=metric(cells[pool+'_'+prefix+'A_'+r].sum()/den.sum(),cells[pool+'_'+prefix+'B_'+r].sum()/den.sum(),cells[pool+'_'+prefix+'AB_'+r].sum()/den.sum())
                for k in ['pA','pB','support','confidence']:assert 0<=m[k]<=1
                assert m['support']<=min(m['pA'],m['pB'])+1e-12
                assert m['support']>=max(0,m['pA']+m['pB']-1)-1e-12
                row={'dimension':name,'primary':primary,'rule':r,'pool':pool,'weighting':weighting,'common_strata_n':len(cells),'observed_coverage':cov['observed_listing_coverage'],'host_coverage':cov['observed_host_coverage'],'weight_coverage':cov['observed_weight_coverage'],'included_observed_n':len(sub),'included_observed_hosts_n':int(sub.host_id.nunique()),**{k+'_standardized':v for k,v in m.items()},'independent_support_within_strata':expected,'lift_within_strata':float(conditional_lift),'conditional_baseline_B':float(baseline_conditional_b),**{k+'_raw_full':v for k,v in globalmetrics[(pool,r,weighting)].items()},**{k+'_raw_common':v for k,v in restricted.items()}}
                results.append(row)
                # Independent per-record weighted summation, not groupby aggregation.
                qlookup=q.to_dict();masslookup=cells[pool+('_n' if weighting=='listing' else '_w')].to_dict()
                baseweights=sub.host_weight.to_numpy() if weighting=='host' else np.ones(len(sub))
                keys=sub.set_index(dims).index
                rowweights=np.array([qlookup[key]/masslookup[key] for key in keys])*baseweights
                assert abs(rowweights.sum()-1)<1e-10
                independently=metric(float(np.dot(rowweights,sub['A_'+r])),float(np.dot(rowweights,sub['B_'+r])),float(np.dot(rowweights,sub['AB_'+r])))
                assert all(abs(m[k]-independently[k])<1e-10 for k in m)
                if name=='region+room_type+accommodates_band' and weighting=='listing':
                    b={'R1':'L','R2':'W','R3':'LW'}[r]
                    z=oldstd[(oldstd.dimension==name)&(oldstd.bundle==b)].iloc[0]
                    expected_old=z.reference_rate_common if pool=='reference' else z.target_rate_standardized_to_reference
                    assert abs(m['support']-expected_old)<1e-10
                    old_support_checks.append({'pool':pool,'rule':r,'matches_previous_standardized_support':True})
    for key,row in allcells.iterrows():
        key=key if isinstance(key,tuple) else (key,)
        is_eligible=bool(row.reference_n>=30 and row.target_n>=30)
        obj={'dimension':name,'primary':primary,'stratum_key':json.dumps(list(key),ensure_ascii=False),'eligible':is_eligible,**dict(zip(dims,key)),'q_listing':float(row.reference_n/cells.reference_n.sum()) if is_eligible else 0.,'q_host':float(row.reference_w/cells.reference_w.sum()) if is_eligible else 0.}
        for col,v in row.items():obj[col]=float(v) if '_w' in col else int(v)
        cellrows.append(obj)
    casesummary.append(info)
    print(json.dumps({'dimension':name,'included_strata':len(cells),'coverage_reference':info['coverage']['reference']['observed_listing_coverage'],'coverage_target':info['coverage']['target']['observed_listing_coverage']},ensure_ascii=False),flush=True)

pd.DataFrame(results).to_csv(OUT/'연관보정_집계.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(cellrows).to_csv(OUT/'연관보정_층별근거.csv',index=False,encoding='utf-8-sig')
# Independently recompute expected conditional independence from serialized cells.
saved=pd.read_csv(OUT/'연관보정_층별근거.csv');checks=[]
for r in results:
    g=saved[(saved.dimension==r['dimension'])&saved.eligible];p=r['pool'];q=g['q_listing' if r['weighting']=='listing' else 'q_host'];prefix='' if r['weighting']=='listing' else 'w';den=g[p+('_n' if r['weighting']=='listing' else '_w')]
    a=g[p+'_'+prefix+'A_'+r['rule']]/den;b=g[p+'_'+prefix+'B_'+r['rule']]/den;ab=g[p+'_'+prefix+'AB_'+r['rule']]/den
    m=metric(float(sum(q*a)),float(sum(q*b)),float(sum(q*ab)));baseline=float(sum(q*a*b))
    assert abs(m['support']/baseline-r['lift_within_strata'])<1e-10
    assert all(abs(v-r[k+'_standardized'])<1e-10 for k,v in m.items())
    checks.append({'dimension':r['dimension'],'rule':r['rule'],'pool':p,'weighting':r['weighting'],'pass':True})
mapping_hash_after=sha(OUT/'후보매핑.csv.gz');assert mapping_hash_before==mapping_hash_after
summary={'task_id':PLAN['task_id'],'followup':PLAN['followup'],'created_at':stamp(),'criteria_sha256':sha(OUT/'연관보정_사전기준.json'),'primary_dimension':'+'.join(PLAN['primary_stratum']),'cases':casesummary,'primary_results':[r for r in results if r['primary']],'all_results':results,'definitions':PLAN['metrics'],'interpretation':PLAN['interpretation'],'candidate_mapping_changed':False}
dump('연관보정_결과요약.json',summary)
dump('연관보정_검산.json',{'task_id':PLAN['task_id'],'created_at':stamp(),'raw_reproduction':rawchecks,'previous_three_factor_support':old_support_checks,'saved_cell_recalculation':checks,'independent_row_weight_sums_verified':True,'all7_observation_masks_equal':True,'no_missing_zero_imputation':True,'candidate_mapping_sha256_before':mapping_hash_before,'candidate_mapping_sha256_after':mapping_hash_after,'candidate_mapping_unchanged':True,'all_pass':True})
print('PRIMARY',json.dumps(summary['primary_results'],ensure_ascii=False),flush=True)
"""
SOURCE_META['adjustment'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/adjust_associations.py', 'sha256': 'a955ca50a8e208f34158ce834cfc193d900d49744b4be0193cdddfdb48d3c281', 'language': 'python'}


### sector3 — 3번 단독 보조 분석
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/sector3_associations.py`
SHA256: `7af94d484cce9bcae99eeb4df08032fcbc5e6633dc83132b49defda076828fdd`


In [13]:
SOURCE_CODE['sector3'] = """import hashlib,json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

ROOT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
COMMON=ROOT/'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/공통분석표.csv.gz'
OUT=ROOT/'outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
OLD=json.loads((OUT/'분석기준_사전고정.json').read_text())
ADJ=json.loads((OUT/'연관보정_사전기준.json').read_text())
RULES=OLD['fixed_association_rules'];FUNCS=OLD['functions']
def stamp():return datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
def dump(name,obj):(OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def metric(a,b,ab):
    return {'pA':float(a),'pB':float(b),'support':float(ab),'confidence':float(ab/a) if a else None,'lift':float(ab/(a*b)) if a*b else None}

# Freeze the new sensitivity scope before examining its frequencies.
PLAN={
 'task_id':OLD.get('task_id','POOLED_LONG_PRODUCT_20261003'),'followup':'sector3_only_association_sensitivity','frozen_at':stamp(),
 'purpose':'최근 리뷰가 있는 장기 설정 공급인 3번만으로 기존 세 시설 규칙을 재계산한다. 최근 리뷰는 실제 장기예약 또는 장기상품 성공을 뜻하지 않음.',
 'reference':'sector==3; minimum_nights>=30 and number_of_reviews_ltm>0',
 'comparison':'sector 3+6 pooled and sector 1+2+4+5 pooled',
 'rules':RULES,'functions':FUNCS,'primary_stratum':ADJ['primary_stratum'],'sensitivity_strata':ADJ['sensitivity_strata'],
 'common_support':'3번과 중단기형에 각각 시설 관측30숙소 이상인 공통층. 통합3+6은3번을 포함하므로 동일층에 포함. 시설등록률이나 연관지표를 보고 층을 선택하지 않음.',
 'composition':'각 공통층의3번 관측 구성비를3번·통합3+6·중단기형 모두에 적용. 동일층·동일구성에서3번과통합을 비교.',
 'host_weight':'3번전체, 통합3+6전체, 중단기형전체 각각1/해당집단내host숙소수 재계산. 관측·공통층 제한 전에 계산. 구성비 역시3번 호스트가중 관측비중을 공통 적용.',
 'metrics':ADJ['metrics'],'observation':ADJ['observations'],
 'split':'기존host split의3번 원지표를train/validation/test별로 기술. 자료노출 이력이 있어 독립확증 아님. 새 후보선정이나 재선정에 사용하지 않음.',
 'selection':'기존시설 후보 정의·후보매핑 유지. 세규칙만 재계산하고 조합 탐색 범위를 넓히지 않음.',
 'interpretation':'공급시설의 동시등록을 설명하는 민감도 분석.3vs6 성공·실패 비교, 실제장기예약·예약효과·인과 추론 없음.'}
assert not (OUT/'3번단독_사전기준.json').exists(), 'A frozen plan already exists; do not overwrite it.'
dump('3번단독_사전기준.json',PLAN)
cp=json.loads((OUT/'현재작업체크포인트.json').read_text())
cp.update(status='in_progress',stage='sector3_only_sensitivity',followup=PLAN['followup'],updated_at=stamp(),remaining=['sector3 raw and common-composition metrics','independent verification','report and manifest'])
dump('현재작업체크포인트.json',cp)
dump('진행상태.json',{'task_id':cp['task_id'],'status':cp['status'],'stage':cp['stage'],'updated_at':cp['updated_at'],'completed':cp['completed'],'remaining':cp['remaining']})

mapping_before=sha(OUT/'후보매핑.csv.gz')
cols=['id','host_id','source','availability_365','sector','split','minimum_nights','number_of_reviews_ltm','region','room_type','property_type','accommodates']+['function__'+f for f in FUNCS]
d=pd.read_csv(COMMON,usecols=cols,dtype={'id':'string','host_id':'string'},encoding='utf-8-sig')
assert len(d)==677678 and d.id.is_unique and d.source.eq(0).all() and d.availability_365.gt(0).all()
assert d.loc[d.sector.eq(3),'minimum_nights'].ge(30).all() and d.loc[d.sector.eq(3),'number_of_reviews_ltm'].gt(0).all()
assert d.groupby('host_id').split.nunique().eq(1).all()
d['accommodates_band']=np.select([d.accommodates.between(1,2),d.accommodates.between(3,4),d.accommodates.between(5,6),d.accommodates.ge(7)],['1-2','3-4','5-6','7+'],default='unknown')
ob=d[['function__'+f for f in FUNCS]].notna()
assert ob.eq(ob.iloc[:,0],axis=0).all().all()
d['observed']=ob.all(axis=1)
for r in RULES:
    rid=r['id'];d['A_'+rid]=d[['function__'+f for f in r['antecedent']]].eq(1).all(axis=1).astype(int)
    d['B_'+rid]=d[['function__'+f for f in r['consequent']]].eq(1).all(axis=1).astype(int)
    d['AB_'+rid]=d['A_'+rid]*d['B_'+rid]

pop={};valid={};population={}
for name,sectors in [('sector3',[3]),('pooled',[3,6]),('target',[1,2,4,5])]:
    g=d[d.sector.isin(sectors)].copy();g['host_weight']=1/g.groupby('host_id').id.transform('size')
    assert abs(g.host_weight.sum()-g.host_id.nunique())<1e-7
    for r in RULES:
        for prefix in ['A','B','AB']:g['w'+prefix+'_'+r['id']]=g['host_weight']*g[prefix+'_'+r['id']]
    pop[name]=g;valid[name]=g[g.observed].copy();v=valid[name]
    population[name]={'all_n':len(g),'all_hosts_n':int(g.host_id.nunique()),'observed_n':len(v),'unobserved_n':len(g)-len(v),'observed_hosts_n':int(v.host_id.nunique()),'regions_n':int(g.region.nunique()),'observed_weight':float(v.host_weight.sum())}
assert population['pooled']['observed_n']==37701 and population['target']['observed_n']==636938
raw=[];globalmetrics={};oldraw=pd.read_csv(OUT/'조건부동시등록_3규칙.csv');rawchecks=[]
for name in valid:
    splits=['all','train','validation','test'] if name=='sector3' else ['all']
    for split in splits:
        g=valid[name] if split=='all' else valid[name][valid[name].split.eq(split)]
        assert len(g)>0
        for rule in RULES:
            rid=rule['id'];counts={k:int(g[k+'_'+rid].sum()) for k in ['A','B','AB']}
            for weight in ['listing','host']:
                den=len(g) if weight=='listing' else g.host_weight.sum();prefix='' if weight=='listing' else 'w'
                m=metric(*(g[prefix+k+'_'+rid].sum()/den for k in ['A','B','AB']))
                row={'pool':name,'split':split,'rule':rid,'weighting':weight,'observed_n':len(g),'observed_hosts_n':int(g.host_id.nunique()),'nA':counts['A'],'nB':counts['B'],'nAB':counts['AB'],'weight_denominator':float(den),**m};raw.append(row)
                if split=='all':globalmetrics[(name,rid,weight)]=m
                if name in ['pooled','target']:
                    prev=oldraw[(oldraw.pool==('reference' if name=='pooled' else 'target'))&(oldraw.split=='all')&(oldraw.rule==rid)].iloc[0]
                    for k in ['support','confidence','lift']:assert abs(m[k]-prev[('host_' if weight=='host' else '')+k])<1e-10
                    rawchecks.append({'pool':name,'rule':rid,'weighting':weight,'matches_existing':True})

cases=[PLAN['primary_stratum']]+PLAN['sensitivity_strata'];results=[];cellrows=[];caseinfo=[];recordchecks=[]
for no,dims in enumerate(cases):
    dimension='+'.join(dims);aggs=[]
    spec={'n':('id','size'),'hosts_n':('host_id','nunique'),'w':('host_weight','sum')}
    for r in RULES:
        for pre in ['A','B','AB','wA','wB','wAB']:
            k=pre+'_'+r['id'];spec[k]=(k,'sum')
    for pool,g in valid.items():
        a=g.groupby(dims,dropna=False).agg(**spec);a.columns=[pool+'_'+k for k in a.columns];aggs.append(a)
    allcells=aggs[0].join(aggs[1:],how='outer').fillna(0)
    ok=(allcells.sector3_n>=30)&(allcells.target_n>=30);cells=allcells[ok].copy()
    assert len(cells)>0 and cells.pooled_n.ge(cells.sector3_n).all()
    qn=cells.sector3_n/cells.sector3_n.sum();qh=cells.sector3_w/cells.sector3_w.sum()
    assert abs(qn.sum()-1)<1e-12 and abs(qh.sum()-1)<1e-12
    info={'dimension':dimension,'primary':no==0,'included_strata_n':len(cells),'excluded_strata_n':int((~ok).sum()),'coverage':{}}
    sub={}
    for pool,g in valid.items():
        sub[pool]=g[g.set_index(dims).index.isin(cells.index)];s=sub[pool]
        assert len(s)==int(cells[pool+'_n'].sum())
        cov={'all_observed_n':len(g),'included_observed_n':len(s),'listing_coverage':len(s)/len(g),'included_hosts_n':int(s.host_id.nunique()),'host_coverage':s.host_id.nunique()/g.host_id.nunique(),'included_observed_weight':float(s.host_weight.sum()),'weight_coverage':float(s.host_weight.sum()/g.host_weight.sum())};info['coverage'][pool]=cov
        for r in RULES:
            rid=r['id']
            for weight,q in [('listing',qn),('host',qh)]:
                prefix='' if weight=='listing' else 'w';den=cells[pool+('_n' if weight=='listing' else '_w')]
                pa=cells[pool+'_'+prefix+'A_'+rid]/den;pb=cells[pool+'_'+prefix+'B_'+rid]/den;pab=cells[pool+'_'+prefix+'AB_'+rid]/den
                m=metric(float((q*pa).sum()),float((q*pb).sum()),float((q*pab).sum()))
                expected=float((q*pa*pb).sum())
                assert 0<=m['support']<=min(m['pA'],m['pB'])+1e-12
                assert m['support']>=max(0,m['pA']+m['pB']-1)-1e-12
                assert all(0<=m[k]<=1 for k in ['pA','pB','confidence'])
                restricted=metric(*(float(cells[pool+'_'+prefix+k+'_'+rid].sum()/den.sum()) for k in ['A','B','AB']))
                results.append({'dimension':dimension,'primary':no==0,'rule':rid,'pool':pool,'weighting':weight,'common_strata_n':len(cells),'included_n':len(s),'listing_coverage':cov['listing_coverage'],'host_coverage':cov['host_coverage'],'weight_coverage':cov['weight_coverage'],**{k+'_standardized':v for k,v in m.items()},'independent_support_within_strata':expected,'lift_within_strata':m['support']/expected,**{k+'_raw_full':v for k,v in globalmetrics[(pool,rid,weight)].items()},**{k+'_raw_common':v for k,v in restricted.items()}})
                if no==0:
                    qi=q.to_dict();di=den.to_dict();keys=s.set_index(dims).index
                    base=np.ones(len(s)) if weight=='listing' else s.host_weight.to_numpy()
                    rw=np.array([qi[k]/di[k] for k in keys])*base
                    assert abs(rw.sum()-1)<1e-10
                    independent=metric(*(float(np.dot(rw,s[k+'_'+rid])) for k in ['A','B','AB']))
                    assert all(abs(m[k]-independent[k])<1e-10 for k in m)
                    recordchecks.append({'pool':pool,'rule':rid,'weighting':weight,'record_weight_recalculation':True})
    for key,row in allcells.iterrows():
        key=key if isinstance(key,tuple) else (key,);eligible=bool(row.sector3_n>=30 and row.target_n>=30)
        rec={'dimension':dimension,'primary':no==0,'stratum_key':json.dumps(list(key),ensure_ascii=False),'eligible':eligible,**dict(zip(dims,key)),'q_listing':float(row.sector3_n/cells.sector3_n.sum()) if eligible else 0.,'q_host':float(row.sector3_w/cells.sector3_w.sum()) if eligible else 0.}
        for c,x in row.items():rec[c]=float(x) if '_w' in c else int(x)
        cellrows.append(rec)
    caseinfo.append(info)
    print(json.dumps({'dimension':dimension,'strata_n':len(cells),'coverage':{k:round(v['listing_coverage'],4) for k,v in info['coverage'].items()}},ensure_ascii=False),flush=True)

pd.DataFrame(raw).to_csv(OUT/'3번단독_원지표.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(results).to_csv(OUT/'3번단독_연관보정.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(cellrows).to_csv(OUT/'3번단독_층별근거.csv',index=False,encoding='utf-8-sig')
saved=pd.read_csv(OUT/'3번단독_층별근거.csv');checks=[]
for r in results:
    g=saved[(saved.dimension==r['dimension'])&saved.eligible];p=r['pool'];q=g['q_listing' if r['weighting']=='listing' else 'q_host'];pre='' if r['weighting']=='listing' else 'w';den=g[p+('_n' if r['weighting']=='listing' else '_w')]
    a=g[p+'_'+pre+'A_'+r['rule']]/den;b=g[p+'_'+pre+'B_'+r['rule']]/den;ab=g[p+'_'+pre+'AB_'+r['rule']]/den
    m=metric(float(sum(q*a)),float(sum(q*b)),float(sum(q*ab)));ex=float(sum(q*a*b))
    assert all(abs(m[k]-r[k+'_standardized'])<1e-10 for k in m)
    assert abs(m['support']/ex-r['lift_within_strata'])<1e-10
    checks.append({'dimension':r['dimension'],'pool':p,'rule':r['rule'],'weighting':r['weighting'],'pass':True})
for dim,g in saved.groupby('dimension'):
    for p in population:assert int(g[p+'_n'].sum())==population[p]['observed_n']
    assert abs(g.q_listing.sum()-1)<1e-10 and abs(g.q_host.sum()-1)<1e-10
after=sha(OUT/'후보매핑.csv.gz');assert mapping_before==after
primary_cells=saved[saved.primary&saved.eligible]
summary={'task_id':PLAN['task_id'],'followup':PLAN['followup'],'created_at':stamp(),'criteria_sha256':sha(OUT/'3번단독_사전기준.json'),'population':population,'primary_dimension':'+'.join(PLAN['primary_stratum']),'primary_regions_n':int(primary_cells.region.nunique()),'primary_property_types_n':int(primary_cells.property_type.nunique()),'primary_room_types':sorted(primary_cells.room_type.unique().tolist()),'cases':caseinfo,'raw_results':raw,'primary_results':[r for r in results if r['primary']],'all_results':results,'definitions':PLAN['metrics'],'candidate_mapping_changed':False,'interpretation':PLAN['interpretation']}
dump('3번단독_결과요약.json',summary)
dump('3번단독_검산.json',{'created_at':stamp(),'population_definition_verified':True,'host_weights_recomputed_per_pool':True,'host_split_unique':True,'all7_observation_masks_identical':True,'no_na_zero_imputation':True,'existing_pooled_target_reproduced':rawchecks,'independent_record_recalculation':recordchecks,'serialized_cell_recalculation':checks,'all_case_observed_counts_reconciled':True,'candidate_mapping_sha256_before':mapping_before,'candidate_mapping_sha256_after':after,'candidate_mapping_unchanged':True,'all_pass':True})
print('POPULATION',json.dumps(population,ensure_ascii=False),flush=True)
print('RAW_SECTOR3',json.dumps([r for r in raw if r['pool']=='sector3' and r['split']=='all'],ensure_ascii=False),flush=True)
print('PRIMARY_R3',json.dumps([r for r in results if r['primary'] and r['rule']=='R3'],ensure_ascii=False),flush=True)
"""
SOURCE_META['sector3'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/sector3_associations.py', 'sha256': '7af94d484cce9bcae99eeb4df08032fcbc5e6633dc83132b49defda076828fdd', 'language': 'python'}


### cluster — 군집수 선택·학습·실루엣·ARI·민감도·배정
원본: `/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/analyze.py`
SHA256: `e590706f6b6f4681871f93b22387eab86301a419bfcb24ab4095ccc25e2f3aaf`


In [14]:
SOURCE_CODE['cluster'] = """\"\"\"Fixed, outcome-blind facility clustering; writes only beside this file.\"\"\"
from pathlib import Path
import os
os.environ.setdefault('OMP_NUM_THREADS', '2')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '2')
import sys, json, hashlib, itertools, datetime, html, traceback
import numpy as np
import pandas as pd
import joblib
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from threadpoolctl import threadpool_limits

OUT = Path(__file__).resolve().parent
CONTRACT_PATH = OUT.parent / '00_공통분석계약.json'
PLAN_PATH = OUT / '실행전계획.json'
FEATURES = ['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']
FCOLS = ['function__' + s for s in FEATURES]
ITEMS = ['kitchen','stove','oven','microwave','refrigerator','cooking_basics','dishes_silverware','washer','wifi','workspace']
ICOLS = ['item__' + s for s in ITEMS]
ROOMS = ['Entire home/apt','Private room']
KEYS = ['country','region','room_type','property_type','accommodates']
KOREAN = dict(zip(FEATURES, ['취사공간','가열기기','냉장고','조리도구·식기','세탁기','와이파이','업무공간']))
PATTERNS = np.array(list(itertools.product([0.,1.],[0.,1.],[0.,1.],[0.,.5,1.],[0.,1.],[0.,1.],[0.,1.])))
RADIX = [2,2,2,3,2,2,2]
NPAT = len(PATTERNS)
assert NPAT == 192


def clean(x):
    if isinstance(x, dict): return {str(k):clean(v) for k,v in x.items()}
    if isinstance(x, (list,tuple,np.ndarray)): return [clean(v) for v in x]
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    if isinstance(x, (float,np.floating)): return float(x) if np.isfinite(x) else None
    if isinstance(x, Path): return str(x)
    return x


def sha(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(2**20), b''): h.update(block)
    return h.hexdigest()


def now():
    return datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).isoformat()


def js(name, data, immutable=False):
    p = OUT / name
    if immutable and p.exists(): raise RuntimeError(f'고정 파일을 덮어쓸 수 없음: {name}')
    q = p.with_name(p.name+'.tmp')
    q.write_text(json.dumps(clean(data),ensure_ascii=False,indent=2,allow_nan=False)+'\\n')
    q.replace(p)


def csv(name, data):
    d = data if isinstance(data,pd.DataFrame) else pd.DataFrame(data)
    p = OUT/name
    q = p.with_name(p.name+'.tmp')
    d.to_csv(q,index=False,compression='gzip' if name.endswith('.gz') else None)
    q.replace(p)


def status(state, done=False, **kwargs):
    js('진행상태.json',dict(status=state,done=done,updated_at_kst=now(),**kwargs))


def record_event(event, **kwargs):
    with (OUT/'실행순서.jsonl').open('a') as f:
        f.write(json.dumps(clean(dict(event=event,at_kst=now(),**kwargs)),ensure_ascii=False)+'\\n')


def numeric(s):
    # Common table boolean flags may be serialized as True/False or 1/0.
    return pd.to_numeric(s.replace({'True':1,'False':0,'true':1,'false':0,True:1,False:0}),errors='coerce')


def encode(x):
    digits = x.copy()
    digits[:,3] *= 2
    digits = digits.astype(int)
    ans = np.zeros(len(x),dtype=np.int16)
    for i,r in enumerate(RADIX): ans = ans*r+digits[:,i]
    return ans


assert np.array_equal(encode(PATTERNS),np.arange(NPAT))


def counts(codes, weight=None):
    return np.bincount(codes,weights=weight,minlength=NPAT).astype(float)


def fit(count, k, weight, seed, plan):
    use = count>0
    if int(use.sum()) < k: raise ValueError('관측 패턴 수가 k보다 적음')
    a = plan['algorithm']
    return KMeans(n_clusters=k,random_state=seed,n_init=a['n_init'],max_iter=a['max_iter'],
                  tol=a['tol'],algorithm=a['algorithm']).fit(PATTERNS[use]*weight,sample_weight=count[use])


def sil(code, labels_by_pattern, weight, sample_pos):
    if not len(sample_pos): return None
    labels = labels_by_pattern[code[sample_pos]]
    if not 1<len(np.unique(labels))<len(labels): return None
    return float(silhouette_score(PATTERNS[code[sample_pos]]*weight,labels))


def sample_pos(n, plan):
    return np.sort(np.random.default_rng(plan['validation']['silhouette_seed']).choice(
        n,min(n,plan['validation']['silhouette_sample_max']),replace=False))


def ari(a,b):
    return float(adjusted_rand_score(a,b)) if len(a)>1 else None


def read_common(contract):
    ready_path = Path(contract['common_ready'])
    if not ready_path.exists():
        status('awaiting_common_data',preparation_done=True,
               completed=['공통계약 확인','실행전 상세설계 고정','실행 코드 준비'],
               pending=['공통 READY와 해시 검산','실제 군집 학습·검증','3/6 투영·비교범위','보고·최종 검산'])
        print('awaiting_common_data',flush=True)
        return None
    ready = json.loads(ready_path.read_text())
    common = Path(contract['common_output'])
    verification_path = common.parent/'검산.json'
    if not common.exists() or not verification_path.exists():
        raise RuntimeError('READY는 있으나 공통표 또는 검산 파일이 없음')
    verification = json.loads(verification_path.read_text())
    # Fail closed on explicit negative readiness; require affirmative QA and declared file hash.
    if ready.get('done') is False or ready.get('ready') is False or verification.get('all_passed') is False:
        raise RuntimeError('공통 데이터 완료/검산 상태가 통과가 아님')
    digest = sha(common)
    strings = []
    def leaves(x):
        if isinstance(x,dict):
            for v in x.values(): leaves(v)
        elif isinstance(x,list):
            for v in x: leaves(v)
        elif isinstance(x,str): strings.append(x)
    leaves(ready)
    if digest not in strings:
        raise RuntimeError('READY에 공통분석표 SHA256 일치값이 확인되지 않음; 스키마 확인 필요')
    affirmative = verification.get('all_passed') is True
    if not affirmative:
        raise RuntimeError('공통 검산의 명시적 통과 필드를 확인해야 함')
    assert ready['contract_sha256']==sha(CONTRACT_PATH)
    assert ready['output_sha256']['검산.json']==sha(verification_path)
    assert all(c['passed'] for c in verification['checks'])
    required = ['id','host_id','sector','split','observation_state']+KEYS+FCOLS+ICOLS
    wanted = set(required+['minimum_nights','number_of_reviews_ltm','availability_365',
                           'raw_registered_count','unclassified_token_count','amenity_count_recounted'])
    d = pd.read_csv(common,usecols=lambda c:c in wanted,dtype={'id':'string','host_id':'string'},low_memory=False)
    missing = set(required)-set(d)
    if missing: raise RuntimeError(f'필수 컬럼 누락: {sorted(missing)}')
    assert len(d)==contract['population']['n'] and d.id.is_unique and d.id.notna().all()
    assert d.host_id.notna().all() and d.groupby('host_id').split.nunique(dropna=False).eq(1).all()
    assert set(d.split)=={'train','validation','test'}
    assert d.sector.value_counts().sort_index().to_dict()=={int(k):v for k,v in contract['population']['sector_n'].items()}
    if 'minimum_nights' in d and 'number_of_reviews_ltm' in d:
        assert (d.minimum_nights.ge(1)&d.minimum_nights.eq(d.minimum_nights.round())).all()
        sector = np.select([d.minimum_nights.le(3),d.minimum_nights.le(29)],[1,2],default=3)
        sector += 3*d.number_of_reviews_ltm.eq(0).to_numpy()
        assert np.array_equal(sector,d.sector) and d.number_of_reviews_ltm.ge(0).all()
    if 'availability_365' in d: assert d.availability_365.gt(0).all()
    for c in ICOLS+FCOLS: d[c] = numeric(d[c])
    for c in ICOLS: assert d[c].dropna().isin([0,1]).all(),c
    for c in FCOLS:
        assert d[c].dropna().isin([0,.5,1] if c=='function__tools' else [0,1]).all(),c
    heat_items=d[['item__stove','item__oven','item__microwave']]
    heat=pd.Series(np.nan,index=d.index)
    heat.loc[heat_items.eq(1).any(axis=1)]=1
    heat.loc[heat_items.eq(0).all(axis=1)]=0
    expected = pd.DataFrame({
        'kitchen':d.item__kitchen,
        'heating_food':heat,
        'refrigerator':d.item__refrigerator,
        'tools':(d.item__cooking_basics+d.item__dishes_silverware)/2,
        'washer':d.item__washer,'wifi':d.item__wifi,'workspace':d.item__workspace})
    assert np.allclose(d[FCOLS].to_numpy(float),expected.to_numpy(float),equal_nan=True), '공통 기능값 공식 불일치'
    known = d.observation_state.eq('valid') & d[ICOLS].notna().all(axis=1) & d[FCOLS].notna().all(axis=1)
    d['facility_observed'] = known
    d['room_type_supported'] = d.room_type.isin(ROOMS)
    d['cluster_eligible'] = known & d.room_type_supported
    d['information_insufficient'] = ~known
    d['exclusion_reason'] = np.select(
        [~known & ~d.room_type_supported,~known,~d.room_type_supported],
        ['facility_unknown_and_room_unsupported','facility_information_unknown','room_type_outside_scope'],default='')
    d['pattern_code'] = pd.Series(pd.NA,index=d.index,dtype='Int64')
    d.loc[known,'pattern_code'] = encode(d.loc[known,FCOLS].to_numpy(float))
    d['population_role'] = np.where(d.sector.isin([1,2,4,5]),'fit_population','reference_projection')
    provenance = {'common_path':str(common),'common_sha256':digest,'ready_sha256':sha(ready_path),
                  'common_qa_sha256':sha(verification_path),'contract_sha256':sha(CONTRACT_PATH),
                  'plan_sha256':sha(PLAN_PATH),'script_sha256':sha(__file__)}
    js('입력검산.json',{'all_pass':True,'n':len(d),'known_n':int(known.sum()),
                         'eligible_n':int(d.cluster_eligible.sum()),'provenance':provenance})
    record_event('common_verified',**provenance)
    return d,provenance


def calibration(model, count, weight, labelmap):
    lab = model.predict(PATTERNS*weight)
    dist = np.linalg.norm(PATTERNS*weight-model.cluster_centers_[lab],axis=1)
    threshold = {}
    for i in range(model.n_clusters):
        idx = np.where((lab==i)&(count>0))[0]
        order = idx[np.argsort(dist[idx],kind='stable')]
        cum = np.cumsum(count[order])
        threshold[i] = float(dist[order[np.searchsorted(cum,.99*cum[-1],side='left')]])
    raw = model.cluster_centers_/weight
    return {'model':model,'weight':weight,'features':FEATURES,'labelmap':labelmap,
            'train_pattern_counts':count,'labels_by_pattern':lab,'distances_by_pattern':dist,
            'distance_threshold99':threshold,'feature_min':PATTERNS[count>0].min(axis=0),
            'feature_max':PATTERNS[count>0].max(axis=0),'raw_centroids':raw}


def choose_and_freeze(d, plan, provenance):
    frozen_path = OUT/'선택고정.json'
    if frozen_path.exists():
        frozen = json.loads(frozen_path.read_text())
        assert frozen['provenance']==provenance, '고정 이후 입력·계획·코드가 변경됨'
        models = {}
        for room,s in frozen['selected'].items():
            if s['k'] is not None:
                p = OUT/s['model_file']
                assert sha(p)==s['model_sha256']
                models[room] = joblib.load(p)
        return frozen,models,pd.read_csv(OUT/'군집평가.csv').to_dict('records')
    weight = np.sqrt(np.array(plan['transformation']['primary_squared_weights']))
    candidates, selected, models = [], {}, {}
    for room in ROOMS:
        q = d[d.cluster_eligible & d.room_type.eq(room) & d.sector.isin([1,2,4,5])]
        train, val = q[q.split.eq('train')],q[q.split.eq('validation')]
        ct = train.pattern_code.to_numpy(int); cv = val.pattern_code.to_numpy(int)
        cnt = counts(ct); vp = sample_pos(len(val),plan)
        hosts = np.sort(train.host_id.unique()); hostvals = train.host_id.to_numpy()
        local_models = {}
        for k in plan['algorithm']['k']:
            status('fitting_train_validation',room_type=room,k=k,preparation_done=True)
            mods = [fit(cnt,k,weight,s,plan) for s in plan['algorithm']['seeds']]
            labs = [m.predict(PATTERNS*weight) for m in mods]
            base = labs[0]
            saris = [ari(a[cv],b[cv]) for a,b in itertools.combinations(labs,2)]
            nt = np.bincount(base[ct],minlength=k); nv = np.bincount(base[cv],minlength=k)
            sv = sil(cv,base,weight,vp)
            failures = []
            tests = {'validation_silhouette':sv is not None and sv>=.2,
                     'train_n':nt.min()>=100,'validation_n':nv.min()>=30,
                     'train_share':nt.min()/nt.sum()>=.05,'validation_share':nv.min()/nv.sum()>=.05,
                     'seed_median':np.median(saris)>=.9,'seed_min':min(saris)>=.8}
            failures += [x for x,ok in tests.items() if not ok]
            basic = all(tests.values()); haris=[]; waris={}
            if basic:
                for seed in plan['validation']['host_resampling']['seeds']:
                    hs = np.random.default_rng(seed).choice(hosts,max(1,int(len(hosts)*.8)),replace=False)
                    use = pd.Series(hostvals).isin(hs).to_numpy()
                    hm = fit(counts(ct[use]),k,weight,11,plan)
                    haris.append(ari(base[cv],hm.predict(PATTERNS[cv]*weight)))
                for name,sq in plan['transformation']['sensitivity_squared_weights'].items():
                    w = np.sqrt(sq)
                    wm = fit(cnt,k,w,11,plan)
                    waris[name] = ari(base[cv],wm.predict(PATTERNS[cv]*w))
            hp = bool(haris and np.median(haris)>=.8 and min(haris)>=.65)
            wp = bool(waris and min(waris.values())>=.65)
            if basic and not hp: failures.append('host_resampling')
            if basic and not wp: failures.append('weight_sensitivity')
            item = dict(room_type=room,k=k,train_n=len(train),validation_n=len(val),
                        train_unique_patterns=int((cnt>0).sum()),silhouette_validation=sv,
                        silhouette_sample_n=len(vp),seed_ari_median=np.median(saris),seed_ari_min=min(saris),
                        train_min_cluster_n=nt.min(),validation_min_cluster_n=nv.min(),
                        train_min_share=nt.min()/nt.sum(),validation_min_share=nv.min()/nv.sum(),
                        basic_pass=basic,host_aris=haris,host_ari_median=np.median(haris) if haris else None,
                        host_ari_min=min(haris) if haris else None,weight_aris=waris,
                        weight_ari_min=min(waris.values()) if waris else None,
                        host_pass=hp,weight_pass=wp,fully_pass=basic and hp and wp,
                        failure_reasons=';'.join(failures),sensitivity_run=basic)
            candidates.append(item);local_models[k]=mods[0]
            csv('군집평가.csv',candidates)
            print(json.dumps(clean(item),ensure_ascii=False),flush=True)
        passed = [c for c in candidates if c['room_type']==room and c['fully_pass']]
        if not passed:
            selected[room] = {'k':None,'selection_status':'not_adopted','reason':'no_fully_passing_candidate'}
            continue
        best = max(c['silhouette_validation'] for c in passed)
        choice = min((c for c in passed if c['silhouette_validation']>=best-.03),key=lambda c:c['k'])
        model = local_models[choice['k']]
        raw = model.cluster_centers_/weight
        order = sorted(range(model.n_clusters),key=lambda i:tuple(raw[i]))
        labelmap = {i:('E' if room==ROOMS[0] else 'P')+f'_K{model.n_clusters}_{j+1}' for j,i in enumerate(order)}
        bundle = calibration(model,cnt,weight,labelmap)
        agg = float(np.sum(cnt*bundle['distances_by_pattern']**2))
        full = float(np.sum(bundle['distances_by_pattern'][ct]**2))
        assert np.isclose(agg,full,rtol=1e-12,atol=1e-8)
        assert np.isclose(agg,model.inertia_,rtol=1e-8,atol=1e-6)
        bundle['aggregation_check']={'aggregated':agg,'expanded':full,'inertia':float(model.inertia_),'all_pass':True}
        fname = '모형_'+('E' if room==ROOMS[0] else 'P')+'.joblib'
        joblib.dump(bundle,OUT/fname)
        models[room]=bundle
        selected[room]={'k':model.n_clusters,'selection_status':'selected_pending_test',
                        'choice':choice,'model_file':fname,'model_sha256':sha(OUT/fname),
                        'labelmap':labelmap,'aggregation_check':bundle['aggregation_check']}
    frozen = {'version':'1.0','frozen_at_kst':now(),'provenance':provenance,
              'fit_sectors':[1,2,4,5],'features':FEATURES,'selected':selected,
              'test_evaluated_at_freeze':False,'reference_evaluated_at_freeze':False,
              'no_outcome_or_reference_selection':True,'plan_file':PLAN_PATH.name}
    js('선택고정.json',frozen,immutable=True)
    record_event('all_models_frozen',selection_sha256=sha(frozen_path),selected=selected)
    return frozen,models,candidates


def evaluate_test(d, models, plan, frozen_sha):
    path = OUT/'시험평가.json'
    if path.exists():
        saved = json.loads(path.read_text())
        assert saved['selection_sha256']==frozen_sha
        return saved
    result={}
    for room in ROOMS:
        if room not in models:
            result[room]={'adopted':False,'reason':'no_fully_passing_candidate','test_evaluated':False}
            continue
        b=models[room]; q=d[d.cluster_eligible & d.room_type.eq(room)&d.sector.isin([1,2,4,5])&d.split.eq('test')]
        code=q.pattern_code.to_numpy(int); labels=b['labels_by_pattern'][code]
        n=np.bincount(labels,minlength=b['model'].n_clusters)
        sample=sample_pos(len(q),plan); st=sil(code,b['labels_by_pattern'],b['weight'],sample)
        passed=st is not None and st>=.2 and n.min()>=30 and n.min()/n.sum()>=.05
        result[room]={'adopted':passed,'reason':'test_pass' if passed else 'test_gate_failed',
                      'test_evaluated':True,'test_n':len(q),'test_cluster_n':n,'silhouette_test':st,
                      'silhouette_sample_n':len(sample),'test_min_share':n.min()/n.sum()}
    saved={'selection_sha256':frozen_sha,'evaluated_at_kst':now(),'results':result,'no_reselection':True}
    js('시험평가.json',saved,immutable=True)
    record_event('test_evaluated',selection_sha256=frozen_sha,results=result)
    return saved


def apply_models(d,models,test,frozen_sha):
    assert sha(OUT/'선택고정.json')==frozen_sha
    d['diagnostic_cluster']='';d['cluster_label']='';d['model_adopted']=False
    d['distance_to_centroid']=np.nan;d['train_distance99_threshold']=np.nan
    d['train_pattern_n']=pd.Series(pd.NA,index=d.index,dtype='Int64')
    d['ood_evaluated']=False
    for c in ['ood_outside_feature_range','ood_novel_training_pattern','ood_distance','ood_any']:
        d[c]=pd.Series(pd.NA,index=d.index,dtype='boolean')
    d['assignment_status']=np.where(d.cluster_eligible,'no_selected_model',d.exclusion_reason)
    record_event('reference_projection_started',selection_sha256=frozen_sha)
    for room,b in models.items():
        ix=d.cluster_eligible & d.room_type.eq(room);q=d.loc[ix]
        code=q.pattern_code.to_numpy(int); labels=b['labels_by_pattern'][code]
        dist=b['distances_by_pattern'][code]
        thresholds=np.array([b['distance_threshold99'][int(i)] for i in labels])
        outside=((PATTERNS[code]<b['feature_min'])|(PATTERNS[code]>b['feature_max'])).any(axis=1)
        novel=b['train_pattern_counts'][code]==0
        far=dist>thresholds+1e-12
        d.loc[ix,'diagnostic_cluster']=[b['labelmap'][int(i)] for i in labels]
        adopted=test['results'][room]['adopted']
        if adopted:
            d.loc[ix,'cluster_label']=d.loc[ix,'diagnostic_cluster']
            d.loc[ix,'model_adopted']=True
        d.loc[ix,'assignment_status']='assigned_adopted' if adopted else 'selected_but_test_failed'
        d.loc[ix,'distance_to_centroid']=dist;d.loc[ix,'train_distance99_threshold']=thresholds
        d.loc[ix,'train_pattern_n']=b['train_pattern_counts'][code].astype(int)
        d.loc[ix,'ood_evaluated']=True
        for c,x in [('ood_outside_feature_range',outside),('ood_novel_training_pattern',novel),
                    ('ood_distance',far),('ood_any',outside|novel|far)]: d.loc[ix,c]=x
    d['supported_for_cluster_interpretation']=d.model_adopted & ~d.ood_any.fillna(True)
    return d


def composition(d,models,plan):
    results=[]
    for room,b in models.items():
        q=d[d.cluster_eligible & d.room_type.eq(room)&d.sector.isin([1,2,4,5])]
        tr=q[q.split.eq('train')];va=q[q.split.eq('validation')]
        ct=tr.pattern_code.to_numpy(int);cv=va.pattern_code.to_numpy(int)
        base=b['labels_by_pattern'][cv];k=b['model'].n_clusters;w=b['weight']
        groups={'host_equal':tr.host_id,'sector_equal':tr.sector,
                'region_equal':tr.country.astype('string').fillna('<NA>')+' / '+tr.region.astype('string').fillna('<NA>')}
        for name,key in groups.items():
            n=key.value_counts(dropna=False); rw=(1/key.map(n)).to_numpy(float)
            m=fit(counts(ct,rw),k,w,11,plan)
            other=m.predict(PATTERNS[cv]*w)
            tab=np.zeros((k,k),dtype=np.int64);np.add.at(tab,(base,other),1)
            ia,ib=linear_sum_assignment(-tab);mapping={int(j):int(i) for i,j in zip(ia,ib)}
            remapped=np.array([mapping[int(j)] for j in other]);gap=0.
            for i,j in zip(ia,ib):gap=max(gap,float(np.max(np.abs(b['raw_centroids'][i]-m.cluster_centers_[j]/w))))
            overall=ari(base,other);disagree=float(np.mean(base!=remapped))
            for sector in ['all',1,2,4,5]:
                mask=np.ones(len(va),bool) if sector=='all' else va.sector.eq(sector).to_numpy()
                if not mask.any():continue
                results.append({'room_type':room,'sensitivity':name,'validation_sector':sector,
                    'n':int(mask.sum()),'ari':ari(base[mask],other[mask]),
                    'mapped_disagreement_fraction':float(np.mean(base[mask]!=remapped[mask])),
                    'max_matched_raw_centroid_gap':gap,
                    'descriptive_flag':overall<.65 or disagree>.2 or gap>=.25,
                    'not_used_to_reselect':True})
    csv('구성민감도.csv',pd.DataFrame(results,columns=['room_type','sensitivity','validation_sector','n','ari',
        'mapped_disagreement_fraction','max_matched_raw_centroid_gap','descriptive_flag','not_used_to_reselect']))
    return results


def profiles(d):
    rows=[]
    assigned=d[d.diagnostic_cluster.ne('')]
    # Pooled fit profile defines descriptive names; reference profiles never rename models.
    names={}
    for (room,cl),g in assigned[assigned.sector.isin([1,2,4,5])].groupby(['room_type','diagnostic_cluster']):
        mean=g[FCOLS].mean().to_numpy(); low=[KOREAN[FEATURES[i]] for i,x in enumerate(mean[:6]) if x<=.3]
        broad=bool((mean[:6]>=.7).all())
        name='생활기능 등록이 폭넓은 유형' if broad else ('·'.join(low)+' 등록이 적은 유형' if low else '생활기능 등록이 혼합된 유형')
        if mean[6]>=.7:name+=' / 업무공간 등록 많음'
        names[cl]=name
    for (room,cl,role,sp),g in assigned.groupby(['room_type','diagnostic_cluster','population_role','split']):
        hc=g.host_id.value_counts();rc=g.groupby(['country','region'],dropna=False).size().sort_values(ascending=False)
        hw=g.host_id.map(1/hc).to_numpy(float)
        row={'room_type':room,'cluster':cl,'name':names[cl],'population_role':role,'split':sp,'n':len(g),
             'model_adopted':bool(g.model_adopted.all()),'hosts':len(hc),'largest_host_share':hc.max()/len(g),
             'regions':len(rc),'largest_region_share':rc.iloc[0]/len(g),
             'top_regions':'; '.join(f'{x[0]}/{x[1]}:{v}' for x,v in rc.head(3).items()),
             'ood_n':int(g.ood_any.fillna(False).sum()),'distance_median':g.distance_to_centroid.median()}
        for s in FEATURES:
            row[s+'_registered_mean']=g['function__'+s].mean()
            row[s+'_host_equal_mean']=np.average(g['function__'+s],weights=hw)
        for sec in range(1,7):row[f'sector{sec}_n']=int(g.sector.eq(sec).sum())
        if 'raw_registered_count' in g:row['registered_item_count_median']=g.raw_registered_count.median()
        rows.append(row)
    cols=['room_type','cluster','name','population_role','split','n','model_adopted','hosts','largest_host_share',
          'regions','largest_region_share','top_regions','ood_n','distance_median']
    prof=pd.DataFrame(rows) if rows else pd.DataFrame(columns=cols)
    csv('군집프로필.csv',prof)
    reg=d.groupby(['country','region','room_type','sector','split','cluster_label','diagnostic_cluster','assignment_status'],dropna=False).size().reset_index(name='n')
    csv('지역_섹터별규모.csv',reg)
    flow=d.groupby(['sector','split','room_type','facility_observed','cluster_eligible','assignment_status'],dropna=False).size().reset_index(name='n')
    csv('표본흐름.csv',flow)
    # Supports are individual questions; not manufactured clusters or cost estimates.
    core=d[FCOLS[:6]]
    missing=(core<1).sum(axis=1).astype('Int64').mask(~d.facility_observed)
    d['not_fully_registered_core_function_n']=missing
    d['support_question_route']='정보 또는 범위 확인'
    d.loc[d.cluster_eligible & missing.eq(0),'support_question_route']='실제 이용 확인 후 장기 상품조건 검토'
    d.loc[d.cluster_eligible & missing.eq(1),'support_question_route']='한 기능의 기재·이용대안·보완비 확인'
    d.loc[d.cluster_eligible & missing.ge(2),'support_question_route']='여러 기능의 이용대안·부담·의향 확인'
    routes=d.groupby(['sector','room_type','cluster_label','support_question_route'],dropna=False).size().reset_index(name='n')
    csv('지원질문_분포.csv',routes)
    strategies=[]
    for cl,name in names.items():
        g=assigned[assigned.diagnostic_cluster.eq(cl)&assigned.sector.isin([1,2,4,5])]
        questions=['등록된 시설의 실제 작동·공용/전용·사용시간·이용료 확인',
                   '연속 제공기간·호스트 의향·월 요금·공과금·유지보수·종료 절차 확인',
                   '주소별 규정·허가·계약 형태 및 실제 체류 수요는 별도 검증']
        if g.room_type.iloc[0]=='Private room':questions.insert(1,'공용 주방·세탁·업무공간 이용권과 다른 거주자와의 규칙 확인')
        for s in FEATURES:
            if g['function__'+s].mean()<=.3:questions.append(KOREAN[s]+' 미등록이 기재누락인지 실제 이용 제한인지 먼저 확인')
        strategies.append({'cluster':cl,'name':name,'fit_population_n':len(g),
            'adopted':bool(g.model_adopted.all()),'questions':questions,'one_cluster_one_route':False})
    js('군집별_지원질문.json',{'annotation_index':1,'strategies':strategies,
                            'warning':'등록 프로필에 따른 확인 질문. 장기 적합성·개선효과·투자비는 검증되지 않음.'})
    return d,prof,reg,flow,routes,names


def support_table(q,keys):
    cols=keys+['n_3','hosts_3','largest_host_share_3','n_6','hosts_6','largest_host_share_6','supported']
    q=q[q.sector.isin([3,6])].dropna(subset=keys)
    if not len(q):return pd.DataFrame(columns=cols)
    tab=q.groupby(keys+['sector']).agg(n=('id','size'),hosts=('host_id','nunique')).reset_index()
    large=q.groupby(keys+['sector','host_id']).size().groupby(level=list(range(len(keys)+1))).max().rename('largest_host_n').reset_index()
    tab=tab.merge(large,on=keys+['sector'],validate='one_to_one')
    tab['largest_host_share']=tab.largest_host_n/tab.n
    result=tab[tab.sector.eq(3)].drop(columns='sector').merge(
        tab[tab.sector.eq(6)].drop(columns='sector'),on=keys,suffixes=('_3','_6'),how='outer')
    for side in [3,6]:
        for col in [f'n_{side}',f'hosts_{side}',f'largest_host_n_{side}']:result[col]=result[col].fillna(0).astype(int)
    result['supported']=(result.n_3.ge(20)&result.n_6.ge(20)&result.hosts_3.ge(5)&result.hosts_6.ge(5)
                         &result.largest_host_share_3.le(.2)&result.largest_host_share_6.le(.2))
    return result


def tag(d,table,keys):
    supported=table.loc[table.supported,keys].drop_duplicates()
    if supported.empty:return np.zeros(len(d),bool)
    z=d[keys].merge(supported.assign(_supported=True),on=keys,how='left',validate='many_to_one')
    assert len(z)==len(d)
    return z._supported.fillna(False).to_numpy(bool)


def comparison_support(d):
    ref=d[d.sector.isin([3,6])]
    alltab=support_table(ref,KEYS)
    csv('동일조건_3vs6_지원.csv',alltab)
    clustered=support_table(ref[ref.model_adopted],KEYS+['cluster_label'])
    csv('동일군집조건_3vs6_지원.csv',clustered)
    inrange=support_table(ref[ref.supported_for_cluster_interpretation],KEYS+['cluster_label'])
    csv('OOD제외_동일군집조건_3vs6_지원.csv',inrange)
    d['reference_exact_supported']=tag(d,alltab,KEYS)
    d['reference_exact_cluster_supported']=tag(d,clustered,KEYS+['cluster_label']) & d.model_adopted
    d['reference_exact_cluster_supported_without_ood']=tag(d,inrange,KEYS+['cluster_label']) & d.supported_for_cluster_interpretation
    # Pattern overlap is presence, not the 20-listing support rule.
    pkeys=KEYS+['pattern_code'];knownref=ref[ref.facility_observed].dropna(subset=pkeys)
    patterns=knownref.groupby(pkeys).sector.nunique().reset_index(name='reference_sector_count')
    patterns['supported']=patterns.reference_sector_count.eq(2)
    d['same_condition_pattern_present_in_both_3_6']=tag(d,patterns,pkeys)
    d['reference_exact_and_pattern_overlap']=d.reference_exact_supported & d.same_condition_pattern_present_in_both_3_6
    cols=['reference_exact_supported','reference_exact_cluster_supported','reference_exact_cluster_supported_without_ood',
          'same_condition_pattern_present_in_both_3_6','reference_exact_and_pattern_overlap']
    summary=d.groupby(['sector','room_type']).agg(n=('id','size'),**{c:(c,'sum') for c in cols}).reset_index()
    csv('비교지원_적용겹침.csv',summary)
    return d,{'exact_supported_strata':int(alltab.supported.sum()),
              'exact_cluster_supported_strata':int(clustered.supported.sum()),
              'exact_cluster_no_ood_supported_strata':int(inrange.supported.sum()),
              'fit_population_n':int(d.sector.isin([1,2,4,5]).sum()),
              'fit_exact_supported_n':int(d.loc[d.sector.isin([1,2,4,5]),'reference_exact_supported'].sum()),
              'fit_exact_and_pattern_overlap_n':int(d.loc[d.sector.isin([1,2,4,5]),'reference_exact_and_pattern_overlap'].sum())}


def build_report(summary,candidates,prof,sensitivity,routes):
    def tbl(df):return df.to_html(index=False,border=0,escape=True,float_format=lambda v:f'{v:.3f}')
    evaluation=pd.DataFrame(candidates)
    keep=['room_type','k','silhouette_validation','train_min_share','validation_min_share',
          'seed_ari_min','host_ari_min','weight_ari_min','fully_pass','failure_reasons']
    explanation='시설등록 조합이 실제로 구분되는지 확인하고, 각 유형에서 무엇을 먼저 물어봐야 하는지 정리했습니다. 최소 숙박일은 유지하는 전략입니다.'
    selections=[]
    for room,s in summary['selected'].items():
        t=summary['test']['results'][room]
        selections.append({'숙소종류':room,'선택k':s['k'],'최종채택':t['adopted'],'사유':t['reason']})
    body=f'''<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>생활·업무 시설등록 군집 확장검증</title>
<style>body{{font-family:system-ui,sans-serif;background:#f7f8fb;color:#182c38;max-width:1200px;margin:auto;padding:32px;line-height:1.65}}section{{background:white;padding:24px;margin:20px 0;border-radius:12px;overflow-x:auto}}table{{border-collapse:collapse;font-size:13px;min-width:650px}}td,th{{padding:8px;border-bottom:1px solid #dde4e8;text-align:left}}th{{background:#edf3f7}}.note{{padding:16px;background:#fff5dc;border-radius:8px}}h1{{font-size:30px}}h2{{font-size:21px}}a{{color:#175c91}}</style>
<h1>생활·업무 시설등록 군집 확장검증</h1><p>2026-10-03 · 전체 677,678곳 · 학습 모집단1·2·4·5번639,774곳 · 장기설정 참고3·6번37,904곳</p>
<p>{explanation}</p><p class="note">미등록은 실제 시설 부재와 다릅니다. 군집은 장기예약 성공·법적 적격·수익증가를 판정하지 않습니다. 3·6번의 차이도 실제 장기 예약실적의 차이가 아닙니다.</p>
<section><h2>선택 결과</h2>{tbl(pd.DataFrame(selections))}<p>적격 {summary['eligible_n']:,}곳, 공식군 배정 {summary['adopted_assigned_n']:,}곳. 미관측·범위 밖·미채택도 전체 숙소표에 보존했습니다.</p></section>
<section><h2>왜 이런 방식으로 나눴나</h2><p>취사공간·가열기기·냉장고·조리도구/식기·세탁기·와이파이·업무공간의 7기능만 사용했습니다. 숙소 전체와 개인실은 별도로 학습했습니다. 취사·세탁·연결/업무 세 묶음에 같은 최대 거리 비중을 주고, 다른 두 가중치에서도 결과가 유지되는지 확인했습니다.</p><p>리뷰·지역·최소박·가용일수는 군집 특징과 선택에서 제외했습니다. 1/2/4/5의 기존 호스트 분할 train/validation으로 두 모형을 먼저 고정하고, test와3/6은 그 뒤에 적용했습니다.</p><p>같은 시설 벡터를 실제 숙소 수로 가중하여 계산량만 줄였습니다. 192개 가능한 패턴을 숙소192곳으로 취급하지 않았습니다. 실루엣은 같은 validation 표본 최대1,500곳에서 계산했습니다. 값이 높아도 전략 효과가 입증되는 것은 아닙니다.</p></section>
<section><h2>후보 비교와 안정성</h2>{tbl(evaluation[keep])}<p>실루엣0.2·최소군집비중5%·시드/호스트/가중치 기준을 모두 통과한 후보 중 최고 실루엣과0.03 이내인 최소k를 선택했습니다. 미실행 민감도는 기본 기준 탈락을 뜻하며 통과로 취급하지 않았습니다.</p></section>
<section><h2>실제 등록 프로필</h2>{tbl(prof)}<p>각 기능 수치는 등록값의 평균입니다. 조리도구·식기의0.5는 한 항목만 등록된 경우입니다. 학습·검증·시험과 장기 참고집단을 분리해서 표시했습니다.</p></section>
<section><h2>호스트·섹터·지역 비중의 영향</h2>{tbl(pd.DataFrame(sensitivity))}<p>실제 숙소 건수를 유지한 주모형과, 호스트/섹터/지역 총가중치를 각각 같게 한 결과의 차이입니다. 민감도가 크다고 유리한 결과로 재선택하지 않았습니다.</p></section>
<section><h2>어떤 도움을 검토할 수 있나</h2><p>생활기능이 폭넓게 등록된 숙소는 실제 이용 가능성을 확인한 뒤 월요금·공과금·제공기간·계약 운영을 검토합니다. 일부 기능이 미등록된 숙소는 기재누락과 이용대안을 먼저 확인합니다. 여러 기능이 미등록된 숙소는 투자비로 환산하지 않고 이용대안·부담·호스트 의향부터 확인합니다.</p>{tbl(routes)}<p>위 구분은 군집을 정답에 맞춘 결과가 아니라 개별 확인 질문입니다. 한 군집이 하나의 지원 경로와 같지는 않습니다. <a href="군집별_지원질문.json">군집별 질문</a></p></section>
<section><h2>장기설정 숙소와 어디까지 비교되나</h2><pre>{html.escape(json.dumps(summary['support'],ensure_ascii=False,indent=2))}</pre><p>국가·지역·숙소종류·세부유형·수용인원이 같은 층에서3·6 각각20숙소·5호스트 이상, 최대호스트20% 이하인 경우만 비교 지원으로 표시했습니다. 동일 군집 지원과 동일 기능패턴 겹침은 별도입니다. 이것은 장기 수요 또는 전환 효과가 아닙니다.</p></section>
<section><h2>계속할 일과 현재 한계</h2><p>장기 참고집단의 리뷰 관련성은 회귀 담당 결과와 연결해야 합니다. 상품 제안은 실제 시설 사용권·연속 제공일·호스트 의향·주소별 법규·비용·수요를 확인한 뒤 정해야 합니다. 새 호스트나 미래 시점의 외부 검증은 수행하지 않았습니다.</p><p><a href="실행전계획.json">실행 전 계획</a> · <a href="선택고정.json">선택 고정</a> · <a href="결과요약.json">결과 수치</a> · <a href="검산.json">검산</a> · <a href="숙소별_군집.csv.gz">전체 숙소별 결과</a></p></section></html>'''
    (OUT/'결과보고서.html').write_text(body)


def main():
    contract=json.loads(CONTRACT_PATH.read_text());plan=json.loads(PLAN_PATH.read_text())
    assert contract['cluster_plan']['folder']==str(OUT)
    loaded=read_common(contract)
    if loaded is None:return
    d,provenance=loaded
    original_ids=d.id.copy()
    with threadpool_limits(limits=2):
        frozen,models,candidates=choose_and_freeze(d,plan,provenance)
        frozen_sha=sha(OUT/'선택고정.json')
        test=evaluate_test(d,models,plan,frozen_sha)
        d=apply_models(d,models,test,frozen_sha)
        status('post_freeze_profiles_and_support',preparation_done=True)
        sensitivity=composition(d,models,plan)
        d,prof,reg,flow,routes,names=profiles(d)
        d,support=comparison_support(d)
    # Keep the complete common data provenance, but the handoff need not repeat every control variable.
    handoff_cols=['id','host_id','sector','split']+KEYS+['observation_state','facility_observed','room_type_supported',
        'cluster_eligible','information_insufficient','exclusion_reason','population_role','pattern_code',
        'cluster_label','diagnostic_cluster','model_adopted','assignment_status','distance_to_centroid',
        'train_distance99_threshold','train_pattern_n','ood_evaluated','ood_outside_feature_range',
        'ood_novel_training_pattern','ood_distance','ood_any','supported_for_cluster_interpretation',
        'reference_exact_supported','reference_exact_cluster_supported','reference_exact_cluster_supported_without_ood',
        'same_condition_pattern_present_in_both_3_6','reference_exact_and_pattern_overlap',
        'not_fully_registered_core_function_n','support_question_route']+FCOLS
    handoff=d[handoff_cols].copy();csv('숙소별_군집.csv.gz',handoff)
    checks={
        'all_ids_preserved':handoff.id.equals(original_ids),'all_rows':len(handoff)==677678,
        'unique_ids':handoff.id.is_unique,'host_split_consistent':d.groupby('host_id').split.nunique().eq(1).all(),
        'scope_rows':int(d.sector.isin([1,2,4,5]).sum())==639774,
        'reference_rows':int(d.sector.isin([3,6]).sum())==37904,
        'no_unknown_assignment':d.loc[~d.cluster_eligible,'diagnostic_cluster'].eq('').all(),
        'official_only_adopted':d.loc[~d.model_adopted,'cluster_label'].eq('').all(),
        'unknown_ood_not_false':d.loc[~d.ood_evaluated,'ood_any'].isna().all(),
        'unknown_function_count_na':d.loc[~d.facility_observed,'not_fully_registered_core_function_n'].isna().all(),
        'profile_denominator':int(prof.n.sum())==int(d.diagnostic_cluster.ne('').sum()),
        'regional_denominator':int(reg.n.sum())==len(d),'flow_denominator':int(flow.n.sum())==len(d),
        'route_denominator':int(routes.n.sum())==len(d),'selection_unchanged':sha(OUT/'선택고정.json')==frozen_sha,
        'common_unchanged':sha(contract['common_output'])==provenance['common_sha256'],
        'plan_unchanged':sha(PLAN_PATH)==provenance['plan_sha256'],
        'model_hashes_unchanged':all(sha(OUT/s['model_file'])==s['model_sha256'] for s in frozen['selected'].values() if s['k'] is not None),
        'aggregation_objective':all(b['aggregation_check']['all_pass'] for b in models.values())}
    assert all(checks.values()),checks
    summary={'done':False,'analysis_done':True,'date_kst':'2026-10-03','population_n':len(d),'fit_population_n':639774,'reference_n':37904,
             'eligible_n':int(d.cluster_eligible.sum()),'facility_unknown_n':int((~d.facility_observed).sum()),
             'room_outside_scope_n':int((~d.room_type_supported).sum()),'adopted_assigned_n':int(d.model_adopted.sum()),
             'ood_evaluated_n':int(d.ood_evaluated.sum()),'ood_any_n':int(d.ood_any.fillna(False).sum()),
             'selected':frozen['selected'],'test':test,'support':support,'names':names,
             'sensitivity':sensitivity,'selection_sha256':frozen_sha,'provenance':provenance,
             'not_proven':['실제 시설·이용권','장기수요·장기예약','법적 적격','개선효과·매출증가'],
             'pending_followup':['3vs6 조정리뷰관련성 결과 연결','지역별 상품조건·개별확인 통합'],
             'model_runtime_note':'요청 Astra/xhigh. 현재 런타임 모델은 이 코드에서 검증하지 않음.'}
    js('결과요약.json',summary)
    js('검산.json',{'all_pass':True,'checks':checks,'selection_sha256':frozen_sha,'provenance':provenance,
                   'verification_kind':'계산·표본·고정절차 검산. 물리·법적·인과 검증 아님.'})
    build_report(summary,candidates,prof,sensitivity,routes)
    # A separate final verifier is required before publishing the completion gate.
    record_event('analysis_outputs_written',rows=len(d),selection_sha256=frozen_sha)
    status('awaiting_final_verification',analysis_done=True,done=False,
           pending=['저장 결과 재읽기 검산','HTML 정적 확인','최종 READY 인계'])
    print(json.dumps(clean({k:summary[k] for k in ['population_n','eligible_n','adopted_assigned_n','ood_any_n','support']}),ensure_ascii=False),flush=True)


if __name__=='__main__':
    try:main()
    except Exception as e:
        status('error',done=False,error_type=type(e).__name__,error=str(e))
        (OUT/'실행오류.txt').write_text(traceback.format_exc())
        raise
"""
SOURCE_META['cluster'] = {'path': '/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/analyze.py', 'sha256': 'e590706f6b6f4681871f93b22387eab86301a419bfcb24ab4095ccc25e2f3aaf', 'language': 'python'}


### cluster_verify — 군집 결과 별도 검산
원본: `/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/verify_final.py`
SHA256: `fa03e1a3147395752b4d9c3f00bcf1b1c90fbe8732fa2411164a65e96164b85b`


In [15]:
SOURCE_CODE['cluster_verify'] = """\"\"\"Read saved outputs and recompute assignment/calibration/support checks, without refitting.\"\"\"
from pathlib import Path
import os
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
import json,hashlib,re,datetime
import numpy as np
import pandas as pd
import joblib
from threadpoolctl import threadpool_limits

OUT=Path(__file__).resolve().parent
FEATURES=['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']
FCOLS=['function__'+s for s in FEATURES]
KEYS=['country','region','room_type','property_type','accommodates']

def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for x in iter(lambda:f.read(2**20),b''):h.update(x)
    return h.hexdigest()

def js(name,x):
    p=OUT/name;q=p.with_name(p.name+'.tmp')
    q.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False)+'\\n');q.replace(p)

def main():
    s=json.loads((OUT/'결과요약.json').read_text())
    frozen=json.loads((OUT/'선택고정.json').read_text())
    test=json.loads((OUT/'시험평가.json').read_text())
    common_path=s['provenance']['common_path']
    boolcols=['model_adopted','facility_observed','cluster_eligible','ood_evaluated','ood_any',
              'ood_outside_feature_range','ood_novel_training_pattern','ood_distance','supported_for_cluster_interpretation',
              'reference_exact_supported','reference_exact_cluster_supported','reference_exact_cluster_supported_without_ood',
              'same_condition_pattern_present_in_both_3_6','reference_exact_and_pattern_overlap']
    out=pd.read_csv(OUT/'숙소별_군집.csv.gz',dtype={'id':'string','host_id':'string',**{c:'boolean' for c in boolcols}})
    base=pd.read_csv(common_path,usecols=['id','host_id','sector','split','observation_state']+KEYS+FCOLS,
                     dtype={'id':'string','host_id':'string'})
    checks={}
    def check(name,value):
        checks[name]=bool(value)
        if not checks[name]:raise AssertionError(name)
    check('rows_and_order',len(out)==677678 and out.id.equals(base.id))
    check('metadata_matches',out[['host_id','sector','split']+KEYS].equals(base[['host_id','sector','split']+KEYS]))
    check('functions_and_na_preserved',np.allclose(out[FCOLS],base[FCOLS],equal_nan=True))
    check('ids_unique_and_host_disjoint',out.id.is_unique and out.groupby('host_id').split.nunique().eq(1).all())
    check('common_hash',sha(common_path)==s['provenance']['common_sha256'])
    check('plan_hash',sha(OUT/'실행전계획.json')==s['provenance']['plan_sha256'])
    check('script_hash',sha(OUT/'analyze.py')==s['provenance']['script_sha256'])
    check('freeze_hash',sha(OUT/'선택고정.json')==s['selection_sha256']==test['selection_sha256'])
    check('frozen_before_test_and_reference',not frozen['test_evaluated_at_freeze'] and not frozen['reference_evaluated_at_freeze'])
    events=[json.loads(x) for x in (OUT/'실행순서.jsonl').read_text().splitlines()]
    event_names=[x['event'] for x in events]
    check('event_order',event_names.index('all_models_frozen')<event_names.index('test_evaluated')<event_names.index('reference_projection_started'))
    chosen=frozen['selected'];checked_rows=0;calibration_rows=0
    with threadpool_limits(limits=2):
        for room,select in chosen.items():
            q=out[out.room_type.eq(room)&out.cluster_eligible].copy()
            if select['k'] is None:
                check(room+'_unselected_no_labels',q.cluster_label.isna().all() and q.diagnostic_cluster.isna().all())
                check(room+'_unselected_no_distance',q.distance_to_centroid.isna().all() and q.ood_any.isna().all())
                continue
            check(room+'_model_hash',sha(OUT/select['model_file'])==select['model_sha256'])
            b=joblib.load(OUT/select['model_file']);x=q[FCOLS].to_numpy(float)
            labels=b['model'].predict(x*b['weight'])
            expected=np.array([b['labelmap'][int(i)] for i in labels])
            check(room+'_full_direct_assignment',np.array_equal(expected,q.diagnostic_cluster.to_numpy()))
            check(room+'_official_labels',np.array_equal(expected,q.cluster_label.to_numpy()) if test['results'][room]['adopted'] else q.cluster_label.isna().all())
            distances=np.linalg.norm(x*b['weight']-b['model'].cluster_centers_[labels],axis=1)
            check(room+'_full_direct_distance',np.allclose(distances,q.distance_to_centroid,rtol=1e-12,atol=1e-12))
            train=q[q.sector.isin([1,2,4,5])&q.split.eq('train')]
            xt=train[FCOLS].to_numpy(float);lt=b['model'].predict(xt*b['weight'])
            dt=np.linalg.norm(xt*b['weight']-b['model'].cluster_centers_[lt],axis=1)
            check(room+'_expanded_objective',np.isclose(np.dot(dt,dt),b['model'].inertia_,rtol=1e-8,atol=1e-6))
            for cluster in range(b['model'].n_clusters):
                quantile=float(np.quantile(dt[lt==cluster],.99,method='inverted_cdf'))
                check(room+f'_threshold_{cluster}',np.isclose(quantile,b['distance_threshold99'][cluster],atol=1e-12))
            patterns_train=train.groupby(FCOLS).size().rename('_train_n').reset_index()
            cc=q[FCOLS].merge(patterns_train,on=FCOLS,how='left',validate='many_to_one')._train_n.fillna(0).to_numpy(int)
            check(room+'_train_pattern_counts',np.array_equal(cc,q.train_pattern_n.to_numpy(int)))
            check(room+'_novel_pattern_flag',np.array_equal(cc==0,q.ood_novel_training_pattern.to_numpy(bool)))
            check(room+'_feature_ood_flag',np.array_equal(((x<xt.min(axis=0))|(x>xt.max(axis=0))).any(axis=1),q.ood_outside_feature_range.to_numpy(bool)))
            check(room+'_distance_ood_flag',np.array_equal(distances>q.train_distance99_threshold.to_numpy()+1e-12,q.ood_distance.to_numpy(bool)))
            check(room+'_ood_union',q.ood_any.equals(q.ood_outside_feature_range|q.ood_novel_training_pattern|q.ood_distance))
            checked_rows+=len(q);calibration_rows+=len(train)
    check('missing_not_assigned',out.loc[~out.cluster_eligible,'diagnostic_cluster'].isna().all())
    check('ineligible_ood_missing',out.loc[~out.ood_evaluated,'ood_any'].isna().all())
    check('registered_questions_na',out.loc[~out.facility_observed,'not_fully_registered_core_function_n'].isna().all())
    check('all_sector_counts',out.sector.value_counts().to_dict()=={1:418643,2:43259,3:9555,4:144542,5:33330,6:28349})
    prof=pd.read_csv(OUT/'군집프로필.csv')
    check('profile_n',prof.n.sum()==out.diagnostic_cluster.notna().sum())
    for file in ['지역_섹터별규모.csv','표본흐름.csv','지원질문_분포.csv']:
        check(file+'_n',pd.read_csv(OUT/file).n.sum()==len(out))
    # Independently check every supported comparison layer against the listing output.
    support_files=[('동일조건_3vs6_지원.csv',KEYS,np.ones(len(out),bool)),
        ('동일군집조건_3vs6_지원.csv',KEYS+['cluster_label'],out.model_adopted.to_numpy(bool)),
        ('OOD제외_동일군집조건_3vs6_지원.csv',KEYS+['cluster_label'],out.supported_for_cluster_interpretation.to_numpy(bool))]
    supported_checked=0
    for file,keys,mask in support_files:
        table=pd.read_csv(OUT/file)
        reference=out[mask&out.sector.isin([3,6]).to_numpy()].dropna(subset=keys)
        calc=reference.groupby(keys+['sector']).agg(n=('id','size'),hosts=('host_id','nunique'))
        largest=reference.groupby(keys+['sector','host_id']).size().groupby(level=list(range(len(keys)+1))).max()
        expected_support=set()
        for row in table.itertuples(index=False):
            valid=True
            for sector in [3,6]:
                key=tuple(getattr(row,c) for c in keys)+(sector,)
                if key not in calc.index:valid=False;continue
                n=int(calc.loc[key,'n']);hosts=int(calc.loc[key,'hosts']);maxhost=int(largest.loc[key])
                check(file+f'_counts_{supported_checked}_{sector}',n==getattr(row,f'n_{sector}') and hosts==getattr(row,f'hosts_{sector}'))
                valid=valid and n>=20 and hosts>=5 and maxhost/n<=.2
            check(file+f'_support_{supported_checked}',bool(row.supported)==valid)
            if valid:expected_support.add(tuple(getattr(row,c) for c in keys))
            supported_checked+=1
    # Final report static validation only; no claim of visual browser rendering.
    page=(OUT/'결과보고서.html').read_text()
    refs=re.findall(r'href="([^"]+)"',page)
    check('report_local_links_exist',all((OUT/r).exists() for r in refs if not r.startswith(('http','#'))))
    check('report_html_structure','<html' in page and '</html>' in page and 'lang="ko"' in page)
    check('report_no_placeholder',all(t not in page for t in ['TODO','undefined','[object Object]']))
    result={'all_pass':True,'check_count':len(checks),'checks':checks,'assignment_rows_recomputed':checked_rows,
            'train_calibration_rows_recomputed':calibration_rows,'support_rows_checked':supported_checked,
            'no_new_fit':True,'no_visual_render_claim':True,'selection_sha256':s['selection_sha256']}
    js('저장후_독립계산검산.json',result)
    qa=json.loads((OUT/'검산.json').read_text());qa['saved_output_recheck']=dict(all_pass=True,
        check_count=len(checks),assignment_rows_recomputed=checked_rows,report_visual_render_verified=False)
    js('검산.json',qa)
    s['done']=True;s['final_verification_passed']=True;s['report_visual_render_verified']=False
    js('결과요약.json',s)
    files=['숙소별_군집.csv.gz','선택고정.json','시험평가.json','군집평가.csv','군집프로필.csv',
           '지역_섹터별규모.csv','비교지원_적용겹침.csv','결과요약.json','검산.json','저장후_독립계산검산.json','결과보고서.html']
    ready={'ready':True,'done':True,'status':'READY','date_kst':'2026-10-03','rows':len(out),
           'data_path':str(OUT/'숙소별_군집.csv.gz'),'data_sha256':sha(OUT/'숙소별_군집.csv.gz'),
           'common_data_sha256':s['provenance']['common_sha256'],'selection_sha256':s['selection_sha256'],
           'selected_k':{room:v['k'] for room,v in chosen.items()},
           'adopted_room_types':[room for room,v in test['results'].items() if v['adopted']],
           'output_sha256':{f:sha(OUT/f) for f in files},'all_checks_passed':True,
           'cluster_column':'cluster_label','diagnostic_column':'diagnostic_cluster',
           'missing_label_semantics':'빈 cluster_label은 임의 군집0이 아님. 개인실 미채택/시설미관측/범위밖을 assignment_status로 구분.',
           'regression_handoff':'공식 군집 분석은 model_adopted=true에서 가능. OOD 포함/제외 및 exact 비교지원은 별도 표시. Private room 공식군 없음.',
           'not_a_success_label':True,'no_new_fit_in_verification':True,
           'pending_followup':s['pending_followup']}
    js('인계.json',ready)
    js('READY.json',ready)
    js('진행상태.json',{'status':'complete','done':True,'date_kst':'2026-10-03',
                        'analysis_done':True,'final_verification_passed':True,
                        'pending_other_owners':s['pending_followup'],
                        'report_visual_render_verified':False})
    print(json.dumps({'all_pass':True,'checks':len(checks),'assignment_rows':checked_rows,
                      'train_calibration_rows':calibration_rows,'support_rows':supported_checked},ensure_ascii=False))

if __name__=='__main__':main()
"""
SOURCE_META['cluster_verify'] = {'path': '/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/verify_final.py', 'sha256': 'fa03e1a3147395752b4d9c3f00bcf1b1c90fbe8732fa2411164a65e96164b85b', 'language': 'python'}


### cluster_report — 군집 보고서 생성
원본: `/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/build_report.py`
SHA256: `53775316f2bca492f5a9c3ff12d6a026cc8254b969560d909f85b51e9a05f697`


In [16]:
SOURCE_CODE['cluster_report'] = """\"\"\"Korean explanation of frozen results; no refitting or selection changes.\"\"\"
from pathlib import Path
import json,html
import pandas as pd

OUT=Path(__file__).resolve().parent
FEATURES=['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']
LABELS=['취사공간','가열기기','냉장고','조리도구·식기','세탁기','와이파이','업무공간']
FCOLS=['function__'+x for x in FEATURES]

def table(d):
    return '<div class="scroll">'+d.to_html(index=False,border=0,escape=True,na_rep='—')+'</div>'

def pct(x):return f'{100*x:.1f}%'
def num(x):return f'{int(x):,}'

def main():
    s=json.loads((OUT/'결과요약.json').read_text())
    d=pd.read_csv(OUT/'숙소별_군집.csv.gz',dtype={'id':'string','host_id':'string'})
    common=pd.read_csv(s['provenance']['common_path'],usecols=['id','raw_registered_count'],dtype={'id':'string'})
    d=d.merge(common,on='id',validate='one_to_one')
    fit=d[d.sector.isin([1,2,4,5])]
    official=fit[fit.model_adopted]
    val=pd.read_csv(OUT/'군집평가.csv')
    sens=pd.read_csv(OUT/'구성민감도.csv')
    profiles=[];detail=[];strategy=[]
    for cl,g in official.groupby('cluster_label'):
        p={'유형':cl,'숙소 수':num(len(g))}
        p.update({label:pct(g[c].mean()) for c,label in zip(FCOLS,LABELS)})
        p['시설목록 길이 중앙값']=num(g.raw_registered_count.median())
        profiles.append(p)
        hc=g.host_id.value_counts();rc=g.region.value_counts()
        detail.append({'유형':cl,'지역 수':len(rc),'최대지역':rc.index[0],'최대지역 비중':pct(rc.iloc[0]/len(g)),
                       '호스트 수':num(len(hc)),'최대호스트 비중':pct(hc.iloc[0]/len(g)),
                       '일반적 학습범위 밖':num(g.ood_any.fillna(False).sum()),
                       '동일조건·군집 비교지원':num(g.reference_exact_cluster_supported.sum())})
        for route,q in g.groupby('support_question_route'):
            strategy.append({'유형':cl,'개별 확인 경로':route,'숙소 수':num(len(q)),'유형 내 비중':pct(len(q)/len(g))})
    ptab=table(pd.DataFrame(profiles));dtab=table(pd.DataFrame(detail));stab=table(pd.DataFrame(strategy))
    for row in profiles:
        assert row['유형'].startswith('E_'), '보고서 설명은 이번 E4 채택 결과에 맞춘 것이므로 재검토 필요'
    assert s['selected']['Entire home/apt']['k']==4 and s['selected']['Private room']['k'] is None
    candidate=[]
    for r in val.itertuples():
        candidate.append({'숙소종류':'숙소 전체' if r.room_type=='Entire home/apt' else '개인실','군집 수':r.k,
                          '검증 실루엣':f'{r.silhouette_validation:.3f}',
                          '시드 일치 최소':f'{r.seed_ari_min:.3f}',
                          '호스트 재표집 최소':'미실행' if pd.isna(r.host_ari_min) else f'{r.host_ari_min:.3f}',
                          '가중치 변경 최소':'미실행' if pd.isna(r.weight_ari_min) else f'{r.weight_ari_min:.3f}',
                          '선택 기준 통과':'통과' if r.fully_pass else '미통과'})
    sensitivity=[]
    for r in sens[sens.validation_sector.eq('all')].itertuples():
        sensitivity.append({'변경한 비중':{'host_equal':'호스트마다 동일','sector_equal':'1·2·4·5번마다 동일','region_equal':'지역마다 동일'}[r.sensitivity],
                             '군집 일치도(ARI)':f'{r.ari:.3f}','대응 후 다른 군 배정':pct(r.mapped_disagreement_fraction),
                             '최대 특징평균 차이':f'{r.max_matched_raw_centroid_gap*100:.1f}%p'})
    overlap=[]
    for sec,g in fit.groupby('sector'):
        overlap.append({'섹터':sec,'전체 숙소':num(len(g)),
                        '동일조건 비교지원':num(g.reference_exact_supported.sum()),
                        '동일조건+동일군집 지원':num(g.reference_exact_cluster_supported.sum()),
                        '동일조건+기능패턴 겹침':num(g.reference_exact_and_pattern_overlap.sum())})
    conditions=int(fit.reference_exact_supported.sum());cluster_support=int(fit.reference_exact_cluster_supported.sum())
    pattern_support=int(fit.reference_exact_and_pattern_overlap.sum())
    adopted=len(official);private=int((fit.room_type.eq('Private room')&fit.cluster_eligible).sum())
    all_core=int((official.not_fully_registered_core_function_n==0).sum())
    one_core=int((official.not_fully_registered_core_function_n==1).sum())
    many_core=int((official.not_fully_registered_core_function_n>=2).sum())
    counts={'eligible_fit_n':int(fit.cluster_eligible.sum()),'adopted_fit_n':adopted,
            'private_not_adopted_fit_n':private,'whole_adopted_all6_n':int(d.model_adopted.sum()),
            'official_fit_complete_core_n':all_core,'official_fit_one_core_not_full_n':one_core,
            'official_fit_multiple_core_not_full_n':many_core,
            'fit_same_condition_n':conditions,'fit_same_condition_cluster_n':cluster_support,
            'fit_same_condition_pattern_n':pattern_support,
            'cluster_counts_fit':official.cluster_label.value_counts().sort_index().to_dict()}
    (OUT/'설명용_핵심수치.json').write_text(json.dumps(counts,ensure_ascii=False,indent=2)+'\\n')
    report=f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>장기 상품 검토를 위한 시설등록 유형 | 2026.10.03</title>
<style>
:root{{--ink:#172e39;--muted:#526976;--line:#dce5e9;--blue:#165f74;--gold:#885d10}}*{{box-sizing:border-box}}
body{{margin:0;background:#f2f5f6;color:var(--ink);font:16px/1.7 system-ui,-apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo",sans-serif}}
main{{max-width:1160px;margin:auto;padding:42px 24px 70px}}header{{margin-bottom:28px}}h1{{font-size:clamp(27px,4vw,40px);line-height:1.3;letter-spacing:-1px;margin:12px 0 20px}}
h2{{font-size:23px;line-height:1.4;margin:0 0 16px}}h3{{font-size:19px;margin:24px 0 10px}}p{{margin:12px 0}}.meta,.muted{{color:var(--muted)}}.meta{{font-size:14px}}
.lead{{font-size:20px;max-width:950px}}section{{background:#fff;border:1px solid var(--line);padding:28px;margin:20px 0;border-radius:16px}}
.cards{{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin:24px 0}}.card{{padding:20px;background:#e7f0f3;border-radius:12px}}.card b{{font-size:30px;display:block;line-height:1.35}}.card small{{color:var(--muted)}}
.note{{background:#fff3d7;border-left:4px solid #bb8b29;padding:15px 18px;border-radius:4px}}.good{{background:#e9f4ec;border-left-color:#52866a}}
.scroll{{overflow:auto;margin:16px 0}}table{{border-collapse:collapse;width:100%;font-size:14px}}td,th{{padding:12px 10px;border-bottom:1px solid var(--line);text-align:left;vertical-align:top}}th{{background:#edf3f5;white-space:nowrap}}td{{min-width:82px}}td:first-child{{font-weight:600}}a{{color:var(--blue)}}li{{margin:7px 0}}code{{background:#edf2f4;padding:2px 4px;border-radius:3px;font-size:13px}}
.flow{{display:flex;flex-wrap:wrap;gap:10px;padding:16px 0}}.flow span{{padding:10px 14px;border:1px solid var(--line);border-radius:9px;background:#f7fafb}}
footer{{color:var(--muted);font-size:14px;margin-top:26px}}@media(max-width:700px){{main{{padding:25px 14px}}section{{padding:21px 17px}}.cards{{grid-template-columns:1fr}}.lead{{font-size:18px}}}}
</style></head><body><main>
<header><div class="meta">26-2 1차 인콘 · 2026.10.03 · 전략3 시설등록 유형 확장검증</div>
<h1>장기 상품 검토를 위한<br>시설등록 유형</h1>
<p class="lead"><strong>숙소 전체는 4개 유형을 채택했습니다.</strong> 개인실은 가중치를 바꾸면 구분이 크게 달라져 채택을 보류했습니다. 이 결과는 호스트에게 물어볼 사항을 구분하는 근거이며, 장기 판매가 잘될 숙소의 판정은 아닙니다.</p>
<div class="cards"><div class="card"><small>1·2·4·5번 검토 모집단</small><b>639,774곳</b><span>기존 최소 숙박일 유지</span></div>
<div class="card"><small>검토 모집단 중 공식 군집 배정</small><b>{num(adopted)}곳</b><span>숙소 전체 · 4개 유형</span></div>
<div class="card"><small>군집 채택을 보류한 적격 개인실</small><b>{num(private)}곳</b><span>지원 불가능하다는 뜻은 아님</span></div></div>
<p class="note">시설값0은 ‘등록되지 않음’입니다. 시설이 실제로 없거나 공사·투자가 필요하다는 뜻이 아닙니다. 월 단위 생활·학업·업무 체류 상품 추가를 검토하되, 기존 최소 숙박일을30박으로 바꾸는 분석은 아닙니다.</p></header>

<section><h2>1. 누구를 어떻게 나눴나</h2>
<p>예약 가능한 날짜가 있는 전체677,678곳 중, 최소 숙박일1~29박인1·2·4·5번639,774곳을 군집 학습 모집단으로 삼았습니다. 실제 학습에는 그중 기존 호스트 분할의 train만 사용했습니다. 최소30박인3·6번37,904곳은 군집 확정 후 비교용으로 적용했습니다.</p>
<div class="scroll"><table><thead><tr><th>최근12개월 리뷰</th><th>최소1~3박</th><th>최소4~29박</th><th>최소30박 이상</th></tr></thead><tbody>
<tr><td>1개 이상</td><td>1번 · 418,643곳</td><td>2번 · 43,259곳</td><td>3번 · 9,555곳</td></tr>
<tr><td>0개</td><td>4번 · 144,542곳</td><td>5번 · 33,330곳</td><td>6번 · 28,349곳</td></tr></tbody></table></div>
<p>숙소 전체와 개인실을 따로 분석했습니다. 투입한 시설은 <strong>취사공간·가열기기·냉장고·조리도구/식기·세탁기·와이파이·업무공간</strong>의7기능입니다. 가열기기는 stove/oven/microwave 중 하나 이상 등록된 경우입니다. 조리도구·식기는 둘 다 등록1, 하나만 등록0.5, 둘 다 미등록0입니다.</p>
<div class="flow"><span>기존 호스트 분할 유지</span><span>train 학습 · validation 선택</span><span>두 숙소종류의 선택·모형 해시 고정</span><span>test 한 번 검증</span><span>3·6번 동일 모형 적용</span></div>
<p>리뷰·평점·지역·최소박·가용일·법규는 군집 특징이나 선택 기준에 넣지 않았습니다. 시설 정보 미관측3,039곳과 지원하지 않는 숙소종류5,146곳은 제외 이유를 보존했습니다. 두 사유가 겹치는34곳이 있어 전체 고유 제외는8,151곳입니다. 적격 전체669,527곳 중 개인실118,148곳은 모형 미채택이며, 숙소 전체551,379곳에 공식군을 부여했습니다. 이 숫자는 비교용3·6번도 포함합니다.</p></section>

<section><h2>2. 발견한 네 유형과 먼저 확인할 것</h2>
<p>아래 수치는1·2·4·5번에서 공식군이 배정된522,248곳 기준입니다. 비율은 기능 등록값 평균이며, 실제 시설 점검 결과가 아닙니다.</p>
{ptab}
<div class="scroll"><table><thead><tr><th>유형</th><th>읽을 수 있는 특징</th><th>먼저 제공할 도움</th></tr></thead><tbody>
<tr><td>E_K4_1<br>113,073곳</td><td>세탁기 미등록. 나머지 생활기능 등록 정도는 숙소마다 다름.</td><td>세탁기 기재 누락인지, 공용 세탁이나 외부 이용 대안이 있는지 확인. 세탁기 설치를 바로 권하지 않음.</td></tr>
<tr><td>E_K4_2<br>61,309곳</td><td>주방과 세탁기는 대부분 등록됐지만, 가열기기·냉장고·도구 세부정보가 적음. 전체 시설목록도 중앙값13개로 짧음.</td><td>시설목록과 실제 취사 가능 범위를 먼저 대조. 장기생활 시설이 부족하다는 판단보다 등록정보 확인이 우선.</td></tr>
<tr><td>E_K4_3<br>178,794곳</td><td>생활기능 등록이 폭넓고 업무공간은 미등록.</td><td>생활 기능의 실제 이용과 제공기간·월요금·공과금 검토. 업무 목적 상품을 원하면 책상·작업환경부터 확인.</td></tr>
<tr><td>E_K4_4<br>169,072곳</td><td>생활기능 등록이 폭넓고 업무공간도 등록.</td><td>업무공간·통신의 실제 사용 조건을 확인한 뒤 생활·업무 체류 상품의 요금·운영 조건 검토. 수요 우위는 아직 미검증.</td></tr>
</tbody></table></div>
<p class="note">E_K4_3·4의 평균이 높아도 모든 숙소에 모든 기능이 등록된 것은 아닙니다. 두 유형을 통째로 ‘장기 적합’으로 선정해서는 안 됩니다. 또한 E_K4_4가 E_K4_3보다 수익성이 높다는 결론은 나오지 않았습니다.</p>
<p>이 군집은 주로 세탁기·취사 세부정보·업무공간 등록으로 구분됩니다. 이를 설명하는 간단한 체크리스트도 충분히 유용할 수 있습니다. 군집의 가치는 반복되는 조합을 정리한 데 있으며, 복잡한 기법을 썼다는 사실 자체가 추가 성과 근거는 아닙니다.</p></section>

<section><h2>3. 요청한 지원 방향과 어떻게 연결하나</h2>
<p>처음부터 ‘장기 적합·일부 보완·대규모 보완’의 세 군집을 만들지 않았습니다. 발견한 유형 위에 <strong>개별 숙소의 미등록 기능을 확인하는 경로</strong>를 붙였습니다. 업무공간은 선택적 활용 항목으로 보고, 기본 생활기능6개를 기준으로 질문을 정리했습니다.</p>
<div class="cards"><div class="card"><small>기본6기능 모두 등록</small><b>{num(all_core)}곳</b><span>실제 이용 확인 후 상품조건 검토</span></div><div class="card"><small>한 기능이 완전 등록되지 않음</small><b>{num(one_core)}곳</b><span>누락·이용대안·보완비 확인</span></div><div class="card"><small>두 기능 이상 완전 등록되지 않음</small><b>{num(many_core)}곳</b><span>여러 기능의 이용대안·부담·의향 확인</span></div></div>
{stab}
<p>이 경로는 지원 순서를 논의하기 위한 설명 기준입니다. ‘한 기능 미등록=보완비가 적다’ 또는 ‘여러 기능 미등록=장기에 부적합하다’는 뜻이 아닙니다. 개인실도 동일한 개별 확인은 가능하지만, 이번 군집 유형을 강제로 부여하지 않았습니다.</p></section>

<section><h2>4. 실루엣이 높으면 무엇을 믿을 수 있나</h2>
<p>숙소 전체4군집의 validation 실루엣은<strong>0.561</strong>, 선택에 쓰지 않은 test는<strong>0.562</strong>였습니다. 같은 기능 조합이 반복되고 세탁기·업무공간처럼0/1로 갈리는 특징이 있어 이 공간에서 구분이 뚜렷하게 나타납니다. 값이 높다는 것은 현재 시설 표현과 거리 기준에서 잘 구분된다는 뜻입니다.</p>
<p>가능한 기능패턴은192개이며, 숙소 전체 train에서는190개가 관측됐습니다. 동일 패턴을 실제 숙소 건수로 가중하여 계산량만 줄였습니다. 실루엣은 숙소 수를 기준으로 동일 validation 표본 최대1,500곳에서 산출했습니다. 군집 최소 규모와 비중은 전체 행으로 검증했습니다.</p>
{table(pd.DataFrame(candidate))}
<p>숙소 전체는4·5군집이 모든 기준을 통과했습니다. 두 실루엣 차이가0.001보다 작아, 사전 규칙인 ‘최고값과0.03 이내에서 가장 적은 군집 수’에 따라4개를 택했습니다. 기존 분석이4개였기 때문에 선택한 것이 아닙니다.</p>
<p><strong>개인실은 채택하지 않았습니다.</strong> 4군집은 다른 가중치와의 일치도 최소0.622로 사전 기준0.65에 못 미쳤고, 5군집은 시드 일치도 최소0.697로 기준0.80에 미달했습니다. 2·3군집도 가중치 안정성을 통과하지 못했습니다. 문턱을 낮추거나 변수를 빼서 원하는 결과로 바꾸지 않았습니다.</p>
<p class="note">실루엣·안정성 통과는 장기 수요, 실제 예약, 시설 보완효과, 수익증가를 검증한 결과가 아닙니다. 기존 호스트 분할을 재사용했으므로 미래·외부 표본의 독립 검증으로 부르지 않습니다.</p></section>

<section><h2>5. 대형 호스트나 특정 지역이 만든 결과인가</h2>
<p>선택을 고정한 뒤 호스트·섹터·지역의 비중을 각각 같게 하여 같은4군집을 비교했습니다. 배정이 달라진 비율은0.3~0.7% 수준으로, 전체 숙소형의 주요 구분은 유지됐습니다.</p>
{table(pd.DataFrame(sensitivity))}
{dtab}
<p>각 유형이53개 지역 모두에서 관측됐고, 한 지역이 유형 전체를 지배하는 모습은 보이지 않았습니다. 다만 같은 등록 유형이라도 지역별 수요·계약·규제 조건은 다릅니다. 위 안정성은 지역별 사업 성과가 같다는 뜻이 아닙니다.</p>
<p>‘일반적 학습범위 밖’은 train에서 없는 기능패턴, 특징 범위 밖, 또는 소속 중심까지 거리가 해당 train군99%점보다 큰 경우입니다. 전체6섹터 공식군 중4,893곳이며, 검토 모집단1/2/4/5에서는4,061곳입니다. 별도 표시한 진단 신호이며 실제 부적합 판정은 아닙니다.</p></section>

<section><h2>6. 장기 설정 숙소와의 비교는 제한적이다</h2>
<p>3·6번은 최소30박 설정 여부와 최근 리뷰 유무로 나눈 집단입니다. 실제 장기 투숙 또는 성공 여부를 뜻하지 않습니다. 국가·지역·숙소종류·세부유형·수용인원이 같은 층에서 각 집단20숙소·5호스트 이상, 최대호스트 비중20% 이하일 때 비교 지원으로 표시했습니다.</p>
<div class="cards"><div class="card"><small>동일한 기본조건에서3/6 비교 지원</small><b>{num(conditions)}곳</b><span>검토 모집단의{pct(conditions/len(fit))}</span></div><div class="card"><small>같은 군집까지 맞춘 비교 지원</small><b>{num(cluster_support)}곳</b><span>검토 모집단의{pct(cluster_support/len(fit))}</span></div><div class="card"><small>기본조건 지원 + 양쪽 동일패턴 실재</small><b>{num(pattern_support)}곳</b><span>검토 모집단의{pct(pattern_support/len(fit))}</span></div></div>
{table(pd.DataFrame(overlap))}
<p>세 수치는 서로 다른 조건으로 계산했습니다. 마지막 수치는 양쪽에서 같은 기능패턴이 관측된다는 뜻이며, 그 패턴마다20건 이상 지원된다는 뜻은 아닙니다. 같은 군집 조건의 비교 지원은 전체의 일부이므로, 장기 참고집단에서 발견할 관계를522,248곳 전체로 일반화하면 안 됩니다.</p>
<p>이 단계에서는3·6번의 조정된 리뷰 관련성을 분석하지 않았습니다. 회귀 담당의 결과를 받아 시설·군집·지역의 관계와 비교 가능 범위를 함께 검토해야 합니다.</p></section>

<section><h2>7. 지금 채택할 것과 남겨둘 것</h2>
<ul><li><strong>채택:</strong> 숙소 전체4개 시설등록 유형을 호스트 확인 질문과 상담 순서를 정리하는 분류로 사용합니다.</li>
<li><strong>보류:</strong> 개인실의 공식 군집, 시설 미등록에 따른 투자비·장기 적합성 판정, 군집별 장기성공 확률 산정은 하지 않습니다.</li>
<li><strong>후속:</strong> 실제 시설 이용권, 연속 제공 가능 기간, 호스트 의향, 월요금·비용, 지역별 상품·계약·규제, 실제 수요를 확인합니다.</li>
<li><strong>효과 검증:</strong> 상담을 제공한 뒤 상품 등록·문의·예약·기여이익의 변화를 관찰하는 후속 실행 검증이 필요합니다.</li></ul>
<p>기존 최소 숙박일을 유지하며 장기 체류 상품을 추가하는 전략입니다. 다른 판매 경로와 중복될 수 있고, 군집 분석이 그 중복을 없애지는 않습니다.</p>
<p><a href="실행전계획.json">실행 전 계획</a> · <a href="선택고정.json">선택 고정·모형 해시</a> · <a href="군집평가.csv">후보 평가</a> · <a href="군집프로필.csv">상세 프로필</a> · <a href="숙소별_군집.csv.gz">전체 숙소별 결과</a> · <a href="결과요약.json">결과 요약</a> · <a href="검산.json">검산</a></p></section>
<footer>시설등록 데이터에 기반한 내부 탐색·검증 결과. 시설의 실제 상태·주소별 적격·예약·수익은 직접 관측하지 않았습니다. 보고서는 정적 구조·링크·수치 검사를 거치며 브라우저 시각 렌더링 확인과는 구별됩니다.</footer>
</main></body></html>'''
    (OUT/'결과보고서.html').write_text(report)
    print(json.dumps(counts,ensure_ascii=False))

if __name__=='__main__':main()
"""
SOURCE_META['cluster_report'] = {'path': '/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집/build_report.py', 'sha256': '53775316f2bca492f5a9c3ff12d6a026cc8254b969560d909f85b51e9a05f697', 'language': 'python'}


### regulatory_candidates — 53지역 시설 후보 집계와 규제 경로 연결
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_regulatory_candidates.py`
SHA256: `bde01e621fd3040a2ca2332f86e95fbe459f3d2c95eb52472353422e74f65b5c`


In [17]:
SOURCE_CODE['regulatory_candidates'] = """import hashlib,html,json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd

CWD=Path('/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities')
BASE=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs')
COMMON=BASE/'InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/공통분석표.csv.gz'
OLD=BASE/'InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
OUT=CWD/'outputs/규제혜택_생활업무_주력후보_20261003';OUT.mkdir(parents=True,exist_ok=True)
FUNCS=['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def write(name,obj):(OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
stamp=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
criteria={'created_at':stamp,'population':'source==0 & availability_365>0 & sector in [1,2,4,5]','room_type':'Entire home/apt','all_required_functions_full':FUNCS,'tools':'cooking_basics and dishes_silverware both registered (function__tools==1)','NA':'Does not satisfy full registration; not replaced by zero in source','candidate_meaning':'상품 참여 안내를 위한 시설 후보. 개별 규제·세금 적격 또는 실제 제공 가능성 확정 수가 아님.','selection_logic':'시설기준은 사용자가 확정한 7기능을 유지. 지역은 명시적 규제/세제 이득과 주거계약 후 관광운영 경로로 비교. 수량을 늘리기 위해 시설 조건을 완화하지 않음.','regulatory_review_scope':'53지역 시설수를 전량 집계. 주요 유망 경로를 프랑스·이탈리아·스페인·그리스·스위스와 빈·오슬로에서 검토.53지역 모든 법령을 전수 검증한 결과는 아님.','original_minimum_nights':'원래 관광상품 설정을 유지하는 별도 주거상품 검토. 분석표와 기존 후보매핑은 수정하지 않음.'}
write('선별기준.json',criteria)
cols=['id','host_id','country','region','source','availability_365','sector','room_type','minimum_nights']+['function__'+f for f in FUNCS]
d=pd.read_csv(COMMON,usecols=cols,dtype={'id':'string','host_id':'string'})
assert len(d)==677678 and d.id.is_unique and d.source.eq(0).all() and d.availability_365.gt(0).all()
t=d[d.sector.isin([1,2,4,5])].copy();assert len(t)==639774
t['full7']=t[['function__'+f for f in FUNCS]].eq(1).all(axis=1)
assert int(t.full7.sum())==135910
mapping=pd.read_csv(OLD/'후보매핑.csv.gz',dtype={'id':'string'})
joined=t[['id','full7']].merge(mapping[['id','bundle_LW']],on='id',validate='one_to_one',how='outer',indicator=True)
assert joined['_merge'].eq('both').all() and joined.full7.eq(joined.bundle_LW.eq(1)).all()
t['candidate']=t.room_type.eq('Entire home/apt')&t.full7
rows=[];sectorrows=[]
for (country,region),g in t.groupby(['country','region']):
    c=g[g.candidate];e=g[g.room_type.eq('Entire home/apt')]
    rows.append({'country':country,'region':region,'target_n':len(g),'entire_home_n':len(e),'entire_home_observed7_n':int(e[['function__'+f for f in FUNCS]].notna().all(axis=1).sum()),'candidate_n':len(c),'candidate_hosts_n':int(c.host_id.nunique()),'candidate_share_of_entire_home':len(c)/len(e) if len(e) else None})
    for sector,s in g.groupby('sector'):sectorrows.append({'region':region,'sector':int(sector),'target_n':len(s),'candidate_n':int(s.candidate.sum())})
R=pd.DataFrame(rows).sort_values('candidate_n',ascending=False);assert len(R)==53
prior=pd.read_csv(CWD/'work/regulatory_candidates_all53.csv')
assert R.set_index('region').candidate_n.to_dict()==prior.set_index('region').candidate_n.to_dict()
R.to_csv(OUT/'53지역_집전체_생활업무_후보수.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(sectorrows).to_csv(OUT/'지역_섹터별_후보수.csv',index=False,encoding='utf-8-sig')
names={'pays-basque':'프랑스 바스크','venice':'베네치아','malaga':'말라가','sevilla':'세비야','valencia':'발렌시아','bordeaux':'보르도','lyon':'리옹','paris':'파리','crete':'크레타','south-aegean':'남에게해','athens':'아테네','thessaloniki':'테살로니키','geneva':'제네바','vaud':'보주','vienna':'빈','oslo':'오슬로','barcelona':'바르셀로나','girona':'지로나','mallorca':'마요르카','menorca':'메노르카'}
judgments={
 'pays-basque':('주력','학생 최소9개월 + 여름 관광3개월','조건부 보상 의무 예외','규제 적용24지자체·학생 거주·연간 임시허가'),
 'venice':('조건부 주력 개발','실제 한시 근무·연수 기간의 주거 + 계약종료 후 관광','적격 협약임대의 IMU우대. 적용기간0.76%×75%=0.57%','현행 지역협약·임대료·실제 임시사유·지적분류·관광 신고'),
 'malaga':('운영형 확장','2개월 초과 실제 주거 + 신고된 관광기간','관광기간과 주거기간을 나누는 운영 경로','추가 세금·허가 특례 미확인. 기존 관광 운영권과 현행 주거계약 확인'),
 'sevilla':('운영형 확장','2개월 초과 실제 주거 + 신고된 관광기간','관광기간과 주거기간을 나누는 운영 경로','추가 세금·허가 특례 미확인. 기존 관광 운영권과 현행 주거계약 확인'),
 'valencia':('운영형 후순위','관광 신고기간과 별도 실제 주거기간','계절별 영업기간 변경 경로','주거임대를 관광 신고기간에 제공하면 등록취소 사유. 추가 특례 미확인'),
 'bordeaux':('주력 보류','학생·주거와 관광 교대','장기 제공에 따른 보상 예외 미확인','관광 복귀의 보상 요건 별도'),
 'lyon':('주력 보류','학생·주거와 관광 교대','장기 제공에 따른 특례 미확인','소형 개인주택 예외는 장기상품 제공 때문에 생기는 혜택이 아님'),
 'paris':('기존 보류 유지','주거와 관광 교대','실제 주거주지·거주기간 조건에 따라 달라짐','이번에 전체 파리 규정을 재검증하지 않음'),
 'geneva':('주력 보류','플랫폼 주거와 관광 교대','플랫폼 장기상품의90일 제외 근거 미확인','일반 주거계약을 플랫폼 장기예약과 동일시하지 않음'),
 'vaud':('주력 보류','플랫폼 주거와 관광 교대','플랫폼 장기상품의90일 제외 근거 미확인','연속 또는 간헐적인 플랫폼 대여 모두의 기간 기준 확인'),
 'vienna':('주력 보류','주거와 관광 교대','90일 예외는 자기 주거지를 포기하지 않는 홈셰어링','임대인 직접거주 조건을 학생9개월 임대로 대체할 수 있다고 확인하지 못함'),
 'oslo':('주력 보류','주거와 관광 교대','단기90일 기준만으로 고정기간 주거상품 이점 확정 불가','일반 주거의 최소계약기간과 예외를 개별 확인해야 함')}
for k in ['crete','south-aegean','athens','thessaloniki']:judgments[k]=('연간 교대형 보류','관광에서 장기 주거로 전환하는 별도 상품','적격 장기전환 소득세 면제','3년 내 관광 재개시 면제 소급 상실. 매년 관광 교대와 충돌')
for k in ['barcelona','girona','mallorca','menorca']:judgments[k]=('주력 보류','계절 주거와 관광 교대','이번검토에서 상품 추가에 따른 특별 이점 미확인','허가·주거계약 조건은 기간만으로 우회되지 않음')
decision=[]
for region,(status,product,benefit,condition) in judgments.items():
    row=R[R.region.eq(region)].iloc[0]
    decision.append({'region':region,'지역':names[region],'후보수':int(row.candidate_n),'고유호스트수':int(row.candidate_hosts_n),'판단':status,'상품방향':product,'제도상_근거':benefit,'참여확인':condition})
pd.DataFrame(decision).to_csv(OUT/'지역별_제안판단.csv',index=False,encoding='utf-8-sig')
primary=t[t.candidate&t.region.isin(['pays-basque','venice'])]
assert len(primary)==3020 and primary.groupby('region').size().to_dict()=={'pays-basque':1563,'venice':1457}
expand=t[t.candidate&t.region.isin(['malaga','sevilla','valencia'])]
assert len(expand)==6036
for name,g in [('주력개발_시설후보_ID.jsonl',primary),('운영형확장_시설후보_ID.jsonl',expand)]:
    # JSONL preserves long listing identifiers as strings.
    data=g[['id','region','sector','room_type']].copy();data['status']='시설기준 참여안내 후보'
    data.to_json(OUT/name,orient='records',lines=True,force_ascii=False)
    check=pd.read_json(OUT/name,lines=True,dtype={'id':'string'})
    assert check.id.tolist()==g.id.tolist() and check.id.is_unique

sources={
 'basque':('바스크 혼합임대 공식 규정 제2조·4.1조','https://www.communaute-paysbasque.fr/fileadmin/user_upload/mediatheque/Tourisme/CAPaysbasqueCDU_Rglt_et_Annexes_modifie2025.pdf'),
 'imu2026':('베네치아2026 IMU세율','https://www.comune.venezia.it/it/content/aliquote-calcolo-e-scadenze-imu-2026'),
 'imu_rate':('2026에도 유지되는 공식IMU세율표','https://www.comune.venezia.it/sites/comune.venezia.it/files/documenti/Tributi/IMU2020/Prospetto%20aliquote%20MEF%202025.pdf'),
 'imu_reduction':('베네치아 협약임대 세액75% 적용','https://www.comune.venezia.it/it/node/29687'),
 'agreement':('베네치아2026 지역협약','https://www.comune.venezia.it/it/content/patti-territoriali'),
 'temporary':('이탈리아 임시주거 계약 시행령 제2조','https://www.gazzettaufficiale.it/atto/serie_generale/caricaArticolo?art.codiceRedazionale=17A01858&art.dataPubblicazioneGazzetta=2017-03-15&art.flagTipoArticolo=0&art.idArticolo=2&art.idGruppo=0&art.idSottoArticolo=1&art.idSottoArticolo1=10&art.progressivo=0&art.versione=1'),
 'veneto':('베네토 관광 영업기간·일시중단 공식FAQ18~26','https://elezioni2020.regione.veneto.it/documents/10813/47451/FAQ%2B%2B29%2Bgennaio%2B2026%2Blocatore%2Bturistico.pdf/29d69227-d0be-4a29-b2e5-84b3d9217906'),
 'andalusia':('안달루시아 관광임대 개정령','https://www.juntadeandalucia.es/boja/2024/24/50'),
 'valencia':('발렌시아 관광규정 제65조·78조','https://www.boe.es/buscar/act.php?id=BOE-A-2018-8950&p=20260702&tn=1'),
 'nice':('니스 공식 혼합임대','https://www.nicecotedazur.org/services/logement/autorisations-de-changements-dusage/location-mixte/'),
 'nicequota':('니스 공식 허가FAQ','https://www.nicecotedazur.org/services/logement/autorisations-de-changements-dusage/foire-aux-questions/'),
 'oslo':('노르웨이 정부 주거계약 최소기간 설명','https://www.regjeringen.no/no/dokumenter/rundskriv-h-0802/id462342/'),
 'vienna':('빈 공식 홈셰어링 예외 설명','https://www.wien.gv.at/pdf/ma37/merkblatt-verwendung-wohnungen-kurzzeitvermietung.pdf')}
research={name:json.loads((CWD/'work'/name).read_text()) for name in ['italy_regulatory_findings.json','france_regulatory_findings.json','southern_regulatory_findings.json','swiss_followup.json']}
write('공식근거와_검토기록.json',{'checked_on':'2026-10-03','primary_source_index':sources,'research':research})
summary={'created_at':stamp,'criteria':criteria,'reference_regions':['pays-basque','venice'],'primary_facility_candidates':3020,'primary_unique_hosts':int(primary.host_id.nunique()),'all_53_region_candidates':int(R.candidate_n.sum()),'decisions':decision,'nice':{'in_dataset':False,'candidate_n':None,'recommendation':'새 데이터 확보 후 규제혜택형 우선 확장','benefit':'학생·관광 혼합임대 보상면제·허가수 제한 없음·쿼터 제외','condition':'CAF대상 학생,최소5~6개월 거주,여름3개월,CROUS협약,임대료상한,DPE A~E,사전허가'},'recommended_development':'바스크 학생9+3와 베네치아 실제 임시근무·연수 주거상품을 주력으로 개발. 베네치아는 현행협약·세금적격·실제 만기반환·관광신고 확인을 전제로 함. 말라가·세비야·발렌시아는 운영형 확장후보.'}
write('결과요약.json',summary)
write('검산.json',{'completed_at':stamp,'input_rows':len(d),'target_rows':len(t),'unique_ids':True,'existing_bundle_LW_mapping_reproduced':True,'all_room_LW_n':135910,'entire_home_LW_n':int(t.candidate.sum()),'all53_prior_count_match':True,'basque_n':1563,'venice_n':1457,'primary_n':3020,'expansion_n':6036,'exported_identifiers_roundtrip_exact':True,'source_sha256':sha(COMMON),'original_mapping_sha256':sha(OLD/'후보매핑.csv.gz'),'old_files_modified':False})

def e(v):return html.escape(str(v))
def link(key):title,url=sources[key];return f'<a href="{e(url)}">{e(title)}</a>'
def table(head,rows):return '<div class="table-wrap"><table><thead><tr>'+''.join('<th>'+e(x)+'</th>' for x in head)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+e(x)+'</td>' for x in row)+'</tr>' for row in rows)+'</tbody></table></div>'
mainrows=[[names[x],f'{int(R[R.region.eq(x)].iloc[0].candidate_n):,}',judgments[x][0],judgments[x][1],judgments[x][2]] for x in ['pays-basque','venice','malaga','sevilla','valencia']]
mainrows.append(['니스','현재 데이터 없음','데이터 확보 후 우선 확장','학생5~6개월 이상 + 여름3개월','보상면제·허가수 제한 없음·쿼터 제외'])
holdrows=[[z['지역'],f"{z['후보수']:,}",z['판단'],z['제도상_근거'],z['참여확인']] for z in decision if '보류' in z['판단']]
css='''*{box-sizing:border-box}body{margin:0;background:#f3f5f7;color:#192b39;font:16px/1.75 -apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo",sans-serif}main{max-width:1180px;margin:30px auto;padding:0 20px}header{background:#153c50;color:white;padding:36px;border-radius:16px}h1{font-size:32px;line-height:1.35;margin:8px 0 16px}h2{font-size:23px;line-height:1.45;margin:0 0 18px}h3{font-size:18px;margin:20px 0 8px}section{background:white;padding:30px;margin:20px 0;border-radius:12px;border:1px solid #dce5eb}p{margin:12px 0}.lead{font-size:21px;font-weight:600}.small{font-size:13px;color:#516779}.note{border-left:4px solid #7eafb8;background:#eff7f7;padding:15px 18px;margin:18px 0}.table-wrap{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:14px;min-width:800px}th,td{padding:13px 11px;text-align:left;vertical-align:top;border-bottom:1px solid #dce5eb}th{background:#eaf2f6}td:nth-child(2){white-space:nowrap}a{color:#12618b;overflow-wrap:anywhere}header p{color:#e4f1f5}.download{display:flex;flex-wrap:wrap;gap:12px}.download a{padding:13px 17px;background:#eef5f8;border-radius:6px}li{margin:8px 0}footer{font-size:12px;color:#516779;padding:15px 5px}@media(max-width:650px){main{padding:0 10px;margin:10px auto}header,section{padding:22px}h1{font-size:27px}}'''
body=f'''<header><div>InsightStay · 공식자료 확인 2026.10.03</div><h1>바스크와 함께 발전시킬 장기상품 후보</h1><p class="lead">주력 개발: 바스크 1,563곳 + 베네치아 1,457곳 = 3,020곳</p><p>집 전체형에 생활·업무 시설을 모두 갖춘 중단기 숙소를 대상으로, 지역별 제도상 이점과 운영 경로를 연결했습니다.</p></header>
<section><h2>제안 후보와 역할</h2>{table(['지역','시설 후보 수','개발 판단','상품 방향','호스트에게 제안할 이유'],mainrows)}<p>주력은 바스크의 혼합임대 보상 예외와 베네치아의 적격 주거계약 세제 우대입니다. 스페인 세 지역은 교대 운영 경로가 확인된 확장 후보로 두었습니다. ‘추가 세금·허가 혜택’과 ‘계절별 운영 가능성’은 다른 근거입니다.</p><p class="small">모든 수치는 현재 분석표의 시설 기준 참여안내 후보입니다. 지역명만으로 개별 허가·세금 적격을 판정하지 않았습니다. 니스는53지역에 없어 후보 수를0으로 처리하지 않았습니다.</p></section>
<section><h2>공통 시설 조건</h2><p>기존1·2·4·5번 가운데 집 전체형이고, <strong>주방·가열기기·냉장고·식기 및 조리도구·세탁기·Wi-Fi·전용 업무공간</strong>이 모두 등록된 숙소입니다. 가열기기는 스토브·오븐·전자레인지 중 하나 이상, 식기 및 조리도구는 두 원항목 모두 등록된 경우입니다.</p><p>source=0, availability_365&gt;0인 기존 공통표를 사용했습니다. 기존 minimum_nights를 유지하면서 별도 주거상품을 제안하는 방향입니다. 기존에 확정한 생활·업무 7기능 기준을 유지했습니다.</p></section>
<section><h2>바스크: 학생 주거와 여름 관광의 혼합임대</h2><p><strong>1,563곳</strong>에 학생 최소9개월·여름 관광3개월 상품 참여를 안내합니다. 적용24지자체의 해당 주택에서 학생 거주를 증명하고 임시허가를 받으면 보상 의무 예외를 이용할 수 있습니다. 같은 시설을 두 시기의 목적에 맞춰 안내하는 구조입니다.</p><p>신청 단계에서 소재지,학생 제공기간과 거주증빙,허가 조건을 확인합니다. 이 단계에서 적격 여부가 정해지므로1,563곳 전체에 혜택이 이미 확정됐다고 표시하지 않습니다. {link('basque')}</p></section>
<section><h2>베네치아: 실제 임시 근무·연수 기간의 주거상품</h2><p><strong>1,457곳</strong>을 바스크와 함께 조건부 주력으로 발전시킵니다. 제안은 ‘실제 한시적 근무·연수 등의 주거 필요에 맞춰 협약임대료로 제공하고, 계약 종료와 반환 후 관광 운영기간으로 돌아가는 상품’입니다.</p><h3>어떤 이점이 있는가</h3><p>베네치아는 적격 협약임대에 시 부동산세 IMU 우대세율0.76%를 적용하고, 해당 세액을75%로 줄입니다. 따라서 적용기간의 실효세율은 <strong>0.57%</strong>입니다. 공식표의 일반 주택 비교세율은1.10%입니다. 지적분류A/2·A/3·A/4·A/5·A/7,등록계약과 현행협약 적합확인 등 요건을 충족해야 하며 관광 운영기간까지 일괄0.57%로 계산하지 않습니다. {link('imu2026')} · {link('imu_rate')} · {link('imu_reduction')}</p><h3>상품 기간과 관광 복귀</h3><p>적법한 임시 주거계약은 실제 사유에 맞춰 최대18개월 안에서 정합니다.30일을 넘는 계약에는 임시 사유 증빙을 붙이고 협약 임대료를 따릅니다. 학생계약은 첫 만기의 자동갱신 조항 때문에9+3의 기본 계약으로 쓰기 어렵습니다. 베네치아는 고정9+3보다 실제 근무·연수 종료시점에 맞춘 상품이 적합합니다. {link('temporary')} · {link('agreement')}</p><p>베네토는 관광 영업기간의 사전 변경과 시작·종료일을 정한 일시중단을 신고하는 절차를 명시합니다. 이 절차와 적법한 주거계약 종료를 연결해 복귀 일정을 설계합니다. 예약달력만 닫는 것은 해당 행정신고를 대신하지 않습니다. 관광 중단기간에도 관광객 0명 신고와 등록 상태를 관리해야 합니다. 12개월 연속 미신고하면 CIR·CIN이 말소되어 관광 재개 시 신규 신고가 필요합니다. {link('veneto')}</p><div class="note">개발 전 확인할 핵심은 실제 임시 사유,2026 지역협약에 맞는 임대료와 계약 적합확인,세금 대상 주택인지,관광 신고 상태입니다.2026.7.28 새 지역협약의 존재와 공식 양식 경로는 확인했으나 새 협약PDF 본문은 확보하지 못했습니다. 세부 허용 사유·임대료표는 후보별 확인 항목으로 남깁니다.</div><p>세금 우대는 제안 이유가 되지만 총수익 증가는 아직 계산하지 않았습니다. 실제 협약임대료와 포기하는 관광매출,공과금·관리비용을 비교해 참여를 결정할 수 있게 합니다.</p></section>
<section><h2>스페인: 교대 운영을 위한 확장 후보</h2><p><strong>말라가2,266곳·세비야2,107곳</strong>은 동일 임차인에게2개월을 초과해 제공하는 기간이 관광임대 규정에서 제외되고,관광 영업기간을 별도로 신고하는 구조를 활용할 수 있습니다. 유효한 관광 운영권을 가진 숙소에 실제 학업·근무 사유의 주거상품을 추가하는 방향입니다. 장기 제공으로 새로운 관광허가나 세금 우대를 얻는다고 확인한 것은 아닙니다. {link('andalusia')}</p><p><strong>발렌시아1,663곳</strong>도 계절별 운영기간을 나누는 후보입니다.11일 이상이라는 관광 분류 기준만으로 주거상품을 만들 수는 없습니다. 실제 주거계약과 관광 신고기간을 구별해야 하며,관광 신고기간에 주거임대를 제공하면 관광등록 취소 사유가 될 수 있습니다. {link('valencia')}</p><p>이 세 지역은 ‘명확한 특별혜택을 앞세운 주력’보다 기존 관광 운영의 비수기 상품 확장 대상으로 두겠습니다. 현행 주거계약과 소재지의 관광 제한,정해진 임대료·임차인 권리까지 반영해야 합니다.</p></section>
<section><h2>니스: 새 데이터 확보 후 가장 유사한 확장 후보</h2><p>공식 혼합임대 제도는 학생 거주와 여름3개월 관광을 결합할 때 보상면제와 허가수 제한 없는 경로를 제공합니다. 현행FAQ는 이 경로의 쿼터 제외도 설명합니다. 바스크와 유사한 규제 이점이지만 현재 데이터에 니스 숙소가 없어 후보 수를 계산할 수 없습니다. {link('nice')} · {link('nicequota')}</p><p>CAF대상 학생의 최소5~6개월 거주,CROUS협약,방 수별 임대료 상한,DPE A~E와 사전허가가 주요 조건입니다. 이를 바스크의9+3과 같은 상품으로 표기하지 않습니다. 새로운 지역 데이터를 확보한다면 우선 검토할 대상입니다.</p></section>
<section><h2>공급이 있어도 이번 주력에서 보류한 지역</h2>{table(['지역','시설 후보','판단','확인 결과','운영에 미치는 의미'],holdrows)}<p>그리스의 적격 장기전환 소득세 면제는 명확한 제도지만3년 내 관광으로 돌아가면 면제를 소급 상실하므로 매년 관광과 교대하는 전략에 넣지 않았습니다. 장기 전환을 원하는 호스트에게는 별도 상품으로 검토할 수 있습니다. 프랑스·스위스 등의 세부 공식출처와 검토 범위는 아래 근거기록에 수록했습니다.</p></section>
<section><h2>주력 후보를 상품으로 발전시키는 순서</h2><ol><li><strong>시설 후보3,020곳에 맞춘 두 상품안을 준비합니다.</strong> 바스크는 학생9+3,베네치아는 실제 임시근무·연수 기간의 협약 주거입니다.</li><li><strong>참여 의향과 제공 가능한 기간·희망 월금액을 받습니다.</strong> 바스크의 소재지·혼합임대 요건,베네치아의 계약·세금·관광신고 조건에 맞춰 상담합니다.</li><li><strong>후보별 기존 관광 수입과 장기상품의 순수입을 비교합니다.</strong> 실제 예약자료가 없으므로 현재 가격·리뷰 수로 수익 증가를 확정하지 않습니다.</li><li><strong>두 상품의 기간과 달력을 연결합니다.</strong> 주거 제공기간,관광 신고기간,계약 종료·반환일을 기준으로 중복 예약을 막고 관광 복귀일을 안내합니다.</li></ol></section>
<section><h2>집계와 근거 파일</h2><div class="download"><a href="53지역_집전체_생활업무_후보수.csv">53지역 시설 후보 수</a><a href="지역별_제안판단.csv">지역별 제안 판단</a><a href="주력개발_시설후보_ID.jsonl">주력3,020곳 내부 연결ID</a><a href="운영형확장_시설후보_ID.jsonl">스페인 확장6,036곳 내부 연결ID</a><a href="공식근거와_검토기록.json">공식자료와 상세 검토</a><a href="검산.json">집계 검산</a></div><p class="small">ID는 원본과 내부 연결하기 위한 문자열로 보존했습니다. 고유ID 중복,지역 합계,기존 생활·업무 매핑과의 일치,출력 후ID 보존을 검산했습니다. 원분석표·이전 매핑은 변경하지 않았습니다.</p></section><footer>데이터: 기존 공통분석표677,678행 중1·2·4·5번639,774행. 지역별 시설 후보는 동일한 고정 조건으로 계산했습니다.</footer>'''
(OUT/'주력후보_제안보고서.html').write_text('<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>바스크와 함께 발전시킬 장기상품 후보</title><style>'+css+'</style></head><body><main>'+body+'</main></body></html>',encoding='utf-8')
print(json.dumps({'output':str(OUT),'all53_candidates':int(R.candidate_n.sum()),'primary_n':len(primary),'expansion_n':len(expand),'primary_unique_hosts':int(primary.host_id.nunique()),'files':[p.name for p in OUT.iterdir()]},ensure_ascii=False))
"""
SOURCE_META['regulatory_candidates'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_regulatory_candidates.py', 'sha256': 'bde01e621fd3040a2ca2332f86e95fbe459f3d2c95eb52472353422e74f65b5c', 'language': 'python'}


### regression — 조정 로지스틱 회귀 전체
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_생활업무시설_조정회귀_20261003/analyze.py`
SHA256: `edc1fc2b357b2b2e58b4d3d316c0c52cdc4db63a6b496678ad882399bda90c1b`


In [18]:
SOURCE_CODE['regression'] = """import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key]='2'
from pathlib import Path
import json, hashlib, warnings, sys
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.special import expit
from scipy.stats import norm
from patsy import build_design_matrices, dmatrices

OUT=Path(__file__).resolve().parent
PROJECT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
SRC=PROJECT/'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터/공통분석표.csv.gz'
BASE=OUT.parent/'7개지역_장기상품_통합전략_20261003'
REGIONS=['london','pays-basque','oslo','munich','malaga','sevilla','valencia']
FUNC=['function__'+x for x in ['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']]
KEYS=['region','property_type','accommodates']
plan={
 'question':'같은 지역·상품 조건에서 생활·업무 7기능 완전 등록이 장기 설정 공급에 더 흔한가?',
 'frozen_before_model_fit':True,
 'prior_information':'기존 보고서의 등록률 및 이번 표본 진단을 이미 읽음. 독립 사전등록 연구가 아니며 계수 계산 전 분석 선택을 고정함.',
 'source':str(SRC), 'regions':REGIONS,
 'population':'source=0, availability_365>0, room_type=Entire home/apt, 7기능 모두 관측',
 'primary_comparison':'최근 리뷰 양수에서 3번 vs 1·2번. 결과가 시설 등록이므로 예측용 train/test 분할 없음.',
 'outcome':'derived bundle_registered: 7개 function__ 값 모두 1이면 1, 아니면 0. 모든 7값 관측 필요.',
 'exposure':'derived long_setting: minimum_nights>=30; 현재 설정이며 실제 체류 길이 아님.',
 'overlap':'region × property_type × accommodates 정확 조합에 두 설정 집단이 적어도 1곳씩 존재하는 범위. 이는 충분 표본 판정과 별개.',
 'primary_model':'M1 binomial-logit GLM: bundle_registered ~ long_setting + C(region)+C(property_type)+C(accommodates). 공통 계수의 탐색적 통합 추정.',
 'uncertainty':'host_id로 묶은 sandwich 표준오차, 유한표본 보정; delta method 95% 구간. 표준화 대상의 공변량은 고정 취급.',
 'estimand':'비교 지원 장기 집단의 상품 구성에서 long_setting=1/0일 때 예측 등록률 평균 및 차이. 설정 변경 효과/인과효과가 아님.',
 'model_sequence':['M0 동일 표본 조정 전','M1 지역·상품 조정','M1_CC 침실·욕실 관측 표본의 동일 기본 조정','M2_CC 동일 관측 표본에 침실·욕실 범주 추가'],
 'bed_bath':'원 관측 bedrooms_observed: 1/2/3/4+; bathrooms_observed: 0.5/1/1.5/2/2.5/3+. 경계 사이 값은 아래 구간으로 묶고 원 분포 저장.',
 'prespecified_sensitivity':['S_HOST 같은 표본에서 호스트당 총 가중치 1','S_ALL 3+6 vs 1+2+4+5, 최근 리뷰 유무 및 장기설정 상호작용 포함'],
 'support_audit':'각 exact층 양측>=20숙소,>=5호스트, 최대 호스트 비중<=20%를 별도로 집계. 전체 회귀에 이 기준을 만족한다고 주장하지 않음.',
 'regional_reporting':'공통 계수를 지역별 검증으로 제시하지 않음. 양측 30곳 이상, 각 결과값 최소10곳, 최소20호스트인 지역만 개별 기본 회귀를 탐색적으로 실행. 해석은 통합 결과의 이질성 점검이고 다중검정 보정 포함.',
 'excluded':'price, estimated occupancy, maximum_nights, 호스트 평점, 시설 구성요소·amenity_count는 주요 X에서 제외. outcome 정의 피처의 자기예측 방지.',
 'stop_rules':'수렴 실패/분리/설계행렬 특이 발생 시 추정 실패 기록. 유리한 결과를 위한 모델 탐색 금지.',
 'interpretation':'회귀의 통계적 관련성만. 장기 수요·실제 장기 예약·매출·법적 적격 검증 불가.'
}
(OUT/'실행전계획.json').write_text(json.dumps(plan,ensure_ascii=False,indent=2))
cols=['id','source','host_id','sector','minimum_nights','number_of_reviews_ltm','availability_365','region','room_type','property_type','accommodates','bedrooms_observed','bathrooms_observed']+FUNC
d=pd.read_csv(SRC,usecols=cols,dtype={'id':'string','host_id':'string'})
assert d.id.is_unique and len(d)==677678
assert d.source.eq(0).all() and d.availability_365.gt(0).all()
all_region=d[d.region.isin(REGIONS)].copy()
home=all_region[all_region.room_type.eq('Entire home/apt')].copy()
home['facilities_observed']=home[FUNC].notna().all(axis=1)
home['bundle_registered']=home[FUNC].eq(1).all(axis=1).astype(int)
home['long_setting']=home.minimum_nights.ge(30).astype(int)
home['review_positive']=home.number_of_reviews_ltm.gt(0).astype(int)
assert home.long_setting.eq(home.sector.isin([3,6])).all()
assert home.review_positive.eq(home.sector.isin([1,2,3])).all()
assert home.host_id.notna().all()

# Candidate IDs must reproduce the supplied report; no new selection criteria.
candidate=home[home.sector.isin([1,2,4,5])&home.facilities_observed&home.bundle_registered.eq(1)]
base_ids=pd.read_csv(BASE/'7개지역_시설후보_ID.csv',dtype={'id':'string'})
assert set(candidate.id)==set(base_ids.id) and len(candidate)==23222
valid=home[home.facilities_observed].copy()
raw=[]
for (r,s),g in home.groupby(['region','sector']):
    v=g[g.facilities_observed];h=v.host_id.value_counts()
    raw.append(dict(region=r,sector=int(s),n_all=len(g),n_observed=len(v),host_n=v.host_id.nunique(),yes=int(v.bundle_registered.sum()),rate=float(v.bundle_registered.mean()) if len(v) else None,max_host_share=float(h.iloc[0]/len(v)) if len(v) else None))
pd.DataFrame(raw).to_csv(OUT/'지역섹터_표본과등록률.csv',index=False,encoding='utf-8-sig')
positive=valid[valid.review_positive.eq(1)].copy()
support_records=[]
def overlap(frame,label):
    rows=[]
    for key,g in frame.groupby(KEYS,dropna=False):
        row=dict(zip(KEYS,key))
        for flag in [0,1]:
            a=g[g.long_setting.eq(flag)];hc=a.host_id.value_counts()
            row.update({f'n_{flag}':len(a),f'host_n_{flag}':a.host_id.nunique(),f'max_host_share_{flag}':float(hc.iloc[0]/len(a)) if len(a) else np.nan})
        row['overlap']=row['n_0']>0 and row['n_1']>0
        row['strict_support']=all(row[f'n_{f}']>=20 and row[f'host_n_{f}']>=5 and row[f'max_host_share_{f}']<=.2 for f in [0,1])
        row['comparison']=label;rows.append(row)
    s=pd.DataFrame(rows);support_records.extend(rows)
    good=s.loc[s.overlap,KEYS]
    r=frame.merge(good,on=KEYS,how='inner',validate='many_to_one').reset_index(drop=True)
    return r,s
primary,support=overlap(positive,'recent_review_positive')
all_supported,support_all=overlap(valid,'all_review_states')
pd.DataFrame(support_records).to_csv(OUT/'정확상품조합_비교지원.csv',index=False,encoding='utf-8-sig')
all_supported['bedrooms_band']=pd.cut(all_supported.bedrooms_observed,[-np.inf,1,2,3,np.inf],labels=['1','2','3','4+']).astype('string')
all_supported['bathrooms_band']=pd.cut(all_supported.bathrooms_observed,[-np.inf,.5,1,1.5,2,2.5,np.inf],labels=['<=0.5','1','1.5','2','2.5','3+']).astype('string')
primary['bedrooms_band']=pd.cut(primary.bedrooms_observed,[-np.inf,1,2,3,np.inf],labels=['1','2','3','4+']).astype('string')
primary['bathrooms_band']=pd.cut(primary.bathrooms_observed,[-np.inf,.5,1,1.5,2,2.5,np.inf],labels=['<=0.5','1','1.5','2','2.5','3+']).astype('string')
complete=primary[primary.bedrooms_observed.notna()&primary.bathrooms_observed.notna()].copy()
# Recheck overlap after excluding missing values; same comparison frame for M1_CC/M2_CC.
complete,support_cc=overlap(complete,'bed_bath_complete')
pd.DataFrame(support_records).to_csv(OUT/'정확상품조합_비교지원.csv',index=False,encoding='utf-8-sig')
for col in ['bedrooms_band','bathrooms_band']:
    complete[col]=complete[col].astype(str)
flow=[]
for name,f in [('7regions_entire',home),('facilities_observed',valid),('positive_reviews',positive),('primary_overlap',primary),('bed_bath_complete_overlap',complete),('all_states_overlap',all_supported)]:
    for r,g in [('ALL',f),*list(f.groupby('region'))]:
        flow.append(dict(stage=name,region=r,n=len(g),long_n=int(g.long_setting.sum()),shortmid_n=int(g.long_setting.eq(0).sum()),host_n=g.host_id.nunique(),long_host_n=g.loc[g.long_setting.eq(1),'host_id'].nunique()))
pd.DataFrame(flow).to_csv(OUT/'표본흐름.csv',index=False,encoding='utf-8-sig')
print('Prepared',len(primary),'primary, long',int(primary.long_setting.sum()),'complete',len(complete),flush=True)

RESULTS=[];COEFS=[];fits={}
def contrast(result,frame,reference,weights=None,review_state=None):
    a=reference.copy();b=reference.copy();a['long_setting']=1;b['long_setting']=0
    if review_state is not None:
        a['review_positive']=review_state;b['review_positive']=review_state
    xa=np.asarray(build_design_matrices([result._analysis_design_info],a)[0]);xb=np.asarray(build_design_matrices([result._analysis_design_info],b)[0])
    beta=np.asarray(result.params);cov=np.asarray(result.cov_params());pa=expit(xa@beta);pb=expit(xb@beta)
    w=np.ones(len(reference)) if weights is None else np.asarray(weights)
    w=w/w.sum();p1=float(w@pa);p0=float(w@pb)
    gradient=(w[:,None]*(pa*(1-pa))[:,None]*xa).sum(0)-(w[:,None]*(pb*(1-pb))[:,None]*xb).sum(0)
    se=float(np.sqrt(max(gradient@cov@gradient,0)));delta=p1-p0
    return dict(standardization_n=len(reference),adjusted_long_rate=p1,adjusted_shortmid_rate=p0,difference_pp=100*delta,ci_low_pp=100*(delta-1.95996398454*se),ci_high_pp=100*(delta+1.95996398454*se),p_difference=float(2*norm.sf(abs(delta/se))) if se>0 else None)

def fit(name,frame,formula,weighted=False,reference=None,notes=''):
    f=frame.copy().reset_index(drop=True)
    counts=f.host_id.map(f.host_id.value_counts()).to_numpy(float)
    weights=1/counts if weighted else np.ones(len(f))
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        ym,xm=dmatrices(formula,f,return_type='dataframe')
        assert len(ym)==len(f)
        model=sm.GLM(ym,xm,family=sm.families.Binomial(),freq_weights=weights if weighted else None)
        res=model.fit(maxiter=100,tol=1e-9,cov_type='cluster',cov_kwds={'groups':f.host_id.to_numpy(),'use_correction':True})
        res._analysis_design_info=xm.design_info
    warning_messages=[str(w.message) for w in captured]
    X=np.asarray(res.model.exog)
    rank=int(np.linalg.matrix_rank(X))
    invalid=(not res.converged) or rank<X.shape[1] or not np.isfinite(res.params).all() or max(abs(res.params))>25
    if invalid:
        raise RuntimeError(f'{name} unsuitable model, converged={res.converged}, rank={rank}, p={X.shape[1]}, maxbeta={max(abs(res.params))}')
    if reference is None: reference=f[f.long_setting.eq(1)].copy()
    refw=None
    if weighted:
        refw=reference.host_id.map(dict(zip(f.host_id,weights))).to_numpy(float)
    out=dict(model=name,n=len(f),long_n=int(f.long_setting.sum()),shortmid_n=int(f.long_setting.eq(0).sum()),host_n=f.host_id.nunique(),long_host_n=f.loc[f.long_setting.eq(1),'host_id'].nunique(),regions=f.region.nunique(),formula=formula,weighted=weighted,converged=bool(res.converged),iterations=int(res.fit_history['iteration']),parameters=X.shape[1],rank=rank,notes=notes,warnings=warning_messages)
    out.update(contrast(res,f,reference,refw))
    coef=float(res.params['long_setting']);se=float(res.bse['long_setting'])
    out.update(odds_ratio=float(np.exp(coef)),odds_ratio_low=float(np.exp(coef-1.95996398454*se)),odds_ratio_high=float(np.exp(coef+1.95996398454*se)),p_coefficient=float(res.pvalues['long_setting']))
    if not weighted:
        null=sm.GLM(f.bundle_registered,np.ones((len(f),1)),family=sm.families.Binomial()).fit()
        out['mcfadden_pseudo_r2']=float(1-res.llf/null.llf)
    # Independently check the host score sandwich, including weighted score and Hessian.
    prob=np.asarray(res.fittedvalues);y=f.bundle_registered.to_numpy(float)
    info=X.T@((weights*prob*(1-prob))[:,None]*X)
    bread=np.linalg.inv(info);codes,hosts=pd.factorize(f.host_id)
    score=np.zeros((len(hosts),X.shape[1]));np.add.at(score,codes,(weights*(y-prob))[:,None]*X)
    corr=len(hosts)/(len(hosts)-1)*(len(f)-1)/(len(f)-X.shape[1])
    independent=bread@(score.T@score)@bread*corr
    out['sandwich_max_abs_error']=float(np.max(np.abs(independent-np.asarray(res.cov_params()))))
    assert out['sandwich_max_abs_error']<1e-7
    if 'long_setting:review_positive' in res.params:
        out['by_review_state']={}
        for v in [0,1]:
            ref=f[f.long_setting.eq(1)&f.review_positive.eq(v)]
            out['by_review_state'][str(v)]=contrast(res,f,ref,review_state=v)
    RESULTS.append(out)
    for term in res.params.index:
        COEFS.append(dict(model=name,term=term,coefficient=float(res.params[term]),host_cluster_se=float(res.bse[term]),p=float(res.pvalues[term])))
    fits[name]=(res,f,reference)
    print(name,json.dumps({k:out[k] for k in ['n','long_n','difference_pp','ci_low_pp','ci_high_pp','odds_ratio','p_coefficient']},ensure_ascii=False),flush=True)
    return res,out

basic='bundle_registered ~ long_setting + C(region) + C(property_type) + C(accommodates)'
fit('M0_unadjusted',primary,'bundle_registered ~ long_setting')
fit('M1_primary',primary,basic)
fit('M1_CC_same_sample',complete,basic)
fit('M2_CC_bed_bath',complete,basic+' + C(bedrooms_band) + C(bathrooms_band)')
fit('S_HOST_equal_host',primary,basic,weighted=True,notes='모형 적합 표본에서 호스트당 가중치 합계 1. 장기 참조 표준화에도 같은 가중치 적용.')
fit('S_ALL_review_interaction',all_supported,'bundle_registered ~ long_setting * review_positive + C(region) + C(property_type) + C(accommodates)',notes='모든 최근 리뷰 상태를 포함한 보조 비교. long_setting 주계수는 리뷰0 집단.')

# Regional regressions are not forced when long-setting references are sparse.
eligibility=[]
for r,g in primary.groupby('region'):
    a=g[g.long_setting.eq(1)];b=g[g.long_setting.eq(0)]
    yes=int(a.bundle_registered.sum());no=len(a)-yes
    eligible=len(a)>=30 and a.host_id.nunique()>=20 and min(yes,no)>=10 and len(b)>=30 and b.host_id.nunique()>=20 and min(b.bundle_registered.sum(),len(b)-b.bundle_registered.sum())>=10
    row=dict(region=r,n_long=len(a),host_long=a.host_id.nunique(),bundle_yes_long=yes,bundle_no_long=no,n_shortmid=len(b),eligible=bool(eligible),reason='exploratory regional model' if eligible else '장기 참조의 숙소·호스트·결과값 최소 기준 미충족')
    eligibility.append(row)
    if eligible:
        fit('REGION_'+r,g,'bundle_registered ~ long_setting + C(property_type) + C(accommodates)',notes='지역별 탐색 추정. 비교 표본 수가 충분한 지역만 수행.')
pd.DataFrame(eligibility).to_csv(OUT/'지역별_회귀가능범위.csv',index=False,encoding='utf-8-sig')
from statsmodels.stats.multitest import multipletests
regional=[r for r in RESULTS if r['model'].startswith('REGION_')]
if regional:
    adj=multipletests([r['p_coefficient'] for r in regional],method='holm')[1]
    for r,pv in zip(regional,adj):r['regional_holm_p']=float(pv)

# Supported exact cells are a diagnostic, not a redefinition of the primary model.
strict=support[support.strict_support]
strict_n=primary.merge(strict[KEYS],on=KEYS,how='inner')
checks={
 'source_ids_unique':bool(d.id.is_unique),'source_n':len(d),'baseline_candidate_id_set_identical':True,'baseline_candidate_n':len(candidate),
 'all_entire_home':bool(primary.room_type.eq('Entire home/apt').all()),
 'primary_all_review_positive':bool(primary.number_of_reviews_ltm.gt(0).all()),
 'primary_7functions_observed':bool(primary[FUNC].notna().all().all()),
 'strict_support_cells':len(strict),'strict_support_long_n':int(strict_n.long_setting.sum()),'strict_support_shortmid_n':int(strict_n.long_setting.eq(0).sum()),
 'strict_support_regions':strict.region.unique().tolist(),
 'primary_common_cells':len(support[support.overlap]),
 'primary_long_missing_excluded':int(home[home.sector.eq(3)].shape[0]-positive[positive.sector.eq(3)].shape[0]),
 'primary_long_no_overlap_excluded':int(positive.long_setting.sum()-primary.long_setting.sum()),
 'sha256_source':hashlib.sha256(SRC.read_bytes()).hexdigest(),
 'python':sys.executable,'versions':{'numpy':np.__version__,'pandas':pd.__version__,'statsmodels':sm.__version__},
 'runtime_note':'Bundled Python lacks statsmodels/scipy. Existing system Python with those capabilities was used. No packages installed.',
 'same_complete_sample_models':True,'all_model_sandwich_independently_checked':True,
 'no_classification_clustering_or_revenue_model':True,
 'confidence_intervals':'Host-cluster robust Wald/delta intervals; covariate distribution used for standardization treated as fixed; no claim of random causal assignment.'
}
result={'done':True,'analysis_date':'2026-10-03','plan':plan,'checks':checks,'flow':flow,'regional_eligibility':eligibility,'models':RESULTS}
(OUT/'결과요약.tmp').write_text(json.dumps(result,ensure_ascii=False,indent=2))
(OUT/'결과요약.tmp').replace(OUT/'결과요약.json')
pd.DataFrame([{k:v for k,v in r.items() if k not in ['by_review_state','warnings']} for r in RESULTS]).to_csv(OUT/'회귀결과.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(COEFS).to_csv(OUT/'회귀계수.csv',index=False,encoding='utf-8-sig')
(OUT/'검산.json').write_text(json.dumps(checks,ensure_ascii=False,indent=2))
primary[['id','host_id','region','sector','property_type','accommodates','bundle_registered','long_setting']].to_csv(OUT/'회귀대상ID.csv.gz',index=False,compression='gzip')
print('DONE',json.dumps(checks,ensure_ascii=False),flush=True)
"""
SOURCE_META['regression'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_생활업무시설_조정회귀_20261003/analyze.py', 'sha256': 'edc1fc2b357b2b2e58b4d3d316c0c52cdc4db63a6b496678ad882399bda90c1b', 'language': 'python'}


### regression_report — 회귀 표·그림·HTML 생성
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_생활업무시설_조정회귀_20261003/report.py`
SHA256: `010c103617aee6e38ca6df2f7c14597be231f7fc0ed42a286932db1e26af22db`


In [19]:
SOURCE_CODE['regression_report'] = """from pathlib import Path
import json,html
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

P=Path(__file__).resolve().parent
d=json.loads((P/'결과요약.json').read_text())
m={r['model']:r for r in d['models']}
ids=pd.read_csv(P/'회귀대상ID.csv.gz',dtype={'id':'string','host_id':'string'})
base=P.parent/'7개지역_장기상품_통합전략_20261003'
cand=pd.read_csv(base/'7개지역_시설후보_ID.csv',dtype={'id':'string','host_id':'string'})
rawrates=ids.groupby('long_setting').bundle_registered.mean()
assert abs(rawrates[1]-m['M0_unadjusted']['adjusted_long_rate'])<1e-10
assert abs(rawrates[0]-m['M0_unadjusted']['adjusted_shortmid_rate'])<1e-10
assert all(r['converged'] and r['rank']==r['parameters'] for r in d['models'])
assert all(r['ci_low_pp']<=r['difference_pp']<=r['ci_high_pp'] for r in d['models'])
assert m['M1_CC_same_sample']['n']==m['M2_CC_bed_bath']['n']
overlap=int(cand.id.isin(ids.id).sum())
conclusion={
 'headline':'7개 지역에서 생활·업무 조합이 장기 설정 공급에 더 흔하다는 일반적 근거는 확보하지 못했다.',
 'primary':'비교 가능한 최근 리뷰 양수 집 전체 42,757곳(장기 216곳, 중단기 42,541곳). 조정 등록률 차이 +4.87%p, 95% 구간 -2.97~+12.71%p.',
 'bed_bath':'침실·욕실 추가 후 +3.89%p, -4.63~+12.42%p. 관측 표본 장기 184곳으로 축소. 동일 표본의 기본 조정은 +4.65%p.',
 'host_weight':'호스트마다 같은 총 가중치로 계산하면 +11.86%p, +3.66~+20.06%p. 주 분석과 가중 대상이 다르고 결과가 민감하므로 이 수치만 채택하지 않는다.',
 'regional':'런던·발렌시아만 사전에 정한 최소 기준에 따라 지역별 탐색 추정. 둘 다 95% 구간이 0 포함. 나머지 5지역 개별 회귀 결론 보류.',
 'all_review_states':'3+6 통합과 중단기형의 보조 분석은 전체 조정 차이 -3.06%p(-7.66~+1.55). 동일 모델의 리뷰양수 부분은 +4.59%p(-3.24~+12.41), 리뷰0 부분은 -5.95%p(-11.13~-0.76). 집단을 바꾸면 질문도 달라진다.',
 'keep':'23,222곳은 기존 고정 시설 기준의 접촉 후보로 재현됨. 실제 장기 성공·법적 적격 후보로 재명명하지 않는다.',
 'revise':'장기 설정 공급에서 흔한 조합을 모델로 입증했다는 설명은 채택하지 않는다. 취사·세탁·학업·업무 기능을 갖춘 공급에 지역별 상품을 제안하는 설계로 설명한다.',
 'not_proven':'등록시설의 무용성이나 장기상품 효과가 없다는 결론도 아니다. 실제 장기 예약·수요·수익에 관한 검증은 이번 결과가 제공하지 않는다.',
 'candidate_n_in_primary_model':overlap,
 'next':'기존 7지역 보고서의 모델 근거 문구를 수정하고 기능 기반 상품 제안과 관측 근거를 구분한다. 새로운 분류·군집화 우선 실행은 권하지 않는다.',
 'recommended_next_setting':'Sol / Extra high, 통합 한 세션. 현재 실행 모델 설정을 변경했다고 주장하지 않음.'
}
d['conclusion']=conclusion
d['checks']['unadjusted_rates_independently_reconciled']=True
d['checks']['candidate_primary_overlap_n']=overlap
(P/'결과요약.json').write_text(json.dumps(d,ensure_ascii=False,indent=2))
(P/'검산.json').write_text(json.dumps(d['checks'],ensure_ascii=False,indent=2))

fonts=font_manager.findSystemFonts()
chosen=next((f for f in fonts if 'AppleGothic' in f),None)
if chosen:
    font_manager.fontManager.addfont(chosen)
    plt.rcParams['font.family']=font_manager.FontProperties(fname=chosen).get_name()
plt.rcParams['axes.unicode_minus']=False
labels=['조정 전 · 같은 표본','지역·상품 조정 · 주 분석','침실·욕실 관측 · 기본 조정','침실·욕실 추가 조정','호스트 동일 가중 · 보조 분석']
names=['M0_unadjusted','M1_primary','M1_CC_same_sample','M2_CC_bed_bath','S_HOST_equal_host']
fig,ax=plt.subplots(figsize=(10,4.8))
for i,(name,label) in enumerate(zip(names,labels)):
    v=m[name];x=v['difference_pp'];color='#1b626b' if i!=4 else '#a66a18'
    ax.errorbar(x,4-i,xerr=[[x-v['ci_low_pp']],[v['ci_high_pp']-x]],fmt='o',capsize=5,color=color,markersize=7)
    ax.text(22,4-i,f"{x:+.1f} [{v['ci_low_pp']:+.1f}, {v['ci_high_pp']:+.1f}]",va='center',fontsize=10)
ax.axvline(0,color='#7b8790',linestyle='--',linewidth=1)
ax.set_yticks(range(5),labels[::-1]);ax.set_xlim(-8,34);ax.set_xticks([-5,0,5,10,15,20]);ax.set_ylim(-.7,4.7)
ax.set_xlabel('등록률 차이 (%p) · 장기 설정 - 중단기 설정')
ax.set_title('생활·업무 7기능 등록률 차이와 95% 신뢰구간',loc='left',pad=17,fontweight='bold')
for side in ['top','right','left']:ax.spines[side].set_visible(False)
ax.grid(axis='x',alpha=.15);fig.tight_layout();fig.savefig(P/'조정등록률차이.png',dpi=180,bbox_inches='tight');plt.close(fig)

def fmt(v,n=1):return f'{v:.{n}f}'
def ci(r):return f"[{r['ci_low_pp']:+.2f}, {r['ci_high_pp']:+.2f}]"
def row(r,label):
    return f"<tr><td>{label}</td><td>{r['n']:,} / {r['long_n']:,}</td><td>{r['adjusted_long_rate']*100:.2f}% / {r['adjusted_shortmid_rate']*100:.2f}%</td><td>{r['difference_pp']:+.2f}%p</td><td>{ci(r)}</td></tr>"
modeltable='<table><thead><tr><th>비교</th><th>전체 / 장기 n</th><th>장기 / 중단기 등록률</th><th>차이</th><th>95% 구간 (%p)</th></tr></thead><tbody>'+''.join(row(m[k],l) for k,l in zip(names,labels))+'</tbody></table>'
kn={'london':'런던','pays-basque':'프랑스 바스크','oslo':'오슬로','munich':'뮌헨','malaga':'말라가','sevilla':'세비야','valencia':'발렌시아'}
regional='<table><thead><tr><th>지역</th><th>장기 참조</th><th>장기 호스트</th><th>중단기 참조</th><th>개별 추정</th></tr></thead><tbody>'
for r in d['regional_eligibility']:
    mod=m.get('REGION_'+r['region']);val=f"{mod['difference_pp']:+.2f}%p {ci(mod)}" if mod else '표본 부족으로 보류'
    regional+=f"<tr><td>{kn[r['region']]}</td><td>{r['n_long']:,}</td><td>{r['host_long']:,}</td><td>{r['n_shortmid']:,}</td><td>{val}</td></tr>"
regional+='</tbody></table>'
report=f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>7개 지역 생활·업무 시설 조정 회귀</title><style>
body{{margin:0;background:#f4f7f7;color:#193139;font:16px/1.8 -apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo",sans-serif}}main{{max-width:1080px;margin:35px auto;padding:36px 42px;background:white;border-radius:16px}}h1{{font-size:30px;line-height:1.4}}h2{{font-size:21px;margin-top:36px}}.lead{{padding:22px;background:#edf6f4;border-left:5px solid #1b626b}}.meta{{color:#60727a;font-size:14px}}table{{width:100%;border-collapse:collapse;margin:18px 0;font-size:14px}}th,td{{padding:11px;text-align:left;border-bottom:1px solid #dce4e5;vertical-align:top}}th{{background:#edf2f3}}code{{background:#eef1f3;padding:2px 4px}}img{{max-width:100%;height:auto}}.warn{{background:#fff7e9;padding:18px}}a{{color:#176b77}}details{{margin:20px 0}}@media(max-width:700px){{main{{padding:22px 18px;margin:0}}.scroll{{overflow-x:auto}}table{{min-width:680px}}}}
</style></head><body><main><p class="meta">InsightStay · 2026-10-03 · 기존 7개 지역 보고서의 추가 회귀 검토</p>
<h1>생활·업무 시설 조합을<br>장기 설정 공급의 특징이라고 설명할 수 있는가?</h1>
<div class="lead"><strong>주 분석에서 뚜렷한 차이를 입증하지 못했다.</strong><br>지역·상품 조건 조정 후 +4.87%p였지만 95% 구간은 −2.97~+12.71%p였다. 호스트 가중치에 따라 결과가 달라 일반화에도 한계가 있다. 기존 23,222곳은 기능 기준의 접촉 후보로 유지할 수 있으나, 모델이 검증한 장기 성공 후보로 설명할 수 없다.</div>
<h2>1. 왜 했나</h2><p>기존 연관분석의 구성보정에는 이번 7개 지역이 포함되지 않았다. 현재 후보 조건인 생활·업무 조합이 실제로 이 7개 지역의 장기 설정 공급에서 더 흔한지 추가로 확인했다. 새 시설 묶음이나 새 분류·군집 모델을 찾는 작업은 하지 않았다.</p>
<h2>2. 무엇을 비교했나</h2><p>런던·프랑스 바스크·오슬로·뮌헨·말라가·세비야·발렌시아에서 <code>source == 0</code>, <code>availability_365 &gt; 0</code>, <code>room_type == "Entire home/apt"</code>인 숙소를 사용했다. 주 분석은 최근 리뷰 양수인 <strong>3번(장기)과 1·2번(중단기)</strong>의 비교다. 리뷰 유무에 따른 등록 차이를 장기 설정 특징과 혼동하지 않기 위해서다.</p>
<p>장기 집 전체 220곳 중 시설 관측이 안 된 3곳을 제외해 217곳이 남았다. 같은 지역·숙소 종류·수용인원에 비교 상대가 없는 1곳을 더 제외한 <strong>장기 216곳과 중단기 42,541곳</strong>이 주 분석 대상이다. 장기 호스트는 155명이다. 표본 수는 최종 후보 수가 아니라 시설 미등록 숙소도 포함한 비교 표본 수다.</p>
<h2>3. 회귀에서 무엇을 설명했나</h2><div class="scroll"><table><tr><th>역할</th><th>실제 사용</th><th>이유</th></tr><tr><td>종속변수</td><td>파생변수 <code>bundle_registered</code>: 주방·가열기기·냉장고·조리도구와 식기·세탁기·Wi-Fi·업무공간 모두 등록하면 1</td><td>시설 조합의 등록 여부를 설명</td></tr><tr><td>핵심 독립변수</td><td>파생변수 <code>long_setting</code>: <code>minimum_nights &gt;= 30</code>이면 1</td><td>장기 설정과 중단기 설정의 차이</td></tr><tr><td>기본 조정</td><td><code>region</code>, <code>property_type</code>, <code>accommodates</code>를 범주형으로 포함</td><td>지역과 상품 구성이 다른 영향을 구분</td></tr><tr><td>이미 고정한 조건</td><td><code>room_type</code>: 집 전체만 사용</td><td>값이 같으므로 회귀에 중복 투입하지 않음</td></tr><tr><td>추가 조정</td><td><code>bedrooms_observed</code>, <code>bathrooms_observed</code></td><td>원본 침실·욕실의 관측값만 사용한 파생 컬럼. 결측 표본 제외</td></tr><tr><td>불확실성</td><td><code>host_id</code> 단위 군집 강건 표준오차</td><td>같은 호스트의 숙소를 완전히 독립인 자료로 취급하지 않음. 호스트 특성 자체를 통제했다는 뜻은 아님</td></tr></table></div>
<p>0/1 결과이므로 이항 로지스틱 회귀를 썼다. 예측 분류기를 만드는 목적이 아니므로 AP·AUC로 채택 여부를 고르지 않았다. 장기 참조 숙소의 상품 구성에 맞춰 두 설정 집단의 모형상 등록률을 계산하고 그 차이와 신뢰구간을 보고했다. 이 계산은 현재 집단 간 관련성 비교이며 minimum_nights를 바꾸는 인과효과가 아니다.</p>
<h2>4. 결과</h2><div class="scroll">{modeltable}</div><img src="조정등록률차이.png" alt="주 분석과 침실·욕실 추가 분석의 신뢰구간은 0을 포함하고 호스트 동일 가중 분석은 양의 차이를 보인다.">
<p>침실·욕실을 추가하면 표본이 줄어든다. 따라서 같은 관측 표본의 기본 조정(+4.65%p)과 추가 조정(+3.89%p)을 나란히 비교했다. 감소분 전체를 침실·욕실의 영향이라고 해석하지 않았다. 두 결과 모두 0을 포함한다.</p>
<div class="warn"><strong>0을 포함한다는 뜻:</strong>차이가 없다고 입증한 것은 아니다. 현재 표본과 모형으로는 양의 차이가 분명하다고 결론내리기 어렵다. 큰 양의 차이도 일부 포함하는 넓은 구간이므로 효과의 부재를 확정하지 않는다.</div>
<h2>5. 호스트 비중을 바꾸면 왜 달라지나</h2><p>주 분석은 숙소 한 곳마다 같은 비중을 준다. 보조 분석은 해당 비교 표본에서 한 호스트의 모든 숙소 가중치 합계를 1로 만든다. 다숙소 호스트가 결과에 미치는 비중이 줄고 표준화 기준도 바뀐다. 보조 분석에서는 +11.86%p가 남았지만, 이 결과만 골라 전체 숙소에 적용할 수 없다. 숙소 중심과 호스트 중심 분석의 민감성을 함께 공개한다.</p>
<h2>6. 지역별로 말할 수 있는 범위</h2><div class="scroll">{regional}</div><p>지역별 탐색 회귀는 장기 표본 30곳·호스트 20명 이상, 시설 조합 등록/미등록 각각 최소 10곳 등 계수 계산 전 고정한 최소 기준을 충족한 런던과 발렌시아만 실행했다. 이 기준 자체가 검증력을 보장하지 않는다. 두 지역 모두 신뢰구간이 0을 포함했고, 두 지역의 계수 검정에 Holm 보정을 적용해도 유의하지 않았다.</p>
<p>더 엄격하게 정확 상품층마다 양측 20숙소·5호스트 이상, 한 호스트 비중 20% 이하를 요구하면 런던의 1개 층(장기 28곳·중단기 5,138곳)만 남는다. 주 회귀의 60개 공통 상품층은 단순히 양쪽 표본이 존재하는 범위이며, 모두 충분한 비교 표본을 확보한 층이라는 뜻이 아니다.</p>
<h2>7. 최근 리뷰가 없는 장기 설정까지 합치면</h2><p>3·6번과 1·2·4·5번을 비교하는 보조 회귀에는 최근 리뷰 유무와 장기 설정의 상호작용을 넣었다. 전체 조정 차이는 −3.06%p [−7.66, +1.55]였다. 같은 보조 모형의 리뷰 양수 부분은 +4.59%p [−3.24, +12.41], 리뷰 0 부분은 −5.95%p [−11.13, −0.76]였다. 참조 집단을 바꾸면 결과가 달라지므로 장기 설정 공급 전체의 보편적 특징이라는 설명을 뒷받침하지 않는다. 최근 리뷰 양수가 실제 장기 예약의 성공을 뜻하지 않는다는 한계도 그대로다.</p>
<h2>8. 전략에 어떻게 반영하나</h2><p><strong>유지:</strong>취사·세탁·학업·업무 기능을 등록한 공급을 찾고, 지역별 제도와 상품 조건을 연결한다. 기존 23,222개 후보 ID는 이번에도 모두 동일하게 재현됐다.</p><p><strong>채택하지 않을 설명:</strong>“장기 숙소의 성공 시설 조합을 회귀로 입증했으며, 같은 조합을 갖춘 중단기 숙소의 장기 성과도 높을 것이다.” 이번 회귀는 이 주장을 제공하지 않는다.</p><p><strong>발표에 쓸 수 있는 문장:</strong>“생활·업무 시설을 갖춘 공급을 기능 기준으로 선별했다. 선정 7개 지역의 장기 설정 공급과도 비교했지만, 숙소 단위 조정 분석에서는 조합 등록률 차이가 명확하지 않아 장기 특화나 성공 예측으로 해석하지 않았다.”</p>
<p>주 분석에 실제 들어간 기존 후보는 리뷰 양수 12,867곳이다. 23,222곳 전체에 대한 개별 검증이나 적격 판정은 아니다. 주소별 허가, 실제 시설 품질, 연속 빈 기간, 참여 의사, 장기 수요와 수익은 이번 분석의 범위 밖이다.</p>
<details><summary>추정·검산 세부사항</summary><p>모든 회귀는 수렴했고 설계행렬은 완전 순위다. 숙소 수와 고정 후보 ID를 기존 보고서와 대조했다. 조정 전 등록률은 직접 집계와 일치했다. host_id별 점수 합과 정보행렬로 sandwich 공분산을 독립 재계산해 라이브러리 결과와 일치하는지 확인했다. 가중 회귀의 라이브러리 경고는 이 별도 계산으로 점검했다.</p><p>침실은 1/2/3/4+ 구간, 욕실은 ≤0.5/1/1.5/2/2.5/3+ 구간으로 포함했다. 완전 관측 표본에서 비교 상대가 남는지 다시 확인했다. 전체 시설 수는 종속변수의 구성요소를 포함하므로 주요 조정 변수에 넣지 않았다. 표준화 대상의 공변량 분포는 고정으로 보고 delta-method 신뢰구간을 계산했다.</p><p>주 모형의 McFadden 의사 R²는 {m['M1_primary']['mcfadden_pseudo_r2']:.4f}, 침실·욕실 추가 모형은 {m['M2_CC_bed_bath']['mcfadden_pseudo_r2']:.4f}다. 선형 회귀의 설명 분산 비율 R²와 같은 뜻이 아니며, 표본도 달라 두 수치를 성능 향상으로 직접 비교하지 않는다.</p></details>
<h2>9. 다음 단계</h2><p>새 분류·군집화를 반복하기보다 기존 보고서에 이번 결과를 반영하고 시설 기준을 기능 요구사항으로 설명하는 것이 적절하다. 필요한 다음 작업은 통합 문구 수정이며 Sol / Extra high 한 세션을 권장한다. 기존 통합 HTML은 이번 작업에서 덮어쓰지 않았다.</p>
<p><a href="회귀결과.csv">회귀 수치표</a> · <a href="지역별_회귀가능범위.csv">지역별 표본 범위</a> · <a href="결과요약.json">상세 결과·방법</a></p></main></body></html>'''
(P/'조정회귀_결과보고서.html').write_text(report)
print('Report created. Numeric checks passed. Candidate overlap:',overlap)
"""
SOURCE_META['regression_report'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_생활업무시설_조정회귀_20261003/report.py', 'sha256': '010c103617aee6e38ca6df2f7c14597be231f7fc0ed42a286932db1e26af22db', 'language': 'python'}


### report — 7지역 후보 검산·원지표 연결·지역별 HTML
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_seven_region_report.py`
SHA256: `766cd5de9e16a889b099b46e5e6ed44ab54ef475f2e2ea5d8960928f7b0100ac`


In [20]:
SOURCE_CODE['report'] = """from pathlib import Path
import json, html, shutil, hashlib
import pandas as pd

ROOT = Path('/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities')
OUT = ROOT / 'outputs/7개지역_장기상품_통합전략_20261003'
OUT.mkdir(parents=True, exist_ok=True)
BASE = Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs')
ASSOC = BASE / 'InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
R = pd.read_csv(ASSOC/'3번단독_원지표.csv')
D = pd.read_csv(ASSOC/'3번단독_연관보정.csv')
S = pd.read_csv(ASSOC/'3번단독_층별근거.csv')
V = pd.read_csv(ASSOC/'묶음_탐색확인.csv')
J = json.loads((ASSOC/'3번단독_결과요약.json').read_text())
PD = pd.read_csv(ASSOC/'연관보정_집계.csv')
PS = pd.read_csv(ASSOC/'연관보정_층별근거.csv')
PJ = json.loads((ASSOC/'연관보정_결과요약.json').read_text())
C = pd.read_pickle(ROOT/'work/seven_region_candidates.pkl')
F = pd.read_csv(ROOT/'outputs/규제혜택_생활업무_주력후보_20261003/53지역_집전체_생활업무_후보수.csv')
regions = ['london','pays-basque','oslo','munich','malaga','sevilla','valencia']
names = dict(zip(regions,['런던','프랑스 바스크','오슬로','뮌헨','말라가','세비야','발렌시아']))
expected = dict(zip(regions,[13317,1563,1700,606,2266,2107,1663]))
F = F.set_index('region').loc[regions].reset_index()
assert C.id.is_unique and len(C)==23222 and C.host_id.nunique()==13255
assert C.groupby('region').size().to_dict()==expected
assert C.source.eq(0).all() and C.availability_365.gt(0).all()
assert C.sector.isin([1,2,4,5]).all() and C.room_type.eq('Entire home/apt').all()
fc = [c for c in C if c.startswith('function__')]
assert len(fc)==7 and C[fc].eq(1).all().all()
assert not any('maximum_nights' in c or 'long_term_stays' in c for c in C)
sensitivity_regions=sorted(S.loc[S.primary & S.eligible,'region'].unique())
common_regions=sorted(PS.loc[PS.primary & PS.eligible,'region'].unique())
assert not set(regions)&set(sensitivity_regions)
assert len(sensitivity_regions)==12 and len(common_regions)==27
assert sorted(set(regions)&set(common_regions))==['london','valencia']
for r in R.loc[R.weighting.eq('listing')].itertuples():
    assert abs(r.support-r.nAB/r.observed_n)<1e-10
    assert abs(r.confidence-r.nAB/r.nA)<1e-10
    assert abs(r.lift-r.confidence/r.pB)<1e-10
for r in PD.loc[PD.primary].itertuples():
    assert abs(r.confidence_standardized-r.support_standardized/r.pA_standardized)<1e-10
    assert abs(r.lift_standardized-r.support_standardized/(r.pA_standardized*r.pB_standardized))<1e-10
    assert abs(r.lift_within_strata-r.support_standardized/r.independent_support_within_strata)<1e-10
assert len(PS.loc[PS.primary & PS.eligible])==163

def e(x): return html.escape(str(x))
def n(x): return f'{int(x):,}'
def pct(x): return f'{100*x:.1f}%'
def table(head, rows):
    return '<div class="table-wrap"><table><thead><tr>'+''.join('<th>'+e(c)+'</th>' for c in head)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+e(c)+'</td>' for c in row)+'</tr>' for row in rows)+'</tbody></table></div>'
def raw(rule,pool='sector3'):
    return R.loc[(R.rule==rule)&(R.pool==pool)&R.weighting.eq('listing')&R.split.eq('all')].iloc[0]
def adj(rule,pool='sector3',weight='listing'):
    return D.loc[(D.rule==rule)&(D.pool==pool)&D.weighting.eq(weight)&D.primary].iloc[0]
def pooled_adj(rule,pool='reference',weight='listing'):
    return PD.loc[(PD.rule==rule)&(PD.pool==pool)&PD.weighting.eq(weight)&PD.primary].iloc[0]
def link(url,title): return f'<a href="{html.escape(url,quote=True)}" target="_blank" rel="noopener">{e(title)} ↗</a>'

src = {
 'london':('https://www.cityoflondon.gov.uk/services/planning/planning-enforcement/short-term-letting','City of London · 단기임대와 90박'),
 'camden':('https://www.camden.gov.uk/short-term-lettings-planning-permission','Camden · 단기임대 계획허가'),
 'renters':('https://www.gov.uk/guidance/renters-rights-act-an-overview-for-landlords','영국 정부 · Renters’ Rights Act'),
 'basque':('https://www.communaute-paysbasque.fr/logement-et-urbanisme/la-location-dun-meuble-de-tourisme-au-pays-basque','바스크 광역연합 · 관광가구주택 안내'),
 'basque_rule':('https://www.communaute-paysbasque.fr/fileadmin/user_upload/mediatheque/Tourisme/CAPaysbasqueCDU_Rglt_et_Annexes_modifie2025.pdf','바스크 · 현행 용도변경 규정 및 혼합형 부속서'),
 'oslo':('https://www.regjeringen.no/no/dokumenter/-24-25-og-46-departementet-svarer-pa-sporsmal-om-fremutleie-av-eierseksjoner/id2850243/','노르웨이 정부 · 구분소유 주택 단기임대 해석'),
 'oslo_days':('https://www.regjeringen.no/no/dokumenter/prop.-36-l-20182019/id2621295/?ch=10','노르웨이 정부 · 연 90일·연속 30일 경계 설명'),
 'oslo_use':('https://www.regjeringen.no/no/dokumenter/veileder-om-korttidsutleie/id3153672/?ch=3','노르웨이 정부 · 2026 단기임대와 실제 용도 안내'),
 'oslo_lease':('https://www.htu.no/artikler/om-tidsbestemte-og-tidsubestemte-leiekontrakter','노르웨이 임대차분쟁위원회 · 계약기간'),
 'munich':('https://stadt.muenchen.de/rathaus/stadtrecht/vorschrift/970/version2/0.html','뮌헨 · 주거용도 보호조례 §4'),
 'munich_city':('https://stadt.muenchen.de/service/info/fachbereich-bestandssicherung/1076745/','뮌헨 · 휴가임대와 8주 안내'),
 'munich_case':('https://www.gesetze-bayern.de/Content/Document/Y-300-Z-BECKRS-B-2024-N-18848','바이에른 고등행정법원 · 12 B 23.2195'),
 'bgb549':('https://www.gesetze-im-internet.de/bgb/__549.html','독일 민법 §549 · 실제 임시사용'),
 'bgb575':('https://www.gesetze-im-internet.de/bgb/__575.html','독일 민법 §575 · 유기 주거계약'),
 'andalusia':('https://www.juntadeandalucia.es/boja/2024/24/50','안달루시아 · Decreto 31/2024'),
 'andalusia_pdf':('https://www.juntadeandalucia.es/sites/default/files/inline-files/2025/04/Decreto%2028-2016%20de%202%20de%20febrero%20de%20las%20viviendas%20de%20uso%20turistico.pdf','안달루시아 · 관광주택 통합 규정'),
 'andalusia_faq':('https://www.juntadeandalucia.es/organismos/turismojusticiadesregulacionyadministracionlocal/areas/turismo/registro-turismo/establecimientos-servicios/paginas/faq-viviendas-turismo.html','안달루시아 · 운영기간·주거임대 공식 FAQ'),
 'malaga':('https://www.malaga.eu/visorcontenido/ANUDocumentDisplayer/178062/COMUNICADO.pdf','말라가 · 2025 신규 관광주택 중단 공고'),
 'malaga2026':('https://www.malaga.eu/el-ayuntamiento/notas-de-prensa/detalle-de-la-nota-de-prensa/index.html?id=178224','말라가 · 2026 주거·숙박 용도 계획 변경 안내'),
 'sevilla':('https://www.urbanismosevilla.org/paginas/limitacion-viviendas-uso-turistico','세비야 · 108개 동네별 관광주택 10% 상한'),
 'sevilla_use':('https://www.urbanismosevilla.org/areas/atencion-al-ciudadano/guia-de-tramitacion/viviendas-fines-turisticos','세비야 · 관광주택의 숙박 용도 및 절차'),
 'valencia':('https://www.boe.es/buscar/act.php?id=BOE-A-2018-8950&p=20260702&tn=1','발렌시아 관광법 · 제65조·제78조'),
 'spain':('https://www.boe.es/buscar/act.php?id=BOE-A-1994-26003&p=20261001&tn=1','스페인 주택임대차법 · 제7조·제9bis조'),
}
def cites(*keys): return '<p class="sources">공식 근거 · '+' · '.join(link(*src[k]) for k in keys)+'</p>'

sectors=table(['섹터','최소 숙박 설정','최근 12개월 리뷰','숙소 수','이번 역할'],[
 [1,'3박 이하','있음','418,643','상품 추가 후보'],[2,'4~29박','있음','43,259','상품 추가 후보'],
 [3,'30박 이상','있음','9,555','3+6 기본 참조 · 단독 보조 검증'],[4,'3박 이하','없음','144,542','상품 추가 후보'],
 [5,'4~29박','없음','33,330','상품 추가 후보'],[6,'30박 이상','없음','28,349','3+6 기본 참조에 포함']])
facilities=table(['분류','기능','완전 등록의 기준','장기상품에서의 역할'],[
 ['생활 L','주방','주방 등록','일상적인 식사 준비'],['생활 L','음식 가열기기','스토브·오븐·전자레인지 중 하나 이상','조리·재가열'],
 ['생활 L','냉장고','냉장고 등록','식재료 보관'],['생활 L','조리도구·식기','조리 기본도구 AND 식기·수저','조리와 식사'],
 ['생활 L','세탁기','세탁기 등록','반복적인 의류 세탁'],['업무 W','Wi-Fi','Wi-Fi 등록','통신·온라인 수업·업무'],
 ['업무 W','전용 업무공간','전용 작업공간 등록','책상에서 업무·학습']])
rnames={'R1':'식생활 4기능 → 세탁기','R2':'Wi-Fi → 전용 업무공간','R3':'생활 5기능 → 업무 2기능'}
pnames={'pooled':'3+6 장기 설정 통합','target':'1·2·4·5 중단기형','sector3':'3번 단독 · 보조 검증'}
raw_rows=[]
for rid in rnames:
    for pool in pnames:
        x=raw(rid,pool)
        raw_rows.append([rnames[rid],pnames[pool],n(x.nAB),pct(x.support),pct(x.confidence),f'{x.lift:.2f}'])
raw_table=table(['규칙 A → B','집단','A·B 동시등록','지지도','신뢰도','향상도'],raw_rows)
pooled_names={'reference':'3+6 장기 설정 통합','target':'1·2·4·5 중단기형'}
adj_table=table(['집단','보정 지지도','보정 신뢰도','보정 향상도','층내 기대 대비 향상도'],[
 [pooled_names[p],pct(pooled_adj('R3',p).support_standardized),pct(pooled_adj('R3',p).confidence_standardized),f'{pooled_adj("R3",p).lift_standardized:.2f}',f'{pooled_adj("R3",p).lift_within_strata:.2f}'] for p in pooled_names])
cover=next(x for x in PJ['cases'] if x['primary'])['coverage']
coverage_table=table(['집단','원지표의 관측 분모','보정 비교 포함','포함률'],[[pooled_names[p],n(cover[p]['all_observed_n']),n(cover[p]['included_observed_n']),pct(cover[p]['observed_listing_coverage'])] for p in pooled_names])
sensitivity_table=table(['집단 · 3번 기준 공통 40층','보정 신뢰도','보정 향상도','층내 기대 대비 향상도'],[
 [pnames[p],pct(adj('R3',p).confidence_standardized),f'{adj("R3",p).lift_standardized:.2f}',f'{adj("R3",p).lift_within_strata:.2f}'] for p in pnames])
val_rows=[]
for b,label in [('L','생활형'),('W','업무형'),('LW','생활·업무형')]:
    tr=V[(V.bundle==b)&V.phase.eq('train')].iloc[0];va=V[(V.bundle==b)&V.phase.eq('validation')].iloc[0]
    val_rows.append([label,pct(tr.support_observed),pct(va.support_observed),pct(tr.host_weighted_support_observed),pct(va.host_weighted_support_observed)])
val_table=table(['사전 정의 묶음','탐색 지지도','확인 지지도','탐색 호스트가중','확인 호스트가중'],val_rows)
candidate_table=table(['지역','중단기 집 전체','생활·업무 후보','후보 비율','후보 호스트'],[[names[x.region],n(x.entire_home_n),n(x.candidate_n),pct(x.candidate_share_of_entire_home),n(x.candidate_hosts_n)] for x in F.itertuples()])

analysis = f'''
<section id="analysis"><div class="eyebrow">01 · 시설에서 후보까지</div><h2>연관분석은 무엇을 확인했고, 후보는 어떻게 정했나</h2>
<p class="lead">생활형과 업무형은 시설의 기능을 기준으로 먼저 정의했다. 연관분석으로 그 기능들이 실제 공급에서 얼마나 함께 등록되는지 확인하고, 최종적으로는 <strong>생활 5기능과 업무 2기능을 모두 갖춘 집 전체</strong>를 선별했다.</p>
<h3>1. 3·6번을 기본 참조로, 3번 단독을 보조 검증으로 본다</h3>
<p>분석 모집단은 기존 공통 분석표에서 <code>source=0</code>, <code>availability_365&gt;0</code>인 677,678개 숙소다. source 코드는 기존 데이터의 표본 구분을 유지했다. 최소 숙박 설정과 최근 리뷰 유무로 아래 여섯 집단을 구분했다.</p>{sectors}
<p><strong>장기 숙박으로 설정된 공급의 시설 구성을 참고한다</strong>는 목적에 맞춰 3·6번 통합 37,904곳을 기본 참조 집단으로 유지한다. 최근 리뷰는 실제 장기 예약의 성공·실패를 판별하지 못하므로, 리뷰가 없는 6번도 포함한다. 이후 수행한 3번 단독 분석은 ‘최근 리뷰가 있는 공급만 보아도 같은 해석이 유지되는가’를 확인하는 보조 검증이다.</p>
<h3>2. 생활형·업무형·생활·업무형은 서로 어떻게 다른가</h3>{facilities}
<div class="bundles"><div><b>생활형 L</b><p>주방 + 가열기기 + 냉장고 + 조리도구·식기 + 세탁기</p><small>먹고, 보관하고, 씻어 쓰는 생활의 기본 기능</small></div><div><b>업무형 W</b><p>Wi-Fi + 전용 업무공간</p><small>일하거나 공부할 수 있는 기능</small></div><div class="selected"><b>최종 후보: 생활·업무형 L∩W</b><p>생활 5기능 + 업무 2기능</p><small>총 7기능을 모두 완전 등록한 숙소</small></div></div>
<p>업무형에는 생활형 시설이 없어도 포함될 수 있고, 생활형에도 업무시설이 없어도 포함될 수 있다. 즉, 서로 배타적인 세 군집이 아니라 <strong>두 기능 묶음과 그 교집합</strong>이다. 최종 주력 후보는 교집합만 사용한다.</p>
<p>여기서 ‘7기능’은 시설 원항목 7개와 정확히 같지는 않다. 가열기기는 여러 기기 중 하나면 충족하지만, 조리도구·식기는 두 원항목이 모두 있어야 충족한다. 도구 관련 한 항목만 등록된 0.5는 완전 충족으로 세지 않는다. 0은 미등록, 결측은 판단 불가이며 실제 시설 부재와 구분했다. Wi-Fi 속도나 작업공간의 품질까지 확인한 것은 아니다.</p>
<h3>3. 이번에 확인한 A → B 조합은 정확히 3개다</h3>
{table(['규칙','A: 먼저 갖춘 조건','B: 함께 갖추었는지 확인한 조건','확인 목적'],[
 ['R1','주방·가열기기·냉장고·조리도구·식기','세탁기','식생활과 세탁이 함께 갖춰지는가'],
 ['R2','Wi-Fi','전용 업무공간','통신과 작업공간이 함께 갖춰지는가'],
 ['R3','생활형 5기능 전체','Wi-Fi와 전용 업무공간 모두','생활과 업무가 결합되는가']])}
<p><strong>시설 묶음 3개와 방향성 규칙 3개를 사전에 고정</strong>했다. 모든 시설 조합을 자동 탐색한 뒤 최고의 조합을 골랐거나, 군집 알고리즘이 생활형·업무형을 발견한 분석은 아니다. A → B의 화살표는 조건부 비율을 계산하는 방향이며 A가 B를 발생시킨다는 뜻도 아니다.</p>
<h3>4. 지지도·신뢰도·향상도를 실제 숫자로 읽기</h3>
<p>3·6번 통합 집단 중 7기능의 등록 상태를 관측할 수 있는 숙소는 <strong>37,701곳</strong>이다. 이 중 생활형 A는 <strong>12,263곳</strong>, 업무형 B는 <strong>15,531곳</strong>, 둘 다 갖춘 A∩B는 <strong>7,239곳</strong>이다. 관측이 불가능한 203곳은 지표 분모에서 제외했다.</p>
<div class="metric-grid"><div><b>지지도 19.2%</b><code>7,239 ÷ 37,701</code><p>전체 관측 숙소 중 생활·업무가 모두 등록된 비율. 이 조합의 공급 규모를 설명한다.</p></div><div><b>신뢰도 59.0%</b><code>7,239 ÷ 12,263</code><p>생활형 숙소 중 업무시설도 갖춘 비율. 통계적 확신이나 예약 성공률이라는 뜻은 아니다.</p></div><div><b>향상도 1.43</b><code>59.0% ÷ (15,531 ÷ 37,701)</code><p>같은 집단의 업무형 기본 등록률 41.2%와 비교한다. 생활형에서 업무형이 약 1.43배 자주 관측된다.</p></div></div>
<p>향상도 1은 두 묶음이 독립적으로 등록되는 경우의 기준이다. 1보다 크면 양의 등록 연관, 1보다 작으면 그 반대다. <strong>1.43은 같은 집단의 독립등록 기준 대비 약 43% 높은 결합을 뜻한다.</strong> 중단기형보다 43% 높다거나 예약·매출이 43% 증가한다는 의미는 아니다.</p>
<h3>5. 세 규칙을 세 집단에서 비교한 원지표</h3>
<p>원지표 분모는 3+6번 37,701곳, 중단기형 636,938곳, 보조 검증인 3번 9,516곳이다. 같은 시설 관측 기준과 지표 정의로 계산한 숙소 단위 결과다.</p>{raw_table}
<p><strong>생활형 → 업무형의 향상도는 장기 설정 통합 집단 1.43, 중단기형 1.16으로, 장기 설정 통합 집단에서 더 높게 관측됐다.</strong> 이번 주장을 뒷받침하는 핵심 비교다. 다만 지지도는 19.2%와 21.3%로 통합 장기형이 오히려 낮다. ‘장기형에 생활·업무 시설이 더 많이 보급돼 있다’와 ‘각 집단의 기본 등록률 대비 결합이 더 강하다’는 서로 다른 주장이다.</p>
<p>다른 규칙까지 모두 장기형의 향상도가 높았던 것은 아니다. 식생활 → 세탁기는 장기 설정 통합 1.13, 중단기형 1.14이고, Wi-Fi → 업무공간은 1.09와 1.03이다. 따라서 결론의 범위는 <strong>생활 5기능과 업무 2기능의 결합</strong>으로 명확히 한정한다.</p>
<h3>6. 지역·숙소 구성을 맞춘 뒤에도 같은 해석인가</h3>
<p>지역 차이가 섞이는 문제를 줄이기 위해 <strong>지역 × 객실 형태 × 숙소 종류 × 수용인원 구간</strong>을 동시에 맞췄다. 3+6번과 중단기형 각각 시설 관측 숙소가 30곳 이상인 공통층 163개를 사용하고, 두 집단 모두 같은 <strong>3+6번의 공통층 구성비</strong>로 다시 계산했다. 3번 단독 보조 검증의 40개 공통층 분석과 구분된다.</p>
<p>핵심 규칙인 ‘생활형 → 업무형’의 결과는 아래와 같다. 표준화 향상도와 층내 기대 대비 향상도는 계산 기준이 다르므로 나란히 구분했다.</p>{adj_table}
<p>공통 구성에서의 향상도는 <strong>장기 설정 통합 1.43, 중단기형 1.17</strong>로 차이가 유지된다. 같은 세부 조건 안에서 독립적으로 등록될 때의 기대 동시등록을 기준으로 삼으면 각각 <strong>1.26과 1.16</strong>이다. 다숙소 호스트의 영향을 줄인 가중 분석에서도 층내 기대 대비 향상도는 1.24와 1.13으로 같은 방향이다. 지표의 계산 기준이 다르므로 ‘보정 후 1.43’과 ‘층내 기준 1.26’을 같은 지표처럼 혼용하지 않는다.</p>
<div class="callout"><b>본문에서 사용할 결론</b><p>장기 설정 공급인 3·6번 통합 집단에서 생활시설과 업무시설 결합의 향상도는 1.43으로, 중단기형의 1.16보다 높게 관측됐다. 지역·숙소 구성을 맞춘 비교에서도 같은 방향이 유지됐다. 이 결과를 생활·업무 기능 묶음의 참조 근거로 활용하고, 실제 상품 제안 후보는 집 전체와 7기능의 <strong>고정 조건</strong>으로 선별했다.</p></div>
<p><strong>3번 단독에서는 차이가 유지되지 않았다.</strong> 원향상도는 3번과 중단기형 모두 약 1.16이며, 3번 기준으로 보정한 별도 비교에서도 비슷했다. 따라서 통합 장기 설정 공급의 관측 결과로 표현하고, 최근 리뷰가 있는 장기 공급 전체에 같은 차이가 확립됐다고 확대하지 않는다. 통계적 유의성이나 장기 예약 성공·고객 선호를 입증한 결과도 아니다.</p>
{coverage_table}
<p><strong>보정 비교는 27개 지역의 공통층에 한정된다.</strong> 이번 전략의 7개 지역 중 런던과 발렌시아의 일부 층이 포함되며, 나머지 5개 지역은 포함되지 않는다. 이것도 7개 지역 각각에서 더 높은 향상도를 입증한 결과는 아니다. 지역별 시설 후보 수는 직접 집계하고, 보정된 연관성은 전체 분석의 참조 근거로 활용한다.</p>
<details><summary>보정 계산식과 포함 지역 보기</summary><p>공통층별 가중치를 q, 각 층의 등록률을 P로 쓰면 지지도 = ΣqP(A∩B), 신뢰도 = ΣqP(A∩B)/ΣqP(A), 표준화 향상도 = ΣqP(A∩B)/[ΣqP(A) × ΣqP(B)]다. 층내 기대 대비 향상도는 ΣqP(A∩B)/Σq[P(A)P(B)]다. 층별 향상도를 단순 평균하지 않았다.</p><p>q는 공통층 안의 3+6번 구성비다. 호스트 가중치는 각 전체 집단 안에서 1/해당 호스트 숙소 수로 계산하고, q도 통합 참조 집단의 호스트가중 구성비를 사용했다. 보정 포함 지역: {e(', '.join(common_regions))}.</p></details>
<details><summary>3번 단독 보조 검증: 통합 결과가 그대로 유지되는가</summary><p>최근 리뷰가 있는 3번만 보면 원지지도 33.5%, 신뢰도 64.8%, 향상도 1.16이다. 3번과 중단기형 각각 관측 30곳 이상인 12지역·40개 공통층에서 3번 구성비로 표준화한 결과는 아래와 같다. 통합 3+6번도 이 표 안에서는 같은 40개 층으로 제한했다.</p>{sensitivity_table}<p>3번과 중단기형의 보정 향상도는 약 1.16으로 비슷하고, 층내 기준은 1.15와 1.17이다. 호스트가중 층내 향상도도 둘 다 약 1.12다. 기본 분석인 통합 163층 비교와 가중치·포함 표본이 다르므로 그 표와 숫자를 직접 이어 비교하지 않는다.</p><p>이 보조 검증에는 이번 전략의 7개 지역이 포함되지 않는다. 기존 보고서의 ‘12지역이며 7개 지역은 미포함’ 설명은 이 3번 단독 보조 검증의 범위다. 3번 관측 표본은 런던 120, 바스크 29, 오슬로 30, 뮌헨 17, 말라가 9, 세비야 4, 발렌시아 88곳이다.</p></details>
<details><summary>초기 3+6 분석의 내부 안정성 확인 보기</summary><p>시설 묶음을 정한 뒤 기존 탐색·확인 분할에서 등록 비중이 크게 달라지는지 확인했다. 탐색 관측 분모는 26,658, 확인 관측 분모는 5,939다.</p>{val_table}<p>사전 기준은 탐색 지지도 10% 이상·500숙소 이상·200호스트 이상, 확인 지지도 10% 이상·100숙소 이상·50호스트 이상이었다. 호스트 가중 지지도도 각 10% 이상, 두 분할 차이 5%p 이내·비율 0.8~1.25 조건을 세 묶음 모두 통과했다. 기존 데이터와 분할을 사용한 내부 안정성 확인이며 새로운 외부 표본의 검증은 아니다. 이 절차는 3번 단독 보정과도 구분된다.</p></details>
<h3>7. 시설 기준을 실제 상품 제안 후보로 연결하기</h3>
<div class="flow"><div><b>639,774</b><span>1·2·4·5번</span></div><i>→</i><div><b>135,910</b><span>생활·업무 7기능</span></div><i>→</i><div><b>123,030</b><span>집 전체 · 53지역</span></div><i>→</i><div class="accent"><b>23,222</b><span>이번 7지역</span></div></div>
<p>기존 관광상품의 <code>minimum_nights</code>를 유지하면서 별도의 주거상품을 제안한다. <strong>maximum_nights, 그 기본값 여부, 장기숙박 허용 항목은 선정 조건에서 제외</strong>했다. 호스트 평점 역시 추가 구별력이 확인되지 않아 핵심 선정 조건에 쓰지 않았다.</p>{candidate_table}
<p>후보 비율은 각 지역 중단기형 집 전체 숙소를 분모로 한다. 7지역 합계는 23,222개 숙소, 지역 간 중복을 제거한 13,255개 호스트다. 지역별 호스트 수는 동일 호스트가 여러 지역에 있을 수 있어 단순 합산하지 않는다. 런던·바스크 14,880곳에 추가 5지역 8,342곳을 더한 규모다.</p>
<p class="note">이는 등록 시설이 맞는 접촉 후보 수다. 관광 허가, 소유 형태, 주소별 규정, 연속 예약 가능 기간, 호스트 참여 의사까지 확인된 실행 가능 물량은 아직 아니다. 분석표의 지역 키와 법적 관할 경계가 일치하는지도 주소로 확인해야 한다. 특히 바스크 1,563곳 전체가 혼합형 특례 대상 24개 코뮌 안에 있다는 뜻은 아니다.</p>
</section>'''

region_data = [
dict(key='london', tagline='관광 90박을 채웠다면, 다음 고객은 런던에서 생활할 사람', kind='관광 상한 뒤 주거상품', product='91박 이상 실제 거주 상품',
 regulation='<p>런던은 일정 요건 아래 주택의 단기 숙박 사용을 연간 합계 90박까지 계획허가 없이 허용한다. Council Tax 납부 의무자 조건 등이 있고, 상한을 넘는 단기 숙박 운영은 계획허가가 필요하다. 같은 사람이 장기간 실제 거주하는 주거임대는 단기 숙박 운영과 구분된다.</p>'+cites('london','camden'),
 mechanism='<p><strong>관광객에게 더 이상 단기 숙박으로 팔 수 없는 기간을, 런던에서 몇 달 생활할 사람에게 주거상품으로 제안한다.</strong> 상품 설계에서는 같은 실제 거주자의 연속 91박 이상을 기준으로 삼는다. 이는 법적 경계 혼동을 줄이기 위한 운영 기준이며 모든 임대차 의무를 없애는 조건은 아니다.</p><p>법인 예약 하나에 직원들이 매주 교대하는 형태는 같은 사람의 장기 거주로 간주하지 않는다. 주방·세탁기와 업무공간이 모두 있는 13,317곳을 우선 접촉해 파견·이전 근무자의 실제 주거 수요를 연결한다.</p>',
 scenario=[('관광 실적 확인','가상의 호스트가 해당 연도 상반기에 관광 숙박 합계 90박을 사용했다. 90박은 연속 3개월이 아니라 여러 예약의 합계일 수 있다.'),('주거상품 제안','7월부터 약 6개월 런던에서 근무할 한 사람에게 같은 집의 주거상품을 제안한다. 생활·업무 시설을 확인하고 적합한 주거계약을 체결한다.'),('캘린더 전환','입주 기간은 관광상품에서 닫는다. 관광상품의 기존 최소 숙박 설정을 일괄 91박으로 바꾸는 대신 별도 상품·계약으로 관리한다.'),('이후 운영','계약기간 숫자만으로 퇴거를 전제하지 않는다. 적법하게 주거계약이 끝나고 실제 반환된 뒤에, 해당 연도의 남은 관광 한도를 확인해 다시 운영한다.')],
 calendar=[('관광','해당 연도 합계 최대 90박'),('주거','같은 사람의 연속 91박 이상 실제 거주'),('재개','적법한 종료·반환 후 판단')],
 condition='<p>2026년 5월 1일부터 적용된 일반적인 assured 임대차는 기간의 종료만으로 자동 퇴거시키는 구조가 아니다. 따라서 ‘관광 3개월 + 주거 9개월을 매년 강제 반복’으로 약속하지 않는다. 기업 임대 등 별도 계약 구조도 실제 점유자와 임대차 권리를 따로 확인해야 한다. 런던의 주력 메시지는 <strong>관광 상한 이후의 주거상품 추가</strong>다.</p>'+cites('renters')),
dict(key='pays-basque', tagline='학기에는 학생의 집, 여름에는 관광객의 숙소', kind='명시적인 혼합 운영 특례', product='학생 최소 9개월 + 여름 관광 3개월',
 regulation='<p>프랑스 바스크의 규제 대상 24개 코뮌에서는 주거주지가 아닌 주택의 관광 임대에 용도변경과 보상이 요구된다. 보상은 같은 코뮌에서 동일 면적 이상의 적격 비주거 공간을 주거로 전환하는 방식이다. 다만 <strong>학생에게 최소 9개월 임대하고 여름 3개월 관광 임대</strong>하는 혼합형에는 보상 없는 임시 허가 경로가 있다.</p>'+cites('basque','basque_rule'),
 mechanism='<p><strong>학생 주거를 먼저 확보함으로써 혼합형 허가 경로를 이용하고, 여름 관광 운영을 연결한다.</strong> 일반 주거주지의 연간 120일·90일 한도를 소진한 뒤 장기로 파는 방식과는 출발점이 다르다. 이 지역은 학생 9개월과 관광 3개월을 결합하는 제도 자체가 전략의 직접적인 근거다.</p><p>생활·업무 시설 후보 1,563곳 중 대상 코뮌, 학생 수요 접근성, 혼합형 신청 의사를 확인한 숙소를 제안한다. 주방·세탁기는 학생 생활을, Wi-Fi·업무공간은 학업을 뒷받침한다.</p>',
 scenario=[('학생 상품부터 모집','가상의 숙소를 2026년 10월~2027년 6월, 9개월간 학생에게 임대한다. 재학 증빙과 실제 학생 임대계약을 준비한다.'),('혼합형 허가 신청','코뮌·소유자·학생이 참여하는 정해진 절차를 밟고 2027년 여름 운영을 위한 신청을 3월 1일 전 제출한다.'),('여름 관광 운영','학생 임대가 적법하게 끝나고 숙소가 반환되면, 허가 범위 안에서 7~9월 관광상품을 운영한다.'),('다음 학기 준비','다음 학생과 주거계약을 준비하고 연간 허가 갱신 조건을 충족한다. 첫해 허가만으로 이후 매년 자동 운영되는 것은 아니다.')],
 calendar=[('학생 주거','10월~다음 해 6월 · 9개월'),('관광','7~9월 · 3개월'),('갱신','다음 학기 학생·연간 허가')],
 condition='<p>혼합형 허가는 연간·개인별·양도 불가 방식이며 규정상 개인과 법인 소유자 모두의 경로가 제시돼 있다. 학생 자격, 9개월 임대, 코뮌 협약 및 허가 절차가 핵심이다. 후보 1,563곳은 바스크 지역 전체의 시설 모수이므로 <strong>24개 코뮌 주소 대조 후 특례 대상 수를 다시 확정</strong>해야 한다.</p>'+cites('basque_rule')),
dict(key='oslo', tagline='짧게 빌려줄 수 있는 날을 다 썼다면, 생활할 세입자로 전환', kind='관광 상한 뒤 실제 주거 전환', product='구분소유 주택의 가구 포함 주거임대',
 regulation='<p>구분소유 공동주택인 sameie의 집 전체 단기임대는 기본적으로 연 90일이며, 정관으로 60~120일 범위에서 조정할 수 있다. 이 규정의 단기임대는 연속 30일 이하이므로 31일 이상 임대는 이 조항의 단기일수 산정과 구분된다. 주택조합 borettslag에는 다른 규칙이 적용된다.</p>'+cites('oslo','oslo_days'),
 mechanism='<p><strong>해당 건물의 관광 한도를 사용한 호스트에게, 새로 이주한 근로자나 학생의 실제 집으로 전환하는 상품을 제안한다.</strong> 31일 관광 예약을 반복하면 자동으로 주거가 되는 구조가 아니다. 정부 안내도 실제 이용 방식에 따라 숙박업성 사용으로 판단될 수 있다고 설명한다.</p><p>생활·업무 시설 후보 1,700곳에서 먼저 소유 형태와 정관을 확인한다. 이 규정이 적용되는 sameie만 해당 상한 경로로 다루고, 나머지는 각 건물 규칙에 맞는 별도 주거 경로를 검토한다.</p>'+cites('oslo_use'),
 scenario=[('건물 규칙 확인','가상의 sameie 주택이 정관상 90일 한도를 적용받고, 올해 그 한도를 사용했다.'),('생활 고객 연결','오슬로로 이주한 근로자가 가구가 있는 집을 찾는다. 취사·세탁·업무가 가능한 집을 실제 거주지로 제안한다.'),('주거계약 체결','예를 들어 7월부터 기간을 정하지 않은 주거임대를 시작한다. 관광 캘린더는 입주 기간 동안 닫고 주거상품으로 관리한다.'),('장기 관계 유지','연말에 관광 한도가 새로 생겨도 세입자에게 자동 퇴거를 요구하지 않는다. 계약의 적법한 종료와 반환 이후 운영을 다시 결정한다.')],
 calendar=[('관광','건물별 연 60~120일 범위'),('주거','실제 거주 임대 시작'),('유지','관광 복귀일을 미리 고정하지 않음')],
 condition='<p>일반 유기 주거계약은 원칙적으로 최소 3년이며 법정 예외가 있다. 따라서 임의의 9개월 계약으로 매년 관광에 복귀하는 모델보다 <strong>관광 상한 뒤 주거로 전환</strong>하는 모델이 분명하다. 1,700곳 모두가 sameie이거나 동일한 90일 상한을 적용받는다는 뜻은 아니다.</p>'+cites('oslo_lease')),
dict(key='munich', tagline='관광은 허용 범위 안에서, 나머지는 근무·연수자의 생활 공간으로', kind='관광 용도와 주거 용도 구분', product='연수·파견 근무자의 실제 임시주거',
 regulation='<p>뮌헨의 주거용도 보호조례는 주택을 연 8주, 즉 56일을 넘겨 방문객 숙박에 사용하는 경우 용도전용 문제를 다룬다. 초과 관광 사용에는 허가가 필요하다. 시 안내는 실제 자기 집을 부재 기간에 빌려주는 사례 등을 구분하므로 모든 투자용 숙소가 무조건 56일을 확보하는 것으로 해석하지 않는다.</p>'+cites('munich','munich_city'),
 mechanism='<p><strong>관광으로 더 쓸 수 없는 기간을, 뮌헨에서 생활하는 연수·파견 근무자의 주거기간으로 연결한다.</strong> 핵심은 숙박일수를 늘리는 것보다 실질적인 생활 사용으로 바꾸는 데 있다. 관련 판결도 제한된 기간이라는 이유만으로 주거 사용이 배제되는 것은 아니라고 구분한다. 이 판결이 모든 집 전체 상품을 일괄 승인한 것은 아니다.</p><p>606개 시설 후보에서 적법한 관광 사용 경로와 주거 가능 상태를 확인한다. 개인 생활이 가능한 주방·세탁기와 근무에 필요한 책상·통신을 상품의 구체적인 구성으로 제시한다.</p>'+cites('munich_case'),
 scenario=[('관광 사용 확인','가상의 주택은 적법한 경로로 올해 관광 숙박 56일을 이미 사용했다.'),('실제 생활 수요 확인','4개월 현장 연수 때문에 뮌헨에 머무는 사람이 자기 생활 공간을 찾는다. 연수 일정과 실제 거주 목적을 확인한다.'),('주거로 제공','적합한 임대차 형태를 정하고, 해당 기간은 주거상품으로 제공한다. 방문객을 교체하는 숙박 운영과 구분한다.'),('종료 후 재판단','유효한 계약 종료와 실제 반환이 확인되면 관광 가능 일수를 다시 확인한다. 이미 올해 56일을 소진했다면 그해 관광을 추가하지 않는다.')],
 calendar=[('관광','적법한 경로의 연 8주 이내'),('주거','예: 4개월 연수자의 생활'),('재개','반환·해당 연도 허용량 확인')],
 condition='<p>주거 임대계약에 ‘4개월 뒤 관광 재개’만 적는 것으로 유효한 유기계약이 되는 것은 아니다. 일반 유기계약의 제한과 실제 임시사용에 관한 예외를 구분해야 한다. <strong>실제 임시주거 사유와 계약 형태가 맞는 경우</strong>에만 기간을 정한 시나리오를 적용하고, 나머지는 일반 주거임대로 설계한다.</p>'+cites('bgb549','bgb575')),
dict(key='malaga', tagline='관광 운영기간 밖에는, 몇 달 일하러 온 사람의 집으로', kind='기존 관광 공급에 주거기간 추가', product='3~6개월 근무·프로젝트 주거',
 regulation='<p>말라가는 신규 관광주택을 2025년 8월 23일부터 최대 3년 또는 새 계획의 조기 시행 때까지 중단하는 조치를 발표했다. 안달루시아 규정은 <strong>동일 임차인에게 연속 2개월을 초과해 임대하는 주택</strong>을 관광주택 규정의 적용 범위에서 제외하며, 관광 운영기간은 연중 또는 특정 기간으로 신고할 수 있다.</p>'+cites('malaga','andalusia_pdf'),
 mechanism='<p><strong>이미 적법하게 관광 운영 중인 숙소에 3~6개월 실제 근무 주거를 추가한다.</strong> 호스트가 관광으로 운영할 기간과 주거로 제공할 기간을 먼저 정한 뒤, 주거기간에 실제 임시 근무자를 연결한다. 새로운 관광 허가를 얻거나 관광 규제를 면제받는 전략으로 설명하지 않는다.</p><p>2,266개 시설 후보 중 기존 관광 등록과 주거 사용이 모두 가능한 집을 우선 확인한다. 지역 계획상 숙박업 전용 용도인 공간을 주택 임대처럼 제공하지 않도록 주소별 용도 확인이 필요하다.</p>'+cites('malaga2026'),
 scenario=[('기간 선택','가상의 기존 적법 관광주택 호스트가 여름 등을 관광 운영기간으로 선택한다. 선택한 달은 수요 분석으로 확정된 성수기가 아니라 시나리오 가정이다.'),('실제 근무자 매칭','10월~다음 해 3월의 6개월 프로젝트로 말라가에 오는 한 사람을 연결하고 근무·임시거주 사유를 확인한다.'),('주거기간 분리','실제 주거계약을 체결하고 관광 신고기간을 그 기간과 겹치지 않게 관리한다. 관광 광고·예약도 신고된 관광 운영기간에만 받는다.'),('반환 후 운영 선택','적법한 종료와 반환 후 관광 등록·용도가 유효하면 신고한 관광기간에 운영한다. 주거 수요가 이어지면 다음 주거상품을 검토할 수 있다.')],
 calendar=[('관광','호스트가 신고한 관광기간'),('주거','예: 10~3월 · 6개월 근무'),('다음 운영','관광 등록·용도 유효성 확인')],
 condition='<p>‘2개월 초과’는 관광 규정의 경계이며 주거계약의 모든 요건을 대신하지 않는다. 스페인 공통 임시주거 요건을 적용한다. 안달루시아 공식 FAQ는 1년 이상 주거임대와 관광 등록의 병존에도 제한을 설명하므로, 이 보고서는 <strong>6개월 실제 임시주거와 신고기간 분리</strong>로 상품을 구체화했다.</p>'+cites('andalusia','andalusia_faq')),
dict(key='sevilla', tagline='관광을 하지 않는 학기에는, 학생·연구자의 집으로', kind='기간을 나눈 학기형 주거상품', product='3~6개월 학업·연구 체류 주거',
 regulation='<p>세비야는 108개 동네 각각에서 가족주택 수 대비 관광주택의 최대 비중을 10%로 제한한다. 이는 숙소별 연간 관광 일수 상한과 다르다. 안달루시아의 동일인 연속 2개월 초과 임대 제외 및 관광 운영기간 신고 규칙도 적용된다. 시의 관광주택은 도시계획상 숙박 용도로 취급되므로 주거 사용 가능성을 함께 확인해야 한다.</p>'+cites('sevilla','andalusia_pdf','sevilla_use'),
 mechanism='<p><strong>기존 적법 관광 공급의 일부 기간을 교환학생·방문 연구자의 주거상품으로 구성한다.</strong> 10% 상한을 학생 임대로 우회해서 신규 관광 허가를 받는 방식이 아니다. 시설 후보 2,107곳 중 관광과 주거를 기간별로 운영할 수 있는 용도가 확인된 집에만 제안한다.</p><p>말라가가 근무·프로젝트 고객을 우선 가정한다면, 세비야는 학기 일정이 있는 학생·연구자를 첫 상품 고객으로 가정한다. 실제 수요 규모는 아직 검증하지 않았고, 이 구분은 상품 개발 방향이다.</p>',
 scenario=[('대상 숙소 확인','가상의 기존 적법 관광주택에 주거 임대가 가능한지 확인한다. 숙박업 용도만 허용된 숙소는 이 혼합 시나리오에서 제외한다.'),('학기 고객 연결','10월~다음 해 3월의 6개월 연구 방문자나 교환학생에게 생활·업무 시설을 갖춘 집을 제안한다. 학교·연구기관의 일정과 사유를 확인한다.'),('주거기간 운영','해당 기간의 실제 임시주거 계약과 관광 신고기간을 구분한다. 주거기간에는 같은 집을 관광객에게 동시에 팔지 않는다.'),('선택한 관광기간 재개','주거계약의 적법한 종료·반환 후, 예를 들어 4월부터 호스트가 선택한 관광 운영기간에 판매한다. 4월 수익 상승은 가정하지 않는다.')],
 calendar=[('학업·연구','예: 10~3월 · 6개월'),('관광','예: 4월 이후 선택한 신고기간'),('선행 조건','주소별 관광·주거 용도 확인')],
 condition='<p>세비야에서 가장 먼저 풀어야 할 것은 <strong>동일 숙소의 기간별 주거 사용이 가능한가</strong>다. 지역 관광법의 2개월 경계만으로 도시계획상 용도 문제까지 해결된다고 볼 수 없다. 양립이 어려운 숙소는 관광 복귀를 전제로 하지 않는 주거 전환을 별도로 검토한다.</p>'+cites('sevilla_use')),
dict(key='valencia', tagline='관광으로 등록한 달과, 생활할 사람에게 빌려주는 달을 분리', kind='관광 신고기간을 나눈 주거상품', product='2~6개월 학업·근무 임시주거',
 regulation='<p>발렌시아 자치주 관광법은 동일인에게 연속 10일 이하로 제공하는 집 전체를 관광주택의 정의에 포함하고, 11일 이상은 이 정의에서 제외한다. 그러나 <strong>신고된 관광 운영기간에 주거·계절 임대를 하는 것은 관광 등록 말소 사유</strong>가 될 수 있다. 기간 경계뿐 아니라 신고 캘린더를 함께 다뤄야 한다.</p>'+cites('valencia'),
 mechanism='<p><strong>관광 운영기간을 먼저 조정한 뒤, 그 밖의 기간에 실제 학업·근무자를 위한 주거상품을 판매한다.</strong> 11박 상품을 장기주거로 이름만 바꾸는 방식이 아니라, 임시거주 사유가 확인되는 2~6개월 상품을 제안한다. 1,663개 시설 후보에서 유효한 관광 등록, 시의 용도 적합성, 기간 변경 가능성을 확인한다.</p><p>발렌시아의 제안은 달력을 나누는 절차까지 포함한다. 예약 사이트에서 날짜만 막는 것과 행정상 관광 운영기간을 바꾸는 것은 다르다.</p>',
 scenario=[('두 기간 설계','가상의 숙소는 다음 해 1~6월을 주거, 7~8월을 관광 운영기간으로 제안한다. 나머지 기간은 소유자가 선택한다.'),('신고기간 먼저 조정','주거상품을 시작하기 전에 유효한 관광 등록의 운영기간을 실제 계획에 맞게 조정하고 관광·주거 기간이 겹치지 않게 한다.'),('실제 거주자 입주','6개월 연구·근무 일정이 있는 같은 거주자와 적합한 임시주거 계약을 체결한다. 취사·세탁·업무 시설을 점검한다.'),('관광 재개','계약의 적법한 종료와 반환 뒤, 유효한 관광 등록과 신고된 기간에 한해 관광상품을 운영한다.')],
 calendar=[('주거','예: 1~6월 · 6개월'),('관광','예: 신고된 7~8월'),('핵심 절차','주거 입주 전 신고기간 조정')],
 condition='<p>10일·11일 경계는 발렌시아 관광 규정의 분류 기준이다. 전국 임시주거법의 실제 사유·기간 요건과 시의 용도·등록 조건은 별도로 충족해야 한다. <strong>11박이면 자유롭게 장기 임대할 수 있다</strong>는 메시지는 쓰지 않는다.</p>'+cites('valencia','spain')),
]

def region_section(d, index):
    key=d['key']; row=F[F.region.eq(key)].iloc[0]
    steps=''.join(f'<li><b>{e(title)}</b><p>{e(body)}</p></li>' for title,body in d['scenario'])
    cal=''.join(f'<div><b>{e(label)}</b><span>{e(body)}</span></div>' for label,body in d['calendar'])
    return f'''<article class="region" id="{key}"><div class="region-top"><div><div class="eyebrow">{index:02d} · {e(d['kind'])}</div><h2>{e(names[key])}</h2><p class="lead">{e(d['tagline'])}</p></div><div class="region-number">{n(expected[key])}<span>시설 후보 · {n(row.candidate_hosts_n)}호스트</span></div></div>
    <div class="product"><b>제안 상품</b> {e(d['product'])}</div><h3>어떤 규제가 적용되는가</h3>{d['regulation']}<h3>그 규정을 상품 설계에 어떻게 활용하는가</h3>{d['mechanism']}<h3>호스트 한 명의 운영 시나리오</h3><p class="note">실제 계약·예약 사례가 아닌 상품 설계 예시다. 표시한 월은 수요나 수익의 검증 결과가 아니다.</p><div class="calendar">{cal}</div><ol class="scenario">{steps}</ol><h3>이 시나리오가 성립하는 핵심 조건</h3>{d['condition']}</article>'''

overview=table(['지역','시설 후보','규제와 연결하는 방식','구체적인 제안 상품'],[[names[d['key']],n(expected[d['key']]),d['kind'],d['product']] for d in region_data])
strategies=f'''<section id="strategies"><div class="eyebrow">02 · 지역별 상품 설계</div><h2>같은 시설 조건, 서로 다른 일곱 가지 운영 경로</h2><p class="lead">공통으로 가져가는 것은 생활·업무 시설과 집 전체라는 공급 조건이다. 관광 상한, 학생 혼합형 특례, 관광 운영기간 신고는 지역마다 다르게 사용한다.</p>{overview}
<div class="callout"><b>스페인 3개 지역에 공통으로 적용할 계약 기준</b><p>2026년 10월 1일부터 적용되는 현행 스페인 임시주거 규정은 실제로 증명할 수 있는 임시 이주 사유를 계약에 명시하도록 하고, 기간은 31일 초과·원칙적으로 12개월 이내로 정한다. 따라서 말라가·세비야·발렌시아는 학업·근무 사유를 확인한 실제 주거상품으로 구성한다. 사유가 없는 짧은 계약을 반복하는 방식은 이 설계에 포함하지 않는다.</p>{cites('spain')}</div></section>'''+''.join(region_section(d,i+1) for i,d in enumerate(region_data))

closing='''<section id="execution"><div class="eyebrow">03 · 실행과 검증</div><h2>지금 증명한 것과 다음에 확인할 것</h2>
<p>이번 데이터가 직접 뒷받침하는 것은 <strong>시설이 함께 등록되는 방식과 조건에 맞는 공급 규모</strong>다. 규제 검토가 뒷받침하는 것은 지역별로 검토할 수 있는 상품 경로다. 두 근거를 연결해 23,222개 접촉 후보를 만들었지만, 이를 바로 예약 가능한 장기 공급 수로 바꾸지는 않는다.</p>'''+table(['확인한 내용','확인 범위'],[
 ['공급 구성','정의한 3묶음과 3규칙의 등록 빈도·신뢰도·향상도'],['구성보정','3+6 기본 분석: 27지역·163층. 3번 보조 검증: 12지역·40층. 7지역 각각의 검증은 아님'],
 ['접촉 후보','7지역, 중단기형, 집 전체, 생활·업무 7기능 완전 등록: 23,222곳'],['지역별 제안','공식 규정에 맞춘 관광·주거 상품 경로와 가상 시나리오'],
 ['아직 확인하지 않은 것','주소별 적격, 호스트 동의, 연속 가용, 실제 장기 수요·예약·수익 효과']])+'''
<h3>호스트에게 제안되는 흐름은 간단하게 유지한다</h3><ol class="scenario"><li><b>시설 후보를 찾는다</b><p>이번 23,222곳에서 생활·업무 시설의 실제 사용 가능 여부를 확인한다.</p></li><li><b>주소와 운영 조건을 확인한다</b><p>관광 허가·소유 형태·주거 사용 가능 여부를 지역별 핵심 조건에 대조한다. 바스크는 24개 코뮌, 오슬로는 소유 형태, 세비야는 관광·주거 용도의 양립을 먼저 확인한다.</p></li><li><b>주거상품을 제안한다</b><p>관광상품은 기존 최소 숙박 설정을 유지한다. 입주 목적·기간·계약 조건을 갖춘 별도 상품으로 제안하고 같은 날짜의 중복 판매를 막는다.</p></li><li><b>실제 결과로 다음 수치를 만든다</b><p>접촉 → 법적·운영 적격 확인 → 호스트 동의 → 연속 가용 확인 → 상품 등록 → 실제 입주 순으로 규모를 다시 집계한다.</p></li></ol>
<p><code>availability_365</code>는 연속 몇 개월 비어 있는지를 알려주지 않는다. <code>365-availability_365</code>를 관광 판매일수로 쓰지 않고, 리뷰 수도 예약일수로 바꾸지 않는다. 관광 상한 관리는 플랫폼 간 실제 숙박 이력으로 따로 확인해야 한다. 최대 숙박일 설정은 이번 선별에서 제외했지만, 호스트가 장기상품을 수락할 때는 필요한 기간을 실제로 제공할 수 있어야 한다.</p>
<h3>주력 후보를 발전시키는 순서</h3><p><strong>바스크는 설명이 가장 선명한 제도형 사례</strong>로, 학생 9개월과 관광 3개월을 앞세운다. <strong>런던은 13,317곳의 규모를 가진 주거상품 확장 사례</strong>로, 관광 상한 이후의 실제 거주 수요 연결을 앞세운다. 추가 5개 지역도 제안 대상에 포함하되, 오슬로·뮌헨은 실제 주거와 계약 형태, 스페인 3개 도시는 관광 등록기간과 주소별 용도를 먼저 확인한다.</p>
<p>이 우선순위는 확인된 후보 규모와 운영 경로의 명확성을 바탕으로 한 실행 제안이다. 지역별 장기 수요의 크기나 수익성 순위라는 의미는 아니다.</p></section>'''

counts=F.copy();counts.insert(0,'지역명',counts.region.map(names))
counts.to_csv(OUT/'7개지역_시설후보_집계.csv',index=False,encoding='utf-8-sig')
C[['region','id','host_id','sector','room_type']+fc].sort_values(['region','id']).to_csv(OUT/'7개지역_시설후보_ID.csv',index=False,encoding='utf-8-sig')
R.loc[R.split.eq('all')&R.weighting.eq('listing')].to_csv(OUT/'연관분석_원지표.csv',index=False,encoding='utf-8-sig')
PD.loc[PD.primary].to_csv(OUT/'연관분석_구성보정.csv',index=False,encoding='utf-8-sig')
D.loc[D.primary].to_csv(OUT/'3번단독_보조검증_구성보정.csv',index=False,encoding='utf-8-sig')
V.to_csv(OUT/'초기묶음_내부안정성.csv',index=False,encoding='utf-8-sig')
shutil.copyfile(ASSOC/'분석기준_사전고정.json',OUT/'시설묶음_정의와초기분석기준.json')
evidence={
 'report_date':'2026-10-03','legal_review_date':'2026-10-03','scrape_date':'원자료 수집일은 확인하지 않음; 보고서 날짜와 구분',
 'selection':'source==0 & availability_365>0 & sector in [1,2,4,5] & room_type==Entire home/apt & seven function values==1',
 'excluded_filters':['maximum_nights','maximum_nights default flag','item__long_term_stays','host ratings'],
 'candidate_n':len(C),'unique_hosts_n':int(C.host_id.nunique()),'counts':expected,
 'sector_counts':{str(k):int(v) for k,v in C.sector.value_counts().sort_index().items()},
 'primary_reference':[3,6],'sensitivity_reference':[3],
 'adjustment_included_regions':common_regions,'selected_regions_in_primary_adjustment':sorted(set(regions)&set(common_regions)),
 'primary_common_strata_n':163,'sensitivity_common_strata_n':40,'sensitivity_included_regions':sensitivity_regions,
 'selected_regions_in_sensitivity_adjustment':[],
 'interpretation':'생활·업무 등록시설 기준의 접촉 후보. 법적 적격·연속 가용·예약 효과 미확정.',
 'sources':[dict(key=k,url=u,title=t) for k,(u,t) in src.items()],
 'analysis_source_files':[str(ASSOC/f) for f in ['3번단독_원지표.csv','연관보정_집계.csv','연관보정_층별근거.csv','연관보정_결과요약.json','3번단독_연관보정.csv','3번단독_층별근거.csv','묶음_탐색확인.csv']],
 'checks':{'candidate_unique_ids':True,'fixed_seven_facilities':True,'regional_sum':True,'association_formula_check':True,'standardized_formula_check':True,'primary_and_sensitivity_coverage_checked':True}}
(OUT/'선별기준_출처_검산.json').write_text(json.dumps(evidence,ensure_ascii=False,indent=2),encoding='utf-8')
appendix='''<section id="evidence"><div class="eyebrow">04 · 근거자료</div><h2>숫자를 다시 확인할 수 있는 자료</h2><p>본문의 수치는 기존 공통 분석표와 저장된 연관분석 결과에 연결했다. 시설 후보 ID를 중복 확인하고, 지역별 합계와 연관분석 원지표 계산식을 검산했다. 보정 결과는 기존 산출물을 사용했으며 새 모델을 적합하지 않았다.</p><div class="downloads">'''+''.join(f'<a download href="{e(filename)}">{e(label)} ↓</a>' for filename,label in [
 ('7개지역_시설후보_집계.csv','지역별 후보 집계'),('7개지역_시설후보_ID.csv','23,222개 후보 ID'),('연관분석_원지표.csv','세 규칙 원지표'),('연관분석_구성보정.csv','3+6 기본 분석 · 구성보정'),('3번단독_보조검증_구성보정.csv','3번 단독 · 보조 검증'),('초기묶음_내부안정성.csv','초기 묶음 내부 안정성'),('시설묶음_정의와초기분석기준.json','시설 정의·초기 기준'),('선별기준_출처_검산.json','현재 선별기준·출처·검산')])+'''</div><p class="note">규제 검토·문서 작성일: 2026-10-03. 이 날짜는 원자료 수집일을 뜻하지 않는다. 초기 기준 파일은 분석 설계 이력이고, 현재 후보 조건은 ‘현재 선별기준’ 파일 및 본문을 따른다. 각 지역의 법적 설명은 공식 출처와, 이를 상품으로 연결한 제안·가상 시나리오를 구분했다.</p></section>'''

css='''
:root{--ink:#183338;--muted:#617379;--paper:#f4f3ed;--teal:#087f78;--line:#dce5e2;--light:#eaf5f1;--gold:#d6ad56}*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:28px}body{margin:0;background:var(--paper);color:var(--ink);font:16px/1.8 -apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo","Malgun Gothic",sans-serif;word-break:keep-all;overflow-wrap:anywhere}a{color:#006d73;text-decoration-thickness:1px;text-underline-offset:3px}a:hover{color:#014447}header{background:#133a3c;color:white;padding:64px max(32px,calc((100vw - 1120px)/2)) 48px}header h1{font-size:clamp(30px,4vw,48px);line-height:1.25;letter-spacing:-1.6px;max-width:800px;margin:12px 0 22px}header p{max-width:880px;color:#e1efea;font-size:18px}.eyebrow{font-size:12px;font-weight:750;letter-spacing:2px;color:var(--teal);text-transform:uppercase}header .eyebrow{color:#9ed3c5}.hero-stats{display:flex;gap:60px;padding-top:18px}.hero-stats b{font-size:34px;display:block;line-height:1.3}.hero-stats span{font-size:13px;color:#bee0d6}.layout{max-width:1440px;display:grid;grid-template-columns:220px minmax(0,1fr);gap:42px;margin:0 auto;padding:42px 40px 80px}aside{align-self:start;position:sticky;top:24px;font-size:13px}aside a{display:block;text-decoration:none;padding:7px 12px;border-left:2px solid transparent;color:#53686c}aside a:hover{border-color:var(--teal);background:#e5eeea}aside strong{display:block;margin:14px 12px 6px;color:#173d40}main{min-width:0}section,.region{background:white;border:1px solid var(--line);border-radius:12px;padding:38px 42px;margin:0 0 28px}h2{font-size:28px;line-height:1.45;letter-spacing:-.8px;margin:9px 0 20px}h3{font-size:20px;line-height:1.55;margin:34px 0 12px;letter-spacing:-.4px}p{margin:12px 0 18px}.lead{font-size:18px;line-height:1.8;color:#36565a}.note,small{font-size:13px;color:var(--muted)}code{font:13px/1.5 ui-monospace,SFMono-Regular,monospace;background:#eff4f2;padding:2px 5px;border-radius:4px;color:#375c5e}.table-wrap{overflow-x:auto;margin:18px 0 24px;border:1px solid var(--line);border-radius:7px}table{border-collapse:collapse;width:100%;font-size:13px;min-width:590px;text-align:left}th{background:#eef4f1;color:#284e50;padding:11px 13px;font-size:12px;font-weight:700;vertical-align:top}td{padding:12px 13px;border-top:1px solid #e4ebe7;vertical-align:top}tr:nth-child(even) td{background:#fbfcfa}.bundles,.metric-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:24px 0}.bundles>div,.metric-grid>div{background:#f3f6f2;border:1px solid var(--line);border-radius:8px;padding:20px 18px}.bundles .selected{background:var(--light);border-color:#74b5a4}.bundles b,.metric-grid b{font-size:17px;color:#126b64}.bundles p,.metric-grid p{font-size:14px;line-height:1.75;margin-bottom:10px}.metric-grid code{font-size:12px;display:block;line-height:1.7;margin-top:12px}.callout{background:#eaf5f1;border-left:4px solid var(--teal);padding:22px 25px;margin:24px 0}.callout p:last-child{margin-bottom:0}.callout>b{font-size:17px}.flow{display:flex;align-items:center;gap:12px;margin:25px 0}.flow>div{flex:1;background:#f0f4f0;border-radius:8px;padding:16px 10px;text-align:center}.flow b{font-size:24px;display:block;line-height:1.3}.flow span{font-size:11px;display:block;margin-top:8px}.flow .accent{color:white;background:var(--teal)}.flow i{font-style:normal;color:#91aaa3}details{margin:18px 0;border:1px solid var(--line);border-radius:7px;padding:16px 20px;font-size:14px}summary{cursor:pointer;font-weight:700;color:#28635f}details p:last-child{margin-bottom:2px}.region-top{display:flex;justify-content:space-between;gap:30px;border-bottom:1px solid var(--line);padding-bottom:20px}.region-top h2{font-size:36px;margin-bottom:8px}.region-top .lead{margin-bottom:0;max-width:560px}.region-number{font-size:33px;font-weight:750;color:var(--teal);white-space:nowrap;align-self:center;text-align:right}.region-number span{display:block;font-size:11px;font-weight:500;color:var(--muted)}.product{background:#f0f5ee;margin:24px 0 0;padding:13px 18px;border-radius:6px;font-size:15px}.product b{color:#39715f;margin-right:14px}.sources{font-size:12px;color:var(--muted);line-height:1.9;margin-top:4px}.calendar{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:8px;margin:20px 0}.calendar>div{background:#e8f3f0;padding:16px 15px;border-top:3px solid #489889;border-radius:4px}.calendar>div:nth-child(2){background:#f4f0e6;border-color:var(--gold)}.calendar b{display:block;font-size:13px}.calendar span{font-size:12px;display:block;margin-top:5px;line-height:1.6}.scenario{padding-left:25px}.scenario li{padding:6px 0 10px 9px}.scenario li::marker{color:var(--teal);font-weight:750}.scenario p{margin:5px 0 0;font-size:15px}.downloads{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px;margin:24px 0}.downloads a{border:1px solid var(--line);border-radius:7px;padding:12px 16px;text-decoration:none;font-size:14px}.downloads a:hover{background:var(--light)}footer{padding:0 28px 34px;text-align:center;font-size:12px;color:var(--muted)}
@media(max-width:1120px){.layout{grid-template-columns:170px minmax(0,1fr);gap:24px;padding:30px 24px}section,.region{padding:30px 28px}.bundles,.metric-grid{grid-template-columns:1fr}.region-top{display:block}.region-number{text-align:left;margin-top:20px}.hero-stats{gap:32px}}
@media(max-width:760px){header{padding:40px 22px}.hero-stats{gap:22px;flex-wrap:wrap}.hero-stats b{font-size:28px}.layout{display:block;padding:20px 12px}aside{position:static;margin:0 8px 24px;display:flex;gap:4px;flex-wrap:wrap}aside strong{width:100%;margin:8px}aside a{padding:4px 8px;border:1px solid #dce5e2;border-radius:4px}section,.region{padding:25px 20px;border-radius:8px}h2{font-size:25px}.lead{font-size:16px}.flow{flex-wrap:wrap;gap:8px}.flow>div{min-width:40%;padding:12px}.flow i{display:none}.calendar{grid-template-columns:1fr}.downloads{grid-template-columns:1fr}p{font-size:15px}table{font-size:12px}th,td{padding:9px}.region-top h2{font-size:32px}}
@media print{body{background:white;font-size:10pt}header{padding:20px;color:#183338;background:white;border-bottom:2px solid #183338}header p,header .eyebrow,.hero-stats span{color:#375c5e}header h1{font-size:26pt}aside{display:none}.layout{display:block;max-width:none;padding:0}section,.region{border:0;padding:24px 0;margin:0;border-radius:0}.region{break-before:page}h2,h3{break-after:avoid}.table-wrap{overflow:visible}table{min-width:0;font-size:9pt}tr{break-inside:avoid}.calendar,.bundles,.metric-grid,.callout{break-inside:avoid}details{display:block}.sources{font-size:8pt}.region-number{font-size:25pt}.downloads,footer{display:none}a{color:#183338;text-decoration:none}}
'''
nav='<strong>시설·연관분석</strong><a href="#analysis">정의 → 지표 → 보정 → 선별</a><strong>7지역 전략</strong><a href="#strategies">전체 비교</a>'+''.join(f'<a href="#{k}">{e(names[k])} · {n(expected[k])}</a>' for k in regions)+'<strong>실행·근거</strong><a href="#execution">실행 흐름과 검증 범위</a><a href="#evidence">집계·후보 ID·원지표</a>'
document=f'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>InsightStay · 시설 연관분석과 7개 지역 장기상품 전략</title><style>{css}</style></head><body><header><div class="eyebrow">InsightStay · Strategy Report · 2026.10.03</div><h1>생활하고 일할 수 있는 집을,<br>지역에 맞는 장기상품으로</h1><p>시설 연관분석의 의미부터 런던·바스크·오슬로·뮌헨·말라가·세비야·발렌시아의 규제, 상품 제안, 운영 시나리오까지.</p><div class="hero-stats"><div><b>7개 지역</b><span>지역별로 다른 운영 경로</span></div><div><b>23,222곳</b><span>집 전체 · 생활·업무 시설 후보</span></div><div><b>3개 규칙</b><span>지지도·신뢰도·향상도 및 구성보정</span></div></div></header><div class="layout"><aside aria-label="목차">{nav}</aside><main>{analysis}{strategies}{closing}{appendix}</main></div><footer>InsightStay · 시설 조건에 따른 공급 후보와 지역별 상품 설계 · 2026-10-03</footer></body></html>'''
path=OUT/'7개지역_장기상품_통합전략.html'
path.write_text(document,encoding='utf-8')
print(json.dumps({'report':str(path),'bytes':path.stat().st_size,'candidate_n':len(C),'hosts':int(C.host_id.nunique()),'regions':expected,'common_regions':common_regions},ensure_ascii=False,indent=2))
"""
SOURCE_META['report'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_seven_region_report.py', 'sha256': '766cd5de9e16a889b099b46e5e6ed44ab54ef475f2e2ea5d8960928f7b0100ac', 'language': 'python'}


### report_update — E4 연결·손익·Python/JS 검산·HTML 통합
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_장기상품_통합전략_20261003/update_cluster_economics.py`
SHA256: `b91ab7cc53b468f523e36fd58f22897471cdb01667844f1d15784921b7f93c81`


In [21]:
SOURCE_CODE['report_update'] = """from pathlib import Path
from datetime import datetime
from html.parser import HTMLParser
import json, hashlib, shutil, re, subprocess, html
import pandas as pd

P=Path(__file__).resolve().parent
F=next(p for p in P.glob('*.html') if '통합' in __import__('unicodedata').normalize('NFC',p.name))
old=F.read_text(); old_sha=hashlib.sha256(F.read_bytes()).hexdigest()
ROOT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
K=ROOT/'outputs/InsightStay_지역특화장기상품_확장검증_20261003/01_상품군집'
R=P.parent/'7개지역_생활업무시설_조정회귀_20261003'
c=pd.read_csv(P/'7개지역_시설후보_ID.csv',dtype={'id':'string','host_id':'string'})
k=pd.read_csv(K/'숙소별_군집.csv.gz',usecols=['id','region','sector','room_type','cluster_label','ood_any'],dtype={'id':'string'})
matched=c.merge(k[['id','cluster_label','ood_any']],on='id',how='left',validate='one_to_one')
assert len(matched)==23222 and matched.cluster_label.eq('E_K4_4').all() and not matched.ood_any.any()
regions=['london','pays-basque','oslo','munich','malaga','sevilla','valencia']
names=dict(zip(regions,['런던','프랑스 바스크','오슬로','뮌헨','말라가','세비야','발렌시아']))
e4=k[k.region.isin(regions)&k.sector.isin([1,2,4,5])&k.room_type.eq('Entire home/apt')&k.cluster_label.eq('E_K4_4')]
summary=[]
for r in regions:
    a=matched[matched.region.eq(r)];e=e4[e4.region.eq(r)]
    summary.append({'region':r,'E4_1245_entire_n':len(e),'fixed_facility_candidate_n':len(a),'candidate_in_E4_n':int(a.cluster_label.eq('E_K4_4').sum()),'candidate_share_of_E4_pct':len(a)/len(e)*100})
assert len(e4)==31224
pd.DataFrame(summary).to_csv(P/'E4_현재후보_지역별연결.csv',index=False,encoding='utf-8-sig')
kj=json.loads((K/'결과요약.json').read_text());rj=json.loads((R/'결과요약.json').read_text())
rm={r['model']:r for r in rj['models']}
assert abs(rm['M1_primary']['difference_pp']-4.872684552002971)<1e-8
defaults=dict(months=9,monthlyRent=1000,displacedNights=45,shortNightlyPrice=120,shortNightlyCost=30,longMonthlyCost=180,hostSetupCost=300,support=100,platformCost=150,hostLongFee=.03,hostShortFee=.03,platformLongFee=.155,platformShortFee=.155,otherHostAlternative=0,otherPlatformLoss=0)
def calc(x):
    gl=x['months']*x['monthlyRent'];gs=x['displacedNights']*x['shortNightlyPrice']
    opp=max(x['displacedNights']*(x['shortNightlyPrice']*(1-x['hostShortFee'])-x['shortNightlyCost']),x['otherHostAlternative'])
    hb=gl*(1-x['hostLongFee'])-x['months']*x['longMonthlyCost']-x['hostSetupCost']-opp
    pb=x['platformLongFee']*gl-x['platformShortFee']*gs-x['platformCost']-x['otherPlatformLoss']
    hf=max(0,(x['months']*x['longMonthlyCost']+x['hostSetupCost']+opp-x['support'])/(x['months']*(1-x['hostLongFee'])))
    pf=max(0,(x['platformShortFee']*gs+x['platformCost']+x['support']+x['otherPlatformLoss'])/(x['months']*x['platformLongFee']))
    return dict(longGMV=gl,displacedGMV=gs,hostOpportunity=opp,hostBeforeSupport=hb,platformBeforeSupport=pb,hostChange=hb+x['support'],platformChange=pb-x['support'],hostRentFloor=hf,platformRentFloor=pf,jointRentFloor=max(hf,pf),supportMin=max(0,-hb),supportMax=pb,supportFeasible=pb>=max(0,-hb),bothNonnegative=hb+x['support']>=0 and pb-x['support']>=0)
scenarios=[{'label':label,'inputs':dict(defaults,displacedNights=n),'outputs':calc(dict(defaults,displacedNights=n))} for label,n in [('대체 판매 0박 가정',0),('대체 판매 45박 가정',45),('대체 판매 90박 가정',90)]]
assert abs(scenarios[1]['outputs']['hostChange']-3022)<1e-8
assert abs(scenarios[1]['outputs']['platformChange']-308)<1e-8
assert abs(scenarios[2]['outputs']['hostChange']+866)<1e-8
assert abs(scenarios[2]['outputs']['platformChange']+529)<1e-8
# An additional euro of support is a transfer: host +1, platform -1.
b=calc(defaults);up=calc(dict(defaults,support=101))
assert abs(up['hostChange']-b['hostChange']-1)<1e-8 and abs(up['platformChange']-b['platformChange']+1)<1e-8
assert not calc(dict(defaults,monthlyRent=0))['bothNonnegative']
assert calc(dict(defaults,otherHostAlternative=10000))['hostChange']<b['hostChange']
tests=[s['inputs'] for s in scenarios]+[dict(defaults,support=0),dict(defaults,monthlyRent=0),dict(defaults,otherHostAlternative=10000,otherPlatformLoss=100),dict(defaults,platformLongFee=.05)]
node='/Users/ohgyuwon/.cache/codex-runtimes/codex-primary-runtime/dependencies/node/bin/node'
code='const {calculateScenario}=require('+json.dumps(str(P/'손익계산.js'))+'); const xs='+json.dumps(tests)+'; process.stdout.write(JSON.stringify(xs.map(calculateScenario)));'
js=json.loads(subprocess.check_output([node,'-e',code],text=True))
for x,a in zip(tests,js):
    py=calc(x)
    for key,v in py.items():
        assert a[key]==v if isinstance(v,bool) else abs(a[key]-v)<1e-7
economics={
 'status':'illustrative_only_not_measured_forecast','unit':'가상 숙소 1곳, 동일한 9개월 평가기간, 금액 예시는 EUR. 지역 실제 임대료·비용을 사용하지 않음.',
 'known_inputs':{'candidate_listings':23222,'unique_candidate_hosts':13255,'assignment_platform_fee_assumption':.155},
 'defaults':defaults,'scenarios':scenarios,
 'fees_note':'15.5%는 과제 수수료 가정. 실제 주거상품에도 적용 가능한지 미검증. 호스트 부담 3%는 별도 설명용 가정이며 실제 Airbnb 요율을 주장하지 않음. 총 플랫폼 수수료 전체를 호스트 비용으로 중복 차감하지 않음.',
 'horizon_note':'9개월은 손익 비교기간일 뿐 일괄 계약기간·자동 반환을 뜻하지 않음. 입주기간의 기존 관광판매는 포기 판매로 차감. 기간 종료 이후의 반환 지연 등 영향은 필요시 비용과 평가기간에 반영.',
 'host_formula':'주거 거래액*(1-호스트 주거 수수료) - 주거 변동비 - 초기비용 - max(포기 관광 순기여, 다른 합법적 대안 순기여) + 지원금',
 'platform_formula':'주거 거래액*플랫폼 주거 수수료 - 포기 관광 거래액*플랫폼 관광 수수료 - 지원금 - 프로그램 비용 - 추가 타숙소 잠식 수수료',
 'support_rule':'호스트 최소 필요 지원금<=플랫폼 최대 감당 지원금일 때만 양측 손익분기 가능한 지원 범위 존재. 참여 의사와 법적 적격·수요는 별도.',
 'not_observed':['동일 기간 실제 포기 예약·거래액','연속 가용기간','주거 수요와 입주 확률','월 주거가격·호스트 비용·업무시간','실제 주거 수수료 계약','법적 적격과 호스트 동의','다른 운영대안과 타숙소 수요 이전'],
 'checks':{'python_javascript_agree':True,'scenarios_n':3,'additional_edge_scenarios_n':4,'support_transfer_identity':True,'no_candidate_count_times_assumed_profit_forecast':True,'constant_existing_costs_cancelled':'장기 상품 전후 동일한 대출·세금·고정비는 제외. 달라지는 항목은 운영비·초기비용에 입력.'}}
(P/'손익분기_가정과계산.json').write_text(json.dumps(economics,ensure_ascii=False,indent=2))
pd.DataFrame([dict(scenario=s['label'],**s['inputs'],**s['outputs']) for s in scenarios]).to_csv(P/'손익분기_가상시나리오.csv',index=False,encoding='utf-8-sig')

rows=''.join(f"<tr><td>{names[r['region']]}</td><td>{r['E4_1245_entire_n']:,}</td><td>{r['fixed_facility_candidate_n']:,}</td><td>{r['candidate_in_E4_n']:,}</td></tr>" for r in summary)
cluster=f'''<section id="cluster-evidence"><div class="eyebrow">01-b · 교육 기법과 후보의 연결</div><h2>기존 군집화에서 현재 후보까지</h2>
<p class="lead">기존 K-means는 시설 구성이 다른 공급 유형을 구분했다. 현재 고정 시설 후보 23,222곳을 숙소 ID로 연결하니 모두 E4였다. 군집은 공급 유형을 설명하고, 최종 후보 수는 고정된 시설 조건으로 정한다.</p>
<div class="table-wrap"><table><thead><tr><th>기존 군집</th><th>관측된 등록 특징</th><th>상품 검토에서의 의미</th></tr></thead><tbody><tr><td>E1</td><td>세탁기 등록이 적음</td><td>세탁 이용 가능성 확인</td></tr><tr><td>E2</td><td>가열기기·냉장고·조리도구·식기 등록이 적음</td><td>실제 취사 가능 범위 확인</td></tr><tr><td>E3</td><td>생활기능 등록이 폭넓음</td><td>생활 목적 상품의 기능 확인</td></tr><tr><td>E4</td><td>생활기능 등록이 폭넓고 업무공간 등록도 많음</td><td>생활·학업·업무 목적 상품을 검토할 유형</td></tr></tbody></table></div>
<h3>어떻게 나눴고 얼마나 구분됐나</h3><p>기존 1·2·4·5번 자료를 사용하고 같은 호스트의 숙소가 학습·검증·시험에 섞이지 않게 나눴다. 입력은 주방·가열기기·냉장고·조리도구와 식기·세탁기·Wi-Fi·업무공간의 7개 기능이다. 지역·리뷰·규제 적격은 군집 입력에 넣지 않았다. 원래 0~1 척도를 유지하면서 취사·세탁·업무의 세 기능 영역에 거리 가중치를 배분한 K-means를 사용했다. 이 가중치는 분석자가 정한 설계이며, 일반 표준화를 했다고 설명하지 않는다.</p>
<p>집 전체형에서 4개 군집을 선택했다. 시험 자료 75,850곳 중 1,500곳으로 계산한 <strong>실루엣 계수는 0.562</strong>였다. 입력 시설에 따른 구분 정도이며 정확도 56.2%나 예약 성공률이 아니다. 초기값 및 기능 가중치 변경에 따른 안정성도 기존 분석에서 확인했다. 이번에는 모형을 다시 학습하지 않고 저장된 배정을 연결했다.</p>
<h3>현재 후보는 E4 전체 중 일부다</h3><div class="table-wrap"><table><thead><tr><th>지역</th><th>1·2·4·5번 집 전체 E4</th><th>7기능 완전 등록 후보</th><th>그중 E4와 연결</th></tr></thead><tbody>{rows}<tr><td><strong>합계</strong></td><td><strong>31,224</strong></td><td><strong>23,222</strong></td><td><strong>23,222</strong></td></tr></tbody></table></div>
<p>E4의 설명은 군집 평균의 특징이다. E4 전체가 7기능을 모두 갖춘 것은 아니다. 현재 후보는 별도의 고정 시설 기준을 충족하고, 그 결과가 모두 E4에 포함된다는 뜻이다. 같은 시설 정보를 쓴 두 결과이므로 독립적인 성공 검증 두 개로 세지 않는다.</p>
<h3>추가 회귀가 알려준 한계</h3><p>별도로 수행한 7지역 조정 회귀에서는 최근 리뷰 양수인 장기 216곳과 중단기 42,541곳을 비교했다. <code>region</code>·<code>property_type</code>·<code>accommodates</code>를 고려한 등록률 차이는 <strong>+4.87%p, 95% 구간 −2.97~+12.71%p</strong>였다. 침실·욕실 추가 조정도 0을 포함했다. 호스트 동일 가중 보조 분석은 +11.86%p [3.66, 20.06]로 달라져, 장기 설정 공급의 보편적 우위라고 일반화하지 않는다. 이 회귀의 3번 참조는 리뷰 차이를 구분하기 위한 별도 보조 질문이며 본문의 3+6 기본 참조를 대체하지 않는다.</p>
<div class="callout"><b>발표에서 연결할 논리</b><p>군집화로 시설 공급 유형을 파악했다. 생활·업무 기능을 완전 등록한 후보를 고정 기준으로 선별했다. 지역의 상품 경로와 호스트·플랫폼 손익 조건을 연결한다. 장기 예약 성공이나 추가 수익이 이미 입증됐다고 설명하지 않는다.</p></div>
<p class="sources">수치 근거: 기존 저장 군집 배정과 현재 후보 ID의 일대일 연결. <a href="E4_현재후보_지역별연결.csv">지역별 연결 집계</a> · <a href="../7개지역_생활업무시설_조정회귀_20261003/조정회귀_결과보고서.html">추가 회귀 전체 결과</a></p></section>'''

def money(v):return f'€{v:,.0f}' if v>=0 else f'−€{abs(v):,.0f}'
scenario_rows=''.join(f"<tr><td>{s['inputs']['displacedNights']}박</td><td>{money(s['outputs']['hostChange'])}</td><td>{money(s['outputs']['platformChange'])}</td><td>약 €{s['outputs']['jointRentFloor']:,.2f}/월</td><td>{'양측 손익분기 충족' if s['outputs']['bothNonnegative'] else '현재 조건으로 미충족'}</td></tr>" for s in scenarios)
def control(key,label,step='1',maxv=None):
    v=defaults[key]*100 if key.endswith('Fee') else defaults[key]
    mx=f' max="{maxv}"' if maxv is not None else ''
    return f'<label>{label}<input data-econ="{key}" type="number" min="0"{mx} step="{step}" value="{v:g}"></label>'
main_controls=''.join([control('months','평가기간 (개월)'),control('monthlyRent','월 주거 거래액 (€)','50'),control('displacedNights','같은 기간 포기하는 관광 판매 (박)'),control('shortNightlyPrice','관광 1박 거래액 (€)','10'),control('longMonthlyCost','주거 월 운영비 (€)','10'),control('support','플랫폼의 호스트 지원금 (€)','10')])
advanced=''.join([control('shortNightlyCost','관광 판매 1박당 변동비 (€)'),control('hostSetupCost','호스트 초기비용·시간가치 (€)'),control('platformCost','플랫폼 추가 프로그램 비용 (€)'),control('hostLongFee','호스트 주거 수수료율 (%)','.1',99),control('hostShortFee','호스트 관광 수수료율 (%)','.1',99),control('platformLongFee','플랫폼 주거 수수료율 (%)','.1',100),control('platformShortFee','플랫폼 관광 수수료율 (%)','.1',100),control('otherHostAlternative','다른 합법적 대안의 기간 순기여 (€)'),control('otherPlatformLoss','추가로 잠식되는 타숙소 수수료 (€)')])
economy=f'''<section id="economics"><div class="eyebrow">03 · 참여 이유와 손익분기</div><h2>호스트와 플랫폼 모두에게 남는가</h2><p class="lead">생활시설을 갖췄다는 이유만으로 호스트가 참여하지는 않는다. 같은 기간에 포기하는 기존 판매, 운영비, 준비 시간을 차감하고도 이익이 남는지 확인해야 한다. 아래는 시장 전망이 아닌 <strong>가상 손익분기 시나리오</strong>다.</p>
<h3>이해관계자에게 제공할 구체적인 가치</h3><div class="table-wrap"><table><thead><tr><th>이해관계자</th><th>제공할 것</th><th>성립 조건</th></tr></thead><tbody><tr><td>호스트</td><td>지역별 상품·계약 절차 안내, 시설 안내 초안, 실제 주거 고객 연결. 필요 시 제한된 초기 지원</td><td>기존 판매와 다른 합법적 대안보다 순기여가 낮지 않고, 제공기간·반환 조건에 동의</td></tr><tr><td>플랫폼</td><td>상품 구성·모집·운영 지원</td><td>새 수수료에서 포기 거래의 수수료·지원금·운영비를 빼도 손익분기 충족</td></tr><tr><td>입주자</td><td>생활·학업·업무 기능과 공과금·기간·계약 조건이 명확한 주거상품</td><td>실제 시설·이용권을 확인하고 기간과 비용이 필요에 부합</td></tr></tbody></table></div>
<h3>같은 기간의 두 선택을 비교한다</h3><p><strong>호스트 증분 순기여</strong> = 주거 거래액에서 호스트 부담 수수료를 뺀 금액 − 주거 운영비 − 준비비용·시간가치 − 포기하는 대안의 순기여 + 지원금.</p><p><strong>플랫폼 증분 기여</strong> = 새 주거 수수료 − 포기하는 기존 관광 수수료 − 지원금 − 추가 프로그램 비용 − 별도로 발생하는 타숙소 잠식 수수료.</p><p>호스트의 비교 대안은 포기 관광판매의 순기여와 다른 합법적 운영 대안의 순기여 중 큰 값으로 둔다. 관광 상한을 소진해도 다른 플랫폼·주거 운영 기회가 있다면 그 가치를 0으로 놓지 않는다. 입주 기간에는 같은 집을 관광객에게 동시에 판매할 수 없으므로 기존 판매를 전액 추가수익으로 중복 합산하지 않는다.</p>
<details><summary>가상 예시의 고정 가정 보기</summary><p>숙소 1곳·평가기간 9개월, 월 주거 거래액 €1,000, 주거 월 운영비 €180, 초기 준비비용 €300(설명 예: 현금비용 €180 + 8시간×€15), 관광 1박 거래액 €120·판매박당 변동비 €30, 호스트 지원 €100, 플랫폼 추가 비용 €150을 가정했다. 다른 대안의 순기여와 추가 타숙소 잠식은 아래 기본 예시에서 0으로 가정했다.</p><p>플랫폼 수수료 15.5%는 과제의 거래액 연동 가정이다. 실제 주거상품의 부과 가능성과 계약은 확인하지 않았다. 호스트 부담 수수료 3%는 별도 설명용 가정이며 실제 Airbnb 요율을 의미하지 않는다. 15.5% 전체를 호스트 비용으로 다시 차감하지 않는다. 모든 금액은 관측값이 아닌 가상 EUR이며 런던·오슬로 등의 실제 통화로 비교하려면 같은 통화로 다시 입력해야 한다.</p><p>9개월은 손익을 비교하는 기간이다. 모든 지역에 9개월 유기계약이 허용되거나 9개월 뒤 자동 반환된다는 뜻은 아니다. 법적으로 허용되는 상품과 제공기간을 확인한 뒤 평가기간을 맞춰야 한다. 운영 전후 동일한 고정비는 상쇄하고 달라지는 비용·세금·반환 지연의 부담은 별도 입력에 포함한다.</p></details>
<h3>가정한 월 €1,000이 항상 이득은 아니다</h3><div class="table-wrap"><table><thead><tr><th>포기 관광판매<br>동일 9개월 안의 가정</th><th>호스트 증분 순기여</th><th>플랫폼 증분 기여</th><th>양측 손익분기 월 거래액</th><th>예시 판단</th></tr></thead><tbody>{scenario_rows}</tbody></table></div><p class="note">손익분기 월 거래액은 현금흐름 시점을 할인하지 않은 기여 기준이다. 숙소 가격 추천이나 실제 수익 전망이 아니다. 0박은 비교기간에 포기하는 적법한 관광 판매와 다른 대안 가치가 모두 0인 경우에만 쓸 수 있다.</p>
<div class="callout"><b>호스트 참여를 유도할 수 있는 지원 범위</b><p>45박을 포기하는 가상 예시에서는 지원금 없이도 호스트 증분 순기여가 €2,922이고, 플랫폼은 최대 €408까지 지원해야 기여가 음수가 되지 않는다. €100을 지원하면 호스트 €3,022, 플랫폼 €308이 남는다. 이 금액은 참여 의사를 보장하지 않는다.</p><p>90박을 포기하면 호스트 손익분기에 최소 €966의 지원이 필요하지만, 플랫폼은 지원 전부터 −€429다. 이 조건에서는 지원금으로 해결할 수 없으므로 월 거래액·비용·대체 기간을 바꾸거나 제안을 보류해야 한다.</p></div>
<h3>가정을 바꿔 손익분기 확인하기</h3><p class="note">아래 입력은 모두 시나리오 가정이다. 페이지에서만 계산되며 숙소 설정이나 원자료를 바꾸지 않는다. 입력한 기간 전체가 제공되고 거래된다는 조건부 계산이며 수요·입주 확률은 추정하지 않는다.</p>
<form id="econ-form" onsubmit="return false"><div class="econ-grid">{main_controls}</div><details><summary>수수료·비용·다른 운영대안 조정</summary><div class="econ-grid">{advanced}</div><p class="note">다른 운영대안은 동일 기간의 순기여를 입력한다. 타숙소 잠식 수수료는 이미 포기 관광 수수료로 차감한 금액과 중복하지 않는다.</p></details><button type="button" id="econ-reset">가상 기본값으로 복원</button></form>
<p id="econ-error" role="alert"></p><div class="metric-grid econ-results" aria-live="polite"><div><b id="econ-host">€3,022</b><p>호스트 증분 순기여</p></div><div><b id="econ-platform">€308</b><p>플랫폼 증분 기여</p></div><div><b id="econ-floor">€779.21 /월</b><p>양측 손익분기 월 거래액</p></div></div><p id="econ-status">가상 기본값에서는 양측 손익분기 조건을 충족한다.</p><p id="econ-support">지원 가능 범위: €0~€408.</p>
<h3>대상 규모와 기대효과를 연결할 때의 기준</h3><p>23,222곳은 접촉 후보이지 상품 등록·입주 완료 수가 아니다. 법적·운영 적격 확인, 호스트 동의, 실제 제공기간, 수요 확보를 거친 대상 수에만 손익을 연결해야 한다. 후보 수 전체에 위 이익을 곱한 기대효과 총액은 제시하지 않는다. 회귀계수·군집 비율을 참여율이나 예약 증가율로 대입하지 않는다.</p><p>교육의 경영 기초 내용은 이처럼 <strong>대상 규모와 실행 단계, 비용, 손익분기 조건</strong>을 연결하는 데 적용한다. 먼저 적격 숙소와 실제 운영 조건을 확인하고, 신규 주거 수수료와 사라진 기존 거래·지원비를 함께 기록하는 검증 계획으로 이어간다.</p></section>'''

extra_css='''\\n.econ-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px;margin:20px 0}.econ-grid label{font-size:13px;color:#36565a}.econ-grid input{display:block;width:100%;padding:11px 12px;margin-top:6px;border:1px solid #afc5c0;border-radius:5px;font:16px system-ui;background:white;color:#183338}#econ-reset{padding:10px 16px;border:1px solid #087f78;background:#edf7f4;border-radius:5px;color:#075f59;cursor:pointer}#econ-error{color:#9b3025}#econ-status{font-weight:700}.econ-results b{font-size:25px}@media(max-width:760px){.econ-grid{grid-template-columns:1fr 1fr}}@media print{#econ-form{display:none}}\\n'''
js_engine=(P/'손익계산.js').read_text()
js_ui='''
const econDefaults=DEFAULTS;
const money=v=>new Intl.NumberFormat('ko-KR',{style:'currency',currency:'EUR',maximumFractionDigits:0}).format(v);
const preciseMoney=v=>new Intl.NumberFormat('ko-KR',{style:'currency',currency:'EUR',minimumFractionDigits:2,maximumFractionDigits:2}).format(v);
function updateEconomics(){
 const x={}; document.querySelectorAll('[data-econ]').forEach(el=>{let v=Number(el.value);if(el.value.trim()==='')v=NaN;x[el.dataset.econ]=el.dataset.econ.endsWith('Fee')?v/100:v;});
 const invalid=Object.values(x).some(v=>!Number.isFinite(v)||v<0)||x.months<=0||x.hostLongFee>=1||x.hostShortFee>=1||x.platformLongFee<=0||x.platformLongFee>1||x.platformShortFee>1;
 if(invalid){document.getElementById('econ-error').textContent='기간은 0보다 크게, 금액은 0 이상으로 입력하세요. 호스트 수수료는 100% 미만, 플랫폼 주거 수수료는 0% 초과 100% 이하입니다.';['host','platform','floor'].forEach(k=>document.getElementById('econ-'+k).textContent='입력 확인');document.getElementById('econ-status').textContent='';document.getElementById('econ-support').textContent='';return;}
 document.getElementById('econ-error').textContent=''; const r=calculateScenario(x);
 document.getElementById('econ-host').textContent=money(r.hostChange);document.getElementById('econ-platform').textContent=money(r.platformChange);document.getElementById('econ-floor').textContent=preciseMoney(r.jointRentFloor)+' /월';
 document.getElementById('econ-status').textContent=r.bothNonnegative?'입력한 가정에서는 양측 손익분기 조건을 충족합니다. 법적 적격·호스트 동의·수요 확보를 뜻하지는 않습니다.':'입력한 가정에서는 한쪽 이상이 손익분기를 충족하지 못합니다. 가격·비용·대체 판매를 조정하거나 제안을 보류합니다.';
 document.getElementById('econ-support').textContent=r.supportFeasible?'양측 손익분기를 만족하는 지원금 범위: '+money(r.supportMin)+' ~ '+money(r.supportMax)+'.':'호스트 최소 필요 지원금 '+money(r.supportMin)+', 플랫폼 지원 전 기여 '+money(r.supportMax)+'. 현재 조건에서는 지원금만으로 양측 손익분기를 맞출 수 없습니다.';
}
document.querySelectorAll('[data-econ]').forEach(el=>el.addEventListener('input',updateEconomics));
document.getElementById('econ-reset').addEventListener('click',()=>{document.querySelectorAll('[data-econ]').forEach(el=>{const k=el.dataset.econ;el.value=econDefaults[k]*(k.endsWith('Fee')?100:1)});updateEconomics()});updateEconomics();
'''.replace('DEFAULTS',json.dumps(defaults))

# Idempotent block replacement; leave all regional law and candidate definitions intact.
s=re.sub(r'<section id="cluster-evidence">.*?</section>','',old,flags=re.S)
s=re.sub(r'<section id="economics">.*?</section>','',s,flags=re.S)
s=re.sub(r'<script id="cluster-economics-script">.*?</script>','',s,flags=re.S)
assert s.count('<section id="strategies">')==1 and s.count('<section id="execution">')==1
s=s.replace('<section id="strategies">',cluster+'<section id="strategies">')
s=s.replace('<section id="execution">',economy+'<section id="execution">')
s=s.replace('<div class="eyebrow">03 · 실행과 검증</div>','<div class="eyebrow">04 · 실행과 검증</div>')
s=s.replace('<div class="eyebrow">04 · 근거자료</div>','<div class="eyebrow">05 · 근거자료</div>')
if '.econ-grid{' not in s:s=s.replace('</style>',extra_css+'</style>',1)
s=s.replace('시설 연관분석의 의미부터 런던·바스크·오슬로·뮌헨·말라가·세비야·발렌시아의 규제, 상품 제안, 운영 시나리오까지.','군집·연관분석과 실제 후보의 연결부터 7개 지역의 상품 경로, 호스트 참여 조건과 손익분기 시나리오까지.')
s=s.replace('<title>InsightStay · 시설 연관분석과 7개 지역 장기상품 전략</title>','<title>InsightStay · 군집 근거와 손익분기를 연결한 7개 지역 장기상품 전략</title>')
if 'href="#cluster-evidence"' not in s:s=s.replace('<strong>7지역 전략</strong>','<a href="#cluster-evidence">군집 E4와 후보 연결</a><strong>7지역 전략</strong>')
if 'href="#economics"' not in s:s=s.replace('<strong>실행·근거</strong>','<strong>실행·근거</strong><a href="#economics">참여 이유·손익분기</a>')
s=s.replace('보정 결과는 기존 산출물을 사용했으며 새 모델을 적합하지 않았다.','연관분석 보정과 군집 배정은 기존 산출물을 사용했다. 추가 조정 회귀는 별도 완료 보고서에서 인용했다. 이번 통합 단계에서는 새 모델을 적합하지 않고 ID 연결과 가상 손익 계산을 검산했다.')
new_links='<a download href="E4_현재후보_지역별연결.csv">E4와 현재 후보 연결 ↓</a><a download href="손익분기_가상시나리오.csv">가상 손익분기 3개 시나리오 ↓</a><a download href="손익분기_가정과계산.json">손익 가정·산식·검산 ↓</a>'
if 'href="E4_현재후보_지역별연결.csv">E4와' not in s:s=s.replace('<div class="downloads">','<div class="downloads">'+new_links,1)
s=s.replace('</body>','<script id="cluster-economics-script">'+js_engine+'\\n'+js_ui+'</script></body>')
for region in regions:
 pattern=rf'<article class="region" id="{re.escape(region)}">.*?</article>'
 assert re.search(pattern,old,re.S).group()==re.search(pattern,s,re.S).group()
assert '3·6번을 기본 참조로' in s
class Check(HTMLParser):
 def __init__(self):super().__init__();self.ids=[];self.refs=[]
 def handle_starttag(self,tag,attrs):
  a=dict(attrs)
  if 'id' in a:self.ids.append(a['id'])
  if tag=='a' and a.get('href','').startswith('#'):self.refs.append(a['href'][1:])
check=Check();check.feed(s);assert len(check.ids)==len(set(check.ids));assert all(a in check.ids for a in check.refs)
assert hashlib.sha256(F.read_bytes()).hexdigest()==old_sha,'Input changed during build; reread before editing'
hist=P/'_history';hist.mkdir(exist_ok=True)
backup=hist/('통합전략_수정전_'+old_sha[:12]+'.html')
if not backup.exists():shutil.copy2(F,backup)
temp=F.with_suffix('.tmp');temp.write_text(s);temp.replace(F)
meta_path=P/'선별기준_출처_검산.json';meta=json.loads(meta_path.read_text())
meta['cluster_economics_addition']={'updated_at':datetime.now().astimezone().isoformat(),'cluster_candidate_n':len(matched),'cluster_label':'E_K4_4','selected_7region_E4_n':len(e4),'candidate_is_subset_of_E4':True,'cluster_test_silhouette':kj['test']['results']['Entire home/apt']['silhouette_test'],'silhouette_sample_n':1500,'silhouette_population_test_n':75850,'host_break_even_is_assumption':True,'regression_limit_disclosed':True,'scenario_inputs_file':'손익분기_가정과계산.json','source_candidate_sha256':hashlib.sha256((P/'7개지역_시설후보_ID.csv').read_bytes()).hexdigest(),'source_cluster_sha256':hashlib.sha256((K/'숙소별_군집.csv.gz').read_bytes()).hexdigest(),'source_cluster_summary':str(K/'결과요약.json'),'source_regression_summary':str(R/'결과요약.json'),'all_regional_article_text_preserved':True,'original_html_sha256':old_sha,'updated_html_sha256':hashlib.sha256(F.read_bytes()).hexdigest(),'numeric_crosscheck_python_javascript':True}
meta_path.write_text(json.dumps(meta,ensure_ascii=False,indent=2))
print(json.dumps({'updated_html':str(F),'candidate_n':len(matched),'E4_7regions_n':len(e4),'default_host_change':b['hostChange'],'default_platform_change':b['platformChange'],'joint_monthly_floor':b['jointRentFloor'],'backup':str(backup),'checks':'numeric Python/JS agreement, IDs, anchors, regional text preservation passed'},ensure_ascii=False))
"""
SOURCE_META['report_update'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_장기상품_통합전략_20261003/update_cluster_economics.py', 'sha256': 'b91ab7cc53b468f523e36fd58f22897471cdb01667844f1d15784921b7f93c81', 'language': 'python'}


### economics_js — 보고서 인터랙티브 계산기의 JavaScript
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_장기상품_통합전략_20261003/손익계산.js`
SHA256: `22ba23927bd5ac598ae81f05f66434dc5b300a936c56c2ed3906b5534d987334`


In [22]:
SOURCE_CODE['economics_js'] = """// Same-period illustrative accounting. No market forecast is encoded here.
function calculateScenario(x) {
  const gl=x.months*x.monthlyRent, gs=x.displacedNights*x.shortNightlyPrice;
  const shortHost=x.displacedNights*(x.shortNightlyPrice*(1-x.hostShortFee)-x.shortNightlyCost);
  const hostOpportunity=Math.max(shortHost,x.otherHostAlternative);
  const hostBeforeSupport=gl*(1-x.hostLongFee)-x.months*x.longMonthlyCost-x.hostSetupCost-hostOpportunity;
  const platformBeforeSupport=x.platformLongFee*gl-x.platformShortFee*gs-x.platformCost-x.otherPlatformLoss;
  const hostChange=hostBeforeSupport+x.support;
  const platformChange=platformBeforeSupport-x.support;
  const hostRentFloor=Math.max(0,(x.months*x.longMonthlyCost+x.hostSetupCost+hostOpportunity-x.support)/(x.months*(1-x.hostLongFee)));
  const platformRentFloor=Math.max(0,(x.platformShortFee*gs+x.platformCost+x.support+x.otherPlatformLoss)/(x.months*x.platformLongFee));
  const supportMin=Math.max(0,-hostBeforeSupport),supportMax=platformBeforeSupport;
  return {longGMV:gl,displacedGMV:gs,hostOpportunity,hostBeforeSupport,platformBeforeSupport,
    hostChange,platformChange,hostRentFloor,platformRentFloor,jointRentFloor:Math.max(hostRentFloor,platformRentFloor),
    supportMin,supportMax,supportFeasible:supportMax>=supportMin,bothNonnegative:hostChange>=0&&platformChange>=0};
}
if (typeof module !== 'undefined' && module.exports) module.exports={calculateScenario};
"""
SOURCE_META['economics_js'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/outputs/7개지역_장기상품_통합전략_20261003/손익계산.js', 'sha256': '22ba23927bd5ac598ae81f05f66434dc5b300a936c56c2ed3906b5534d987334', 'language': 'javascript'}


### association_audit — 시설 후보 독립 스트리밍 검산
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/independent_recount.py`
SHA256: `882a46349bca8d2032cb88cbded483556680c031e2ac5b265ef3512253c85ef3`


In [23]:
SOURCE_CODE['association_audit'] = """import csv,gzip,hashlib,json
from collections import Counter,defaultdict
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

ROOT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26')
COMMON=ROOT/'outputs/InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터'
OUT=ROOT/'outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합'
defs={'L':['kitchen','heating_food','refrigerator','tools','washer'],'W':['wifi','workspace'],'LW':['kitchen','heating_food','refrigerator','tools','washer','wifi','workspace']}
host_totals=defaultdict(Counter);host_observed=defaultdict(Counter);counts=defaultdict(Counter);host_matches=defaultdict(Counter)
missing=Counter();tools=defaultdict(Counter);pool_counts=Counter();rows=0
with gzip.open(COMMON/'공통분석표.csv.gz','rt',encoding='utf-8-sig',newline='') as f:
    for r in csv.DictReader(f):
        rows+=1;p='reference' if r['sector'] in ['3','6'] else 'target';host=r['host_id'];pool_counts[p]+=1;host_totals[p][host]+=1
        vals={x:r['function__'+x] for x in defs['LW']};obs=all(v!='' for v in vals.values())
        if obs:host_observed[p][host]+=1
        else:missing[p]+=1
        tools[p][vals['tools']]+=1
        flags={b:all(vals[x]!='' and float(vals[x])==1 for x in fs) for b,fs in defs.items()}
        flags['ANY']=flags['L'] or flags['W']
        for b,yes in flags.items():
            if yes:counts[p][b]+=1;host_matches[(p,b)][host]+=1
result={'task_id':'POOLED_LONG_PRODUCT_20261003','method':'independent Python csv/gzip streaming; direct equality comparisons; no pandas aggregate reuse','rows':rows,'pools':{}}
summary=json.loads((OUT/'결과요약.json').read_text())
for p in ['reference','target']:
    result['pools'][p]={'n':pool_counts[p],'observed_n':pool_counts[p]-missing[p],'unobserved_n':missing[p],'host_n':len(host_totals[p]),'bundles':{}}
    weighted_obs=sum(n/host_totals[p][h] for h,n in host_observed[p].items())
    for b in ['L','W','LW','ANY']:
        weighted_match=sum(n/host_totals[p][h] for h,n in host_matches[(p,b)].items())
        s=next(x for x in summary['pooled_bundles'] if x['pool']==p and x['bundle']==b)
        v={'match_n':counts[p][b],'host_n':len(host_matches[(p,b)]),'host_weighted_support_observed':weighted_match/weighted_obs}
        v['matches_main']=v['match_n']==s['match_n'] and v['host_n']==s['match_host_n'] and abs(v['host_weighted_support_observed']-s['host_weighted_support_observed'])<1e-9
        assert v['matches_main'],(p,b,v,s)
        result['pools'][p]['bundles'][b]=v
    result['pools'][p]['tools_value_counts']=dict(tools[p])
assert rows==677678
result['all_pass']=True
result['created_at']=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
result['source_hashes']={}
for name in ['공통분석표.csv.gz','피처사전.json','컬럼명세.json']:
    h=hashlib.sha256()
    with open(COMMON/name,'rb') as f:
        for v in iter(lambda:f.read(1024*1024),b''):h.update(v)
    result['source_hashes'][name]=h.hexdigest()
(OUT/'독립재집계.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
check=json.loads((OUT/'검산.json').read_text());check['independent_recount']=result;check['all_pass']=check['all_pass'] and result['all_pass']
(OUT/'검산.json').write_text(json.dumps(check,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(result,ensure_ascii=False))
"""
SOURCE_META['association_audit'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/independent_recount.py', 'sha256': '882a46349bca8d2032cb88cbded483556680c031e2ac5b265ef3512253c85ef3', 'language': 'python'}


### association_report — 초기 시설조합 보고서 생성
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_report.py`
SHA256: `e26a06ae77a285d00f5b60077a7941f57a26fa48a7455036b1cee7c0a1501015`


In [24]:
SOURCE_CODE['association_report'] = """import html,json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd

OUT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합')
S=json.loads((OUT/'결과요약.json').read_text());P=json.loads((OUT/'분석기준_사전고정.json').read_text())
stats=pd.read_csv(OUT/'묶음_전체집계.csv');func=pd.read_csv(OUT/'기능_등록률.csv');groups=pd.read_csv(OUT/'묶음_지역섹터유형별.csv');rules=pd.read_csv(OUT/'조건부동시등록_3규칙.csv');split=pd.read_csv(OUT/'묶음_탐색확인.csv');std=pd.read_csv(OUT/'구성표준화.csv');regstab=pd.read_csv(OUT/'지역_train_validation_안정성.csv');obs=pd.read_csv(OUT/'시설_관측상태.csv');exact=pd.read_csv(OUT/'정확등록패턴_top10.csv');dist=pd.read_csv(OUT/'생활기능_분포.csv')
def n(v):return f'{int(v):,}'
def pc(v):return '관측 불가' if pd.isna(v) else f'{100*v:.1f}%'
def pp(v):return '관측 불가' if pd.isna(v) else f'{100*v:+.1f}%p'
def e(v):return html.escape(str(v))
def get(pool,b,sp='all'):return stats[(stats.pool==pool)&(stats.bundle==b)&(stats.split==sp)].iloc[0]
def table(headers,rows):return '<div class="table-wrap"><table data-cols="'+str(len(headers))+'"><thead><tr>'+''.join('<th>'+e(x)+'</th>' for x in headers)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+str(v)+'</td>' for v in r)+'</tr>' for r in rows)+'</tbody></table></div>'
def details(title,body):return '<details><summary>'+e(title)+'</summary>'+body+'</details>'
labels={'L':'생활형','W':'업무형','LW':'생활·업무 결합형','ANY':'중복 제거 합집합'}
flabels={'kitchen':'주방','heating_food':'취사용 가열','refrigerator':'냉장고','tools':'조리도구·식기 완전등록','washer':'세탁기','wifi':'Wi-Fi','workspace':'전용 작업공간'}
ref=S['populations']['reference'];target=S['populations']['target'];union=get('target','ANY')
core=get('target','LW');l=get('target','L');w=get('target','W')
topref=groups[(groups.pool=='reference')&(groups.dimension=='region')&(groups.bundle=='ANY')].sort_values('pool_n',ascending=False)
top5=topref.head(5);top5_share=top5.pool_n.sum()/ref['n']
eligible=regstab[regstab.reporting_eligible];regional_summary=[]
for b in ['L','W','LW']:
    r=eligible[eligible.bundle==b]
    regional_summary.append({'bundle':b,'eligible_regions_n':len(r),'regions_abs_gap_gt5pp_n':int(r.gap.abs().gt(.05).sum()),'min_gap':float(r.gap.min()),'max_gap':float(r.gap.max())})
S['internal_confirmation']={'selected':['L','W','LW'],'all_predeclared_train_validation_checks_passed':True,'regional_stability':regional_summary,'test_descriptive_sensitivity':[{'bundle':b,'test_support_observed':float(get('reference',b,'test').support_observed),'test_host_weighted_support_observed':float(get('reference',b,'test').host_weighted_support_observed),'test_minus_train_gap':float(get('reference',b,'test').support_observed-get('reference',b,'train').support_observed)} for b in ['L','W','LW']]}
S['main_findings']=[f'대상 중 생활형 {n(l.match_n)}곳, 업무형 {n(w.match_n)}곳, 두 묶음 교집합 {n(core.match_n)}곳. 합집합 {n(union.match_n)}곳·{n(union.match_host_n)}호스트 ({pc(union.support_all)}; 전체 대상 분모).',f'장기 설정 공급의 관측기준 빈도는 생활형 {pc(get("reference","L").support_observed)}, 업무형 {pc(get("reference","W").support_observed)}, 결합형 {pc(get("reference","LW").support_observed)}. 결합형은 정확 7기능 상태패턴 중 가장 빈번함.',f'결합형 호스트 동일가중 빈도는 {pc(get("reference","LW").host_weighted_support_observed)}로 숙소기준보다 낮음. test 숙소기준 {pc(get("reference","LW","test").support_observed)}; 지역별 균일 재현 미확인.',f'지역×room_type×수용인원 공통층에서 결합형 빈도 {pc(std[(std.bundle=="LW")&(std.dimension=="region+room_type+accommodates_band")].iloc[0].reference_rate_common)} 대 {pc(std[(std.bundle=="LW")&(std.dimension=="region+room_type+accommodates_band")].iloc[0].target_rate_standardized_to_reference)}. 장기에만 특유한 묶음으로 해석할 근거 약함.']
S['next_steps']=['내부 후보매핑을 id 문자열로 호스트 집계에 연결하며 지역·섹터별 구성과 중복을 유지','생활형은 시설 접근권한·공용/유료 세탁기·조리 가능범위, 업무형은 Wi-Fi와 전용 작업공간 상태를 확인해 설명문 검토','기존 최소박은 유지하고 상품 설명 추가를 검토. 실제 목적·문의·장기예약 관측을 별도로 모으기 전 수요·매출 효과를 주장하지 않음']
S['evidence_files']=[x.name for x in sorted(OUT.glob('*.csv'))]+['후보매핑.csv.gz','독립재집계.json','검산.json']
(OUT/'결과요약.json').write_text(json.dumps(S,ensure_ascii=False,indent=2),encoding='utf-8')

intro=f'''<header><div class="eyebrow">InsightStay · 2026.10.03</div><h1>장기 설정 공급에서 반복되는<br>생활·업무 시설 묶음</h1><p class="subtitle">30박 이상으로 설정된 3·6번 {n(ref['n'])}곳을 통합해 시설의 동시등록을 확인하고, 1·2·4·5번 {n(target['n'])}곳에 같은 기준을 적용했습니다.</p></header>
<div class="kpis"><div><span>장기 설정 참조 공급</span><strong>{n(ref['n'])}</strong><small>{n(ref['host_n'])}호스트 · 53개 지역</small></div><div><span>상품 추가 검토 대상</span><strong>{n(target['n'])}</strong><small>기존 minimum_nights 유지</small></div><div class="accent"><span>시설 기준 합집합</span><strong>{n(union.match_n)}</strong><small>전체 대상의 {pc(union.support_all)} · {n(union.match_host_n)}호스트</small></div></div>'''

summary_rows=[]
for b in ['L','W','LW','ANY']:
    a=get('reference',b);z=get('target',b)
    summary_rows.append([e(labels[b]),n(a.match_n),pc(a.support_observed),pc(a.host_weighted_support_observed),n(z.match_n),n(z.match_host_n),pc(z.support_all)])
requirements=[['생활형 (L)','주방 + 취사용 가열 + 냉장고 + 조리도구·식기 완전등록 + 세탁기'],['업무형 (W)','Wi-Fi + 전용 작업공간'],['결합형 (LW)','생활형과 업무형 모두. 7기능 완전등록의 교집합']]
section1='<section><h2>같은 시설 기준으로 구분한 검토 범위</h2><p>세 묶음은 탐색 전 고정했고 모두 train·validation 반복성 기준을 통과했습니다. 결합형은 생활형과 업무형의 교집합이므로 추가 숙소 수로 합산하지 않습니다.</p>'+table(['묶음','필수 동시등록'],requirements)+table(['묶음','참조 숙소 수','참조 관측빈도','참조 호스트가중 빈도','대상 숙소 수','대상 고유호스트','대상 전체 대비'],summary_rows)+f'<div class="note">관측빈도 분모: 참조 {n(ref["all7_observed_n"])}곳. 대상 전체 대비 분모: {n(target["n"])}곳. 호스트가중 빈도는 각 pool에서 1/호스트 숙소 수로 재계산한 가중치를 사용합니다. 묶음을 갖춘 호스트의 비율과는 다른 지표입니다.</div></section>'

only_l=int(l.match_n-core.match_n);only_w=int(w.match_n-core.match_n);both=int(core.match_n);unknown=int(union.unknown_membership_n);neither=target['n']-int(union.match_n)-unknown
segments=[('생활형만',only_l,'#287c73'),('업무형만',only_w,'#5377b3'),('두 묶음 모두',both,'#c98142'),('두 묶음 모두 미충족',neither,'#ced4dc'),('관측 불가',unknown,'#725d79')]
stack='<div class="stack" role="img" aria-label="시설 묶음의 중복을 제거한 대상 분포">'+''.join(f'<div style="width:{v/target["n"]*100:.8f}%;background:{c}" title="{e(label)} {n(v)}곳"></div>' for label,v,c in segments)+'</div>'
legend='<div class="legend">'+''.join(f'<div><i style="background:{c}"></i><span>{e(label)}</span><b>{n(v)}곳</b></div>' for label,v,c in segments)+'</div>'
ov=next(x for x in S['overlap'] if x['pool']=='target' and x['bundle_a']=='L' and x['bundle_b']=='W')
section2=f'<section><h2>중복 제거와 관측 불가</h2>{stack}{legend}<p class="equation">{n(l.match_n)} + {n(w.match_n)} − {n(core.match_n)} = <strong>{n(union.match_n)}곳</strong></p><p>생활형 또는 업무형을 갖춘 범위를 합집합으로 제시합니다. 업무형만 해당하는 {n(only_w)}곳에는 생활형 5기능을 모두 요구하지 않습니다. ‘미충족’은 등록 기준의 미충족이며 실제 시설 부재 판정이 아닙니다.</p><p>호스트는 다른 숙소에서 각각 생활형과 업무형을 갖출 수도 있습니다. 두 유형을 보유한 호스트의 교집합은 {n(ov["host_overlap_any_listing_n"])}명이고, 같은 숙소에 두 유형을 모두 갖춘 결합형 호스트는 {n(core.match_host_n)}명입니다. 따라서 호스트 합집합도 따로 중복을 제거했습니다.</p><div class="note">시설을 판정할 수 없는 대상 {n(unknown)}곳은 미등록으로 바꾸지 않고 NA를 보존했습니다. 이 자료에서는 7기능 결측이 함께 발생합니다. 내부 연결용 매핑은 대상 {n(target["n"])}개 id 전체를 담으며, 이 보고서는 집계만 제시합니다.</div></section>'

func_rows=[]
for f in P['functions']:
    rr=func[(func.pool=='reference')&(func.function==f)];tt=func[(func.pool=='target')&(func.function==f)]
    r=rr[rr.state=='완전등록'].iloc[0];t=tt[tt.state=='완전등록'].iloc[0]
    func_rows.append([e(flabels[f]),n(r.n),pc(r.rate_observed),pc(r.host_weighted_rate_observed),pc(t.rate_observed)])
partial=func[(func.pool=='reference')&(func.function=='tools')&(func.state=='부분등록')].iloc[0]
all_dist=dist[(dist.pool=='reference')&(dist.group=='전체7기능')]
distribution_rows=[[e(r.full_registered_functions)+'개' if r.full_registered_functions!='unobserved' else '관측 불가',n(r.n),pc(r.rate_observed)] for r in all_dist.itertuples()]
exact_rows=[]
for r in exact[(exact.pool=='reference')&(exact.split=='all')].itertuples():
    state=' / '.join(f'{getattr(r,f):g}' for f in P['functions'])
    exact_rows.append([r.rank,'<code>'+e(state)+'</code>',n(r.n),pc(r.support_observed)])
obs_rows=[[e(r.pool),e(r.observation_state),n(r.n)] for r in obs.itertuples()]
section3='<section><h2>장기 설정 공급의 등록 상태</h2>'+table(['기능','참조 완전등록 수','참조 관측등록률','호스트가중 등록률','대상 관측등록률'],func_rows)+f'<p>주방 등록률은 90.1%지만 취사용 가열·냉장고·조리도구 완전등록은 각각 53.7%·57.2%·47.5%입니다. 주방 한 항목 등록만으로 나머지 생활 기능이 관측됐다고 간주하지 않았습니다. 조리도구 부분등록은 {n(partial.n)}곳({pc(partial.rate_observed)})이며 생활형의 완전등록 조건에 포함하지 않았습니다.</p><div class="note">0=미등록, NA=관측 판단 불가. tools=0.5는 조리 기본도구 또는 식기·수저 중 하나만 등록, tools=1은 둘 모두 등록입니다. 취사용 가열은 stove/oven/microwave 중 하나 이상이며 난방과 다릅니다. 세탁기 등록에는 공용·유료 항목도 포함되고 주방에는 kitchenette가 포함됩니다.</div>'+details('완전등록 기능 수와 정확 상태패턴',table(['완전등록 기능 수','참조 숙소 수','관측 대비'],distribution_rows)+'<p>상태패턴 순서: 주방 / 취사용 가열 / 냉장고 / 조리도구 / 세탁기 / Wi-Fi / 전용 작업공간. 실제 관측된 정확 상태를 세었으며 모든 부분조합을 탐색한 결과가 아닙니다. 0.5를 별도 상태로 유지했습니다.</p>'+table(['순위','정확 상태','참조 숙소 수','관측빈도'],exact_rows))+details('시설 관측상태',table(['pool','관측 상태','숙소 수'],obs_rows))+ '</section>'

rule_names={'R1':('식생활 4기능 완전등록','세탁기'),'R2':('Wi-Fi','전용 작업공간'),'R3':('생활형 5기능','업무형 2기능')}
rule_rows=[]
for r in rules[(rules.pool=='reference')&(rules.split=='all')].itertuples():
    a,b=rule_names[r.rule]
    rule_rows.append([e(a)+' → '+e(b),n(r.AB_n),pc(r.support),pc(r.confidence),pc(r.consequent_rate),f'{r.lift:.2f}',f'{r.host_lift:.2f}'])
section4='<section><h2>동시등록 관계는 어느 정도인가</h2>'+table(['조건부 등록 방향','A∩B 수','support','confidence','B 전체등록률','lift','호스트가중 lift'],rule_rows)+'<p>생활형을 갖춘 숙소 중 업무형도 등록된 비율은 59.0%이며, 전체 업무형 등록률 41.2% 대비 lift는 1.43입니다. 식생활 4기능과 세탁기, Wi-Fi와 작업공간의 lift는 약 1.13·1.09로, 함께 등록되지만 결합의 추가 강도는 크지 않습니다.</p><div class="note">동일 관측분모 37,701곳에서 support=P(A∩B), confidence=P(B|A), lift=P(B|A)/P(B)로 계산했습니다. 화살표는 조건부 등록률의 방향만 표시합니다. 시간 순서·인과·예약 효과나 같은 묶음에 대한 고객 선호를 뜻하지 않습니다. 모든 지표의 분자·분모와 split별 값은 근거 CSV에 있습니다.</div></section>'

split_rows=[]
for b in ['L','W','LW']:
    tr=get('reference',b,'train');va=get('reference',b,'validation');te=get('reference',b,'test')
    split_rows.append([e(labels[b]),pc(tr.support_observed),pc(va.support_observed),pp(va.support_observed-tr.support_observed),pc(tr.host_weighted_support_observed),pc(va.host_weighted_support_observed),pc(te.support_observed),pc(te.host_weighted_support_observed)])
reg_rows=[[e(labels[r['bundle']]),r['eligible_regions_n'],r['regions_abs_gap_gt5pp_n'],pp(r['min_gap']),pp(r['max_gap'])] for r in regional_summary]
section5='<section><h2>탐색과 확인의 구분, 재현 범위</h2><p>7기능 정의와 세 묶음, 빈도·안정성 기준을 결과 계산 전에 저장했습니다. 호스트가 겹치지 않는 기존 split의 train만으로 반복성을 확인해 탐색 결과를 고정한 뒤 validation을 확인했습니다. 두 집단 사이 호스트 중복은 0명입니다. 적용 대상과 test 빈도는 묶음 선정에 사용하지 않았습니다.</p>'+table(['묶음','train 숙소','validation 숙소','차이','train 호스트가중','validation 호스트가중','test 숙소','test 호스트가중'],split_rows)+'<div class="note">사전 기준: train 관측빈도·호스트가중 빈도 각각 10% 이상, 등록 500숙소·200호스트 이상. validation 각각 10% 이상, 100숙소·50호스트 이상. 두 빈도 모두 절대차 5%p 이내이고 validation/train 비율 0.8–1.25. 세 묶음 모두 통과했습니다. 분석 후 기준을 낮추지 않았습니다.</div><p class="caution">선정 후 기술 점검한 test에서 결합형 숙소 빈도는 14.7%로 train 19.7%보다 5.0%p 낮습니다. 호스트 동일가중 빈도는 13.9%로 train 14.7%와 더 가깝습니다. 참조 전체에서도 결합형 빈도는 숙소기준 19.2%에서 호스트 동일가중 14.5%로 낮아집니다. 여러 숙소를 보유한 호스트의 구성에 민감하므로 전체 비율을 개별 호스트에 그대로 적용할 수 없습니다.</p>'+table(['묶음','train·validation 각각 관측100+ 지역','절대차 5%p 초과 지역','지역 차이 최소','지역 차이 최대'],reg_rows)+'<p>지역별 비교 가능한 10개 지역에서도 통합 수준의 안정성이 균일하게 나타나지는 않았습니다. 소표본 지역을 후보에서 제외하는 기준으로 쓰지는 않았고, 지역별 수와 관측분모를 모두 보존했습니다.</p><div class="note">이 자료와 기존 host split 결과는 과거에 분석·열람된 적이 있습니다. 이번 절차는 탐색과 확인을 구분한 제한된 내부 재현이며 신규 외부 독립확증으로 볼 수 없습니다. test는 선정조건을 사후 변경하는 데 사용하지 않았습니다.</div></section>'

main_std=std[std.dimension=='region+room_type+accommodates_band']
stdrows=[[e(labels[r.bundle]),pc(r.reference_raw_rate),pc(r.target_raw_rate),pc(r.reference_rate_common),pc(r.target_rate_standardized_to_reference),pp(r.standardized_gap)] for r in main_std.itertuples()]
fullstd=[[e(r.dimension),e(labels[r.bundle]),r.included_strata_n,pc(r.reference_coverage),pc(r.target_coverage),pc(r.reference_rate_common),pc(r.target_rate_standardized_to_reference),pp(r.standardized_gap)] for r in std.itertuples()]
room_rows=[]
for r in groups[(groups.pool=='reference')&(groups.dimension=='room_type')&(groups.bundle=='L')].itertuples():
    rr=groups[(groups.pool=='reference')&(groups.dimension=='room_type')&(groups.group==r.group)]
    room_rows.append([e(r.group),n(r.pool_n),pc(r.pool_n/ref['n']),pc(r.support_observed),pc(rr[rr.bundle=='W'].iloc[0].support_observed),pc(rr[rr.bundle=='LW'].iloc[0].support_observed)])
prop=groups[(groups.pool=='reference')&(groups.dimension=='property_type')&(groups.bundle=='ANY')].sort_values('pool_n',ascending=False).head(10)
prop_rows=[[e(r.group),n(r.pool_n),pc(r.pool_n/ref['n']),n(r.match_n)] for r in prop.itertuples()]
accom_rows=[]
for r in groups[(groups.pool=='reference')&(groups.dimension=='accommodates_band')&(groups.bundle=='LW')].itertuples():
    t=groups[(groups.pool=='target')&(groups.dimension=='accommodates_band')&(groups.bundle=='LW')&(groups.group==r.group)].iloc[0]
    accom_rows.append([e(r.group),n(r.pool_n),pc(r.pool_n/ref['n']),pc(r.support_observed),pc(t.support_observed)])
section6=f'<section><h2>지역·숙소 유형 구성이 결과를 주도하는가</h2><p>참조 공급은 상위 5개 지역({", ".join(top5.group)})에 {pc(top5_share)}가 몰려 있습니다. Entire rental unit이 54.3%를 차지하고, Entire home/apt는 77.3%, 1–2인 수용 숙소는 45.6%입니다. 생활형은 Entire home/apt에서 36.4%, Private room에서 19.5%로 다르고, 업무형은 각각 38.7%·49.9%입니다. 전체 비율의 지역·유형 구성을 함께 봐야 합니다.</p>'+table(['묶음','참조 원빈도','대상 원빈도','공통층 참조','참조 구성으로 맞춘 대상','참조−대상'],stdrows)+'<p>지역×room_type×수용인원 구성을 함께 맞춘 공통층에서 결합형은 19.4% 대 19.1%로 차이가 거의 없습니다. 생활형·업무형은 여전히 대상 공급에서 더 흔합니다. 따라서 반복되는 시설 묶음은 찾았지만 장기 설정 공급에만 특유한 구성이라고 해석할 근거는 약합니다.</p><div class="note">두 pool 모두 관측 30숙소 이상인 116개 공통층에서 대상 등록률을 참조 관측구성으로 가중했습니다. 참조 관측행의 95.4%, 대상 관측행의 64.6%를 포함하며 606개 희소·비공통층은 제외됐습니다. 이 비율은 비교 가능 범위이며 대상 후보의 탈락 기준이 아닙니다. 표준화는 인과추정이 아닙니다.</div>'+details('한 요인씩 맞춘 비교와 포함률',table(['요인','묶음','포함 층','참조 포함률','대상 포함률','공통층 참조','표준화 대상','차이'],fullstd))+details('참조 room_type·property_type·수용인원 분포',table(['room_type','참조 수','참조 구성비','생활형','업무형','결합형'],room_rows)+table(['상위 property_type','참조 수','참조 구성비','합집합 숙소 수'],prop_rows)+table(['수용인원','참조 수','참조 구성비','참조 결합형','대상 결합형'],accom_rows))+'</section>'

sectors=[]
sectorlabels={'1':'1–3박 · ltm 양수','2':'4–29박 · ltm 양수','4':'1–3박 · ltm 0','5':'4–29박 · ltm 0'}
for sector in ['1','2','4','5']:
    ss=groups[(groups.pool=='target')&(groups.dimension=='sector')&(groups.group.astype(str)==sector)]
    a=ss[ss.bundle=='ANY'].iloc[0]
    sectors.append([sector,e(sectorlabels[sector]),n(a.pool_n),n(ss[ss.bundle=='L'].iloc[0].match_n),n(ss[ss.bundle=='W'].iloc[0].match_n),n(ss[ss.bundle=='LW'].iloc[0].match_n),n(a.match_n),n(a.match_host_n)])
region_rows=[]
allregions=[]
for rr in groups[(groups.pool=='target')&(groups.dimension=='region')&(groups.bundle=='ANY')].sort_values('match_n',ascending=False).itertuples():
    ss=groups[(groups.pool=='target')&(groups.dimension=='region')&(groups.group==rr.group)]
    row=[e(rr.group),n(rr.pool_n),n(ss[ss.bundle=='L'].iloc[0].match_n),n(ss[ss.bundle=='W'].iloc[0].match_n),n(ss[ss.bundle=='LW'].iloc[0].match_n),n(rr.match_n),n(rr.match_host_n),pc(rr.support_all)]
    allregions.append(row)
region_rows=allregions[:10]
regheads=['지역','대상 전체','생활형','업무형','결합형','합집합','합집합 호스트','전체 대비']
section7='<section><h2>대상 섹터·지역별 적용 범위</h2>'+table(['섹터','정의','대상 전체','생활형','업무형','결합형','합집합','고유호스트'],sectors)+'<div class="note">ltm은 섹터 정의의 맥락 정보입니다. 리뷰 유무를 예측하거나 3·6번을 성공·실패로 비교하지 않았습니다. 각 섹터·지역의 고유호스트 수에는 서로 중복되는 호스트가 있어 단순 합산하지 않습니다.</div><h3>합집합 숙소 수가 많은 10개 지역</h3>'+table(regheads,region_rows)+details('전체 53개 지역',table(regheads,allregions))+'<p>지역별 섹터 교차표와 room_type·property_type·수용인원별 숙소 수·호스트 수·관측률·해당률은 근거 CSV에 모두 포함했습니다.</p></section>'

limits=''.join('<li>'+e(x)+'</li>' for x in S['interpretation_limits'])
nexts=''.join('<li>'+e(x)+'</li>' for x in S['next_steps'])
links=[('묶음_전체집계.csv','통합 집계·분모·가중치'),('묶음_탐색확인.csv','train·validation'),('기능_등록률.csv','7기능 등록률·부분등록'),('조건부동시등록_3규칙.csv','support·confidence·lift'),('묶음_지역섹터유형별.csv','지역·섹터·숙소 유형'),('대상_지역섹터별.csv','대상 53지역×4섹터'),('묶음_중복.csv','숙소·호스트 교집합'),('묶음_배타패턴.csv','중복 없는 상태패턴'),('지역_train_validation_안정성.csv','지역별 내부 안정성'),('구성표준화.csv','표준화 요약'),('구성표준화_층별근거.csv','비교 가능층·제외층'),('생활기능_분포.csv','생활기능 분포'),('시설_관측상태.csv','관측상태'),('정확등록패턴_top10.csv','정확 상태 top 10'),('분석기준_사전고정.json','사전 정의·선정기준'),('결과요약.json','기계 판독 요약'),('검산.json','id·대상 누락·독립 재집계')]
download='<div class="downloads">'+''.join(f'<a href="{e(file)}">{e(label)}<small>{e(file)}</small></a>' for file,label in links)+'</div>'
section8='<section><h2>해석 범위와 다음 연결</h2><p>생활형은 식생활·세탁 기능을 함께 설명할 수 있는 등록 범위, 업무형은 통신·전용 작업공간을 설명할 수 있는 등록 범위입니다. 결합형은 두 설명을 함께 검토할 수 있는 교집합입니다. 이 단계의 후보 구분은 새 군집을 학습한 결과가 아니라 고정된 시설 기준의 적용 결과입니다.</p><ul>'+limits+'</ul><h3>다음 작업</h3><ol>'+nexts+'</ol></section>'
section9='<section><h2>근거와 검산</h2><p>입력은 지정된 공통분석표·피처사전·컬럼명세를 읽기 전용으로 사용했습니다. 677,678행의 id 중복·누락, 53개 지역, 대상 639,774개 id의 매핑 완전성, 7기능 원항목 계산의 일치, 호스트 분할 중복, 통합 pool 가중치, 숙소·호스트 합집합을 검산했습니다. 별도 CSV 스트리밍 계산으로 숙소 수·고유호스트 수·호스트가중 빈도를 다시 확인했습니다.</p>'+download+'<p class="note">입력 출처: InsightStay_지역특화장기상품_확장검증_20261003/00_공통데이터. 입력 SHA-256과 검산 상세는 검산.json 및 독립재집계.json에 기록했습니다. 원자료의 수집일 컬럼이 없으므로 보고서 작성일을 자료 수집일로 해석하지 않습니다.</p></section>'

css='''*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:#eef2f3;color:#1b2935;font-family:-apple-system,BlinkMacSystemFont,"Apple SD Gothic Neo","Noto Sans KR",Arial,sans-serif;line-height:1.75;font-size:15px}main{max-width:1180px;margin:32px auto;background:white;border:1px solid #dce4e7;border-radius:16px;overflow:hidden}header{padding:48px 46px 30px;background:#102f3b;color:#fff}.eyebrow{color:#9dd6d0;font-size:14px;letter-spacing:.08em;font-weight:650}h1{font-size:38px;line-height:1.35;letter-spacing:-.035em;margin:14px 0 20px}.subtitle{max-width:870px;color:#d5e3e9;font-size:17px}.kpis{display:grid;grid-template-columns:repeat(3,1fr);gap:18px;padding:28px 46px 8px}.kpis>div{padding:20px;border:1px solid #dfe7ea;border-radius:10px;background:#f6f9fa}.kpis .accent{background:#eaf5f2;border-color:#c4ded7}.kpis span,.kpis small{display:block}.kpis span{font-size:14px;color:#5b6b77}.kpis strong{display:block;font-size:32px;line-height:1.5;letter-spacing:-.03em}.kpis small{font-size:12px;color:#566771}section{padding:30px 46px;border-top:1px solid #e7edef}section:first-of-type{border-top:0}h2{font-size:23px;line-height:1.4;letter-spacing:-.025em;margin:0 0 18px}h3{font-size:18px;margin:24px 0 12px}p{margin:12px 0 18px}table{width:100%;border-collapse:collapse;font-size:13px;line-height:1.6}th{background:#f0f5f6;font-weight:650;text-align:left;color:#324955}td,th{padding:11px 12px;border-bottom:1px solid #e2e9ec;vertical-align:top}td{font-variant-numeric:tabular-nums}tr:hover td{background:#fafcfc}.table-wrap{overflow-x:auto;margin:16px 0;border:1px solid #dce5e8;border-radius:8px}td:not(:first-child){white-space:nowrap}td:first-child{min-width:85px}td code{font-size:12px}.note{font-size:13px;color:#526570;background:#f3f7f8;padding:13px 16px;border-radius:7px;margin:15px 0}.caution{background:#fff7ec;border-left:3px solid #c98142;padding:15px 18px;border-radius:4px}.equation{font-size:20px;font-weight:550;margin-top:22px}.stack{display:flex;height:42px;border-radius:8px;overflow:hidden;margin:22px 0}.legend{display:grid;grid-template-columns:repeat(3,1fr);gap:12px 20px;font-size:13px}.legend>div{display:flex;gap:8px;align-items:center}.legend i{width:12px;height:12px;flex-shrink:0;border-radius:3px}.legend b{margin-left:auto;white-space:nowrap}details{margin:18px 0;border:1px solid #dce6e9;border-radius:8px;padding:10px 14px}summary{cursor:pointer;font-weight:600;color:#235d62}details[open]>summary{margin-bottom:14px}ul,ol{padding-left:22px}li{margin:8px 0}.downloads{display:grid;grid-template-columns:repeat(3,1fr);gap:10px}.downloads a{color:#235d62;text-decoration:none;font-weight:600;display:block;background:#f3f7f8;border:1px solid #e0e8ea;border-radius:6px;padding:12px;overflow-wrap:anywhere}.downloads small{display:block;font-weight:400;font-size:11px;color:#697b85}footer{font-size:12px;padding:24px 46px;background:#f3f7f8;color:#61727c}@media(max-width:800px){main{margin:0;border-radius:0}header,section{padding:28px 20px}h1{font-size:29px}.kpis{padding:20px;grid-template-columns:1fr}.kpis strong{font-size:29px}.legend,.downloads{grid-template-columns:1fr 1fr}table{min-width:680px}}@media print{body{background:white}main{margin:0;border:0}header{background:white;color:#1b2935}.subtitle{color:#526570}.eyebrow{color:#235d62}details{display:block}details>*{display:block}.downloads{display:none}section{break-inside:avoid}.table-wrap{overflow:visible}h1{font-size:30px}}'''
doc='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>InsightStay 장기통합참조 시설조합</title><style>'+css+'</style></head><body><main>'+intro+section1+section2+section3+section4+section5+section6+section7+section8+section9+'<footer>POOLED_LONG_PRODUCT_20261003 · 시설조합 담당 · 작성일 2026-10-03 (Asia/Seoul)</footer></main></body></html>'
doc=doc.replace('</style>','table[data-cols="2"]{min-width:0}table[data-cols="2"] td:last-child{white-space:normal}table[data-cols="2"] td:first-child{width:24%;min-width:100px}</style>')
(OUT/'결과보고서.html').write_text(doc,encoding='utf-8')
print(json.dumps({'html':str(OUT/'결과보고서.html'),'bytes':len(doc.encode()),'target_union':int(union.match_n),'unknown':unknown,'host_overlap':ov,'regional_stability':regional_summary},ensure_ascii=False))
"""
SOURCE_META['association_report'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/build_report.py', 'sha256': 'e26a06ae77a285d00f5b60077a7941f57a26fa48a7455036b1cee7c0a1501015', 'language': 'python'}


### adjustment_report — 구성 보정 보고서 생성
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/report_adjustment.py`
SHA256: `e8386e477815651446e82e41d05fe01a856fcfe4583fefc7eb4e90306a890b31`


In [25]:
SOURCE_CODE['adjustment_report'] = """import hashlib,html,json,re
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd

OUT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합')
J=json.loads((OUT/'연관보정_결과요약.json').read_text());D=pd.read_csv(OUT/'연관보정_집계.csv');C=pd.read_csv(OUT/'연관보정_층별근거.csv');C=C[C.primary&C.eligible]
def e(x):return html.escape(str(x))
def pc(x):return f'{x*100:.1f}%'
def n(x):return f'{int(x):,}'
def tbl(head,rows):return '<div class="table-wrap"><table><thead><tr>'+''.join('<th>'+e(v)+'</th>' for v in head)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+e(v)+'</td>' for v in row)+'</tr>' for row in rows)+'</tbody></table></div>'
def v(rule,pool,weight='listing',dimension=None):
    sel=D[(D.rule==rule)&(D.pool==pool)&(D.weighting==weight)]
    return sel[sel.dimension==dimension].iloc[0] if dimension else sel[sel.primary].iloc[0]
names={'R1':'식생활 4기능 → 세탁기','R2':'Wi-Fi → 전용 업무공간','R3':'생활형 5기능 → Wi-Fi·전용 업무공간'}
poolnames={'reference':'장기형 (3+6)','target':'중단기형 (1+2+4+5)'}
rows=[];hostrows=[]
for rule in ['R1','R2','R3']:
    for pool in ['reference','target']:
        r=v(rule,pool);h=v(rule,pool,'host')
        rows.append([names[rule],poolnames[pool],pc(r.support_standardized),pc(r.confidence_standardized),f'{r.lift_standardized:.2f}',f'{r.lift_within_strata:.2f}'])
        hostrows.append([names[rule],poolnames[pool],pc(h.support_standardized),pc(h.confidence_standardized),f'{h.lift_standardized:.2f}',f'{h.lift_within_strata:.2f}'])
head=['등록 관계','집단','표준화 지지도','표준화 신뢰도','표준화 향상도','층내 기대 기준 향상도']
primary=next(x for x in J['cases'] if x['primary']);rc=primary['coverage']['reference'];tc=primary['coverage']['target']
r=v('R3','reference');t=v('R3','target');rh=v('R3','reference','host');th=v('R3','target','host')
evidence=[['공통층 수',n(primary['included_strata_n']),n(primary['included_strata_n'])],['지역 수',n(C.region.nunique()),n(C.region.nunique())],['포함 관측 숙소 수',n(rc['included_observed_n']),n(tc['included_observed_n'])],['관측 숙소 대비 포함률',pc(rc['observed_listing_coverage']),pc(tc['observed_listing_coverage'])],['포함 고유호스트',n(rc['included_observed_hosts_n']),n(tc['included_observed_hosts_n'])],['관측 호스트 대비 포함률',pc(rc['observed_host_coverage']),pc(tc['observed_host_coverage'])]]
sensrows=[]
for case in J['cases']:
    a=v('R3','reference',dimension=case['dimension']);b=v('R3','target',dimension=case['dimension'])
    sensrows.append([case['dimension'],pc(case['coverage']['reference']['observed_listing_coverage']),pc(case['coverage']['target']['observed_listing_coverage']),pc(a.support_standardized),pc(b.support_standardized),f'{a.lift_within_strata:.2f}',f'{b.lift_within_strata:.2f}'])

explain=f'''<p>장기형과 중단기형의 시설 결합 경향을 비교할 때는 지역·객실 형태·숙소 종류·수용인원 구성을 맞춰야 합니다. 기존 보고서에서는 시설 묶음의 빈도만 표준화했고, 연관규칙의 신뢰도·향상도는 원빈도를 제시했습니다. 이번 추가 분석에서는 기존 세 규칙의 확률을 같은 구성으로 표준화한 뒤 신뢰도·향상도를 다시 계산했습니다.</p>
<p>주 분석의 조건은 <strong>region × room_type × property_type × 수용인원 구간</strong>입니다. 두 집단 각각 관측 30숙소 이상인 {n(primary['included_strata_n'])}개 공통층에서 장기형의 구성비를 두 집단에 동일하게 적용했습니다. 수용인원 구간은 1–2인, 3–4인, 5–6인, 7인 이상입니다. 시설 조건과 후보 매핑은 유지했습니다.</p>'''
definitions='''<ul><li><strong>표준화 지지도:</strong> 각 공통층의 A·B 동시등록률을 동일 구성비로 가중합니다.</li><li><strong>표준화 신뢰도:</strong> 표준화한 동시등록률을 표준화한 A등록률로 나눕니다.</li><li><strong>표준화 향상도:</strong> 표준화한 동시등록률을 표준화 A등록률×표준화 B등록률로 나눕니다. 기존 향상도 정의를 공통 구성에서 계산한 값입니다.</li><li><strong>층내 기대 기준 향상도:</strong> 동일 세부 조건 안에서 A와 B가 독립적으로 등록될 때 기대되는 동시등록률을 기준으로 삼습니다. 지역·유형마다 A와 B가 모두 흔해서 생기는 동시등록도 구별하는 추가 지표입니다.</li></ul>
<div class="note">층별 비율을 단순 평균하지 않습니다. 먼저 각 층에서 P(A), P(B), P(A∩B)를 계산하고 장기형 공통층 구성 q를 적용합니다. 표준화 향상도는 ΣqP(A∩B) / {ΣqP(A) × ΣqP(B)}이고, 층내 기대 기준 향상도는 ΣqP(A∩B) / Σq{P(A)P(B)}입니다. 두 향상도는 기준이 다르므로 값을 섞어 비교하지 않습니다.</div>'''
findings=f'''<p>생활·업무형의 표준화 지지도는 장기형 <strong>{pc(r.support_standardized)}</strong>, 중단기형 <strong>{pc(t.support_standardized)}</strong>로 비슷합니다. 생활형을 갖춘 숙소 안에서 업무시설도 등록된 비율은 {pc(r.confidence_standardized)} 대 {pc(t.confidence_standardized)}입니다.</p>
<p>같은 조건 안의 기대 동시등록을 기준으로 보면 생활형·업무시설 결합의 향상도는 <strong>{r.lift_within_strata:.2f} 대 {t.lift_within_strata:.2f}</strong>입니다. 장기형의 표준화 향상도 {r.lift_standardized:.2f} 중 일부는 세부 조건별 등록률 차이와 관련됩니다. 나머지 두 규칙의 층내 기대 기준 향상도는 약 1.03–1.05로, 기준 1에 가깝습니다.</p>
<p class="caution">호스트를 동일하게 가중하면 생활형→업무시설 신뢰도는 <strong>{pc(rh.confidence_standardized)} 대 {pc(th.confidence_standardized)}</strong>로 거의 같습니다. 따라서 ‘생활형을 갖춘 장기형 호스트가 업무시설을 더 갖춘다’는 일반적 주장으로 확대하기 어렵습니다. 호스트가중 층내 기대 기준 향상도는 {rh.lift_within_strata:.2f} 대 {th.lift_within_strata:.2f}로 다른데, 이는 각 집단의 조건별 업무시설 기본 등록률도 다르기 때문입니다.</p>'''
coverage=tbl(['비교 범위','장기형','중단기형'],evidence)+f'''<p>공통층은 {n(C.region.nunique())}개 지역, {n(C.property_type.nunique())}개 property_type, Entire home/apt와 Private room에 한정됩니다. Hotel room·Shared room 및 희소·비공통 조건은 주 비교 범위 밖입니다. 포함률은 장기형 관측 숙소의 {pc(rc['observed_listing_coverage'])}, 중단기형의 {pc(tc['observed_listing_coverage'])}입니다. 중단기형 전체로 일반화하지 않습니다.</p>
<div class="note">전체 공급 37,904곳·639,774곳 중 시설 관측 가능 행 37,701곳·636,938곳을 기준으로 포함률을 계산했습니다. 제외된 숙소를 기존 시설 후보에서 탈락시키지는 않았습니다. 지원 범위를 늘리기 위해 관측30숙소 기준을 완화하지 않았습니다.</div>'''
limits='''<p>보정은 관측된 공급 구성의 비교를 더 명확히 하는 데 쓰입니다. 시설 조건을 충족하는 숙소 수는 원집계를 그대로 활용할 수 있습니다. 시설등록의 품질·접근권한·속도는 확인되지 않았고, 고객의 장기 체류 목적·선호·실제 장기예약·수익은 이 분석에서 관측하지 않았습니다. 구성보정만으로 장기 적합성이나 예약 효과를 입증하지 않습니다.</p><p>세 규칙은 사전에 정한 규칙을 재계산했습니다. 새 시설 조합 탐색이나 새로운 후보 선정은 수행하지 않았습니다. 기존 자료·host split의 과거 노출 한계가 있으며, 이번 결과는 기술적 비교이고 통계적 유의성 검정이 아닙니다.</p>'''
downloads='<div class="downloads">'+''.join(f'<a href="{name}">{label}<small>{name}</small></a>' for name,label in [('연관보정_사전기준.json','추가 분석 사전 기준'),('연관보정_집계.csv','표준화·호스트가중·민감도 전체 집계'),('연관보정_층별근거.csv','공통층·제외층·분자·분모'),('연관보정_결과요약.json','결과 요약'),('연관보정_검산.json','원빈도 재현·독립 계산 검산'),('결과보고서.html','기존 전체 보고서')])+'</div>'
mainhtml=(OUT/'결과보고서.html').read_text();css=re.search(r'<style>(.*?)</style>',mainhtml,re.S).group(1)
doc='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>InsightStay 연관분석 구성보정</title><style>'+css+'</style></head><body><main><header><div class="eyebrow">InsightStay · 2026.10.03</div><h1>생활·업무 시설 연관지표의 구성보정</h1><p class="subtitle">장기형과 중단기형을 같은 지역·객실 형태·숙소 종류·수용인원 구성에서 비교했습니다.</p></header><section><h2>왜, 어떻게 보정했는가</h2>'+explain+definitions+'</section><section><h2>같은 구성에서 다시 계산한 세 규칙</h2>'+tbl(head,rows)+findings+'</section><section><h2>호스트 동일가중 확인</h2><p>전체 장기형 pool과 전체 중단기형 pool에서 각각 1/호스트 숙소 수를 재계산했습니다. 각 공통층의 확률과 장기형 공통층 구성비도 이 가중치로 계산했습니다.</p>'+tbl(head,hostrows)+'</section><section><h2>비교 가능한 범위</h2>'+coverage+'</section><section><h2>구성 기준별 민감도</h2>'+tbl(['구성 기준','장기 포함률','중단기 포함률','장기 지지도','중단기 지지도','장기 층내향상도','중단기 층내향상도'],sensrows)+'<p>이 표는 생활형→업무시설 규칙입니다. 요인별로 포함 범위가 달라지므로 단순히 수치가 가장 큰 기준을 선택하지 않았습니다. 사전에 정한 4요인 공통층 비교를 주 결과로 유지했습니다.</p></section><section><h2>어디까지 사용할 수 있는가</h2>'+limits+'</section><section><h2>근거와 검산</h2><p>기존 원빈도 지표와 이전3요인 표준화 지지도 일치, 층별 2×2 합계와 확률 범위, 구성비 합1, 원행별 가중합과 저장된 근거 CSV를 통한 별도 재계산을 검증했습니다. 후보매핑 파일의 SHA-256은 변경되지 않았습니다.</p>'+downloads+'</section><footer>POOLED_LONG_PRODUCT_20261003 · 연관지표 구성보정 추가 분석</footer></main></body></html>'
(OUT/'연관분석_구성보정.html').write_text(doc,encoding='utf-8')
addition='<section id="association-adjustment"><h2>추가 확인: 연관지표의 구성보정</h2>'+explain+tbl(head,rows)+findings+'<div class="note">비교 범위: 장기형 관측숙소의 83.9%, 중단기형의 46.9%. 공통 27지역·163층. 이후 원빈도 표와 구분하여 읽습니다.</div><p><a href="연관분석_구성보정.html">보정 방법·호스트가중·포함 범위·근거 보기</a></p></section>'
if 'id="association-adjustment"' in mainhtml:
    mainhtml=re.sub(r'<section id="association-adjustment">.*?</section>',addition,mainhtml,flags=re.S)
else:mainhtml=mainhtml.replace('<section><h2>동시등록 관계는 어느 정도인가</h2>',addition+'<section><h2>동시등록 관계의 원빈도</h2>',1)
(OUT/'결과보고서.html').write_text(mainhtml,encoding='utf-8')
S=json.loads((OUT/'결과요약.json').read_text());S['association_composition_adjustment']={'summary_path':str(OUT/'연관보정_결과요약.json'),'report_path':str(OUT/'연관분석_구성보정.html'),'primary_dimension':J['primary_dimension'],'coverage':primary['coverage'],'primary_results':J['primary_results'],'conclusion':'같은 조건 안의 기대 동시등록 대비 생활형·업무시설 향상도는 숙소기준 장기1.26 중단기1.16. 호스트 동일가중 신뢰도는49.3% 대49.5%로 비슷함. 주 비교는27지역·163공통층, 관측행 기준83.9%/46.9%에 한정. 후보조건 및 후보매핑 유지.'}
(OUT/'결과요약.json').write_text(json.dumps(S,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({'report':str(OUT/'연관분석_구성보정.html'),'bytes':len(doc.encode()),'main_report_updated':True,'listing_R3_lift_within':{'long':float(r.lift_within_strata),'short_medium':float(t.lift_within_strata)},'host_R3_confidence':{'long':float(rh.confidence_standardized),'short_medium':float(th.confidence_standardized)}},ensure_ascii=False))
"""
SOURCE_META['adjustment_report'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/report_adjustment.py', 'sha256': 'e8386e477815651446e82e41d05fe01a856fcfe4583fefc7eb4e90306a890b31', 'language': 'python'}


### sector3_report — 3번 보조 분석 보고서 생성
원본: `/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/report_sector3.py`
SHA256: `5c9f74b7d6aceaf29f545c4453f5976a33a378eaac3345c86467f746127506bb`


In [26]:
SOURCE_CODE['sector3_report'] = """import html,json,re
from pathlib import Path
import pandas as pd

OUT=Path('/Users/ohgyuwon/.codex/.chatgpt-projects/g-p-6ab22278d1bc81918f8361d1708baa26/outputs/InsightStay_장기통합참조_시설호스트연결_20261003/01_시설조합')
J=json.loads((OUT/'3번단독_결과요약.json').read_text());R=pd.read_csv(OUT/'3번단독_원지표.csv');D=pd.read_csv(OUT/'3번단독_연관보정.csv')
def e(v):return html.escape(str(v))
def pc(v):return f'{v*100:.1f}%'
def n(v):return f'{int(v):,}'
def tbl(head,rows):return '<div class="table-wrap"><table><thead><tr>'+''.join('<th>'+e(v)+'</th>' for v in head)+'</tr></thead><tbody>'+''.join('<tr>'+''.join('<td>'+e(v)+'</td>' for v in row)+'</tr>' for row in rows)+'</tbody></table></div>'
names={'R1':'식생활 4기능 → 세탁기','R2':'Wi-Fi → 전용 업무공간','R3':'생활형 5기능 → Wi-Fi·전용 업무공간'}
pn={'sector3':'3번 단독','pooled':'통합 3+6','target':'중단기형 1+2+4+5'}
def raw(rule,pool='sector3',weight='listing',split='all'):
    return R[(R.rule==rule)&(R.pool==pool)&(R.weighting==weight)&(R.split==split)].iloc[0]
def adj(rule,pool='sector3',weight='listing',dimension=None):
    s=D[(D.rule==rule)&(D.pool==pool)&(D.weighting==weight)]
    return s[s.dimension==dimension].iloc[0] if dimension else s[s.primary].iloc[0]
primary=next(c for c in J['cases'] if c['primary']);p=J['population']['sector3'];a=adj('R3');po=adj('R3','pooled');t=adj('R3','target');h=adj('R3',weight='host');th=adj('R3','target','host');r=raw('R3');pr=raw('R3','pooled')
why=f'''<p>3번 단독 분석은 <strong>최근 12개월 리뷰가 있는 장기 설정 공급</strong>의 시설 구성을 확인하는 데 유용합니다. 다만 minimum_nights≥30은 장기 설정이고, 최근 리뷰는 실제 장기예약이나 장기상품 성공을 증명하지 않습니다. 기존 통합 분석과 구분한 민감도 확인으로 수행했습니다.</p><p>3번 전체 {n(p['all_n'])}숙소·{n(p['all_hosts_n'])}호스트 중 시설 관측 {n(p['observed_n'])}숙소를 계산에 사용했습니다. {n(p['unobserved_n'])}숙소는 시설 관측이 불가능해 분모에서 제외했습니다. 기존의 세 규칙·기능 정의를 유지했으며 새 조합 탐색과 후보 재선정은 수행하지 않았습니다.</p>'''
rawrows=[[names[rid],n(raw(rid).nAB),pc(raw(rid).support),pc(raw(rid).confidence),f'{raw(rid).lift:.2f}'] for rid in names]
rawtable=tbl(['등록 관계','동시등록 숙소','지지도','신뢰도','향상도'],rawrows)
rawexp=f'''<p>지지도는 모든 필요 시설을 관측할 수 있는 {n(p['observed_n'])}숙소 중 A·B가 함께 등록된 비율입니다. 신뢰도는 A가 등록된 숙소 중 B도 등록된 비율이며, 향상도는 신뢰도를 같은 관측 집단의 B등록률로 나눕니다. 통계적 확신이나 판매 성공률을 뜻하지 않습니다.</p><p>생활형→업무시설의 지지도는 통합 집단 {pc(pr.support)}에서 3번 단독 {pc(r.support)}로, 신뢰도는 {pc(pr.confidence)}에서 {pc(r.confidence)}로 높아집니다. 반면 향상도는 {pr.lift:.2f}에서 {r.lift:.2f}로 낮아집니다. 업무시설 B 자체의 등록률이 {pc(pr.pB)}에서 {pc(r.pB)}로 높기 때문에, 높은 신뢰도가 곧 더 강한 연관을 뜻하지는 않습니다.</p>'''
head=['등록 관계','비교 집단','표준화 지지도','표준화 신뢰도','표준화 향상도','층내 기대 기준 향상도']
def adjrows(weight):
    rows=[]
    for rid in names:
        for pool in pn:
            x=adj(rid,pool,weight);rows.append([names[rid],pn[pool],pc(x.support_standardized),pc(x.confidence_standardized),f'{x.lift_standardized:.2f}',f'{x.lift_within_strata:.2f}'])
    return rows
method=f'''<p>지역×객실 형태×숙소 종류×수용인원 구간으로 나누고, 3번과 중단기형 각각 시설 관측 30숙소 이상인 <strong>{n(primary['included_strata_n'])}개 공통층</strong>만 비교했습니다. 통합3+6도 같은 층으로 제한했습니다. 세 집단 모두 이 공통층 안의 <strong>3번 구성비</strong>를 적용했으므로 3번과 통합을 비교할 때도 구성과 범위가 같습니다.</p><p>표준화 지지도는 ΣqP(A∩B), 신뢰도는 ΣqP(A∩B)/ΣqP(A), 향상도는 ΣqP(A∩B)/{{ΣqP(A)×ΣqP(B)}}입니다. 층별 신뢰도나 향상도를 단순 평균하지 않았습니다. 층내 기대 기준 향상도는 ΣqP(A∩B)/Σq{{P(A)P(B)}}이며, 같은 세부 조건 안에서 독립적으로 등록될 때 기대되는 동시등록을 기준으로 삼는 별도 지표입니다.</p>'''
findings=f'''<p>동일한 구성에서 생활형→업무시설 신뢰도는 3번 <strong>{pc(a.confidence_standardized)}</strong>, 통합 <strong>{pc(po.confidence_standardized)}</strong>로 비슷합니다. 3번의 원신뢰도가 통합보다 높다는 관계는 이 구성보정 비교에서 유지되지 않습니다.</p><p>3번과 중단기형의 표준화 향상도는 각각 {a.lift_standardized:.2f}, {t.lift_standardized:.2f}입니다. 같은 조건 안의 기대 동시등록 대비 향상도는 <strong>{a.lift_within_strata:.2f} 대 {t.lift_within_strata:.2f}</strong>입니다. ‘3번에서 생활·업무 결합이 관측된다’는 설명은 가능하지만, 중단기형보다 강하거나 장기에 특화된 결합이라고 주장할 근거로 확대하기 어렵습니다. 통계적 유의성 검정은 수행하지 않았습니다.</p><p>식생활→세탁, Wi-Fi→업무공간은 3번의 층내 기대 기준 향상도가 각각 {adj('R1').lift_within_strata:.2f}, {adj('R2').lift_within_strata:.2f}로 1에 가깝습니다. 시설이 흔하게 함께 등록되는 공급 구성임을 보여주는 수치로 읽어야 합니다.</p>'''
host=f'''<p>가중치는 전체3번, 전체통합3+6, 전체중단기형 각각에서 1/호스트 숙소 수로 새로 계산했습니다. 이전 통합 가중치를 3번에 단순 적용하지 않았습니다. 공통층 구성비도 3번의 호스트가중 관측비중으로 공통 적용했습니다.</p><p>생활형→업무시설의 3번 신뢰도는 {pc(h.confidence_standardized)}, 중단기형은 {pc(th.confidence_standardized)}입니다. 그러나 층내 기대 기준 향상도는 <strong>{h.lift_within_strata:.2f} 대 {th.lift_within_strata:.2f}</strong>로 거의 같습니다. 신뢰도 차이에는 업무시설의 기본 등록률 차이도 포함됩니다.</p>'''
coverage_rows=[]
for pool in pn:
    c=primary['coverage'][pool];pp=J['population'][pool]
    coverage_rows.append([pn[pool],n(pp['all_n']),n(pp['observed_n']),n(c['included_observed_n']),pc(c['listing_coverage']),n(c['included_hosts_n']),pc(c['host_coverage'])])
coverage=tbl(['집단','전체 숙소','시설 관측 숙소','공통층 숙소','관측 숙소 포함률','공통층 호스트','관측 호스트 포함률'],coverage_rows)+f'''<p>주 비교는 <strong>{n(J['primary_regions_n'])}지역·{n(J['primary_property_types_n'])}개 숙소 종류</strong>와 Entire home/apt·Private room에 한정됩니다. 3번 관측 숙소의 {pc(primary['coverage']['sector3']['listing_coverage'])}, 중단기형의 {pc(primary['coverage']['target']['listing_coverage'])}만 포함됩니다. 기존 통합 주 분석보다 비교 범위가 줄어들어, 특히 중단기형 전체를 대표하는 결과로 쓸 수 없습니다. 제외된 숙소를 기존 후보에서 탈락시키지는 않았습니다.</p>'''
sens=[]
for case in J['cases']:
    dim=case['dimension'];x=adj('R3',dimension=dim);y=adj('R3','target',dimension=dim)
    sens.append([dim,n(case['included_strata_n']),pc(case['coverage']['sector3']['listing_coverage']),pc(case['coverage']['target']['listing_coverage']),f'{x.lift_within_strata:.2f}',f'{y.lift_within_strata:.2f}'])
splits=[]
for split in ['train','validation','test']:
    x=raw('R3',split=split);y=raw('R3',weight='host',split=split)
    splits.append([split,n(x.observed_n),n(x.nAB),pc(x.support),pc(x.confidence),f'{x.lift:.2f}',pc(y.support),f'{y.lift:.2f}'])
lim='''<p>3번을 최근 리뷰가 있는 장기 설정 공급의 주 참고 집단으로 설명하고, 3+6을 전체 장기 설정 공급에서의 보완 확인으로 두는 방식은 가능합니다. 다만 이를 채택한다면 참고 집단의 목적이 달라졌음을 명시해야 합니다. 3번을 ‘장기판매 성공 집단’으로 명명할 수는 없습니다.</p><p>이번 결과는 시설 등록의 동시발생을 설명합니다. 시설의 품질·사용권한·업무환경 수준, 실제 장기예약, 장기 적합성, 고객 선호 및 예약·수익 효과를 확인한 결과가 아닙니다. 시설0은 미등록이고 실제 부재가 아니며, 결측은0으로 대체하지 않았습니다. 조리도구는 두 구성 항목 모두 등록된1만 충족으로 처리했습니다. 기존 고정시설 조건과 후보매핑은 유지했습니다.</p>'''
downloads='<div class="downloads">'+''.join(f'<a href="{file}">{label}<small>{file}</small></a>' for file,label in [('3번단독_사전기준.json','계산 전 고정 기준'),('3번단독_원지표.csv','원지표·기존split 결과'),('3번단독_연관보정.csv','보정·호스트가중·민감도 결과'),('3번단독_층별근거.csv','공통층·제외층·분자·분모'),('3번단독_결과요약.json','결과 요약'),('3번단독_검산.json','독립 재계산 검산'),('결과보고서.html','기존 전체 보고서')])+'</div>'
main=(OUT/'결과보고서.html').read_text();css=re.search(r'<style>(.*?)</style>',main,re.S).group(1)
doc='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>InsightStay 3번 단독 연관분석</title><style>'+css+'</style></head><body><main><header><div class="eyebrow">InsightStay · 2026.10.03</div><h1>3번 단독으로 다시 본 시설 동시등록</h1><p class="subtitle">최근 리뷰가 있는 장기 설정 공급에서 기존 세 규칙을 재계산했습니다.</p></header><section><h2>왜 3번을 따로 확인했는가</h2>'+why+'</section><section><h2>3번 전체 관측 숙소의 세 규칙</h2>'+rawtable+rawexp+'</section><section><h2>같은 범위와 구성에서의 비교</h2>'+method+tbl(head,adjrows('listing'))+findings+'</section><section><h2>호스트를 동일하게 가중했을 때</h2>'+host+tbl(head,adjrows('host'))+'</section><section><h2>비교 가능한 범위</h2>'+coverage+'</section><section><h2>구성 기준을 바꿔 확인한 결과</h2>'+tbl(['구성 기준','공통층 수','3번 포함률','중단기 포함률','3번 층내향상도','중단기 층내향상도'],sens)+'<p>생활형→업무시설 규칙의 결과입니다. 사전에 고정한4요인 비교를 주 분석으로 유지했습니다. 포함 범위가 다른 분석의 가장 큰 수치만 선택하지 않았습니다.</p></section><section><h2>기존 host split에서의 내부 재현</h2>'+tbl(['split','3번 관측 숙소','동시등록 숙소','지지도','신뢰도','향상도','호스트 지지도','호스트 향상도'],splits)+'<p>생활형→업무시설 규칙의 원지표입니다. 기존 자료와split의 과거 노출 이력이 있어 외부 독립확증이 아닙니다. test까지 본 뒤 새 후보를 선정하지 않았습니다. 전체 세 규칙 결과는 원지표 CSV에 수록했습니다.</p></section><section><h2>해석과 활용</h2>'+lim+'</section><section><h2>근거와 검산</h2><p>기존 통합·중단기 원지표 일치, 숙소·호스트 수와 가중치 합계, 공통층 구성비 합1, 확률 범위, 주 분석의 원행별 독립 가중합, 저장된층별 CSV를 통한 전 지표 재계산을 확인했습니다. 기존 후보매핑 파일의해시는 유지됐습니다.</p>'+downloads+'</section><footer>POOLED_LONG_PRODUCT_20261003 · 3번 단독 민감도 확인</footer></main></body></html>'
(OUT/'3번단독_연관분석.html').write_text(doc,encoding='utf-8')
add='<section id="sector3-association"><h2>추가 확인: 3번 단독 연관분석</h2>'+why+rawtable+rawexp+'<p>동일한 40개 공통층·3번 구성에서 표준화한 생활형→업무시설의 3번 신뢰도는 66.0%, 통합은 67.0%입니다. 3번·중단기형의 층내 기대 기준 향상도는 1.15·1.17이고, 호스트가중에서는 둘 다 1.12입니다. 3번이 장기에 특화된 더 강한 연관을 보인다는 주장으로 확대하기 어렵습니다.</p><div class="note">주 비교는 12지역·40층, 3번 관측 숙소의 75.3%, 중단기형의 18.8%에 한정됩니다. 실제 장기예약이나 장기판매 성공은 확인하지 못합니다.</div><p><a href="3번단독_연관분석.html">3번 단독 결과·보정·포함 범위·근거 보기</a></p></section>'
if 'id="sector3-association"' in main:main=re.sub(r'<section id="sector3-association">.*?</section>',add,main,flags=re.S)
else:main=main.replace('<section id="association-adjustment">',add+'<section id="association-adjustment">',1)
(OUT/'결과보고서.html').write_text(main,encoding='utf-8')
S=json.loads((OUT/'결과요약.json').read_text())
S['sector3_only_association_sensitivity']={'summary_path':str(OUT/'3번단독_결과요약.json'),'report_path':str(OUT/'3번단독_연관분석.html'),'population':J['population'],'primary_coverage':primary['coverage'],'primary_results':J['primary_results'],'conclusion':'3번단독 관측9516숙소에서 생활·업무형3184숙소: 지지도33.5%, 신뢰도64.8%, 향상도1.16. 동일40층·3번구성에서3번/중단기 층내기대기준향상도1.15/1.17, 호스트가중1.12/1.12. 주 비교3번75.3% 중단기18.8% 포함. 장기특화·예약성과 근거로 확대불가. 기존 후보 유지.'}
(OUT/'결과요약.json').write_text(json.dumps(S,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print(json.dumps({'report':str(OUT/'3번단독_연관분석.html'),'sections':9,'main_updated':True,'bytes':len(doc.encode())},ensure_ascii=False))
"""
SOURCE_META['sector3_report'] = {'path': '/Users/ohgyuwon/Documents/Codex/2026-10-03/insightstay-long-product-facilities/work/report_sector3.py', 'sha256': '5c9f74b7d6aceaf29f545c4453f5976a33a378eaac3345c86467f746127506bb', 'language': 'python'}


## 10. 원본 코드 추출과 선택 실행
재실행은 현재 노트북 커널의 Python을 사용한다. 전체 재계산에는 numpy/pandas/scipy/scikit-learn/statsmodels/patsy/joblib/threadpoolctl이 필요하고, 보고서의 추가 Python/JavaScript 대조에는 Node.js가 필요하다. 자동 패키지 설치는 하지 않는다.

재실행을 선택하지 않은 단계는 명시적으로 건너뛴다. 분석 코드의 출력 경로만 바꾸며 기존 원자료·저장 결과에 덮어쓰지 않는다. 보조 보고서·독립검산 원본도 전문으로 추출하지만 선택 실행 목록에는 계산 본체와 최종 보고서 생성만 넣었다.


In [27]:
EXPORT=RUN_DIR/'원본코드';EXPORT.mkdir()
for name,source in SOURCE_CODE.items():
    suffix='.js' if SOURCE_META[name]['language']=='javascript' else '.py'
    assert hashlib.sha256(source.encode()).hexdigest()==SOURCE_META[name]['sha256']
    if suffix=='.py': compile(source,name,'exec')
    (EXPORT/(name+suffix)).write_text(source,encoding='utf-8')
(EXPORT/'출처.json').write_text(json.dumps(SOURCE_META,ensure_ascii=False,indent=2),encoding='utf-8')
print('전문 보존한 코드 파일:',len(SOURCE_CODE))

# 원본 출력 폴더의 파일은 복사본에서만 변경한다.
def copy_flat(src,dst,exclude=()):
    dst.mkdir(parents=True,exist_ok=True)
    for p in src.iterdir():
        if p.is_file() and p.name not in exclude: shutil.copy2(p,dst/p.name)

def replay(stage):
    root=RUN_DIR/'재실행'/stage;root.mkdir(parents=True,exist_ok=False)
    s=SOURCE_CODE[stage]; original=Path(SOURCE_META[stage]['path'])
    destination=root/'out';destination.mkdir()
    args=[]
    if stage=='common':
        contract=load_json(COMMON_DIR.parent/'00_공통분석계약.json')
        contract['facility_plan']['folder']=str(destination)
        (root/'00_공통분석계약.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2))
    elif stage in ['association','adjustment','sector3']:
        copy_flat(ASSOC_DIR,destination)
        s=s.replace(str(ASSOC_DIR.relative_to(PROJECT)),str(ASSOC_DIR.relative_to(PROJECT)))
        # OUT만 치환한다. COMMON 등 기존 읽기전용 입력은 보존한다.
        s=re.sub(r"^OUT\s*=.*$",'OUT = Path('+repr(str(destination))+')',s,count=1,flags=re.M)
        if stage=='association': args=['--phase','full']
    elif stage=='cluster':
        shutil.copy2(CLUSTER_DIR/'실행전계획.json',destination/'실행전계획.json')
        shutil.copy2(COMMON_DIR.parent/'00_공통분석계약.json',root/'00_공통분석계약.json')
    elif stage=='regression':
        copy_flat(REPORT_DIR,root/'7개지역_장기상품_통합전략_20261003')
    elif stage=='report':
        s=re.sub(r"^OUT\s*=.*$",'OUT = Path('+repr(str(destination))+')',s,count=1,flags=re.M)
    elif stage=='report_update':
        copy_flat(REPORT_DIR,destination)
        copy_flat(REGRESSION_DIR,root/'7개지역_생활업무시설_조정회귀_20261003')
    else: raise ValueError('지원하지 않는 재실행 단계: '+stage)
    runner=destination/original.name
    runner.write_text(s,encoding='utf-8')
    import subprocess
    with (root/'실행로그.txt').open('w') as log:
        result=subprocess.run([sys.executable,str(runner),*args],stdout=log,stderr=subprocess.STDOUT)
    if result.returncode: raise RuntimeError(f'{stage} 실패: {root / "실행로그.txt"}')
    return {'stage':stage,'returncode':result.returncode,'output':str(destination)}

allowed=['common','association','adjustment','sector3','cluster','regression','report','report_update']
assert REPLAY_STAGES.issubset(set(allowed))
replayed=[]
for name in allowed:
    if name in REPLAY_STAGES:
        print('재계산 시작:',name,flush=True)
        replayed.append(replay(name));print('완료:',name,flush=True)
    else: print('재계산 생략:',name)


전문 보존한 코드 파일: 17
재계산 생략: common
재계산 생략: association
재계산 생략: adjustment
재계산 생략: sector3
재계산 생략: cluster
재계산 생략: regression
재계산 생략: report
재계산 생략: report_update


### 작성 시 실행 확인
기본 Python 셀을 순서대로 실행해 결과 출력을 저장했다. 추가로 조정회귀 재적합, 7지역 HTML 생성, E4·손익 통합의 선택 실행을 별도 폴더에서 확인했다. 재적합한 회귀 계수에 따른 조정차이와 신뢰구간이 기존 값과 일치했다. 군집 전체 재학습·공통표 원본 재작성 등 나머지 선택 실행은 이번 통합 작업에서 재실행하지 않았다. Jupyter 커널은 설치돼 있지 않아 Python에서 셀을 순차 실행했으며, 원본 분석 코드의 문법과 해시는 모두 검산했다.


## 11. 최종 확인과 해석
- 시설 후보 23,222곳·고유 호스트 13,255명, 모두 7지역 E4의 부분집합이다.
- 시설 동시등록은 계산으로 확인되지만 장기 수요·예약 효과·법적 적격은 확인되지 않았다.
- 회귀 주 분석의 95% 구간은 0을 포함하므로 유리한 민감도 결과만 선택하지 않는다.
- 손익은 가상 조건의 계산이며 실제 증가 매출 추정치가 아니다.
- 본 노트북은 코드를 통합한 산출물이다. 향후 원자료나 시설 정의가 바뀌면 과거 숫자와의 일치 검산은 실패할 수 있다.


In [28]:
assert all(sha(Path(p))==digest for p,digest in input_hashes.items())
verification={'candidate_n':len(candidate),'hosts':int(candidate.host_id.nunique()),'all_candidates_E4':bool(joined.cluster_label.eq('E_K4_4').all()),
              'E4_in_7regions':len(e4),'association_raw_recomputed':True,'primary_standardization_recomputed':True,
              'saved_cluster_quality_read':True,'saved_regression_read':True,'scenario_recomputed':True,
              'original_sources_compiled_and_hash_matched':len(SOURCE_CODE),'replayed_stages':replayed,'inputs_unchanged':True}
(RUN_DIR/'노트북검산.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2),encoding='utf-8')
show(verification)


{'candidate_n': 23222, 'hosts': 13255, 'all_candidates_E4': True, 'E4_in_7regions': 31224, 'association_raw_recomputed': True, 'primary_standardization_recomputed': True, 'saved_cluster_quality_read': True, 'saved_regression_read': True, 'scenario_recomputed': True, 'original_sources_compiled_and_hash_matched': 17, 'replayed_stages': [], 'inputs_unchanged': True}
